# Part 2 — complete source appendix

This notebook embeds the preserved Python code for model fitting, feature
construction, experimental searches, evaluation, figure generation, tests,
launchers, and hand-in preparation. The two analysis notebooks contain their
own table/figure code. The historical Ani notebooks are also included in
`resources/experiments/ani_comparison/source/`.

**Run all writes inspectable copies into `_source_export/`; it does not run
every search or overwrite the canonical research sources.** Each write cell
contains the actual source, with its relative path and hash. The final cell
verifies these copies byte-for-byte against the included implementations.

In [1]:
from pathlib import Path
import hashlib, json, os, runpy, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

locations = [candidate for base in [Path.cwd(), *Path.cwd().parents]
             for candidate in (base, base / "submission/midterm/part2", base / "midterm/part2")]
HANDIN = next(p for p in locations if (p / "01_final_model.ipynb").is_file())
ROOT = HANDIN / "resources"
assert (ROOT / "run.py").is_file(), "Run python part2/packaging/build_handin.py first, or extract the complete hand-in ZIP."
WORKSPACE = runpy.run_path(str(ROOT / "run.py"))["prepare_workspace"](ROOT)
sys.path.insert(0, str(WORKSPACE))
GENERATED = HANDIN / "generated"
GENERATED.mkdir(exist_ok=True)
sha256 = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
from experiments import load_data, metrics, PHASES
data = load_data()
SPRINT = ROOT / "experiments/leaderboard_sprint/runs/4dc0f274d45bce9e"
ledger = json.loads((ROOT / "submissions/leaderboard_results.json").read_text())
pd.set_option("display.max_rows", 16)
pd.set_option("display.max_columns", 12)


In [2]:
export_directory = HANDIN / "_source_export"
export_directory.mkdir(exist_ok=True)
os.chdir(export_directory)
Path('.').mkdir(parents=True, exist_ok=True)
Path('experiments/ani_comparison/source').mkdir(parents=True, exist_ok=True)
Path('packaging').mkdir(parents=True, exist_ok=True)
Path('src').mkdir(parents=True, exist_ok=True)
Path('tests').mkdir(parents=True, exist_ok=True)


## `experiments/ani_comparison/source/part2_best_model.py`

SHA-256: `253cc923e9e4867ddc5d9bbea91b36ccaf28b16ed1ce0de8a3067089778787e1`

In [3]:
%%writefile experiments/ani_comparison/source/part2_best_model.py
import ast
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import normalize
from sklearn.metrics import log_loss, roc_auc_score, average_precision_score, f1_score

DATA = '../part2/'
PHASES = ['I', 'II', 'III']
REASONS = ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']

train = pd.read_csv(DATA + 'outcome_train.csv')
val = pd.read_csv(DATA + 'outcome_val.csv')
test = pd.read_csv(DATA + 'outcome_test.csv')
df = pd.concat([train.assign(split='train'), val.assign(split='val'), test.assign(split='test')], ignore_index=True)
i_tr, i_va, i_te = (np.flatnonzero(df.split == s) for s in ['train', 'val', 'test'])
y, phase, groups = df.label.values, df.phase.values, df.trial_id.values

print(f'train {len(i_tr)}, val {len(i_va)}, test {len(i_te)}')
display(df.iloc[np.r_[i_tr, i_va]].groupby(['split', 'phase']).label.agg(['count', 'mean']).round(3).unstack(0))
print('success rate by start year (train):')
train.groupby('start_year').label.mean().loc[2000:].round(2).to_frame().T

def parse_list(s):
    try:
        out = []
        for x in ast.literal_eval(s):
            out += ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else [x]
        return [str(x) for x in out]
    except (ValueError, SyntaxError):
        return []


def semicolon_tokens(col):
    return df[col].fillna('').map(lambda s: ' '.join(sorted({t.strip() for t in s.split(';') if t.strip()})))


icd = df.icdcodes.fillna('[]').map(parse_list)
text = {
    'criteria': df.criteria.fillna(''),
    'disease': df.diseases_cleaned.fillna('[]').map(lambda s: ' '.join(parse_list(s))),
    'drug': df.drugs.fillna('[]').map(lambda s: ' '.join(parse_list(s))),
    # ICD codes at three granularities: chapter letter, 3-char category, 5-char subcategory
    'icd': icd.map(lambda L: ' '.join(sorted({c[:1] for c in L} | {c[:3] for c in L} | {c[:5] for c in L}))),
    'targets': semicolon_tokens('drug_targets'),
    'pathways': semicolon_tokens('drug_pathways'),
    'actions': semicolon_tokens('drug_actions'),
}
split_tokens = dict(tokenizer=str.split, lowercase=False, token_pattern=None)

F = {
    'criteria': TfidfVectorizer(min_df=5, ngram_range=(1, 2), sublinear_tf=True).fit_transform(text['criteria']),
    'crit_counts': CountVectorizer(min_df=5, ngram_range=(1, 2)).fit_transform(text['criteria']),
    'disease': TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True).fit_transform(text['disease']),
    'drug': TfidfVectorizer(min_df=2, sublinear_tf=True).fit_transform(text['drug']),
    'icd': TfidfVectorizer(min_df=2, **split_tokens).fit_transform(text['icd']),
    'targets': TfidfVectorizer(min_df=3, **split_tokens).fit_transform(text['targets']),
    'pathways': TfidfVectorizer(min_df=5, **split_tokens).fit_transform(text['pathways']),
    'actions': CountVectorizer(min_df=2, binary=True, **split_tokens).fit_transform(text['actions']),
    'meta': sp.csr_matrix(np.column_stack([
        df.matched_drugs.notna(), np.log1p(text['drug'].str.split().str.len()), np.log1p(text['criteria'].str.len()) / 10,
        df.phase_detail.str.contains('/'), df.phase_detail.eq('early phase 1')]).astype(float)),
}
PHASE_1HOT = sp.csr_matrix(pd.get_dummies(df.phase)[PHASES].values.astype(float))
CRIT_NORM = normalize(F['criteria'])
pd.Series({k: v.shape[1] for k, v in F.items()}, name='n_features').to_frame().T

def per_phase(fit, pred, make_model, X):
    p = np.zeros(len(pred))
    for ph in PHASES:
        a, b = fit[phase[fit] == ph], phase[pred] == ph
        p[b] = make_model().fit(X[a], y[a]).predict_proba(X[pred[b]])[:, 1]
    return p


def nb_per_phase(fit, pred):
    return per_phase(fit, pred, lambda: MultinomialNB(alpha=1.0), F['crit_counts'])


def baseline_lr(fit, pred):
    return per_phase(fit, pred, lambda: LogisticRegression(C=1.0, max_iter=3000), F['criteria'])


def pooled_lr(blocks, C):
    X = sp.hstack([F[b] for b in blocks] + [PHASE_1HOT]).tocsr()
    return lambda fit, pred, w=None: (LogisticRegression(C=C, max_iter=3000).fit(X[fit], y[fit], sample_weight=w)
                                      .predict_proba(X[pred])[:, 1])


def pooled_nb(fit, pred, w=None):
    X = sp.hstack([F['crit_counts'], PHASE_1HOT]).tocsr()
    return MultinomialNB(alpha=1.0).fit(X[fit], y[fit], sample_weight=w).predict_proba(X[pred])[:, 1]


def knn_criteria(fit, pred, w=None, k=50):
    p = np.zeros(len(pred))
    for ph in PHASES:
        a, b = fit[phase[fit] == ph], pred[phase[pred] == ph]
        S = (CRIT_NORM[b] @ CRIT_NORM[a].T).toarray()
        idx = np.argpartition(-S, k, axis=1)[:, :k]
        w = np.take_along_axis(S, idx, 1)
        p[phase[pred] == ph] = ((w * y[a][idx]).sum(1) + 1) / (w.sum(1) + 2)
    return p


BASE_MODELS = {
    'criteria LR': pooled_lr(['criteria'], 2.0),
    'disease LR': pooled_lr(['disease'], 1.0),
    'ICD LR': pooled_lr(['icd'], 1.0),
    'drug-name LR': pooled_lr(['drug'], 1.0),
    'drug-knowledge LR': pooled_lr(['targets', 'pathways', 'actions'], 0.3),
    'metadata LR': pooled_lr(['meta'], 1.0),
    'criteria MNB': pooled_nb,
    'criteria kNN': knn_criteria,
}


def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


year = df.start_year.fillna(df.start_year.median()).values


def stacked(fit, pred, tau=None, phase3_year=False, return_meta=False):
    w = np.ones(len(fit)) if tau is None else np.exp((year[fit] - year[fit].max()) / tau)
    w = w / w.mean()
    Z_fit, Z_pred = np.zeros((len(fit), len(BASE_MODELS))), np.zeros((len(pred), len(BASE_MODELS)))
    for j, fn in enumerate(BASE_MODELS.values()):
        for a, b in GroupKFold(5).split(fit, groups=groups[fit]):
            Z_fit[b, j] = fn(fit[a], fit[b], w[a])
        Z_pred[:, j] = fn(fit, pred, w)

    def meta_X(Z, idx):
        ph = pd.get_dummies(pd.Categorical(phase[idx], PHASES)).values.astype(float)
        cols = [logit(Z), ph]
        if phase3_year:
            # capped 2 years past the last training year so the trend is not extrapolated to 2021
            cols.append((ph[:, 2] * (np.minimum(year[idx], year[fit].max() + 2) - 2007) / 3)[:, None])
        return np.hstack(cols)

    meta = LogisticRegression(C=1.0, max_iter=3000).fit(meta_X(Z_fit, fit), y[fit], sample_weight=w)
    p = meta.predict_proba(meta_X(Z_pred, pred))[:, 1]
    return (p, meta, Z_pred) if return_meta else p


MODELS = {'Multinomial NB (per phase)': nb_per_phase, 'Baseline LR (per phase)': baseline_lr,
          'Improved: stacked ensemble': stacked,
          'Final: recency-weighted stack (tau=3)': lambda fit, pred: stacked(fit, pred, tau=3)}

def phase_metrics(idx, p):
    rows = {}
    for ph in PHASES:
        m = phase[idx] == ph
        yt, pp = y[idx][m], p[m]
        rows[ph] = {'F1': f1_score(yt, pp >= 0.5), 'ROC-AUC': roc_auc_score(yt, pp),
                    'PR-AUC': average_precision_score(yt, pp), 'log loss': log_loss(yt, np.clip(pp, 1e-4, 1 - 1e-4))}
    out = pd.DataFrame(rows).T
    out.loc['mean'] = out.mean()
    return out


val_pred = {name: fn(i_tr, i_va) for name, fn in MODELS.items()}
val_table = pd.concat({name: phase_metrics(i_va, p) for name, p in val_pred.items()}, names=['model', 'phase'])
val_table.round(3)

i_all = np.r_[i_tr, i_va]
temporal = {}
for cut in [2009, 2011]:
    t_fit, t_eval = i_all[year[i_all] <= cut], i_all[year[i_all] > cut]
    print(f'<= {cut}: fit n={len(t_fit)} (success rate {y[t_fit].mean():.3f}), '
          f'eval n={len(t_eval)} (success rate {y[t_eval].mean():.3f})')
    temporal[f'fit <= {cut}'] = pd.DataFrame({name: phase_metrics(t_eval, fn(t_fit, t_eval)).loc['mean']
                                              for name, fn in MODELS.items()}).T
pd.concat(temporal, names=['holdout', 'model']).round(4)

reasons = pd.read_csv('part1_best_reason_labels.csv').set_index('ID')[REASONS + ['source']]
failed = df.iloc[i_va].assign(**{n: p for n, p in val_pred.items()})
failed = failed[failed.label == 0].join(reasons, on='trial_id')
print(f'failed val rows: {len(failed)} | matched to Part 1: {failed.source.notna().sum()} '
      f"({(failed.source == 'true').sum()} true labels, {(failed.source == 'predicted').sum()} predicted)")

rows = []
for ph in PHASES:
    g = failed[failed.phase == ph]
    groups_by_reason = {r: g[g[r] == 1] for r in REASONS} | {'No recorded reason': g[g.source.isna()]}
    for reason, s in groups_by_reason.items():
        rows.append({'phase': ph, 'reason': reason, 'n trials': len(s),
                     'n true labels': int((s.source == 'true').sum()), 'n predicted labels': int((s.source == 'predicted').sum()),
                     **{f'error: {n.split(":")[0].split(" (")[0]}': (s[n] >= 0.5).mean() if len(s) else np.nan
                        for n in ['Baseline LR (per phase)', 'Improved: stacked ensemble', 'Final: recency-weighted stack (tau=3)']}})
errors = pd.DataFrame(rows).set_index(['phase', 'reason'])
errors.round(3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, ph in zip(axes, PHASES):
    e = errors.loc[ph]
    x = np.arange(len(e))
    ax.bar(x - 0.2, e['error: Baseline LR'], 0.4, label='Baseline LR')
    ax.bar(x + 0.2, e['error: Improved'], 0.4, label='Improved (stack)')
    ax.set_xticks(x, [f'{r}\n(n={n})' for r, n in zip(e.index, e['n trials'])], rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Phase {ph}')
axes[0].set_ylabel('fraction of failed trials predicted successful')
axes[0].legend()
plt.tight_layout()
plt.savefig('part2_error_by_reason.png', dpi=150)

p_test, meta, Z_test = stacked(i_all, i_te, tau=3, return_meta=True)
submission = pd.DataFrame({'ID': df.ID.values[i_te], 'label': np.clip(p_test, 1e-3, 1 - 1e-3)})
submission.to_csv('part2_best_submission.csv', index=False)

p_alt = stacked(i_all, i_te, tau=3, phase3_year=True)
pd.DataFrame({'ID': df.ID.values[i_te], 'label': np.clip(p_alt, 1e-3, 1 - 1e-3)}).to_csv(
    'part2_alt_phase3_trend_submission.csv', index=False)
p_unweighted = stacked(i_all, i_te)

print('meta-model weights:', dict(zip(list(BASE_MODELS) + PHASES, meta.coef_[0].round(2))))
pd.DataFrame({'train+val success rate': pd.Series(y[i_all]).groupby(phase[i_all]).mean(),
              'test mean: unweighted stack': pd.Series(p_unweighted).groupby(phase[i_te]).mean(),
              'test mean: final (tau=3)': submission.groupby(phase[i_te]).label.mean(),
              'test mean: alt (+ Phase III trend)': pd.Series(p_alt).groupby(phase[i_te]).mean()}).round(3)

Overwriting experiments/ani_comparison/source/part2_best_model.py


## `experiments/ani_comparison/source/part2_first_submission_model.py`

SHA-256: `8a526bb1fa94ba24f48245b27add937c430ff65e3d805518627ce5c067ee8f6a`

In [4]:
%%writefile experiments/ani_comparison/source/part2_first_submission_model.py
import ast
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import normalize
from sklearn.metrics import log_loss, roc_auc_score, average_precision_score, f1_score

DATA = '../part2/'
PHASES = ['I', 'II', 'III']
REASONS = ['Safety', 'Funding', 'Efficacy', 'Recruitment', 'Other']

train = pd.read_csv(DATA + 'outcome_train.csv')
val = pd.read_csv(DATA + 'outcome_val.csv')
test = pd.read_csv(DATA + 'outcome_test.csv')
df = pd.concat([train.assign(split='train'), val.assign(split='val'), test.assign(split='test')], ignore_index=True)
i_tr, i_va, i_te = (np.flatnonzero(df.split == s) for s in ['train', 'val', 'test'])
y, phase, groups = df.label.values, df.phase.values, df.trial_id.values

print(f'train {len(i_tr)}, val {len(i_va)}, test {len(i_te)}')
display(df.iloc[np.r_[i_tr, i_va]].groupby(['split', 'phase']).label.agg(['count', 'mean']).round(3).unstack(0))
print('success rate by start year (train):')
train.groupby('start_year').label.mean().loc[2000:].round(2).to_frame().T

def parse_list(s):
    try:
        out = []
        for x in ast.literal_eval(s):
            out += ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else [x]
        return [str(x) for x in out]
    except (ValueError, SyntaxError):
        return []


def semicolon_tokens(col):
    return df[col].fillna('').map(lambda s: ' '.join(sorted({t.strip() for t in s.split(';') if t.strip()})))


icd = df.icdcodes.fillna('[]').map(parse_list)
text = {
    'criteria': df.criteria.fillna(''),
    'disease': df.diseases_cleaned.fillna('[]').map(lambda s: ' '.join(parse_list(s))),
    'drug': df.drugs.fillna('[]').map(lambda s: ' '.join(parse_list(s))),
    # ICD codes at three granularities: chapter letter, 3-char category, 5-char subcategory
    'icd': icd.map(lambda L: ' '.join(sorted({c[:1] for c in L} | {c[:3] for c in L} | {c[:5] for c in L}))),
    'targets': semicolon_tokens('drug_targets'),
    'pathways': semicolon_tokens('drug_pathways'),
    'actions': semicolon_tokens('drug_actions'),
}
split_tokens = dict(tokenizer=str.split, lowercase=False, token_pattern=None)

F = {
    'criteria': TfidfVectorizer(min_df=5, ngram_range=(1, 2), sublinear_tf=True).fit_transform(text['criteria']),
    'crit_counts': CountVectorizer(min_df=5, ngram_range=(1, 2)).fit_transform(text['criteria']),
    'disease': TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True).fit_transform(text['disease']),
    'drug': TfidfVectorizer(min_df=2, sublinear_tf=True).fit_transform(text['drug']),
    'icd': TfidfVectorizer(min_df=2, **split_tokens).fit_transform(text['icd']),
    'targets': TfidfVectorizer(min_df=3, **split_tokens).fit_transform(text['targets']),
    'pathways': TfidfVectorizer(min_df=5, **split_tokens).fit_transform(text['pathways']),
    'actions': CountVectorizer(min_df=2, binary=True, **split_tokens).fit_transform(text['actions']),
    'meta': sp.csr_matrix(np.column_stack([
        df.matched_drugs.notna(), np.log1p(text['drug'].str.split().str.len()), np.log1p(text['criteria'].str.len()) / 10,
        df.phase_detail.str.contains('/'), df.phase_detail.eq('early phase 1')]).astype(float)),
}
PHASE_1HOT = sp.csr_matrix(pd.get_dummies(df.phase)[PHASES].values.astype(float))
CRIT_NORM = normalize(F['criteria'])
pd.Series({k: v.shape[1] for k, v in F.items()}, name='n_features').to_frame().T

def per_phase(fit, pred, make_model, X):
    p = np.zeros(len(pred))
    for ph in PHASES:
        a, b = fit[phase[fit] == ph], phase[pred] == ph
        p[b] = make_model().fit(X[a], y[a]).predict_proba(X[pred[b]])[:, 1]
    return p


def nb_per_phase(fit, pred):
    return per_phase(fit, pred, lambda: MultinomialNB(alpha=1.0), F['crit_counts'])


def baseline_lr(fit, pred):
    return per_phase(fit, pred, lambda: LogisticRegression(C=1.0, max_iter=3000), F['criteria'])


def pooled_lr(blocks, C):
    X = sp.hstack([F[b] for b in blocks] + [PHASE_1HOT]).tocsr()
    return lambda fit, pred: LogisticRegression(C=C, max_iter=3000).fit(X[fit], y[fit]).predict_proba(X[pred])[:, 1]


def pooled_nb(fit, pred):
    X = sp.hstack([F['crit_counts'], PHASE_1HOT]).tocsr()
    return MultinomialNB(alpha=1.0).fit(X[fit], y[fit]).predict_proba(X[pred])[:, 1]


def knn_criteria(fit, pred, k=50):
    p = np.zeros(len(pred))
    for ph in PHASES:
        a, b = fit[phase[fit] == ph], pred[phase[pred] == ph]
        S = (CRIT_NORM[b] @ CRIT_NORM[a].T).toarray()
        idx = np.argpartition(-S, k, axis=1)[:, :k]
        w = np.take_along_axis(S, idx, 1)
        p[phase[pred] == ph] = ((w * y[a][idx]).sum(1) + 1) / (w.sum(1) + 2)
    return p


BASE_MODELS = {
    'criteria LR': pooled_lr(['criteria'], 2.0),
    'disease LR': pooled_lr(['disease'], 1.0),
    'ICD LR': pooled_lr(['icd'], 1.0),
    'drug-name LR': pooled_lr(['drug'], 1.0),
    'drug-knowledge LR': pooled_lr(['targets', 'pathways', 'actions'], 0.3),
    'metadata LR': pooled_lr(['meta'], 1.0),
    'criteria MNB': pooled_nb,
    'criteria kNN': knn_criteria,
}


def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def stacked(fit, pred, return_meta=False):
    Z_fit, Z_pred = np.zeros((len(fit), len(BASE_MODELS))), np.zeros((len(pred), len(BASE_MODELS)))
    for j, fn in enumerate(BASE_MODELS.values()):
        for a, b in GroupKFold(5).split(fit, groups=groups[fit]):
            Z_fit[b, j] = fn(fit[a], fit[b])
        Z_pred[:, j] = fn(fit, pred)
    ph = lambda idx: pd.get_dummies(pd.Categorical(phase[idx], PHASES)).values.astype(float)
    meta = LogisticRegression(C=1.0, max_iter=3000).fit(np.hstack([logit(Z_fit), ph(fit)]), y[fit])
    p = meta.predict_proba(np.hstack([logit(Z_pred), ph(pred)]))[:, 1]
    return (p, meta, Z_pred) if return_meta else p


MODELS = {'Multinomial NB (per phase)': nb_per_phase, 'Baseline LR (per phase)': baseline_lr, 'Improved: stacked ensemble': stacked}

def phase_metrics(idx, p):
    rows = {}
    for ph in PHASES:
        m = phase[idx] == ph
        yt, pp = y[idx][m], p[m]
        rows[ph] = {'F1': f1_score(yt, pp >= 0.5), 'ROC-AUC': roc_auc_score(yt, pp),
                    'PR-AUC': average_precision_score(yt, pp), 'log loss': log_loss(yt, np.clip(pp, 1e-4, 1 - 1e-4))}
    out = pd.DataFrame(rows).T
    out.loc['mean'] = out.mean()
    return out


val_pred = {name: fn(i_tr, i_va) for name, fn in MODELS.items()}
val_table = pd.concat({name: phase_metrics(i_va, p) for name, p in val_pred.items()}, names=['model', 'phase'])
val_table.round(3)

i_all = np.r_[i_tr, i_va]
year = df.start_year.fillna(df.start_year.median()).values
t_fit, t_eval = i_all[year[i_all] <= 2009], i_all[year[i_all] >= 2010]
temporal = pd.DataFrame({name: phase_metrics(t_eval, fn(t_fit, t_eval)).loc['mean'] for name, fn in MODELS.items()}).T
print(f'fit n={len(t_fit)} (success rate {y[t_fit].mean():.3f}), eval n={len(t_eval)} (success rate {y[t_eval].mean():.3f})')
temporal.round(3)

reasons = pd.read_csv('part1_best_reason_labels.csv').set_index('ID')[REASONS + ['source']]
failed = df.iloc[i_va].assign(**{n: p for n, p in val_pred.items()})
failed = failed[failed.label == 0].join(reasons, on='trial_id')
print(f'failed val rows: {len(failed)} | matched to Part 1: {failed.source.notna().sum()} '
      f"({(failed.source == 'true').sum()} true labels, {(failed.source == 'predicted').sum()} predicted)")

rows = []
for ph in PHASES:
    g = failed[failed.phase == ph]
    groups_by_reason = {r: g[g[r] == 1] for r in REASONS} | {'No recorded reason': g[g.source.isna()]}
    for reason, s in groups_by_reason.items():
        rows.append({'phase': ph, 'reason': reason, 'n trials': len(s),
                     'n true labels': int((s.source == 'true').sum()), 'n predicted labels': int((s.source == 'predicted').sum()),
                     **{f'error: {n.split(":")[0].split(" (")[0]}': (s[n] >= 0.5).mean() if len(s) else np.nan
                        for n in ['Baseline LR (per phase)', 'Improved: stacked ensemble']}})
errors = pd.DataFrame(rows).set_index(['phase', 'reason'])
errors.round(3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, ph in zip(axes, PHASES):
    e = errors.loc[ph]
    x = np.arange(len(e))
    ax.bar(x - 0.2, e['error: Baseline LR'], 0.4, label='Baseline LR')
    ax.bar(x + 0.2, e['error: Improved'], 0.4, label='Improved (stack)')
    ax.set_xticks(x, [f'{r}\n(n={n})' for r, n in zip(e.index, e['n trials'])], rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Phase {ph}')
axes[0].set_ylabel('fraction of failed trials predicted successful')
axes[0].legend()
plt.tight_layout()
plt.savefig('part2_first_error_by_reason.png', dpi=150)

p_test, meta, Z_test = stacked(i_all, i_te, return_meta=True)
submission = pd.DataFrame({'ID': df.ID.values[i_te], 'label': np.clip(p_test, 1e-3, 1 - 1e-3)})
submission.to_csv('part2_first_submission.csv', index=False)

print('meta-model weights:', dict(zip(list(BASE_MODELS) + PHASES, meta.coef_[0].round(2))))
pd.DataFrame({'train+val success rate': pd.Series(y[i_all]).groupby(phase[i_all]).mean(),
              'mean predicted P(success) on test': submission.groupby(phase[i_te]).label.mean()}).round(3)

Overwriting experiments/ani_comparison/source/part2_first_submission_model.py


## `packaging/build_handin.py`

SHA-256: `cde541a7edf9580f3051601825fa93ab302181da8db04c42d4e905370aacb95a`

In [5]:
%%writefile packaging/build_handin.py
"""Materialize and verify a portable Part 2 hand-in without changing research files.

Usage: python part2/packaging/build_handin.py [--skip-execution]
The ZIP contains midterm/part2/. It does not claim to package the Part 1 hand-in.
"""
from pathlib import Path
import argparse
import hashlib
import json
import os
import shutil
import subprocess
import tempfile
import zipfile

REPO = Path(__file__).resolve().parents[2]
HANDIN = REPO / "submission/midterm/part2"
WINNER_SHA = "be29fc849f63ed4c23b31e67c97aaa97c6940f569b0af0920cf9799b428f7636"


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def inventory():
    raw = subprocess.check_output(["git", "ls-files", "--cached", "--others", "--exclude-standard", "-z", "--", "part2"], cwd=REPO)
    return sorted({REPO / os.fsdecode(p) for p in raw.split(b"\0") if p
                   if (REPO / os.fsdecode(p)).is_file()})


def include_resource(source):
    """Keep all inputs/code/trial summaries, plus evidence used by final notebooks.

    Per-fold predictions for every unsuccessful candidate, repeated fit-ID audits,
    and intermediate feature/base-prediction arrays remain in the working project.
    They are outputs that can be regenerated, not additional input datasets.
    """
    relative = source.relative_to(REPO / "part2")
    parts = relative.parts
    if parts[0] in {"logs", "report"}:
        return False
    if parts[0] != "experiments":
        return True
    if {"audits", "bases", "input_snapshots"}.intersection(parts):
        return False
    if source.suffix in {".log", ".joblib"}:
        return False
    if source.suffix == ".npz":
        return str(relative) in {
            "experiments/model_search/linear_validation_predictions.npz",
            "experiments/final_handin/recorded_components.npz",
        }
    if "predictions" in parts:
        return (str(relative).startswith("experiments/leaderboard_sprint/runs/4dc0f274d45bce9e/")
                and "gaps" not in parts
                and source.stem in {"ani_first", "ani_stack", "file10",
                                    "mix_legacy01_0.2", "mix_legacy01_0.3"})
    return True


def materialize(files):
    target = HANDIN / "resources"
    if target.exists() and not (target / "PACKAGE_MANIFEST.json").is_file():
        raise RuntimeError(f"Refusing to replace an unrecognized directory: {target}")
    with tempfile.TemporaryDirectory(prefix="part2-handin-") as temporary:
        stage = Path(temporary) / "resources"
        stage.mkdir()
        entries = {}
        included = [p for p in files if include_resource(p)]
        for source in included:
            relative = source.relative_to(REPO / "part2")
            # Existing report support files are repository-local symlinks. Copy
            # their contents so the exported hand-in needs no external paths.
            assert source.resolve().is_relative_to(REPO), source
            assert ".runtime" not in relative.parts and source.suffix != ".joblib", source
            destination = stage / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            entries[str(relative)] = {"sha256": sha(destination), "bytes": destination.stat().st_size}
        manifest = dict(
            scope="Part 2 only; all supplied inputs, complete source, all trial settings/summaries, selected row predictions",
            source_commit=subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
            note="Includes current working-tree Part 2 changes; file hashes identify the packaged contents.",
            excluded="Intermediate arrays, repeated row predictions/audits, logs and report drafts stay in the working project; regenerate from included code and inputs.",
            working_files=len(files), included_files=len(included),
            omitted_working_files=len(files)-len(included),
            files=entries)
        (stage / "PACKAGE_MANIFEST.json").write_text(json.dumps(manifest, indent=2) + "\n")
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(stage, target)
    assert sha(HANDIN / "part2_best_submission.csv") == WINNER_SHA
    return entries


def execute_notebooks():
    import nbformat
    from nbclient import NotebookClient
    records = []
    for path in sorted(HANDIN.glob("0*.ipynb")):
        print(f"Executing {path.name}", flush=True)
        notebook = nbformat.read(path, as_version=4)
        client = NotebookClient(notebook, timeout=600, kernel_name="python3",
                                resources={"metadata": {"path": str(HANDIN)}})
        client.execute()
        nbformat.write(notebook, path)
        records.append(dict(notebook=path.name, code_cells=sum(c.cell_type == "code" for c in notebook.cells),
                            errors=0))
    result = HANDIN / "results"
    result.mkdir(exist_ok=True)
    for path in (HANDIN / "generated").iterdir():
        if path.is_file() and path.suffix in [".csv", ".png"]:
            shutil.copy2(path, result / path.name)
    verification = dict(notebooks=records, expected_winner_sha256=WINNER_SHA,
                        source_files=len(list((HANDIN / "_source_export").rglob("*.py"))),
                        note="Default execution rescores saved predictions and replays recorded components; see fresh-fit receipt separately.")
    (result / "notebook_verification.json").write_text(json.dumps(verification, indent=2) + "\n")


def archive(entries):
    destination = REPO / "submission/dist/part2-midterm-code.zip"
    destination.parent.mkdir(exist_ok=True)
    paths = sorted([p for p in HANDIN.iterdir() if p.is_file() and p.suffix in [".ipynb", ".csv", ".md"]]
                   + list((HANDIN / "results").glob("*"))
                   + [HANDIN / "resources" / p for p in entries]
                   + [HANDIN / "resources/PACKAGE_MANIFEST.json"])
    temporary = destination.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for path in paths:
            assert path.is_file() and not path.is_symlink(), path
            relative = "midterm/part2/" + path.relative_to(HANDIN).as_posix()
            info = zipfile.ZipInfo(relative, date_time=(2026, 11, 7, 0, 0, 0))
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16
            z.writestr(info, path.read_bytes())
    temporary.replace(destination)
    print(f"Created {destination} ({destination.stat().st_size / 1e6:.1f} MB)", flush=True)
    return destination


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--skip-execution", action="store_true", help="Generate notebooks without executing them.")
    parser.add_argument("--zip-only", action="store_true", help="Repackage already executed notebooks and materialized resources.")
    args = parser.parse_args()
    if args.zip_only:
        entries = json.loads((HANDIN / "resources/PACKAGE_MANIFEST.json").read_text())["files"]
    else:
        from notebooks import build
        files = inventory()
        print(f"Including {sum(include_resource(p) for p in files)} of {len(files)} working files; all input datasets and Python code retained.", flush=True)
        assert all(include_resource(p) for p in files if p.suffix == ".py")
        build(REPO, files)
        entries = materialize(files)
        if not args.skip_execution:
            execute_notebooks()
    archive(entries)


if __name__ == "__main__":
    main()


Overwriting packaging/build_handin.py


## `packaging/capture_recorded.py`

SHA-256: `b3a2758cfb48612ef58b80fab7a0721fda94a1c76b586fc4e9b374ddf11fa46e`

In [6]:
%%writefile packaging/capture_recorded.py
"""Capture original native-model component probabilities for a portable replay.

This requires the historical local caches; the final training notebook does not.
It never derives a component by algebraically subtracting the winning CSV.
"""
import argparse
import hashlib
import json
from pathlib import Path
import runpy
import sys


def capture(root, output):
    root, output = Path(root).resolve(), Path(output).resolve()
    workspace = runpy.run_path(str(root / "run.py"))["prepare_workspace"](root)
    sys.path.insert(0, str(workspace))
    import joblib
    import numpy as np
    import pandas as pd
    from threadpoolctl import threadpool_limits
    from experiments import load_data
    from sprint_export import native_packets, original_components
    paths = dict(ani="experiments/ani_extensions/cache/decision/final/submission_bundle.joblib",
                 legacy="experiments/leaderboard_sprint/cache/legacy_full.joblib")
    sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
    receipt = json.loads((root / "experiments/leaderboard_sprint/runs/4dc0f274d45bce9e/exports/13_file10_legacy30.json").read_text())
    assert sha(root / paths["ani"]) == receipt["original_bundle_sha256"]
    assert sha(root / paths["legacy"]) == receipt["legacy_bundle_sha256"]
    test = load_data()["test"]
    bundle, legacy = (joblib.load(root / paths[k]) for k in ["ani", "legacy"])
    with threadpool_limits(limits=2):
        old, extra = native_packets(bundle, test)
        components = original_components(bundle, old, extra, test)
        legacy01 = legacy["model"].predict(test)["01_balanced_ensemble"]
    p = np.clip(.7*components["file10"] + .3*legacy01, .001, .999)
    recorded = pd.read_csv(root / "submissions/13_file10_legacy30.csv")
    assert recorded.ID.tolist() == test.ID.tolist()
    difference = float(np.max(np.abs(p - recorded.label)))
    assert difference < 1e-12
    output.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(output / "recorded_components.npz", ID=test.ID.to_numpy(dtype=str),
                        ani_stack=components["ani_stack"],
                        fast2_neighbors=components["fast2_neighbors"], legacy01=legacy01)
    audit = dict(kind="Native inference from original fitted caches; not a fresh fit",
                 original_caches={p: sha(root/p) for p in paths.values()},
                 component_archive_sha256=sha(output / "recorded_components.npz"),
                 recorded_submission_sha256=sha(root / "submissions/13_file10_legacy30.csv"),
                 maximum_probability_difference=difference)
    (output / "recorded_components_provenance.json").write_text(json.dumps(audit, indent=2) + "\n")
    print(json.dumps(audit, indent=2))


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--part2-root", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    capture(args.part2_root, args.output)


Overwriting packaging/capture_recorded.py


## `packaging/notebooks.py`

SHA-256: `2c67543301b2054fad8d9c991a86bb2ae8f0db5106b791d424a78f136907ecb0`

In [7]:
%%writefile packaging/notebooks.py
"""Build the three final Part 2 hand-in notebooks from preserved research files."""
from pathlib import Path
import hashlib
import inspect
import json
import textwrap
import nbformat as nbf


SETUP = '''from pathlib import Path
import hashlib, json, os, runpy, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

locations = [candidate for base in [Path.cwd(), *Path.cwd().parents]
             for candidate in (base, base / "submission/midterm/part2", base / "midterm/part2")]
HANDIN = next(p for p in locations if (p / "01_final_model.ipynb").is_file())
ROOT = HANDIN / "resources"
assert (ROOT / "run.py").is_file(), "Run python part2/packaging/build_handin.py first, or extract the complete hand-in ZIP."
WORKSPACE = runpy.run_path(str(ROOT / "run.py"))["prepare_workspace"](ROOT)
sys.path.insert(0, str(WORKSPACE))
GENERATED = HANDIN / "generated"
GENERATED.mkdir(exist_ok=True)
sha256 = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
from experiments import load_data, metrics, PHASES
data = load_data()
SPRINT = ROOT / "experiments/leaderboard_sprint/runs/4dc0f274d45bce9e"
ledger = json.loads((ROOT / "submissions/leaderboard_results.json").read_text())
pd.set_option("display.max_rows", 16)
pd.set_option("display.max_columns", 12)
'''


def md(text):
    return nbf.v4.new_markdown_cell(inspect.cleandoc(text).strip())


def code(text):
    return nbf.v4.new_code_cell(textwrap.dedent(text).strip() + "\n")


def write(path, cells):
    notebook = nbf.v4.new_notebook(cells=cells, metadata={
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python", "version": "3.12"},
    })
    # Stable cell IDs avoid unrelated diffs when regenerating the notebooks.
    for i, cell in enumerate(notebook.cells):
        cell.id = hashlib.sha256(f"{path.name}:{i}".encode()).hexdigest()[:12]
    nbf.write(notebook, path)


def build(repo, source_files):
    repo = Path(repo)
    destination = repo / "submission/midterm/part2"
    destination.mkdir(parents=True, exist_ok=True)
    training_source = (repo / "part2/packaging/train_final.py").read_text().split('if __name__ == "__main__":')[0]
    final = [
        md('''# Part 2 — final outcome model

        **Final scored artifact: file 13, reported Kaggle log loss 0.58465 (lower is better).**
        The exact CSV is `part2_best_submission.csv`. This notebook follows the supplied
        template's flow: data, model, validation, reason-error analysis, improvements,
        and prediction export. The template is a starting point, not a required cell layout.

        Run all cells to independently rescore the saved held-out predictions, regenerate
        figures, and replay the final blend. Set `RUN_FINAL_TRAINING=True` near the end
        to **fit the actual model from raw supplied data**, with no old model caches.
        Saved-probability replay and fresh training are identified separately.

        [Experiment history](02_experiments.ipynb) · [Full implementation](03_source_code.ipynb)
        · [Development narrative](resources/MODEL_DEVELOPMENT.md)
        · [Original template](https://colab.research.google.com/drive/1mECHFXxHQnHAQiq5T3VPxeAjHXwFK7TG)
        '''), code(SETUP),
        md('''## Supplied data and evaluation boundaries

        We have 8,037 training outcomes, 1,013 validation outcomes, and 1,500 unlabeled
        test rows. Outcome label 1 means success. IDs are used for joins and splitting,
        never as predictors. No additional dataset was downloaded and no deep learning
        model was trained. Input data and the supplied pathway catalog are included.

        The primary benchmark refits preprocessing inside each outer/inner training
        split. It uses the official validation split, three forward windows, two
        repeats of three-fold trial-grouped CV, and three longer-gap checks. The
        grouped CV pool contains train + validation, with every evaluation trial
        excluded from its fitting fold. The metric gives phases I, II and III equal weight.

        The final fit uses all 9,050 labeled rows. As in Ani's published recipe, its
        Ani-derived vocabulary/IDF fits use **unlabeled text from all three supplied
        splits**, including test. The original text/tree component fits preprocessing
        on labeled fitting rows. Test outcomes are never available. Strict benchmark
        and published-preprocessing results are reported separately.
        '''), code('''
        protocol = json.loads((ROOT / "experiments/local_benchmark/protocol.json").read_text())
        for split, expected in protocol["data_sha256"].items():
            assert sha256(ROOT / "data" / f"outcome_{split}.csv") == expected
        assert "label" not in data["test"]
        for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
            assert not set(data[a].trial_id) & set(data[b].trial_id)
        display(pd.DataFrame([dict(split=k, rows=len(v), trials=v.trial_id.nunique(),
                                   outcome_labels="label" in v) for k, v in data.items()]))
        display(pd.DataFrame([{k: v for k, v in s.items() if k not in ["fit_ids", "eval_ids"]}
                              for s in protocol["splits"]]))
        '''),
        md('''## Frozen final recipe

        `p13 = 0.70 * p10 + 0.30 * p01`, clipped to [0.001, 0.999].
        File 10 is half Ani's recency stack and half our faster-recency stack with
        eight neighbor features. The original ensemble `p01` combines a
        three-logistic-regression mixture (50%) with CatBoost depth 6 and depth 8
        (25% each). The final weights are frozen, not searched again here.

        The Ani base learners use text TF-IDF, logistic regression, multinomial
        Naive Bayes, and same-phase neighbors. Grouped out-of-fold predictions train
        the logistic stacking layer. Eligibility, disease, drug and ICD text,
        phase, and supplied drug knowledge support the components; stopping reasons
        and Part 1 embeddings are excluded from outcome training.

        The full fresh-fit orchestration appears below. Every model, preprocessing,
        benchmark, and plot implementation is readable in the code appendix.
        '''),
        md('''## Validation against the baseline and both Ani references

        These predictions were generated with the 1,013 official validation rows
        held out of supervised fitting. They are **not** predictions from the final
        model fitted on train + validation. We recompute all metrics from saved row
        probabilities. F1 uses a 0.5 success threshold; PR-AUC is average precision.
        '''), code('''
        validation = data["val"]
        predictions = {"Eligibility LR baseline": np.load(
            ROOT / "experiments/model_search/linear_validation_predictions.npz"
        )["lr_criteria_none_C2_phase0"]}
        names = {"Ani unweighted": "ani_first", "Ani recency": "ani_stack",
                 "File 10": "file10", "File 13 (final)": "mix_legacy01_0.3"}
        for label, name in names.items():
            saved = pd.read_csv(SPRINT / "strict/predictions/official" / f"{name}.csv")
            assert saved[["ID", "trial_id", "phase", "label"]].equals(
                validation[["ID", "trial_id", "phase", "label"]])
            predictions[label] = saved.probability.to_numpy()
        scores = {name: metrics(validation, p) for name, p in predictions.items()}
        assert abs(scores["File 13 (final)"]["phase_mean_logloss"] - 0.5532789641183768) < 1e-12
        phase_table = pd.DataFrame([dict(model=name, phase=phase,
            **{metric: score[phase+"_"+metric] for metric in ["logloss", "f1", "roc_auc", "pr_auc"]})
            for name, score in scores.items() for phase in PHASES])
        display(phase_table)
        display(pd.DataFrame(scores).T[["phase_mean_logloss"]])
        phase_table.to_csv(GENERATED / "validation_phase_metrics.csv", index=False)

        ax = phase_table.pivot(index="phase", columns="model", values="logloss").plot.bar(figsize=(10, 4))
        ax.set(ylabel="Held-out log loss (lower is better)", xlabel="Clinical phase",
               title="Official validation: identical held-out rows")
        ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
        plt.tight_layout()
        plt.savefig(GENERATED / "validation_comparison.png", dpi=160, bbox_inches="tight")
        plt.show()
        '''),
        md('''## Post-hoc stopping-reason analysis

        On known failed validation trials, a false-success error means predicted
        probability ≥ 0.5. The supplied frozen reason table uses known Part 1 labels
        where available and historical inferred labels elsewhere. Their counts are
        shown separately; unmatched trials remain a separate category. Reasons can
        overlap, and inferred reasons add uncertainty. This table is joined **after**
        outcome prediction and does not affect any model score or fit.

        This Part 2 package consumes the existing reason-label export and its provenance;
        it does not train or revise Part 1. No Part 1 checkout is needed to reproduce
        these tables and plots.
        '''), code('''
        from error_analysis import errors_by_reason
        reason_labels = pd.read_csv(ROOT / "experiments/model_search/reason_labels_for_analysis.csv")
        display(json.loads((ROOT / "experiments/model_search/reason_model_provenance.json").read_text()))
        reason_table = pd.concat([errors_by_reason(validation, predictions[name], reason_labels).assign(model=name)
            for name in ["Eligibility LR baseline", "File 13 (final)"]], ignore_index=True)
        display(reason_table)
        reason_table.to_csv(GENERATED / "reason_errors.csv", index=False)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
        for phase, ax in zip(PHASES, axes):
            subset = reason_table[reason_table.phase.eq(phase)]
            subset.pivot(index="reason", columns="model", values="false_success_rate").plot.bar(ax=ax)
            ax.set(title=f"Phase {phase}", xlabel="", ylim=(0, 1))
            ax.tick_params(axis="x", labelrotation=70, labelsize=8)
            ax.legend(fontsize=7)
        axes[0].set_ylabel("False-success rate among failed trials")
        fig.suptitle("Post-hoc reasons; known/inferred counts and Wilson intervals in the table")
        fig.tight_layout()
        fig.savefig(GENERATED / "reason_errors.png", dpi=160, bbox_inches="tight")
        plt.show()
        '''),
        md('''## Exact scored artifact and recorded-component replay

        The small component archive contains native predictions from the original
        fitted models, with their cache hashes recorded. Recombining these arrays
        checks the final recipe and CSV; **it is not a new fit**. We preserve the
        exact scored CSV bytes separately from newly serialized predictions.
        '''), code('''
        artifact = ROOT / "submissions/13_file10_legacy30.csv"
        expected_sha = "be29fc849f63ed4c23b31e67c97aaa97c6940f569b0af0920cf9799b428f7636"
        assert sha256(artifact) == sha256(HANDIN / "part2_best_submission.csv") == expected_sha
        provenance = json.loads((ROOT / "experiments/final_handin/recorded_components_provenance.json").read_text())
        component_path = ROOT / "experiments/final_handin/recorded_components.npz"
        assert sha256(component_path) == provenance["component_archive_sha256"]
        with np.load(component_path) as components:
            assert components["ID"].tolist() == data["test"].ID.tolist()
            file10 = .5*components["ani_stack"] + .5*components["fast2_neighbors"]
            replay = np.clip(.7*file10 + .3*components["legacy01"], .001, .999)
        recorded = pd.read_csv(artifact)
        sample = pd.read_csv(ROOT / "data/part2_sample_submission.csv")
        assert recorded.ID.equals(sample.ID) and recorded.ID.is_unique and len(recorded) == 1500
        replay_error = float(np.max(np.abs(recorded.label - replay)))
        assert replay_error < 1e-12
        pd.DataFrame(dict(ID=data["test"].ID, label=replay)).to_csv(GENERATED / "recorded_blend_replay.csv", index=False)
        (GENERATED / "exact_scored_submission.csv").write_bytes(artifact.read_bytes())
        display(dict(kaggle_score=ledger["current_best"]["kaggle_score"],
                     exact_csv_sha256=expected_sha, recorded_component_replay_max_error=replay_error))
        '''),
        md('''## Fresh final training from raw data

        This code calls the preserved model implementations, refits both stacking
        banks and the original text/tree ensemble, checks disjoint stacking folds,
        and tests native inference after replacing IDs and adding a dummy label.
        It writes to `generated/fresh_fit/`, leaving the scored artifact intact.
        Set the flag in the following cell to run it. This takes substantially longer
        than replaying saved probabilities. Machine/library differences may change
        floating-point results; the receipt records versions and numerical differences.
        '''), code(training_source), code('''
        RUN_FINAL_TRAINING = False
        if RUN_FINAL_TRAINING:
            fresh_receipt = refit_final(ROOT, GENERATED / "fresh_fit")
            display(fresh_receipt)
        else:
            print("Fresh training is off. The checks above rescore held-out predictions and replay recorded components.")
        verified_refit = ROOT / "experiments/final_handin/refit_verification.json"
        if verified_refit.exists():
            print("Recorded verification of the fresh-fit path during hand-in preparation:")
            display(json.loads(verified_refit.read_text()))
        '''),
        md('''## Interpretation and limits

        File 13 improved the reported Kaggle log loss from Ani's 0.58688 to 0.58465.
        File 15 improved some local averages but scored 0.58492, so we retained file 13.
        The final reserve was not submitted. Local validation was repeatedly used for
        selection; bootstrap intervals do not correct for this adaptive search.
        The reported rank and scores are historical user/commit reports, not a live
        leaderboard query. See the experiment notebook for all phases, failures,
        controls, and the distinction between configurations and independent fits.
        ''')]
    write(destination / "01_final_model.ipynb", final)

    history = [md('''# Part 2 — experiments and selection history

        This notebook organizes the complete working evidence by phase. It preserves
        unsuccessful trials and separates local measurements from Kaggle feedback.
        Full summary tables, split protocols, settings, manifests and figures are
        included under `resources/experiments/`, along with the row predictions used
        by these finalized comparisons. Other intermediate prediction arrays and
        repeated fold audits remain in the working project and can be regenerated.
        The original research notebooks are
        under `resources/notebooks/`. The source appendix includes experiment and
        figure-generation implementations, including the final files 11–16 sprint.
        '''), code(SETUP),
        md('''## Experiment inventory

        Counts below are rows in the retained result tables. They are **not** a total
        number of independent trained models: ensemble weights reuse predictions,
        controls recur across phases, and one recipe is evaluated on multiple splits
        and preprocessing tracks. The 318 component and 434 ensemble rows belong to
        the initial search, not to the later common benchmark.
        '''), code('''
        stages = {
            "Initial component search": "model_search/all_component_scores.csv",
            "Initial ensemble search": "model_search/ensemble_scores.csv",
            "Stack proposals and controls": "stack_improvements/runs/097ac8161372126d/strict/summary.csv",
            "Meta extensions and controls": "ani_extensions/meta/76d98f1d5e9d5ff1/strict/summary.csv",
            "Neighbor extensions and controls": "ani_extensions/neighbors/a27759052dfa9571/strict/summary.csv",
            "Files 01–10 on the common benchmark": "all_submissions/runs/8f68385e25e07615/comparison_strict.csv",
            "Sprint: 32 proposals plus controls": "leaderboard_sprint/runs/4dc0f274d45bce9e/summary_strict.csv",
            "Recent trees: 4 proposals plus controls": "leaderboard_sprint/recent_tree/dea847f57c7c07ee/summary_strict.csv",
            "Refinement: 19 proposals plus controls": "leaderboard_sprint/refine/2e5c792462013673/summary_strict.csv",
            "Final slot: 9 proposals plus controls": "leaderboard_sprint/finish/b235582538f63f9c/summary_strict.csv",
        }
        inventory = pd.DataFrame([dict(stage=stage, result_rows=len(pd.read_csv(ROOT / "experiments" / path)),
                                       table=path) for stage, path in stages.items()])
        display(inventory)
        inventory.to_csv(GENERATED / "experiment_inventory.csv", index=False)
        for stage, path in stages.items():
            display(Markdown(f"### {stage}"))
            display(pd.read_csv(ROOT / "experiments" / path))
        '''),
        md('''## Why these changes were tried

        1. Text, recency, structured drug features and tree models supplied diverse
           original components; the first submitted recipes trailed Ani's stacks.
        2. A fixed trial-grouped and chronological benchmark put both Ani winners and
           our proposals on the same rows. Stronger meta regularization, temporal
           features, and many alternative mixtures were retained even when they lost.
        3. Faster recency and additional same-phase neighbor views yielded file 10.
           A half-strength blend kept Ani's stack as an anchor.
        4. The original text/tree ensemble contributed complementary errors. A 20%
           contribution produced file 11; 30% produced the final file 13.
        5. Recent-tree and trend refinements looked attractive locally but file 15
           did not improve Kaggle. File 16's tiny, uncertain gains did not justify
           using the last submission slot.

        [Full rationale and links](resources/MODEL_DEVELOPMENT.md) ·
        [Frozen protocols](resources/docs/README.md) ·
        [Original notebook guide](resources/notebooks/README.md)
        '''),
        md('''## Direct comparison: the same benchmark and preprocessing track

        Lower is better. These rows come from the same strict run, using identical
        outer splits, phase weighting and grouped-CV aggregation. The published
        preprocessing track is shown separately. Local results are selection evidence;
        stronger local averages did not guarantee stronger Kaggle performance.
        '''), code('''
        chosen = ["ani_first", "ani_stack", "file10", "mix_legacy01_0.2", "mix_legacy01_0.3"]
        columns = ["official", "temporal", "worst", "cv", "gap_mean", "gap_worst"]
        comparisons = {}
        for track in ["strict", "published"]:
            table = pd.read_csv(SPRINT / f"summary_{track}.csv").set_index("model")
            comparisons[track] = table.loc[chosen, columns]
            display(Markdown(f"### {track.title()} preprocessing"))
            display(comparisons[track])
        comparisons["strict"].to_csv(GENERATED / "common_benchmark.csv")
        plotted = comparisons["strict"][["temporal", "gap_mean", "cv"]].rename(
            index={"ani_first": "Ani unweighted", "ani_stack": "Ani recency", "file10": "File 10",
                   "mix_legacy01_0.2": "File 11", "mix_legacy01_0.3": "File 13 (final)"},
            columns={"temporal": "Forward windows", "gap_mean": "Longer-gap windows", "cv": "Grouped CV"})
        ax = plotted.plot(marker="o", figsize=(10, 4))
        ax.set(ylabel="Equal-phase log loss", xlabel="Recipe", title="Shared strict benchmark")
        ax.set_xticks(range(len(plotted)), plotted.index, rotation=20, ha="right")
        ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
        plt.tight_layout()
        plt.savefig(GENERATED / "common_benchmark.png", dpi=160, bbox_inches="tight")
        plt.show()
        '''),
        md('''## Verify the main comparisons from row-level predictions

        The following check recomputes every phase log loss on each of the ten original
        splits for the five recipes above, in both preprocessing tracks. It also checks
        split membership and original labels. The longer-gap results and paired
        uncertainty estimates remain in the linked phase reports.
        '''), code('''
        from benchmark_protocol import initialize, resolve
        universe, protocol = initialize()
        verification_rows = []
        for track in ["strict", "published"]:
            for split in protocol["splits"]:
                _, evaluation = resolve(universe, split)
                stored = pd.read_csv(SPRINT / track / f"scores_{split['name']}.csv").set_index("model")
                for model in chosen:
                    frame = pd.read_csv(SPRINT / track / "predictions" / split["name"] / f"{model}.csv")
                    assert frame[["ID", "trial_id", "phase", "label"]].equals(evaluation[["ID", "trial_id", "phase", "label"]])
                    measured = metrics(evaluation, frame.probability.to_numpy())
                    for phase in PHASES:
                        assert abs(measured[phase+"_logloss"] - stored.loc[model, phase+"_logloss"]) < 1e-12
                    assert abs(measured["phase_mean_logloss"] - stored.loc[model, "phase_mean_logloss"]) < 1e-12
                    verification_rows.append(dict(track=track, split=split["name"], model=model,
                                                  phase_mean_logloss=measured["phase_mean_logloss"]))
        verified = pd.DataFrame(verification_rows)
        assert len(verified) == 100
        verified.to_csv(GENERATED / "benchmark_rescoring.csv", index=False)
        print(f"Verified {len(verified)} recipe/split/track evaluations against original row labels.")
        '''),
        md('''## Kaggle feedback and final decision

        Only confirmed submitted artifacts have recorded scores. Local recipes and
        unsubmitted reserves do not inherit scores from similar filenames or weights.
        These are historical user/commit reports; no live leaderboard was queried.
        '''), code('''
        rows = []
        for name, item in ledger["ani_winners"].items():
            rows.append(dict(model=name, kaggle_log_loss=item["kaggle_score"], source=item["source"]))
        for name, item in ledger["submissions"].items():
            assert sha256(ROOT / "submissions" / name) == item["sha256"]
            rows.append(dict(model=name, kaggle_log_loss=item["kaggle_score"], source=item["source"]))
        feedback = pd.DataFrame(rows)
        display(feedback)
        feedback.to_csv(GENERATED / "kaggle_feedback.csv", index=False)
        ax = feedback.set_index("model").kaggle_log_loss.plot(marker="o", figsize=(10, 4))
        ax.set(ylabel="Reported Kaggle log loss", xlabel="Scored artifact")
        labels = ["Ani unweighted", "Ani recency", "File 03", "File 02", "File 10", "File 11", "File 13 (final)", "File 15"]
        ax.set_xticks(range(len(feedback)), labels, rotation=30, ha="right", fontsize=9)
        plt.tight_layout()
        plt.savefig(GENERATED / "kaggle_progress.png", dpi=160, bbox_inches="tight")
        plt.show()
        display(json.loads((ROOT / "experiments/leaderboard_sprint/refine/2e5c792462013673/submission_decision.json").read_text()))
        '''), md('''## Re-running research versus reproducing the final model

        The final model's fresh training is in notebook 01. The complete historical
        research suite is much larger and is not rerun by default. Its original
        notebooks, saved notebook outputs, settings and trial summary tables are
        included in `resources/`. Their execution may require regenerating omitted
        intermediate arrays or using the full working checkout. Use the commands in
        [reproduction instructions](resources/docs/CACHE_REPRODUCTION.md) and
        [organization guide](resources/docs/ORGANIZATION.md), replacing `part2/`
        with this package's `resources/` path. Rebuild omitted intermediate predictions
        and fitted caches first; some historical runners expect earlier phases to
        have completed. The full working project retains those intermediates.

        Historical scripts that recreate Part 1 reason labels require the separate
        Part 1 project. The supplied frozen labels fully reproduce Part 2 post-hoc
        analysis without that step. Historical Git-based provenance/tag checks also
        require the research Git repository; the hand-in uses file hashes instead.
        ''')]
    write(destination / "02_experiments.ipynb", history)

    paths = [p for p in source_files if p.suffix == ".py"]
    appendix = [md('''# Part 2 — complete source appendix

        This notebook embeds the preserved Python code for model fitting, feature
        construction, experimental searches, evaluation, figure generation, tests,
        launchers, and hand-in preparation. The two analysis notebooks contain their
        own table/figure code. The historical Ani notebooks are also included in
        `resources/experiments/ani_comparison/source/`.

        **Run all writes inspectable copies into `_source_export/`; it does not run
        every search or overwrite the canonical research sources.** Each write cell
        contains the actual source, with its relative path and hash. The final cell
        verifies these copies byte-for-byte against the included implementations.
        '''), code(SETUP), code(
        'export_directory = HANDIN / "_source_export"\n'
        'export_directory.mkdir(exist_ok=True)\n'
        'os.chdir(export_directory)\n' + "\n".join(
            f"Path({folder!r}).mkdir(parents=True, exist_ok=True)"
            for folder in sorted({str(p.relative_to(repo / "part2").parent) for p in paths}))) ]
    hashes = {}
    for path in paths:
        relative = str(path.relative_to(repo / "part2"))
        content = path.read_text()
        hashes[relative] = hashlib.sha256(path.read_bytes()).hexdigest()
        appendix.append(md(f"## `{relative}`\n\nSHA-256: `{hashes[relative]}`"))
        cell = nbf.v4.new_code_cell(f"%%writefile {relative}\n" + content)
        cell.metadata["source_path"] = relative
        cell.metadata["source_sha256"] = hashes[relative]
        appendix.append(cell)
    source_lengths = {str(p.relative_to(repo / "part2")): p.stat().st_size for p in paths}
    appendix.append(code(f"expected_sources = {hashes!r}\nsource_lengths = {source_lengths!r}\n" + '''
for relative, expected in expected_sources.items():
    # IPython adds a final newline to %%writefile bodies. Two archived Ani
    # sources originally had no terminal newline; restore that exact EOF.
    exported = export_directory / relative
    content = exported.read_bytes()
    if len(content) == source_lengths[relative] + 1 and content.endswith(b"\\n"):
        exported.write_bytes(content[:-1])
    assert sha256(export_directory / relative) == expected, relative
    assert sha256(ROOT / relative) == expected, relative
os.chdir(HANDIN)
print(f"Verified {len(expected_sources)} complete source files.")
'''))
    write(destination / "03_source_code.ipynb", appendix)


Overwriting packaging/notebooks.py


## `packaging/train_final.py`

SHA-256: `4671e9b38a1016d8db23d3c8c490c7fadd31e0c6b70b6a08d1b7aab2d2a07966`

In [8]:
%%writefile packaging/train_final.py
"""Refit file 13 from the supplied CSVs without reading existing fitted caches.

The same implementation is embedded in the finalized model notebook. Run with
Python 3.12 and part2/requirements/requirements-all-submissions.txt installed.
"""
from pathlib import Path
import argparse
import hashlib
import importlib.metadata
import json
import os
import runpy
import sys
import time

for _key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_key, "2")


def refit_final(part2_root, output_dir):
    """Train the frozen recipe, replay it on feature-only rows, and save a new CSV."""
    root, output = Path(part2_root).resolve(), Path(output_dir).resolve()
    output.mkdir(parents=True, exist_ok=True)
    workspace = runpy.run_path(str(root / "run.py"))["prepare_workspace"](root)
    sys.path.insert(0, str(workspace))
    import numpy as np
    import pandas as pd
    from threadpoolctl import threadpool_limits
    from ani_models import AniFeatures
    from experiments import load_data
    from improvement_models import fit_bank, fit_recipes as fit_base_metas
    from extension_neighbors import fit_extended, fit_recipes as fit_neighbor_metas
    from all_submission_models import LegacySubmissions
    from sprint_export import native_packets, original_components

    started = time.monotonic()
    def progress(message):
        print(f"{time.monotonic() - started:.1f}s: {message}", flush=True)

    data = load_data()
    train = pd.concat([data["train"], data["val"]], ignore_index=True)
    test = data["test"]
    assert len(train) == 9050 and len(test) == 1500
    assert not set(train.trial_id) & set(test.trial_id)
    assert "label" not in test
    # Frozen published-recipe preprocessing: vocabulary/IDF see all supplied
    # texts; supervised fits and grouped stacking folds use labeled rows only.
    with threadpool_limits(limits=2):
        features = AniFeatures().fit(pd.concat(list(data.values()), ignore_index=True))
        progress("fitted vocabulary/IDF on supplied train, validation, and test text")
        old = fit_bank(train, test, features=features, retain_models=True,
                       progress=lambda message: progress("base bank " + message))
        baseline, base_metas = fit_base_metas(old, train, test)
        extra = fit_extended(train, test, old, space=features, retain=True,
                             progress=lambda fold: progress(f"neighbor bank inner {fold}/5"))
        neighbors, neighbor_metas = fit_neighbor_metas(old, extra, train, test, baseline)
        legacy = LegacySubmissions().fit(train, progress=progress)
        legacy_probability = legacy.predict(test)["01_balanced_ensemble"]
        probability = np.clip(.7 * neighbors["fast2_neighbors_half"]
                              + .3 * legacy_probability, .001, .999)

        # Replay actual fitted models after removing outcomes and replacing IDs.
        query = test.copy()
        query["ID"] = "unused"
        query["trial_id"] = "unused"
        query["label"] = 0
        bundle = dict(old=old, extra=extra, base_metas=base_metas,
                      neighbor_metas=neighbor_metas)
        replay_old, replay_extra = native_packets(bundle, query)
        components = original_components(bundle, replay_old, replay_extra, query)
        replay = np.clip(.7 * components["file10"]
                         + .3 * legacy.predict(query)["01_balanced_ensemble"], .001, .999)
    replay_error = float(np.max(np.abs(probability - replay)))
    assert replay_error < 1e-12, replay_error
    assert old["training_ids"] == extra["train_ids"] == legacy.training_ids == train.ID.tolist()
    for packet in (old, extra):
        for fold in packet["audit"]:
            heldout = fold.get("heldout_ids", fold.get("eval_ids"))
            assert not set(fold["fit_ids"]) & set(heldout)
            indexed = train.set_index("ID")
            assert not (set(indexed.loc[fold["fit_ids"], "trial_id"])
                        & set(indexed.loc[heldout, "trial_id"]))

    sample = pd.read_csv(root / "data/part2_sample_submission.csv")
    frame = pd.DataFrame(dict(ID=test.ID, label=probability)).set_index("ID").loc[sample.ID].reset_index()
    assert frame.ID.is_unique and frame.ID.equals(sample.ID)
    assert np.isfinite(frame.label).all() and frame.label.between(0, 1, inclusive="neither").all()
    destination = output / "13_file10_legacy30_refit.csv"
    frame.to_csv(destination, index=False)
    recorded = pd.read_csv(root / "submissions/13_file10_legacy30.csv")
    assert frame.ID.equals(recorded.ID)
    error = float(np.max(np.abs(frame.label - recorded.label)))
    sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
    receipt = dict(
        recipe={"file10": .7, "legacy01": .3}, training_rows=len(train),
        test_rows=len(test), existing_model_caches_read=False,
        native_replay_max_error=replay_error, recorded_csv_max_error=error,
        numerically_matches_recorded_at_1e_7=error < 1e-7,
        recorded_csv_sha256=sha(root / "submissions/13_file10_legacy30.csv"),
        refit_csv_sha256=sha(destination), elapsed_seconds=time.monotonic()-started,
        versions={p: importlib.metadata.version(p) for p in
                  ["numpy", "pandas", "scipy", "scikit-learn", "catboost", "joblib"]},
        data_sha256={p.name: sha(p) for p in sorted((root / "data").glob("*.csv"))},
        source_sha256={p.name: sha(p) for p in sorted((root / "src").glob("*.py"))},
        note="The Kaggle score belongs to the original CSV. A refit does not acquire that score.")
    (output / "refit_verification.json").write_text(json.dumps(receipt, indent=2) + "\n")
    progress(f"saved refit; maximum difference from scored CSV: {error:.3g}")
    return receipt


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--part2-root", type=Path, default=Path(__file__).resolve().parents[1])
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    print(json.dumps(refit_final(args.part2_root, args.output), indent=2))


Overwriting packaging/train_final.py


## `packaging/verify_handin.py`

SHA-256: `0d049dd5f90c45ca9cb9679e3e18ee38de9ff5aa8e5ee0374292d9a39e7ff4a9`

In [9]:
%%writefile packaging/verify_handin.py
"""Extract the Part 2 ZIP away from Git and execute all three notebooks."""
from pathlib import Path
import argparse
import hashlib
import json
import tempfile
import zipfile


def verify(archive):
    import nbformat
    from nbclient import NotebookClient
    archive = Path(archive).resolve()
    sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
    with tempfile.TemporaryDirectory(prefix="isolated-part2-handin-") as temporary:
        root = Path(temporary)
        with zipfile.ZipFile(archive) as zipped:
            assert zipped.testzip() is None
            for name in zipped.namelist():
                assert name.startswith("midterm/part2/") and ".." not in Path(name).parts
                assert not name.endswith(".joblib")
            zipped.extractall(root)
        handin = root / "midterm/part2"
        resources = handin / "resources"
        assert not (root / ".git").exists() and not (root / "part1").exists()
        assert not any(p.is_symlink() for p in root.rglob("*"))
        manifest = json.loads((resources / "PACKAGE_MANIFEST.json").read_text())
        def check_resources():
            for relative, item in manifest["files"].items():
                path = resources / relative
                assert path.is_file() and path.stat().st_size == item["bytes"], relative
                assert sha(path) == item["sha256"], relative
        check_resources()
        records = []
        for path in sorted(handin.glob("0*.ipynb")):
            print(f"Standalone execution: {path.name}", flush=True)
            nb = nbformat.read(path, as_version=4)
            NotebookClient(nb, timeout=600, kernel_name="python3",
                           resources={"metadata": {"path": str(handin)}}).execute()
            records.append(dict(notebook=path.name, errors=0))
        check_resources()
        winner = sha(handin / "part2_best_submission.csv")
        assert winner == "be29fc849f63ed4c23b31e67c97aaa97c6940f569b0af0920cf9799b428f7636"
        assert not list(resources.rglob("*.joblib"))
        return dict(archive=archive.name, archive_sha256=sha(archive),
                    archive_bytes=archive.stat().st_size, notebooks=records,
                    resource_files_verified=len(manifest["files"]),
                    winner_sha256=winner, all_resource_bytes_unchanged=True,
                    required_git=False, required_part1_checkout=False,
                    required_existing_model_caches=False,
                    fresh_fit_receipt=json.loads((resources / "experiments/final_handin/refit_verification.json").read_text()))


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("archive", type=Path)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    report = verify(args.archive)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(report, indent=2) + "\n")
    print(json.dumps({k: v for k, v in report.items() if k != "fresh_fit_receipt"}, indent=2))


Overwriting packaging/verify_handin.py


## `run.py`

SHA-256: `d396a9f2ffc52db9f7b2fdd8b2fa5feca9539fcc5013607273ad5052378e4d93`

In [10]:
%%writefile run.py
"""Run preserved experiments from the organized Part Two checkout.

The ignored execution directory contains byte-identical source copies and links
to canonical data/results. This preserves the paths expected by frozen sources,
notebook cells, model pickles, and provenance checks without duplicating data.
"""
from __future__ import annotations

import json
import os
from pathlib import Path
import re
import subprocess
import sys
from urllib.parse import unquote, urlsplit

ROOT = Path(__file__).resolve().parent
BOOTSTRAP_ID = "part2-layout-setup"


def layout(root=ROOT):
    return json.loads((root / "layout.json").read_text())["legacy_paths"]


def copy_source(source, target):
    content = source.read_bytes()
    if target.exists() and target.read_bytes() == content:
        return
    temporary = target.with_name(f".{target.name}.{os.getpid()}.tmp")
    temporary.write_bytes(content)
    temporary.replace(target)


def link_to(source, target):
    expected = os.path.relpath(source, target.parent)
    if target.is_symlink():
        if os.readlink(target) == expected:
            return
        target.unlink()
    elif target.exists():
        raise RuntimeError(f"Expected a generated link at {target}; move it aside first")
    target.symlink_to(expected, target_is_directory=source.is_dir())


def prepare_workspace(root=ROOT):
    root = Path(root).resolve()
    workspace = root / ".runtime" / "part2"
    workspace.mkdir(parents=True, exist_ok=True)
    for old, new in layout(root).items():
        source, target = root / new, workspace / old
        if source.suffix == ".py" or old == "README.md":
            # __file__.resolve() must stay in the legacy execution directory.
            # Old notebook %%writefile cells can safely rewrite these copies.
            copy_source(source, target)
        else:
            link_to(source, target)
    for folder in ["experiments", "submissions"]:
        link_to(root / folder, workspace / folder)
    link_to(root.parent / "part1", workspace.parent / "part1")
    # New experiments need no entry in the historical migration map.
    for folder, pattern in [("src", "*.py"), ("tests", "test_*.py")]:
        for source in (root / folder).glob(pattern):
            copy_source(source, workspace / source.name)
    return workspace


def update_references(text, old_file, new_file, root=ROOT):
    """Translate original Markdown links and CLI examples to canonical paths."""
    paths = layout(root)
    def translated(target):
        try:
            relative = target.relative_to(root).as_posix()
        except ValueError:
            return target
        return root / paths.get(relative, relative)

    def replace_link(match):
        href = match.group(2)
        parts = urlsplit(href)
        if parts.scheme or not parts.path:
            return match.group(0)
        original = Path(os.path.abspath(old_file.parent / unquote(parts.path)))
        target = translated(original)
        relative = Path(os.path.relpath(target, new_file.parent)).as_posix()
        suffix = ("?" + parts.query if parts.query else "") + ("#" + parts.fragment if parts.fragment else "")
        return match.group(1) + relative + suffix + ")"

    text = re.sub(r"(!?\[[^\]]*\]\()([^)]+)\)", replace_link, text)
    # Historical protocol documents are intentionally excluded by callers.
    scripts = {name for name in paths if name.endswith(".py")}
    text = re.sub(r"(?P<prefix>(?:\.venv/bin/python|python3?)\s+)part2/(?P<script>[\w_]+\.py)",
                  lambda m: m["prefix"] + "part2/run.py " + m["script"]
                  if m["script"] in scripts else m.group(0), text)
    text = re.sub(r"part2/(requirements(?:-[\w-]+)?\.txt)", r"part2/requirements/\1", text)
    text = re.sub(r"part2/(outcome_(?:train|val|test)\.csv)", r"part2/data/\1", text)
    text = re.sub(r"-m unittest discover -s part2 -p ('[^']+'|\"[^\"]+\"|\S+)",
                  r"part2/run.py test \1", text)
    return text


def refresh_report_links(root=ROOT):
    for folder in [root / "experiments", root / "submissions"]:
        for path in folder.rglob("*.md"):
            text = path.read_text()
            updated = update_references(text, path, path, root)
            if updated != text:
                path.write_text(updated)


def install_notebook_bootstrap(root=ROOT):
    source = '''# Locate the organized checkout and use its preserved execution layout.
from pathlib import Path
import os, runpy, sys
_bases = [Path.cwd(), *Path.cwd().parents]
_part2 = next(candidate for base in _bases for candidate in (base, base / "part2")
              if (candidate / "run.py").is_file() and (candidate / "layout.json").is_file())
_workspace = runpy.run_path(str(_part2 / "run.py"))["prepare_workspace"]()
os.chdir(_workspace)
sys.path.insert(0, str(_workspace))
'''
    for path in (root / "notebooks").glob("*.ipynb"):
        notebook = json.loads(path.read_text())
        if any(cell.get("id") == BOOTSTRAP_ID for cell in notebook["cells"]):
            continue
        notebook["cells"].insert(0, {
            "cell_type": "code", "execution_count": None, "id": BOOTSTRAP_ID,
            "metadata": {}, "outputs": [], "source": source.splitlines(keepends=True),
        })
        path.write_text(json.dumps(notebook, indent=1, ensure_ascii=False) + "\n")


def main(argv=None):
    args = list(sys.argv[1:] if argv is None else argv)
    if not args or args[0] in ["-h", "--help"]:
        print("Usage: python part2/run.py SCRIPT.py [arguments]\n"
              "       python part2/run.py test [test_pattern.py]\n"
              "       python part2/run.py list\n\n"
              "Edit code in part2/src; open saved notebooks in part2/notebooks.\n"
              "Training runs only when you select a training script or Make target.")
        return 0
    if args[0] == "list":
        print("\n".join(sorted(path.name for path in (ROOT / "src").glob("*.py"))))
        return 0
    available = {path.name for path in (ROOT / "src").glob("*.py")}
    if args[0] != "test" and args[0] not in available:
        print(f"Unknown script: {args[0]}. Use 'list' to see available scripts.", file=sys.stderr)
        return 2
    workspace = prepare_workspace()
    environment = os.environ.copy()
    environment["PART2_PROJECT_ROOT"] = str(ROOT)
    environment["PYTHONPATH"] = str(workspace) + os.pathsep + environment.get("PYTHONPATH", "")
    if args[0] == "test":
        command = [sys.executable, "-m", "unittest", "discover", "-s", str(workspace),
                   "-p", args[1] if len(args) > 1 else "test_*.py"]
    else:
        command = [sys.executable, str(workspace / args[0]), *args[1:]]
    try:
        return subprocess.run(command, env=environment).returncode
    finally:
        # Report generators retain their frozen source; repair their navigation
        # and notebook startup after they write into the canonical folders.
        refresh_report_links()
        install_notebook_bootstrap()


if __name__ == "__main__":
    raise SystemExit(main())


Overwriting run.py


## `src/all_submission_models.py`

SHA-256: `7d8a51f78d13c64a7cdaef02a4b97b5b408d4164a7c806aa96593c001c8d8832`

In [11]:
%%writefile src/all_submission_models.py
"""Original files 01–06, refitted without tuning on arbitrary benchmark folds."""
import json
from pathlib import Path
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from benchmark_models import OurLinear
from boosted import transform_dense
from ensemble_search import weighted
from experiments import ROOT, matrix

LINEAR_NAMES = ["text_wordwide_C2", "recency_criteria_C2_half6",
                "recency_clinical_C0.5_half3"]
CAP = "lr_criteria_yearcap_C2_phase0"


def recipes():
    result = {}
    for file in ["manifest.json", "followup_manifest.json"]:
        result.update(json.loads((ROOT / "submissions" / file).read_text())["ensembles"])
    return result


class LegacySubmissions:
    """Fit six original base components once, then apply frozen ensemble weights."""
    def fit(self, train, progress=lambda message: None):
        self.training_ids = train.ID.tolist()
        self.training_trial_ids = sorted(train.trial_id.unique().tolist())
        self.ensembles = recipes()
        self.linear = OurLinear().fit(train)
        self.features = self.linear.features
        blocks = self.features.transform(train)
        self.capped = LogisticRegression(C=2, solver="liblinear", max_iter=1000,
                                        tol=1e-5, random_state=42)
        self.capped.fit(matrix(blocks, "criteria", "year_cap"), train.label)
        progress("linear components fitted")
        self.svds = {key: TruncatedSVD(n_components=n, random_state=42, n_iter=5).fit(blocks[key])
                     for key, n in [("criteria", 100), ("disease", 24), ("drug", 24), ("char", 32)]}
        self.mask = np.asarray((blocks["knowledge"] > 0).sum(axis=0)).ravel() >= 15
        dense = transform_dense(train, blocks, self.svds, self.mask,
                                self.features.year_median).drop(columns="year")
        cats = dense.select_dtypes("str").columns.tolist()
        self.trees = {}
        for depth in [6, 8]:
            model = CatBoostClassifier(iterations=1000, depth=depth, learning_rate=.035,
                l2_leaf_reg=6, loss_function="Logloss", random_seed=42, thread_count=4,
                verbose=False, allow_writing_files=False)
            model.fit(dense, train.label, cat_features=cats)
            self.trees[f"catboost_d{depth}_n1000_none"] = model
            progress(f"CatBoost depth {depth} fitted")
        return self

    def components(self, df):
        blocks = self.features.transform(df)
        xs = self.linear.inputs(df, blocks)
        bank = {name: model.predict_proba(x)[:, 1]
                for name, model, x in zip(LINEAR_NAMES, self.linear.models, xs)}
        bank[CAP] = self.capped.predict_proba(matrix(blocks, "criteria", "year_cap"))[:, 1]
        dense = transform_dense(df, blocks, self.svds, self.mask,
                                self.features.year_median).drop(columns="year")
        bank.update({name: model.predict_proba(dense, ntree_end=1000)[:, 1]
                     for name, model in self.trees.items()})
        return bank

    def predict(self, df):
        bank = self.components(df)
        result = {name: weighted(bank, weights, df.phase) for name, weights in self.ensembles.items()}
        result["our_linear"] = sum(w * bank[k] for w, k in zip([.375, .375, .25], LINEAR_NAMES))
        return result


Overwriting src/all_submission_models.py


## `src/all_submissions_benchmark.py`

SHA-256: `fce474d88e9d29b68daf05521a63e4816e1c8a8f66c9cbd2ea03cfda7cc9cab2`

In [12]:
%%writefile src/all_submissions_benchmark.py
"""One fixed benchmark for all ten numbered submissions and both Ani winners."""
import os
for key in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ.setdefault(key, "4")
import argparse
import hashlib
import importlib.metadata
import json
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from all_submission_models import LegacySubmissions, recipes
from benchmark_protocol import initialize, resolve, digest, DIRECTORY
from experiments import ROOT, load_data
from local_benchmark import predict_and_score
from improvement_report import audit_track

OUT = ROOT / "experiments/all_submissions"
EXTENSION = ROOT / "experiments/ani_extensions/decision"
ALIASES = {"ani_first": "ani_first", "ani_stack": "ani_stack", "ani_phase3_trend": "ani_phase3_trend",
           "09_ani_replace_knn5": "replace_knn5", "10_ani_fast2_neighbors_half": "fast2_neighbors_half"}
NAMES = list(recipes()) + ["07_ani_mild_phase3_trend", "08_ani_linear25"] + list(ALIASES)
SOURCES = ["all_submission_models.py", "all_submissions_benchmark.py", "benchmark_models.py",
           "benchmark_protocol.py", "boosted.py", "experiments.py", "ensemble_search.py",
           "local_benchmark.py", "improvement_report.py"]


def register():
    universe, protocol = initialize()
    gaps = json.loads((EXTENSION / "gaps/protocol.json").read_text())
    # Check inherited controls' original model implementations before reusing them.
    source_manifest = ROOT / "experiments/stack_improvements/runs/097ac8161372126d/manifest.json"
    for name, expected in json.loads(source_manifest.read_text())["source_sha256"].items():
        assert digest(ROOT / name) == expected, name
    selection = json.loads((EXTENSION / "selection.json").read_text())
    for path in selection["source_runs"].values():
        for name, expected in json.loads((ROOT / path / "manifest.json").read_text())["source_sha256"].items():
            assert digest(ROOT / name) == expected, name
    inputs = ["submissions/manifest.json", "submissions/followup_manifest.json",
              "submissions/ani_candidates_manifest.json", "submissions/ani_extensions_manifest.json",
              "submissions/leaderboard_results.json"]
    inputs += [str(path.relative_to(ROOT)) for path in sorted((ROOT / "submissions").glob("[0-9][0-9]_*.csv"))]
    manifest = dict(models=NAMES, recipes=recipes(), aliases=ALIASES,
        protocol_sha256=digest(DIRECTORY / "protocol.json"),
        gap_protocol_sha256=digest(EXTENSION / "gaps/protocol.json"),
        policy_sha256=digest(ROOT / "ALL_SUBMISSIONS_PROTOCOL.md"),
        source_sha256={f: digest(ROOT / f) for f in SOURCES},
        input_sha256={f: digest(ROOT / f) for f in inputs},
        versions={p: importlib.metadata.version(p) for p in ["numpy", "pandas", "scipy", "scikit-learn", "catboost", "joblib"]},
        preprocessing=dict(strict="All models training-fold-only",
            published="Original recipe policies: Ani covariate-transductive; legacy components training-fold-only"))
    key = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:16]
    out = OUT / "runs" / key
    out.mkdir(parents=True, exist_ok=True)
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
    (OUT / "current_run.json").write_text(json.dumps(dict(run=key), indent=2) + "\n")
    return universe, protocol, gaps, out


def current():
    return OUT / "runs" / json.loads((OUT / "current_run.json").read_text())["run"]


def original_parity(model, ev, out):
    """Verify this fold adapter against the artifacts actually submitted."""
    errors = []
    components = model.components(ev)
    for family in ["refined", "extra", "boosted", "yearcap"]:
        path = ROOT / "experiments/model_search" / f"{family}_validation_predictions.npz"
        if not path.exists():
            continue
        with np.load(path) as old:
            for name in set(old.files) & set(components):
                errors.append(dict(kind="validation_component", model=name,
                    max_probability_difference=float(np.max(np.abs(old[name] - components[name])))))
    test = load_data()["test"]
    ps = model.predict(test)
    for name in recipes():
        path = ROOT / "submissions" / f"{name}.csv"
        saved = pd.read_csv(path)
        assert saved.ID.tolist() == test.ID.tolist()
        errors.append(dict(kind="submitted_test_file", model=name,
            max_probability_difference=float(np.max(np.abs(saved.label - ps[name]))),
            original_sha256=digest(path)))
    changed = ev.copy()
    changed["ID"] = "unused"; changed["trial_id"] = "unused"; changed["label"] = 1 - changed.label
    before, after = model.predict(ev), model.predict(changed)
    invariance = {name: float(np.max(np.abs(before[name] - after[name]))) for name in before}
    assert max(invariance.values()) == 0
    pd.DataFrame(errors).to_csv(out / "original_replay.csv", index=False)
    (out / "inference_invariance.json").write_text(json.dumps(invariance, indent=2) + "\n")
    assert max(r["max_probability_difference"] for r in errors) < 1e-6, errors
    print("Original replay max error", max(r["max_probability_difference"] for r in errors), flush=True)


def run_split(task):
    split, is_gap, out_string = task
    out = Path(out_string)
    base = out / "gaps" if is_gap else out
    done = all((base / track / f"scores_{split['name']}.csv").exists() for track in ["strict", "published"])
    if done:
        print("cached", split["name"], flush=True)
        return
    universe, _ = initialize(); train, ev = resolve(universe, split)
    cache = OUT / "cache" / out.name
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / f"{split['name']}.joblib"
    started = time.time()
    def progress(message):
        print(split["name"], message, f"{time.time()-started:.1f}s", flush=True)
    if path.exists():
        model = joblib.load(path)
    else:
        model = LegacySubmissions().fit(train, progress)
        joblib.dump(model, path, compress=3)
    assert model.training_ids == train.ID.tolist()
    assert not set(model.training_trial_ids) & set(ev.trial_id)
    legacy = model.predict(ev)
    if split["name"] == "official":
        original_parity(model, ev, out)
    input_hashes = {}
    for track in ["strict", "published"]:
        source = EXTENSION / "gaps" / track if is_gap else EXTENSION / track
        predictions = {name: legacy[name] for name in recipes()}
        for name, alias in ALIASES.items():
            p = source / "predictions" / split["name"] / f"{alias}.csv"
            d = pd.read_csv(p)
            for col in ["ID", "trial_id", "phase", "label"]:
                assert d[col].tolist() == ev[col].tolist(), (p, col)
            predictions[name] = d.probability.to_numpy()
            input_hashes[str(p.relative_to(ROOT))] = digest(p)
        predictions["07_ani_mild_phase3_trend"] = .75 * predictions["ani_stack"] + .25 * predictions["ani_phase3_trend"]
        predictions["08_ani_linear25"] = .75 * predictions["ani_stack"] + .25 * legacy["our_linear"]
        predict_and_score(base / track, split, ev, predictions)
    folder = base / "audits"; folder.mkdir(exist_ok=True)
    audit = dict(training_ids=train.ID.tolist(), evaluation_ids=ev.ID.tolist(),
        preprocessing_fit_ids=model.training_ids, training_trial_ids=model.training_trial_ids,
        reused_predictions_sha256=input_hashes, model_sha256=digest(path),
        legacy_preprocessing="Fitting rows only: vocabulary, IDF, SVD, metadata mask, year statistics",
        seconds=time.time()-started)
    (folder / f"{split['name']}.json").write_text(json.dumps(audit, indent=2) + "\n")
    progress("all 13 recipes scored in both tracks")


def summarize(out, universe, protocol, gaps):
    for is_gap, spec in [(False, protocol), (True, gaps)]:
        for track in ["strict", "published"]:
            folder = out / "gaps" / track if is_gap else out / track
            table, scores, _ = audit_track(folder, universe, spec, names=NAMES)
            table.to_csv(folder / "summary.csv")
            scores.to_csv(folder / "fold_scores.csv", index=False)
            print("gaps" if is_gap else "main", track, table.sort_values("temporal").to_string(), flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--official-only", action="store_true")
    parser.add_argument("--summarize", action="store_true")
    args = parser.parse_args()
    universe, protocol, gaps, out = register()
    tasks = [(s, False, str(out)) for s in protocol["splits"]]
    tasks += [(s, True, str(out)) for s in gaps["splits"]]
    if args.official_only:
        run_split(tasks[0]); return
    if not args.summarize:
        with ProcessPoolExecutor(max_workers=args.workers) as pool:
            list(pool.map(run_split, tasks))
    summarize(out, universe, protocol, gaps)
    print("Results", out, flush=True)


if __name__ == "__main__":
    main()


Overwriting src/all_submissions_benchmark.py


## `src/all_submissions_report.py`

SHA-256: `471bf520f9a60b6838223c01ee7e6ab26a74b284c0b223703350231bb04630b4`

In [13]:
%%writefile src/all_submissions_report.py
"""Build directly comparable tables, paired intervals, figures and a notebook."""
import json
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
from all_submissions_benchmark import current, NAMES, EXTENSION, SOURCES
from benchmark_protocol import initialize, digest
from experiments import ROOT
from improvement_report import audit_track, paired_interval

TARGET = "10_ani_fast2_neighbors_half"


def markdown_table(data):
    def cell(value):
        if isinstance(value, (float, np.floating)):
            return "Unknown" if np.isnan(value) else f"{value:.6f}"
        return str(value)
    return "\n".join(["| " + " | ".join(data.columns) + " |",
                      "| " + " | ".join(["---"]*len(data.columns)) + " |"] +
                     ["| " + " | ".join(cell(x) for x in row) + " |" for row in data.itertuples(index=False, name=None)])


def report():
    out = current(); universe, protocol = initialize()
    gaps = json.loads((EXTENSION / "gaps/protocol.json").read_text())
    ledger = json.loads((ROOT / "submissions/leaderboard_results.json").read_text())
    kaggle = {k: v["kaggle_score"] for k, v in ledger["ani_winners"].items()}
    kaggle.update({Path(k).stem: v["kaggle_score"] for k, v in ledger["submissions"].items()})
    for filename, entry in ledger["submissions"].items():
        assert digest(ROOT / "submissions" / filename) == entry["sha256"]
    best = ledger.get("current_best")
    rank_note = (f"The user reported rank {best['reported_rank']}. "
                 if best and best.get("reported_rank") is not None
                 else "Leaderboard rank was not refreshed for this score. ")
    score_update = (f"**Later Kaggle result:** `{best['submission']}` scored **{best['kaggle_score']:.5f}**, "
        f"improving Ani's 0.58688 by **{best['improvement_over_ani']:.5f}**. "
        + rank_note +
        "This result was reported after the frozen comparison. Local predictions and selection evidence are unchanged. "
        "The current ledger supplies the Kaggle column; its original input snapshot is preserved with the run."
        if best else "Kaggle results are recorded separately from local validation.")
    tables = {}; windows = []; pairs = []; wins = []
    for track in ["strict", "published"]:
        main, scores, frames = audit_track(out / track, universe, protocol, names=NAMES)
        gap, gap_scores, gap_frames = audit_track(out / "gaps" / track, universe, gaps, names=NAMES)
        table = main.join(gap[["temporal", "worst"]].rename(columns={"temporal": "gap_mean", "worst": "gap_worst"}))
        table["kaggle"] = table.index.map(kaggle)
        forward = scores[scores.kind.eq("temporal")].pivot(index="split", columns="model", values="phase_mean_logloss")
        gaps_wide = gap_scores.pivot(index="split", columns="model", values="phase_mean_logloss")
        table["forward_wins_vs_ani"] = forward.lt(forward.ani_stack, axis=0).sum()
        table["gap_wins_vs_ani"] = gaps_wide.lt(gaps_wide.ani_stack, axis=0).sum()
        table["temporal_delta_vs_02"] = table.temporal - table.loc["02_text_heavy", "temporal"]
        table["cv_delta_vs_02"] = table.cv - table.loc["02_text_heavy", "cv"]
        table.to_csv(out / f"comparison_{track}.csv")
        for metric in ["official", "temporal", "cv", "gap_mean"]:
            values = table[metric].to_numpy()
            matrix = pd.DataFrame(values[:, None] - values[None, :],
                                  index=table.index, columns=table.index)
            matrix.to_csv(out / f"pairwise_{track}_{metric}.csv")
        tables[track] = table
        windows += [scores.assign(track=track, suite="main"), gap_scores.assign(track=track, suite="gaps")]
        for reference in NAMES:
            if reference == TARGET:
                continue
            wins.append(dict(track=track, reference=reference,
                forward_wins=int((forward[TARGET] < forward[reference]).sum()),
                forward_windows=len(forward),
                gap_wins=int((gaps_wide[TARGET] < gaps_wide[reference]).sum()),
                gap_windows=len(gaps_wide)))
            for kind in ["temporal", "group_cv"]:
                pairs.append(dict(track=track, suite="main", **paired_interval(frames, TARGET, reference, kind)))
            pairs.append(dict(track=track, suite="gaps", **paired_interval(gap_frames, TARGET, reference, "temporal")))
        print("Compared", track, "file 10 against all 12 alternatives", flush=True)
    pair = pd.DataFrame(pairs)
    win_table = pd.DataFrame(wins)
    win_table.to_csv(out / "file10_window_wins.csv", index=False)
    pair.to_csv(out / "paired_comparisons.csv", index=False)
    pd.concat(windows, ignore_index=True).to_csv(out / "all_fold_scores.csv", index=False)
    figures = out / "figures"; figures.mkdir(exist_ok=True)
    strict = tables["strict"].sort_values("temporal", ascending=False)
    fig, axes = plt.subplots(1, 2, figsize=(11, 7), sharey=True)
    ys = np.arange(len(strict))
    colors = ["#118066" if k == TARGET else "#2f5da8" if k in ["ani_stack", "ani_first"] else "#748397" for k in strict.index]
    for ax, col, title in zip(axes, ["temporal", "cv"], ["Mean of three forward windows", "Repeated grouped CV"]):
        ax.scatter(strict[col], ys, c=colors, s=65)
        ax.axvline(strict.loc["ani_stack", col], color="#2f5da8", linestyle="--", linewidth=1)
        ax.set_xlabel("Equal-phase log loss (lower is better)"); ax.set_title(title)
        ax.grid(axis="x", alpha=.2)
    axes[0].set_yticks(ys, strict.index)
    fig.suptitle("All submissions on identical strict benchmark splits")
    fig.tight_layout(); fig.savefig(figures / "comparison.png", dpi=160)
    fig.savefig(figures / "comparison.pdf"); plt.close(fig)
    cols = ["official", "temporal", "worst", "cv", "gap_mean", "kaggle"]
    displayed = {k: v.loc[NAMES, cols].rename_axis("model").reset_index() for k, v in tables.items()}
    deltas = tables["strict"].loc[TARGET, ["official", "temporal", "cv", "gap_mean"]] - tables["strict"].loc["02_text_heavy", ["official", "temporal", "cv", "gap_mean"]]
    direct_wins = win_table[(win_table.track == "strict") & win_table.reference.isin(["02_text_heavy", "03_phase_specialized", "ani_stack"])]
    relevant_pairs = pair[(pair.track == "strict") & pair.reference.isin(["02_text_heavy", "03_phase_specialized", "ani_first", "ani_stack"])]
    text = f"""# All Part Two submissions on the same fixed benchmark

**Complete:** all ten numbered recipes, both Kaggle-scored Ani references, and
Ani's trend control, on the same ten original splits and three longer-gap splits.
Every table uses identical evaluation trials and equal-phase log loss. Lower is
better. These are frozen recipes; this run did not tune or generate submissions.

{score_update}

## Primary strict comparison

Every model fits preprocessing on its fitting rows only. `official` is the
supplied validation set; `temporal` averages the three original forward windows;
`worst` is the worst of those windows; `cv` pools held-out rows within each of two
grouped-CV repeats; `gap_mean` averages the three longer-gap windows. Known Kaggle
scores belong to the original file hashes, not these fold refits.

{markdown_table(displayed['strict'])}

## File 10 versus the actual non-Ani submissions

File 02 is the best scored non-Ani submission (Kaggle 0.58993); file 03 scored
0.59562. Their full original ensembles, including both CatBoost models, are now
evaluated directly. No linear-component proxy is substituted for either recipe.

For file 10 minus file 02 under strict preprocessing, the log-loss differences
are: supplied validation **{deltas.official:+.6f}**, chronological mean
**{deltas.temporal:+.6f}**, grouped CV **{deltas.cv:+.6f}**, and longer-gap mean
**{deltas.gap_mean:+.6f}**. Negative favors file 10; positive favors file 02.

File 10's number of winning periods, with the same trial IDs in each pair:

{markdown_table(direct_wins.drop(columns='track'))}

An average gain does not imply winning every period. In particular, file 03
beats file 10 in the first two original forward windows, while file 10 wins the
latest window by a larger margin. Retain this temporal variation when judging
whether the gains will transfer to a future cohort.

The old files' single historical temporal scores are not used in this table.
Files 01–06 originally trained their final submissions on 8,037 rows, while Ani
and 07–10 used 9,050. All recipes here receive the same fitting rows in each
split. That resolves the local population mismatch without inventing a Kaggle
score for a hypothetical refit of file 02.

![Chronological and grouped-CV comparison](runs/{out.name}/figures/comparison.png)

## Secondary original-recipe preprocessing comparison

This track preserves Ani/09/10 feature-only transduction and legacy training-only
preprocessing. Files 01–06 are therefore identical to the strict track; 07/08 use
the corresponding Ani predictions. This sensitivity table has matched supervised
fitting and evaluation rows but deliberately retains different original feature
policies. Use the strict table for common-preprocessing comparisons.

{markdown_table(displayed['published'])}

## Paired uncertainty for file 10

Every difference below uses the same held-out trials for both models. Repeated
CV losses are averaged per row before resampling; all phase rows of a trial share
bootstrap weights. These intervals are descriptive and do not correct for earlier
adaptive model selection. The longer-gap labels were also consulted previously.

{markdown_table(relevant_pairs[['reference','suite','kind','delta','lower','upper']])}

The complete [paired comparison table](runs/{out.name}/paired_comparisons.csv)
contains file 10 versus every alternative under both tracks. The
[window-level scores](runs/{out.name}/all_fold_scores.csv) retain every period and
phase, including unfavorable results.
All-model difference matrices are also saved for each metric and track; for
example, [strict chronological differences](runs/{out.name}/pairwise_strict_temporal.csv).
Each matrix cell is row-model loss minus column-model loss; negative favors the row.

## Fidelity, verification and reproduction

Original ensemble weights come directly from the original manifests. The legacy
adapter preserves text features, capped-year behavior, SVD dimensions, fixed tree
counts, recency half-lives and seeds. Vocabulary/IDF, SVD and metadata masks are
rebuilt on each fold's fitting rows. Already-scored Ani/09/10 predictions are
reused with source, input, split and probability checks; they are not refit with
different settings. Original submission CSVs are unchanged.

[Original replay](runs/{out.name}/original_replay.csv) compares all six original
components on supplied validation and all six numbered legacy test CSVs against
the rebuilt official-training model. [Verification](runs/{out.name}/verification.json)
independently rescores all 338 model/split/track results, checks trial boundaries,
fixed blends, source hashes and inference invariance to IDs and outcome labels.

```sh
uv pip install --python .venv/bin/python -r part2/requirements-all-submissions.txt
.venv/bin/python part2/all_submissions_benchmark.py --workers 2
.venv/bin/python part2/all_submissions_verify.py
.venv/bin/python part2/all_submissions_report.py
```

See [comparison policy](../../ALL_SUBMISSIONS_PROTOCOL.md),
[executed notebook](../../all_submissions.ipynb),
[strict machine-readable table](runs/{out.name}/comparison_strict.csv), and
[source/input manifest](runs/{out.name}/manifest.json). Native caches are ignored
by Git and can be rebuilt; per-row held-out predictions and fitting IDs are saved.
This benchmark does not upload submissions. Later user-reported Kaggle results
are joined by exact submission hashes from the current ledger.
"""
    (ROOT / "experiments/all_submissions/RESULTS.md").write_text(text)
    notebook(text, out)
    print(tables["strict"].loc[NAMES, cols].to_string(), flush=True)


def notebook(report_text, out):
    nb = nbformat.v4.new_notebook()
    nb.metadata.kernelspec = dict(display_name="Part 2", language="python", name="part2")
    root_rel = str(out.relative_to(ROOT))
    cells = [nbformat.v4.new_markdown_cell("# All Part Two submissions: direct fixed-benchmark comparison\n\nAll 13 recipes on the same 13 splits. This notebook reads verified results; rebuilding models is explicit."),
             nbformat.v4.new_code_cell("from pathlib import Path\nimport json\nimport pandas as pd\nfrom IPython.display import display, Image\nROOT = Path.cwd() if (Path.cwd() / 'outcome_train.csv').exists() else Path.cwd() / 'part2'\nassert (ROOT / 'outcome_train.csv').exists()\nRUN = ROOT / " + repr(root_rel)),
             nbformat.v4.new_code_cell("display(pd.read_csv(RUN / 'comparison_strict.csv').set_index('model'))"),
             nbformat.v4.new_code_cell("display(pd.read_csv(RUN / 'comparison_published.csv').set_index('model'))"),
             nbformat.v4.new_code_cell("display(Image(filename=str(RUN / 'figures/comparison.png')))"),
             nbformat.v4.new_code_cell("paired = pd.read_csv(RUN / 'paired_comparisons.csv')\ndisplay(paired[(paired.track == 'strict') & paired.reference.isin(['02_text_heavy','03_phase_specialized','ani_first','ani_stack'])])"),
             nbformat.v4.new_code_cell("display(pd.read_csv(RUN / 'original_replay.csv'))\ndisplay(json.loads((RUN / 'verification.json').read_text()))"),
             nbformat.v4.new_markdown_cell(report_text.replace(f"runs/{out.name}/", root_rel + "/").replace("../../", ""))]
    sources = SOURCES + ["all_submissions_verify.py", "all_submissions_report.py", "ani_models.py", "improvement_models.py"]
    embedded = {name: (ROOT / name).read_text() for name in dict.fromkeys(sources)}
    cells.append(nbformat.v4.new_markdown_cell("## Embedded implementation source\n\nSource is included for review. This cell checks equality with the repository and compiles each module without running training."))
    cells.append(nbformat.v4.new_code_cell("embedded_sources = " + repr(embedded) + "\nfor filename, source in embedded_sources.items():\n    assert (ROOT / filename).read_text() == source, filename\n    compile(source, filename, 'exec')\nprint(f'{len(embedded_sources)} source files match and compile')"))
    nb.cells = cells
    nbformat.write(nb, ROOT / "all_submissions.ipynb")


if __name__ == "__main__":
    report()


Overwriting src/all_submissions_report.py


## `src/all_submissions_verify.py`

SHA-256: `70b5d3298838a763e6d5e703429838a8f154d8e172737c685ef5f74c6f40fff3`

In [14]:
%%writefile src/all_submissions_verify.py
"""Independently rescore the complete, common-population comparison."""
import json
import hashlib
import joblib
import numpy as np
import pandas as pd
from all_submissions_benchmark import current, NAMES, EXTENSION, ALIASES
from all_submission_models import recipes
from benchmark_protocol import initialize, resolve, digest, DIRECTORY
from experiments import ROOT, load_data
from improvement_report import audit_track


def verify():
    out = current()
    manifest = json.loads((out / "manifest.json").read_text())
    universe, protocol = initialize()
    gaps = json.loads((EXTENSION / "gaps/protocol.json").read_text())
    assert digest(DIRECTORY / "protocol.json") == manifest["protocol_sha256"]
    assert digest(EXTENSION / "gaps/protocol.json") == manifest["gap_protocol_sha256"]
    assert digest(ROOT / "ALL_SUBMISSIONS_PROTOCOL.md") == manifest["policy_sha256"]
    for category in ["source_sha256", "input_sha256"]:
        for name, expected in manifest[category].items():
            path = ROOT / name
            if name == "submissions/leaderboard_results.json" and digest(path) != expected:
                # Later leaderboard results do not change the frozen benchmark inputs.
                snapshot = out / "input_snapshots" / name
                assert digest(snapshot) == expected, name
                before, after = json.loads(snapshot.read_text()), json.loads(path.read_text())
                for key, value in before.items():
                    if key == "submissions":
                        assert all(after[key][file] == record for file, record in value.items())
                    else:
                        assert after[key] == value
            else:
                assert digest(path) == expected, name
    ledger = json.loads((ROOT / "submissions/leaderboard_results.json").read_text())
    for entry in ledger["ani_winners"].values():
        assert digest(ROOT / "submissions" / entry["submission"]) == entry["submission_sha256"]
    for filename, entry in ledger["submissions"].items():
        assert digest(ROOT / "submissions" / filename) == entry["sha256"]
    count = 0; reused = 0; native = 0
    for is_gap, spec in [(False, protocol), (True, gaps)]:
        parent = out / "gaps" if is_gap else out
        all_frames = {}
        for track in ["strict", "published"]:
            folder = parent / track
            summary, scores, frames = audit_track(folder, universe, spec, names=NAMES)
            saved = pd.read_csv(folder / "summary.csv", index_col="model")
            assert np.allclose(summary.astype(float).sort_index(), saved.loc[summary.index].astype(float).sort_index(), equal_nan=True, atol=1e-12, rtol=0)
            all_frames[track] = frames
            count += len(scores)
            for split in spec["splits"]:
                f = frames[split["name"]]
                p = {name: f[name].probability.to_numpy() for name in NAMES}
                assert np.allclose(p["07_ani_mild_phase3_trend"], .75*p["ani_stack"]+.25*p["ani_phase3_trend"], atol=3e-15, rtol=0)
                # Recover the old linear component independently from two fixed blends.
                linear = 2*p["02_text_heavy"]-p["01_balanced_ensemble"]
                assert np.allclose(p["08_ani_linear25"], .75*p["ani_stack"]+.25*linear, atol=3e-15, rtol=0)
                phase1 = f["02_text_heavy"].phase.eq("I")
                for name in ["03_phase_specialized", "05_text_later_phases", "06_lighter_tree_blend"]:
                    assert np.allclose(p[name][phase1], p["02_text_heavy"][phase1], atol=3e-15, rtol=0)
                source = EXTENSION / "gaps" / track if is_gap else EXTENSION / track
                for name, alias in ALIASES.items():
                    old = pd.read_csv(source / "predictions" / split["name"] / f"{alias}.csv")
                    assert old.ID.tolist() == f[name].ID.tolist()
                    assert np.allclose(old.probability, p[name], atol=1e-15, rtol=0)
                    reused += 1
        for split in spec["splits"]:
            train, ev = resolve(universe, split)
            audit = json.loads((parent / "audits" / f"{split['name']}.json").read_text())
            assert audit["training_ids"] == audit["preprocessing_fit_ids"] == train.ID.tolist()
            assert audit["evaluation_ids"] == ev.ID.tolist()
            assert not set(audit["training_trial_ids"]) & set(ev.trial_id)
            for name, expected in audit["reused_predictions_sha256"].items():
                assert digest(ROOT / name) == expected
            path = ROOT / "experiments/all_submissions/cache" / out.name / f"{split['name']}.joblib"
            if path.exists():
                assert digest(path) == audit["model_sha256"]
                native += 1
            for name in recipes():
                a = all_frames["strict"][split["name"]][name]
                b = all_frames["published"][split["name"]][name]
                assert a.equals(b)
    replay = pd.read_csv(out / "original_replay.csv")
    assert len(replay[replay.kind.eq("validation_component")]) == 6
    assert len(replay[replay.kind.eq("submitted_test_file")]) == 6
    assert replay.max_probability_difference.max() < 1e-6
    invariance = json.loads((out / "inference_invariance.json").read_text())
    assert max(invariance.values()) == 0
    official_cache = ROOT / "experiments/all_submissions/cache" / out.name / "official.joblib"
    byte_replay = {}
    if official_cache.exists():
        # Replay after deserialization, including exact historical CSV rounding.
        model = joblib.load(official_cache)
        test = load_data()["test"]
        predictions = model.predict(test)
        for name in recipes():
            frame = pd.DataFrame(dict(ID=test.ID, label=predictions[name]))
            sha = hashlib.sha256(frame.to_csv(index=False, float_format="%.10f").encode()).hexdigest()
            assert sha == digest(ROOT / "submissions" / f"{name}.csv"), name
            byte_replay[name] = sha
    result = dict(models=len(NAMES), original_splits=len(protocol["splits"]),
        larger_gap_splits=len(gaps["splits"]), preprocessing_tracks=2,
        independently_rescored_model_split_results=count, source_and_input_hashes_match=True,
        same_trial_ids_and_labels_for_every_model=True, legacy_training_boundaries_verified=True,
        unchanged_control_prediction_files=reused, native_cache_hashes_verified=native,
        original_component_and_submission_replay_max_error=float(replay.max_probability_difference.max()),
        label_and_identifier_inference_max_error=max(invariance.values()),
        deserialized_original_csv_byte_replay=byte_replay,
        reporting_source_sha256={name: digest(ROOT / name) for name in
            ["all_submissions_verify.py", "all_submissions_report.py"]},
        current_leaderboard_sha256=digest(ROOT / "submissions/leaderboard_results.json"),
        new_kaggle_submissions=0)
    (out / "verification.json").write_text(json.dumps(result, indent=2) + "\n")
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    verify()


Overwriting src/all_submissions_verify.py


## `src/ani_models.py`

SHA-256: `13639f92744f3d4a1fc38113861bb15cc85d32e26964cf1e5f774a8974956c53`

In [15]:
%%writefile src/ani_models.py
"""Ani's eight-model stack, made reusable for fold-local evaluation.

Adapted from Ani's midterm/part2_best_model.ipynb at commit
1b8bd11c47428fb5453e53cdc275542b92a3b347. The preserved source is in
experiments/ani_comparison/source/. Transductive reproduction passes an
explicit feature space fitted on all available covariates; the benchmark
defaults to fitting its vocabulary and IDF only on the fold's training rows.
"""
from __future__ import annotations
import ast
import numpy as np
import pandas as pd
from scipy import sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import normalize

PHASES=('I','II','III')
BASES=[('criteria LR',['criteria'],2.),('disease LR',['disease'],1.),
       ('ICD LR',['icd'],1.),('drug-name LR',['drug'],1.),
       ('drug-knowledge LR',['targets','pathways','actions'],.3),
       ('metadata LR',['meta'],1.),('criteria MNB',['crit_counts'],None),
       ('criteria kNN',['criteria'],None)]


def parse_list(s):
    try:
        out=[]
        for x in ast.literal_eval(s):
            out+=ast.literal_eval(x) if isinstance(x,str) and x.startswith('[') else [x]
        return [str(x) for x in out]
    except (ValueError,SyntaxError):return []


def text_fields(df):
    icd=df.icdcodes.fillna('[]').map(parse_list)
    text={'criteria':df.criteria.fillna(''),
          'disease':df.diseases_cleaned.fillna('[]').map(lambda s:' '.join(parse_list(s))),
          'drug':df.drugs.fillna('[]').map(lambda s:' '.join(parse_list(s))),
          'icd':icd.map(lambda v:' '.join(sorted({c[:1] for c in v}|{c[:3] for c in v}|{c[:5] for c in v})))}
    for key,col in [('targets','drug_targets'),('pathways','drug_pathways'),('actions','drug_actions')]:
        text[key]=df[col].fillna('').map(lambda s:' '.join(sorted({t.strip() for t in s.split(';') if t.strip()})))
    return text


def onehot(df):return np.column_stack([df.phase.eq(p).to_numpy().astype(float) for p in PHASES])


class AniFeatures:
    def fit(self,df):
        text=text_fields(df);split=dict(tokenizer=str.split,lowercase=False,token_pattern=None)
        self.vectorizers={
            'criteria':TfidfVectorizer(min_df=5,ngram_range=(1,2),sublinear_tf=True),
            'crit_counts':CountVectorizer(min_df=5,ngram_range=(1,2)),
            'disease':TfidfVectorizer(min_df=2,ngram_range=(1,2),sublinear_tf=True),
            'drug':TfidfVectorizer(min_df=2,sublinear_tf=True),
            'icd':TfidfVectorizer(min_df=2,**split),'targets':TfidfVectorizer(min_df=3,**split),
            'pathways':TfidfVectorizer(min_df=5,**split),'actions':CountVectorizer(min_df=2,binary=True,**split)}
        for key,vec in self.vectorizers.items():vec.fit(text['criteria' if key=='crit_counts' else key])
        self.year_median=float(df.start_year.median())
        self.fit_ids=df.ID.tolist()
        return self

    def transform(self,df):
        text=text_fields(df)
        f={key:vec.transform(text['criteria' if key=='crit_counts' else key]) for key,vec in self.vectorizers.items()}
        f['meta']=sp.csr_matrix(np.column_stack([
            df.matched_drugs.notna(),np.log1p(text['drug'].str.split().str.len()),
            np.log1p(text['criteria'].str.len())/10,df.phase_detail.str.contains('/'),
            df.phase_detail.eq('early phase 1')]).astype(float))
        f['phase']=sp.csr_matrix(onehot(df));f['knn']=normalize(f['criteria'])
        return f


def matrix(f,blocks):return sp.hstack([f[k] for k in blocks]+[f['phase']],format='csr')


def fit_base(name,blocks,C,f,df,weights):
    if name=='criteria kNN':
        # Faithfully preserve Ani's implementation: recency does not weight kNN.
        return {p:(f['knn'][df.phase.eq(p).to_numpy()],df.label.to_numpy()[df.phase.eq(p).to_numpy()]) for p in PHASES}
    x=matrix(f,blocks)
    model=MultinomialNB(alpha=1.) if name=='criteria MNB' else LogisticRegression(C=C,max_iter=3000)
    return model.fit(x,df.label.to_numpy(),sample_weight=weights)


def predict_base(name,blocks,model,f,df,k=50):
    if name!='criteria kNN':return model.predict_proba(matrix(f,blocks))[:,1]
    result=np.zeros(len(df))
    for phase,(x,y) in model.items():
        mask=df.phase.eq(phase).to_numpy()
        if not mask.any():continue
        similarities=(f['knn'][mask]@x.T).toarray()
        neighbors=min(k,len(y)-1)
        idx=np.argpartition(-similarities,neighbors,axis=1)[:,:neighbors]
        weights=np.take_along_axis(similarities,idx,1)
        result[mask]=((weights*y[idx]).sum(axis=1)+1)/(weights.sum(axis=1)+2)
    return result


def subset(f,idx):return {k:v[idx] for k,v in f.items()}


class AniStack:
    def __init__(self,tau=3.,meta_C=1.,phase3_year=False,knn_k=50,inner_refit=False):
        self.tau=tau;self.meta_C=meta_C;self.phase3_year=phase3_year;self.knn_k=knn_k;self.inner_refit=inner_refit

    def meta_matrix(self,z,df):
        p=np.clip(z,1e-4,1-1e-4);ph=onehot(df)
        parts=[np.log(p/(1-p)),ph]
        if self.phase3_year:
            year=df.start_year.fillna(self.features.year_median).to_numpy()
            parts.append((ph[:,2]*(np.minimum(year,self.year_cap)-2007)/3)[:,None])
        return np.hstack(parts)

    def fit(self,df,features=None):
        df=df.reset_index(drop=True)
        self.features=features if features is not None else AniFeatures().fit(df)
        f=self.features.transform(df);year=df.start_year.fillna(self.features.year_median).to_numpy()
        self.year_cap=float(year.max()+2)
        w=np.ones(len(df)) if self.tau is None else np.exp((year-year.max())/self.tau)
        w=w/w.mean();z=np.zeros((len(df),len(BASES)));self.models=[];self.fold_audit=[]
        folds=list(GroupKFold(5).split(np.arange(len(df)),groups=df.trial_id.to_numpy()))
        if self.inner_refit:
            for fold,(a,b) in enumerate(folds):
                assert not set(df.trial_id.iloc[a])&set(df.trial_id.iloc[b])
                inner=AniFeatures().fit(df.iloc[a])
                fa,fb=inner.transform(df.iloc[a]),inner.transform(df.iloc[b])
                for j,(name,blocks,C) in enumerate(BASES):
                    m=fit_base(name,blocks,C,fa,df.iloc[a],w[a])
                    z[b,j]=predict_base(name,blocks,m,fb,df.iloc[b],self.knn_k)
                self.fold_audit.append({'fold':fold,'fit_rows':len(a),'heldout_rows':len(b),'trial_overlap':0,'preprocessing':'inner training rows only'})
            self.models=[fit_base(name,blocks,C,f,df,w) for name,blocks,C in BASES]
        else:
            for j,(name,blocks,C) in enumerate(BASES):
                for fold,(a,b) in enumerate(folds):
                    assert not set(df.trial_id.iloc[a])&set(df.trial_id.iloc[b])
                    m=fit_base(name,blocks,C,subset(f,a),df.iloc[a],w[a])
                    z[b,j]=predict_base(name,blocks,m,subset(f,b),df.iloc[b],self.knn_k)
                    if j==0:self.fold_audit.append({'fold':fold,'fit_rows':len(a),'heldout_rows':len(b),'trial_overlap':0,'preprocessing':'shared fitting-population vocabulary'})
                self.models.append(fit_base(name,blocks,C,f,df,w))
        self.meta=LogisticRegression(C=self.meta_C,max_iter=3000).fit(self.meta_matrix(z,df),df.label.to_numpy(),sample_weight=w)
        self.oof_base_predictions=z;self.training_ids=df.ID.tolist()
        return self

    def predict_base_matrix(self,df):
        f=self.features.transform(df)
        return np.column_stack([predict_base(name,blocks,m,f,df,self.knn_k) for (name,blocks,_),m in zip(BASES,self.models)])

    def predict_proba(self,df):return self.meta.predict_proba(self.meta_matrix(self.predict_base_matrix(df),df))[:,1]


Overwriting src/ani_models.py


## `src/benchmark_ani_recipe.py`

SHA-256: `3c1cc9719818cd841d58c1eb8a03274fa460608ee41558e567c75e21fe6a717c`

In [16]:
%%writefile src/benchmark_ani_recipe.py
"""Sensitivity track: faithful Ani preprocessing on the SAME frozen splits.

Unlike the main strict benchmark, this deliberately preserves Ani's feature-only
transduction: train/validation/test text shapes vocabulary and IDF. No evaluation
outcome labels fit a model. Keep this track explicitly separate from strict scores.
"""
import os
os.environ.setdefault('OMP_NUM_THREADS','4')
os.environ.setdefault('OPENBLAS_NUM_THREADS','4')
import copy,json,time
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from ani_models import AniStack
from benchmark_models import OurLinear
from benchmark_protocol import initialize,resolve,digest
from local_benchmark import run_directory,predict_and_score,summarize
from experiments import ROOT


def run():
    universe,protocol=initialize();strict=run_directory(protocol)
    out=strict/'published_recipe_track';out.mkdir(exist_ok=True)
    space=joblib.load(ROOT/'experiments/ani_comparison/transductive_features.joblib')
    (out/'run_manifest.json').write_text(json.dumps({'protocol_sha256':digest(ROOT/'experiments/local_benchmark/protocol.json'),
        'source_commit':'1b8bd11c47428fb5453e53cdc275542b92a3b347',
        'preprocessing':'Faithful Ani: vocabulary and IDF include all available train/validation/test covariates; internal OOF uses that shared vocabulary.',
        'labels':'Evaluation labels excluded from every fit. This is a transductive sensitivity track, not strict prospective validation.'},indent=2))
    for split in protocol['splits']:
        if (out/f"scores_{split['name']}.csv").exists():continue
        start=time.time();train,ev=resolve(universe,split)
        folder=out/'models'/split['name'];folder.mkdir(parents=True,exist_ok=True)
        path=folder/'ani_stack.joblib'
        if path.exists():model=joblib.load(path)
        else:
            model=AniStack(tau=3).fit(train,features=space);joblib.dump(model,path,compress=3)
        z=model.predict_base_matrix(ev)
        p=model.meta.predict_proba(model.meta_matrix(z,ev))[:,1]
        linear_path=strict/'models'/split['name']/'our_linear.joblib'
        if linear_path.exists():linear=joblib.load(linear_path)
        else:linear=OurLinear().fit(train)
        lp=linear.predict_proba(ev)
        trend=copy.copy(model);trend.phase3_year=True
        year=train.start_year.fillna(space.year_median).to_numpy();w=np.exp((year-year.max())/3);w=w/w.mean()
        trend.meta=LogisticRegression(C=1,max_iter=3000).fit(trend.meta_matrix(trend.oof_base_predictions,train),train.label,sample_weight=w)
        tp=trend.meta.predict_proba(trend.meta_matrix(z,ev))[:,1]
        predict_and_score(out,split,ev,{'ani_stack':p,'our_linear':lp,'ani_phase3_trend':tp,
            'blend_ani75_linear25':.75*p+.25*lp,'blend_ani50_linear50':.5*p+.5*lp})
        print(split['name'],'published-recipe sensitivity completed',round(time.time()-start,1),'seconds',flush=True)
        summarize(out,protocol,bootstrap=False)
    summarize(out,protocol)
    print('Published-recipe sensitivity results:',out,flush=True)

if __name__=='__main__':run()


Overwriting src/benchmark_ani_recipe.py


## `src/benchmark_candidates.py`

SHA-256: `6c5042a02e41e125c7370a7785c49885261d90fceb76bcb8a403dc8a1cdf1115`

In [17]:
%%writefile src/benchmark_candidates.py
"""Two conservative, pre-specified blends scored on EVERY frozen split.

No new base-model fitting: use saved out-of-fold predictions from each track.
The 25% linear blend was already in the original batch. The mild year variant
is a conservative interpolation toward Ani's existing Phase III alternative.
"""
import json
import pandas as pd
from benchmark_protocol import initialize,digest
from local_benchmark import run_directory,predict_and_score,summarize
from experiments import ROOT

WEIGHTS={'ani_mild_phase3_trend':{'ani_stack':.75,'ani_phase3_trend':.25},
         'ani_anchored_linear25':{'ani_stack':.75,'our_linear':.25}}


def run():
    _,protocol=initialize();strict=run_directory(protocol)
    for parent in [strict,strict/'published_recipe_track']:
        for split in protocol['splits']:
            assert all((parent/'predictions'/split['name']/f'{model}.csv').exists() for model in ['ani_stack','ani_phase3_trend','our_linear']),f'Complete parent track first: {parent}/{split["name"]}'
        out=parent/'derived_candidates';out.mkdir(exist_ok=True)
        (out/'run_manifest.json').write_text(json.dumps({'parent':str(parent.relative_to(ROOT)),
            'parent_manifest_sha256':digest(parent/'run_manifest.json'),'protocol_sha256':digest(ROOT/'experiments/local_benchmark/protocol.json'),
            'weights':WEIGHTS,'selection_note':'Exploratory, fixed conservative blends after inspecting the initial batch; every saved split is evaluated, not just favorable periods.'},indent=2))
        for split in protocol['splits']:
            folder=parent/'predictions'/split['name']
            banks={name:pd.read_csv(folder/f'{name}.csv') for name in ['ani_stack','ani_phase3_trend','our_linear']}
            df=banks['ani_stack']
            assert all(frame.ID.equals(df.ID) for frame in banks.values())
            ps={'ani_stack':df.probability.to_numpy()}
            ps.update({name:sum(w*banks[key].probability.to_numpy() for key,w in weights.items()) for name,weights in WEIGHTS.items()})
            predict_and_score(out,split,df,ps)
        summarize(out,protocol)
        print('Derived candidates:',out,flush=True)

if __name__=='__main__':run()


Overwriting src/benchmark_candidates.py


## `src/benchmark_final.py`

SHA-256: `37a72d05acb222fc42ed080a1311e3059632f1f34b9b185b6abc6002cc162eb1`

In [18]:
%%writefile src/benchmark_final.py
"""Prepare two optional, Ani-anchored CSVs after the complete benchmark.

The winning published prediction file remains the anchor. Refit our linear
component on train+validation consistently with Ani's final fitting population.
These are prospective candidates, not established leaderboard improvements.
"""
import hashlib,json
import joblib
import numpy as np
import pandas as pd
from benchmark_models import OurLinear
from benchmark_protocol import DIRECTORY,initialize,digest
from experiments import ROOT,load_data


def run():
    initialize();out=ROOT/json.loads((DIRECTORY/'current_run.json').read_text())['directory']
    for track in [out,out/'published_recipe_track']:
        for suffix in ['', 'derived_candidates/']:
            s=pd.read_csv(track/suffix/'summary.csv')
            assert s.complete_comparison.all(),'Finish all comparisons before generating candidates.'
    data=load_data();train=pd.concat([data['train'],data['val']],ignore_index=True)
    assert not set(train.trial_id)&set(data['test'].trial_id)
    source=ROOT/'experiments/ani_comparison/source'
    provenance=json.loads((source.parent/'provenance.json').read_text())
    for file in ['part2_best_submission.csv','part2_alt_phase3_trend_submission.csv']:
        assert digest(source/file)==provenance['files']['midterm/'+file]
    sample=pd.read_csv(ROOT/'part2_sample_submission.csv')
    def ordered(file):
        d=pd.read_csv(source/file)
        assert d.ID.is_unique and set(d.ID)==set(sample.ID)
        return d.set_index('ID').loc[sample.ID,'label'].to_numpy()
    ani=ordered('part2_best_submission.csv');trend=ordered('part2_alt_phase3_trend_submission.csv')
    models=out/'final_models';models.mkdir(exist_ok=True)
    path=models/'our_linear_trainval.joblib'
    if path.exists():linear=joblib.load(path)
    else:
        linear=OurLinear().fit(train);joblib.dump(linear,path,compress=3)
    assert linear.training_ids==train.ID.tolist()
    test=data['test'].set_index('ID',drop=False).loc[sample.ID].reset_index(drop=True)
    lp=linear.predict_proba(test)
    candidates={'07_ani_mild_phase3_trend':.75*ani+.25*trend,'08_ani_linear25':.75*ani+.25*lp}
    recipes={'07_ani_mild_phase3_trend':{'Ani 0.58688 published probabilities':.75,'Ani capped Phase III trend alternative':.25},
             '08_ani_linear25':{'Ani 0.58688 published probabilities':.75,'Our linear component refitted on train+validation':.25}}
    history={p.name:digest(p) for p in (ROOT/'submissions').glob('0[1-6]_*.csv')}
    rows=[]
    for name,p in candidates.items():
        assert np.isfinite(p).all() and ((p>0)&(p<1)).all()
        f=ROOT/'submissions'/f'{name}.csv'
        pd.DataFrame({'ID':sample.ID,'label':p}).to_csv(f,index=False,float_format='%.10f')
        disk=pd.read_csv(f)
        assert len(disk)==1500 and disk.ID.equals(sample.ID) and disk.ID.is_unique
        assert np.max(abs(disk.label-p))<5.1e-11
        rows.append({'submission':f.name,'mean_test_probability':float(p.mean()),'mean_abs_change_from_ani':float(np.abs(p-ani).mean()),
            'sha256':digest(f),'kaggle_score':None})
    for name,sha in history.items():assert digest(ROOT/'submissions'/name)==sha
    pd.DataFrame(rows).to_csv(ROOT/'submissions/ani_candidates_index.csv',index=False)
    (ROOT/'submissions/ani_candidates_manifest.json').write_text(json.dumps({
        'baseline_score':.58688,'baseline_source_commit':provenance['commit'],
        'reference_file_sha256':digest(source/'part2_best_submission.csv'),
        'trend_file_sha256':digest(source/'part2_alt_phase3_trend_submission.csv'),
        'benchmark_run':str(out.relative_to(ROOT)),'linear_final_training_rows':len(train),
        'recipe':recipes,'files':rows,'status':'Unsubmitted, optional experiments. Small local differences and overlapping paired intervals do not establish a Kaggle improvement.',
        'selection':'Compare both strict and faithful-published-recipe tracks. Prefer consistency across chronological windows and grouped CV over supplied-validation F1.'},indent=2))
    print(pd.DataFrame(rows).to_string(index=False))

if __name__=='__main__':run()


Overwriting src/benchmark_final.py


## `src/benchmark_models.py`

SHA-256: `719918c15adb87b3797a1b57a7ce15c0e61e49535988994198acb26df76c741a`

In [19]:
%%writefile src/benchmark_models.py
"""Explicit model adapters for the frozen local benchmark."""
import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from experiments import Features,matrix


class OurLinear:
    """Exactly the linear component shared by submissions 02 and 05."""
    def fit(self,df):
        self.features=Features().fit(df)
        self.wide=TfidfVectorizer(dtype=np.float64,strip_accents='unicode',ngram_range=(1,2),
            min_df=2,max_features=80000,sublinear_tf=True).fit(self.features.documents(df,'criteria'))
        blocks=self.features.transform(df)
        xs=self.inputs(df,blocks)
        self.models=[]
        for x,C,half in zip(xs,[2.,2.,.5],[None,6.,3.]):
            w=None
            if half:
                w=2**((df.start_year.fillna(df.start_year.median()).to_numpy()-df.start_year.max())/half)
                w=w/w.mean()
            model=LogisticRegression(C=C,solver='liblinear',max_iter=1000,tol=1e-5,random_state=42)
            self.models.append(model.fit(x,df.label,sample_weight=w))
        self.training_ids=df.ID.tolist()
        return self

    def inputs(self,df,blocks):
        wide=sparse.hstack([self.wide.transform(self.features.documents(df,'criteria')),blocks['phase']],format='csr')
        return [wide,matrix(blocks,'criteria'),matrix(blocks,'clinical')]

    def predict_proba(self,df):
        xs=self.inputs(df,self.features.transform(df))
        return sum(w*m.predict_proba(x)[:,1] for w,m,x in zip([.375,.375,.25],self.models,xs))


Overwriting src/benchmark_models.py


## `src/benchmark_protocol.py`

SHA-256: `e5eb9190d2681607fc2d5cd305f68c6c63158ab1dd248b49d9ecee8772102ebe`

In [20]:
%%writefile src/benchmark_protocol.py
"""Persistent trial-level split definitions; test outcomes are never accessed."""
from pathlib import Path
import hashlib,json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from experiments import ROOT,load_data

VERSION='part2-local-v1'
DIRECTORY=ROOT/'experiments/local_benchmark'


def digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def initialize():
    DIRECTORY.mkdir(parents=True,exist_ok=True)
    data=load_data();universe=pd.concat([data['train'],data['val']],ignore_index=True)
    fingerprints={s:digest(ROOT/f'outcome_{s}.csv') for s in ['train','val','test']}
    path=DIRECTORY/'protocol.json'
    if path.exists():
        protocol=json.loads(path.read_text())
        assert protocol['version']==VERSION and protocol['data_sha256']==fingerprints,'Data or protocol changed. Create a new protocol version rather than mixing results.'
        return universe,protocol
    splits=[]
    def add(name,kind,tr,ev,repeat=None):
        a,b=universe.iloc[tr],universe.iloc[ev]
        assert not set(a.trial_id)&set(b.trial_id)
        assert not set(a.ID)&set(b.ID)
        assert set(a.phase)==set(b.phase)=={'I','II','III'}
        assert b.groupby('phase').label.nunique().eq(2).all()
        splits.append(dict(name=name,kind=kind,repeat=repeat,fit_ids=a.ID.tolist(),eval_ids=b.ID.tolist(),
            fit_rows=len(a),eval_rows=len(b),eval_phase_counts=b.phase.value_counts().to_dict(),
            fit_year_max=float(a.start_year.max()),eval_year_min=float(b.start_year.min()),eval_year_max=float(b.start_year.max())))
    add('official','official',np.arange(len(data['train'])),np.arange(len(data['train']),len(universe)))
    years=universe.start_year.fillna(2007).to_numpy()
    for cut,end in [(2007,2009),(2009,2011),(2011,2013)]:
        add(f'time_{cut}_to_{cut+1}_{end}','temporal',np.flatnonzero(years<=cut),np.flatnonzero((years>cut)&(years<=end)))
    strata=universe.phase+'_'+universe.label.astype(str)
    for repeat,seed in enumerate([42,43]):
        for fold,(a,b) in enumerate(StratifiedGroupKFold(3,shuffle=True,random_state=seed).split(np.zeros(len(universe)),strata,groups=universe.trial_id)):
            add(f'group_repeat{repeat}_fold{fold}','group_cv',a,b,repeat)
    protocol=dict(version=VERSION,data_sha256=fingerprints,primary_metric='Equal mean of phase I/II/III log losses',
        fitting_universe='outcome_train + outcome_val, with evaluation trials excluded separately in every split',
        preprocessing='Outer and stacking-inner training rows only; no held-out or Kaggle-test text in vocabularies/IDF',
        temporal_selection='Prefer consistent chronological improvement; inspect the worst period and group-CV agreement. No scalar local score guarantees Kaggle performance.',
        note='Official validation and some historical data were consulted in earlier searches. New scores remain model-selection data, not independent evidence of test performance.',
        splits=splits)
    path.write_text(json.dumps(protocol,indent=2))
    return universe,protocol


def resolve(universe,split):
    indexed=universe.set_index('ID',drop=False)
    tr=indexed.loc[split['fit_ids']].reset_index(drop=True)
    ev=indexed.loc[split['eval_ids']].reset_index(drop=True)
    assert not set(tr.trial_id)&set(ev.trial_id)
    if split['kind']=='temporal':assert tr.start_year.max()<ev.start_year.min()
    return tr,ev


if __name__=='__main__':
    universe,protocol=initialize()
    print(pd.DataFrame([{k:v for k,v in s.items() if k not in ['fit_ids','eval_ids']} for s in protocol['splits']]).to_string(index=False))


Overwriting src/benchmark_protocol.py


## `src/benchmark_report.py`

SHA-256: `8b38e872c2fc5f1a59dcefbc01fe5fa2d9613fb9bb95137955f269ef6e408196`

In [21]:
%%writefile src/benchmark_report.py
"""Report and standalone plots for complete benchmark runs."""
import os
from pathlib import Path
os.environ.setdefault('MPLCONFIGDIR',str(Path(__file__).resolve().parent/'experiments/.matplotlib'))
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator,FormatStrFormatter
from benchmark_protocol import DIRECTORY,initialize
from experiments import ROOT

LABELS={'ani_stack':'Ani stack','ani_meta_C0.3':'Stronger meta L2','ani_phase3_trend':'Full Phase III trend',
        'our_linear':'Our linear component','blend_ani75_linear25':'75% Ani + 25% linear',
        'blend_ani50_linear50':'50% Ani + 50% linear','ani_mild_phase3_trend':'75% Ani + 25% trend',
        'ani_anchored_linear25':'75% Ani + 25% linear'}


def markdown(df):
    lines=['| '+' | '.join(df.columns)+' |','| '+' | '.join(['---']*len(df.columns))+' |']
    for row in df.itertuples(index=False,name=None):
        lines.append('| '+' | '.join('' if pd.isna(x) else f'{x:.6f}' if isinstance(x,float) else str(x) for x in row)+' |')
    return '\n'.join(lines)


def run():
    _,protocol=initialize();pointer=json.loads((DIRECTORY/'current_run.json').read_text());out=ROOT/pointer['directory']
    strict=pd.read_csv(out/'summary.csv');published=pd.read_csv(out/'published_recipe_track/summary.csv')
    assert strict.complete_comparison.all() and published.complete_comparison.all(),'Do not report incomplete comparisons as final.'
    figures=out/'figures';figures.mkdir(exist_ok=True)
    plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':150})
    names=[x for x in LABELS if x in set(strict.model) and x!='ani_stack']
    fig,axes=plt.subplots(1,2,figsize=(12,4.6),sharey=True)
    for ax,kind,title in zip(axes,['temporal','group_cv'],['Three chronological windows','Two repeats of grouped 3-fold CV']):
        rows=strict.set_index('model').loc[names]
        point=rows[kind+'_delta'].to_numpy();lo=rows[kind+'_delta_lower'].to_numpy();hi=rows[kind+'_delta_upper'].to_numpy()
        ax.errorbar(point,np.arange(len(names)),xerr=np.vstack([np.maximum(0,point-lo),np.maximum(0,hi-point)]),fmt='o',color='#087f8c',capsize=3)
        ax.axvline(0,color='.4',ls=':');ax.set_title(title);ax.set_xlabel('Log-loss difference from Ani (negative is better)')
        ax.set_yticks(np.arange(len(names)),[LABELS[n] for n in names])
    axes[0].invert_yaxis()
    fig.suptitle('Paired trial-bootstrap intervals; descriptive 95% ranges',fontsize=12)
    fig.tight_layout();fig.savefig(figures/'paired_comparisons.png');plt.close(fig)
    fold=pd.read_csv(out/'fold_scores.csv');temporal=fold[fold.kind.eq('temporal')]
    fig,axes=plt.subplots(1,3,figsize=(13,4.5),sharey=True)
    limits=[]
    for ax,(name,g) in zip(axes,temporal.groupby('split',sort=True)):
        baseline=g[g.model.eq('ani_stack')].phase_mean_logloss.iloc[0]
        rows=g.set_index('model').loc[names]
        delta=rows.phase_mean_logloss.to_numpy()-baseline;limits.extend(delta)
        ax.scatter(delta,np.arange(len(names)),color=['#c36341' if d>0 else '#087f8c' for d in delta])
        ax.axvline(0,color='.4',ls=':');ax.set_title(name.replace('time_','').replace('_to_',' → ').replace('_','–'))
        ax.set_yticks(np.arange(len(names)),[LABELS[n] for n in names]);ax.set_xlabel('Log-loss difference from Ani')
    axes[0].invert_yaxis()
    for ax in axes:
        ax.set_xlim(min(limits)-.002,max(limits)+.002)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
        ax.tick_params(axis='x',labelsize=9)
    fig.tight_layout();fig.savefig(figures/'chronological_windows.png');plt.close(fig)
    columns=['model','official_logloss','temporal_mean_logloss','worst_temporal_logloss','cv_mean_logloss','temporal_mean_delta_vs_ani','cv_mean_delta_vs_ani']
    derived=[]
    for track in [out,out/'published_recipe_track']:
        path=track/'derived_candidates/summary.csv'
        if path.exists():
            d=pd.read_csv(path);d.insert(0,'track','Strict' if track==out else 'Published recipe');derived.append(d)
    exact=json.loads((ROOT/'experiments/ani_comparison/exact_reproduction.json').read_text())
    report=f'''# Part Two benchmark results

**Hold 05.** Its underlying linear component should be judged against Ani on the saved chronological and grouped splits, not inferred to win by extrapolating two leaderboard scores. The best confirmed reference remains Ani's **0.58688**; our reported scores are **02: 0.58993**, **03: 0.59562**.

Ani commit: `1b8bd11c47428fb5453e53cdc275542b92a3b347`. Exact final-submission reproduction has maximum probability difference **{exact[1]['max_submission_probability_difference']:.3g}** from the published CSV. The source notebook, CSVs and hashes are preserved in `experiments/ani_comparison`.

## Strict comparison on the same saved splits

{markdown(strict[columns])}

All ten splits are complete for every model. Features and stacking-inner preprocessing use fitting rows only. This adapts Ani's algorithm to a strict prospective evaluation. The three chronological windows are 2008–2009, 2010–2011 and 2012–2013, always fitting earlier years. Group CV uses two seeded repeats of three trial-grouped folds over the same labeled universe. Combined-phase rows stay together. Log loss is averaged equally across phases; CV is pooled across held-out rows within each repeat.

The 50/50 Ani/linear blend improves supplied validation and grouped CV, yet is worse on chronological mean loss. The linear endpoint is worse across time. This warns against using either validation F1 or random CV alone to choose a submission.

![Paired comparisons](runs/{pointer['run']}/figures/paired_comparisons.png)

Negative differences favor the candidate. Intervals spanning zero indicate inconclusive local evidence. The ranges are conditional, descriptive trial bootstraps, not guarantees for a future cohort or corrections for model selection.

![Chronological windows](runs/{pointer['run']}/figures/chronological_windows.png)

## Faithful published-recipe sensitivity track

{markdown(published[columns])}

This track preserves Ani's feature-only transduction: vocabulary/IDF sees all available train/validation/test text. Evaluation outcome labels never fit a model. These results are kept separate from strict scores. Both tracks reuse the exact same trial splits. Original Ani submission refits supervised models on 9,050 train+validation rows, whereas our initial submitted models fitted only 8,037 training rows. That difference also prevents attributing the leaderboard gap to the algorithm alone.

## Conservative follow-up comparisons

'''
    if derived:
        d=pd.concat(derived,ignore_index=True)
        report+=markdown(d[['track']+columns])+'\n\n'
    report+='''These are exploratory fixed blends, evaluated on every saved split. The mild trend interpolates 25% toward Ani's existing Phase III trend alternative; the linear blend puts 25% weight on our linear component. If using another exploratory submission, prefer optional **07_ani_mild_phase3_trend.csv** over 05: it remains anchored to Ani's known winning file and has favorable point differences in both tracks. Its temporal interval still includes zero. The linear blend improves grouped CV but is less consistent across time. Neither candidate has a known Kaggle score; preserve Ani's 0.58688 as the confirmed reference.

## Repeatability and future candidates

See [LOCAL_TESTING.md](../../LOCAL_TESTING.md) for protocol, commands and limitations. `protocol.json` contains immutable IDs, grouping, seeds, periods and data hashes. Each run records implementation hashes and package versions. Models and exact held-out predictions are cached. `fold_scores.csv`, `summary.csv`, per-fold `audit.json`, and prediction files make independent rescoring possible. Incomplete runs cannot be presented as completed comparisons.

Use this suite for the next candidate batch. Do not choose one favorable validation F1 or change boundaries to make a model win. Freeze the recipe before generating a final CSV; refit final components consistently on train plus validation, while keeping held-out benchmark metrics separate. Kaggle results remain the only direct evidence of performance on its newer cohort.
'''
    (DIRECTORY/'RESULTS.md').write_text(report)
    print('Wrote benchmark report and figures:',DIRECTORY/'RESULTS.md')

if __name__=='__main__':run()


Overwriting src/benchmark_report.py


## `src/boosted.py`

SHA-256: `c2a3da5c682cb7f74fbb78649cd1f7a598b21848d667175aa24d3af039d1029f`

In [22]:
%%writefile src/boosted.py
"""Boosted-tree and forest comparisons using training-only text projections."""
import os
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '4')
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from catboost import CatBoostClassifier
from experiments import ROOT, OUT, load_data, prepare, metrics


def transform_dense(df, blocks, svds, mask, year_median):
    parts=[svds[k].transform(blocks[k]) for k in ['criteria','disease','drug','char']]
    parts.append(blocks['knowledge'][:,mask].toarray())
    text=df.criteria.fillna('')
    parts.append(np.column_stack([np.log1p(text.str.len()),np.log1p(text.str.count('\n')),
        df.drug_smiles.notna().astype(float),df.drug_targets.fillna('').str.count(';'),
        df.drugs.fillna('').str.count(','),df.diseases_cleaned.fillna('').str.count(',')]))
    f=pd.DataFrame(np.hstack(parts).astype(np.float32))
    f.columns=[f'x{i}' for i in range(f.shape[1])]
    for col in ['phase','phase_detail','diseases_cleaned','drugs','matched_drugs']:
        f[col]=df[col].fillna('missing').astype(str).to_numpy()
    f['year']=df.start_year.fillna(year_median).to_numpy()
    return f


def dense_features(tr, ev, test, bt, be, bx):
    cache = OUT / ('dense_temporal.joblib' if len(tr) < 7000 else 'dense_validation.joblib')
    if cache.exists():
        return joblib.load(cache)
    parts = [[], [], []]
    svds = {}
    for key, n in [('criteria', 100), ('disease', 24), ('drug', 24), ('char', 32)]:
        svd = TruncatedSVD(n_components=n, random_state=42, n_iter=5).fit(bt[key])
        svds[key] = svd
        for out, b in zip(parts, [bt, be, bx]):
            out.append(svd.transform(b[key]))
    mask = np.asarray((bt['knowledge'] > 0).sum(axis=0)).ravel() >= 15
    for out, b, d in zip(parts, [bt, be, bx], [tr, ev, test]):
        out.append(b['knowledge'][:, mask].toarray())
        text = d.criteria.fillna('')
        out.append(np.column_stack([np.log1p(text.str.len()), np.log1p(text.str.count('\n')),
            d.drug_smiles.notna().astype(float), d.drug_targets.fillna('').str.count(';'),
            d.drugs.fillna('').str.count(','), d.diseases_cleaned.fillna('').str.count(',')]))
    frames=[]
    for out,d in zip(parts,[tr, ev, test]):
        f=pd.DataFrame(np.hstack(out).astype(np.float32))
        f.columns=[f'x{i}' for i in range(f.shape[1])]
        for col in ['phase', 'phase_detail', 'diseases_cleaned', 'drugs', 'matched_drugs']:
            f[col]=d[col].fillna('missing').astype(str).to_numpy()
        f['year']=d.start_year.fillna(tr.start_year.median()).to_numpy()
        frames.append(f)
    packet=(frames, svds, mask)
    joblib.dump(packet, cache, compress=3)
    return packet


def run():
    data=load_data()
    for split in ['temporal','validation']:
        tr, ev, features, bt, be, bx=prepare(data, split)
        frames,svds,mask=dense_features(tr,ev,data['test'],bt,be,bx)
        scores=[]; preds={}; tests={}; configs={}
        for year in ['none','year']:
            xs=[f.drop(columns=['year']) if year=='none' else f for f in frames]
            cats=xs[0].select_dtypes('str').columns.tolist()
            for depth in [4,6,8]:
                model=CatBoostClassifier(iterations=1000, depth=depth, learning_rate=.035,
                    l2_leaf_reg=6, loss_function='Logloss', random_seed=42, thread_count=4,
                    verbose=False, allow_writing_files=False)
                model.fit(xs[0],tr.label,cat_features=cats)
                for n in [250,600,1000]:
                    name=f'catboost_d{depth}_n{n}_{year}'
                    p=model.predict_proba(xs[1],ntree_end=n)[:,1]
                    pt=model.predict_proba(xs[2],ntree_end=n)[:,1]
                    preds[name]=p; tests[name]=pt
                    configs[name]=dict(family='catboost',depth=depth,iterations=n,year=year)
                    scores.append(dict(name=name,**metrics(ev,p)))
                print(split,year,'catboost',depth, pd.DataFrame(scores).phase_mean_logloss.min(), flush=True)
                pd.DataFrame(scores).sort_values('phase_mean_logloss').to_csv(OUT/f'boosted_{split}_scores.csv',index=False)
                joblib.dump(model, OUT/f'catboost_{split}_d{depth}_{year}.joblib',compress=3)
            nums=[f.drop(columns=cats) for f in xs]
            for leaves,l2 in [(7,10),(15,20),(31,30)]:
                model=HistGradientBoostingClassifier(max_iter=350,max_leaf_nodes=leaves,l2_regularization=l2,
                    learning_rate=.04, max_bins=64,min_samples_leaf=30,early_stopping=False,random_state=42)
                model.fit(nums[0],tr.label)
                name=f'histgb_leaves{leaves}_{year}'
                p=model.predict_proba(nums[1])[:,1]; pt=model.predict_proba(nums[2])[:,1]
                preds[name]=p; tests[name]=pt
                configs[name]=dict(family='histgb',leaves=leaves,l2=l2,year=year)
                scores.append(dict(name=name,**metrics(ev,p)))
                joblib.dump(model, OUT/f'{name}_{split}.joblib',compress=3)
                print(split,name,metrics(ev,p)['phase_mean_logloss'],flush=True)
            for leaf in [5,20]:
                model=ExtraTreesClassifier(n_estimators=500,min_samples_leaf=leaf,max_features=.5,n_jobs=4,random_state=42)
                model.fit(nums[0],tr.label)
                name=f'extratrees_leaf{leaf}_{year}'
                p=model.predict_proba(nums[1])[:,1]; pt=model.predict_proba(nums[2])[:,1]
                preds[name]=p;tests[name]=pt
                configs[name]=dict(family='extratrees',leaf=leaf,year=year)
                scores.append(dict(name=name,**metrics(ev,p)))
                print(split,name,metrics(ev,p)['phase_mean_logloss'],flush=True)
        pd.DataFrame(scores).sort_values('phase_mean_logloss').to_csv(OUT/f'boosted_{split}_scores.csv',index=False)
        np.savez_compressed(OUT/f'boosted_{split}_predictions.npz',**preds)
        np.savez_compressed(OUT/f'boosted_{split}_test_predictions.npz',**tests)
        (OUT/'boosted_configs.json').write_text(json.dumps(configs,indent=2))

if __name__=='__main__':
    run()


Overwriting src/boosted.py


## `src/boosted_robustness.py`

SHA-256: `047eabbe06eb5afd40381957ef6b340baa9c429776c67b18e157cc5f08be0702`

In [23]:
%%writefile src/boosted_robustness.py
"""No-year boosting: random-seed, recency and phase-weight sensitivity."""
import os
os.environ.setdefault('OMP_NUM_THREADS','4')
os.environ.setdefault('OPENBLAS_NUM_THREADS','4')
import json
import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from experiments import OUT, PHASES, load_data, prepare, metrics
from boosted import dense_features


def run():
    data=load_data()
    for split in ['validation','temporal']:
        tr,ev,f,bt,be,bx=prepare(data,split)
        frames,svds,mask=dense_features(tr,ev,data['test'],bt,be,bx)
        xs=[d.drop(columns='year') for d in frames]
        cats=xs[0].select_dtypes('str').columns.tolist()
        scores=[];pred={};test={};configs={}
        for variant,seed in [('seed43',43),('seed44',44),('recency6',43),('phasebalanced',43)]:
            w=np.ones(len(tr))
            if variant=='recency6':
                w=2**((tr.start_year.fillna(tr.start_year.median()).to_numpy()-tr.start_year.max())/6)
            elif variant=='phasebalanced':
                w=tr.phase.map({p:len(tr)/(3*sum(tr.phase==p)) for p in PHASES}).to_numpy()
            w=w/w.mean()
            model=CatBoostClassifier(iterations=1500,depth=6,learning_rate=.035,l2_leaf_reg=6,
                random_seed=seed,thread_count=4,verbose=False,allow_writing_files=False,loss_function='Logloss')
            model.fit(xs[0],tr.label,cat_features=cats,sample_weight=w)
            joblib.dump(model,OUT/f'robust_catboost_{variant}_{split}.joblib',compress=3)
            for n in [800,1500]:
                name=f'robust_catboost_{variant}_n{n}'
                p=model.predict_proba(xs[1],ntree_end=n)[:,1]
                pt=model.predict_proba(xs[2],ntree_end=n)[:,1]
                pred[name]=p;test[name]=pt
                configs[name]=dict(family='robust_catboost',variant=variant,seed=seed,iterations=n)
                scores.append(dict(name=name,**metrics(ev,p)))
                print(split,name,metrics(ev,p)['phase_mean_logloss'],flush=True)
            pd.DataFrame(scores).sort_values('phase_mean_logloss').to_csv(OUT/f'robust_{split}_scores.csv',index=False)
            # Save each completed fit so intermediate results are safely inspectable.
            np.savez_compressed(OUT/f'robust_{split}_predictions.npz',**pred)
            np.savez_compressed(OUT/f'robust_{split}_test_predictions.npz',**test)
            (OUT/'robust_configs.json').write_text(json.dumps(configs,indent=2))

if __name__=='__main__':run()


Overwriting src/boosted_robustness.py


## `src/build_benchmark_notebook.py`

SHA-256: `c2fabcb3e7a0a209f94acc73f20b3c87deddbed5df4d94fb61546d75a6aa7d9d`

In [24]:
%%writefile src/build_benchmark_notebook.py
"""Create an executed notebook containing all local-benchmark source/results."""
import json,os,sys
from pathlib import Path
import nbformat as nbf
from nbclient import NotebookClient

ROOT=Path(__file__).resolve().parent
MODULES=['experiments.py','ani_models.py','benchmark_models.py','benchmark_protocol.py','local_benchmark.py',
         'reproduce_ani.py','benchmark_ani_recipe.py','benchmark_candidates.py','benchmark_final.py',
         'benchmark_report.py','test_local_benchmark.py','verify_benchmark.py']


def run():
    cells=[]
    def md(s):cells.append(nbf.v4.new_markdown_cell(s))
    def code(s,hidden=False):
        c=nbf.v4.new_code_cell(s)
        if hidden:c.metadata['jupyter']={'source_hidden':True}
        cells.append(c)
    md('''# Part Two: repeatable local testing and Ani comparison

**Hold submission 05.** The known reference is Ani's Kaggle **0.58688**, compared with our **02: 0.58993** and **03: 0.59562**. A higher validation F1, or even lower validation log loss, did not establish a better Kaggle model.

This notebook freezes trial-level splits, reproduces Ani's published predictions, compares models on grouped CV and chronological holdouts, and retains uncertainty and source provenance. All implementation code is included below. No Kaggle upload is performed.
''')
    code('''from pathlib import Path
import os,sys,json
ROOT=Path.cwd()
if (ROOT/"AML-Midterm/part2").is_dir():ROOT=ROOT/"AML-Midterm/part2"
assert (ROOT/"outcome_train.csv").exists()
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
os.environ["OMP_NUM_THREADS"]="4"
os.environ["OPENBLAS_NUM_THREADS"]="4"
os.environ["MPLCONFIGDIR"]=str(ROOT/"experiments/.matplotlib")
''')
    md('''## Complete source

Collapsed cells materialize the exact implementation. Training is controlled separately and is off by default.
''')
    for file in MODULES:code('%%writefile '+file+'\n'+(ROOT/file).read_text(),True)
    md('''## Protocol and reproduction

Ten fixed comparisons: supplied validation, three nonoverlapping two-year future windows, and two repeats of trial-grouped three-fold CV. The primary metric is mean log loss with equal weights for Phases I/II/III. Combined-phase rows stay together. All fitting labels and preprocessing are isolated from the held-out trials. The strict stacking pipeline refits feature preprocessing inside its inner folds as well.

The reference recipe separately reproduces Ani's original transductive vocabulary/IDF and final supervised train+validation refit. These two tracks remain explicit; a feature-policy change is not silently attributed to the model alone.
''')
    code('''RUN_TRAINING=False
import subprocess
if RUN_TRAINING:
    for command in [["reproduce_ani.py"],["local_benchmark.py","--suite","full"],
                    ["benchmark_ani_recipe.py"],["benchmark_candidates.py"],
                    ["benchmark_final.py"],["benchmark_report.py"]]:
        subprocess.run([sys.executable]+command,check=True)
import pandas as pd
import numpy as np
from IPython.display import display,Image
from benchmark_protocol import DIRECTORY,initialize
universe,protocol=initialize()
OUT=ROOT/json.loads((DIRECTORY/"current_run.json").read_text())["directory"]
display(pd.DataFrame([{k:v for k,v in s.items() if k not in ["fit_ids","eval_ids"]} for s in protocol["splits"]]))
display(json.loads((OUT/"run_manifest.json").read_text()))
''')
    md('''## Exact Ani reproduction

Source: [Ani commit 1b8bd11](https://github.com/anihadagali7/AML-Midterm/tree/1b8bd11c47428fb5453e53cdc275542b92a3b347). Frozen notebooks and CSVs are preserved under `experiments/ani_comparison/source` with hashes. All three outcome data files match the branch byte for byte.

Ani uses eight pooled field-specific predictors with grouped-OOF logistic stacking and exponential recency tau=3. His final model fits 9,050 train+validation rows; our initial submissions fitted 8,037 training rows. His feature vocabulary/IDF uses all known covariates, including test text. Recency weights do not affect his similarity-weighted kNN component. No stopping-reason inputs enter either outcome classifier.
''')
    code('''display(pd.read_csv(ROOT/"experiments/ani_comparison/exact_reproduction.csv"))
display(json.loads((ROOT/"experiments/ani_comparison/provenance.json").read_text()))
reproduction=json.loads((ROOT/"experiments/ani_comparison/exact_reproduction.json").read_text())
assert reproduction[1]["max_submission_probability_difference"]<1e-7
''')
    md('''## Strict comparison

Read chronological mean, worst period, per-phase changes and grouped CV together. Group-CV scores pool all held-out rows within each repeat before averaging the repeats. F1, ROC-AUC, average precision, Brier score and predicted/observed success rates are secondary diagnostics in `fold_scores.csv`.

The six recipes were evaluated on the same saved splits. Do not select a favorable subset of folds. Small local differences do not establish performance on 2014–2021 trials.
''')
    code('''summary=pd.read_csv(OUT/"summary.csv")
assert summary.complete_comparison.all()
display(summary)
scores=pd.read_csv(OUT/"fold_scores.csv")
display(scores[scores.kind.eq("temporal")][["model","split","phase_mean_logloss","I_logloss","II_logloss","III_logloss"]])
display(Image(filename=str(OUT/"figures/chronological_windows.png")))
''')
    md('''## Paired uncertainty

Intervals resample trials jointly, keeping combined-phase rows together. Repeated-CV losses are averaged per trial before resampling. Temporal intervals average the three periods equally. Negative differences favor a candidate relative to Ani. Intervals spanning zero are inconclusive.

These are descriptive, conditional intervals. They do not correct fully for model selection or represent the unseen cohort shift. The most recent temporal window has only 94 Phase I and 168 Phase III rows.
''')
    code('''display(Image(filename=str(OUT/"figures/paired_comparisons.png")))
display(summary[["model","temporal_delta","temporal_delta_lower","temporal_delta_upper", "group_cv_delta","group_cv_delta_lower","group_cv_delta_upper"]])
''')
    md('''## Faithful published-recipe track

The same labels and trial splits are used, but Ani's original feature-only transduction is preserved. Evaluation labels never fit a predictor. Compare this sensitivity track separately from the strict prospective benchmark.
''')
    code('''published=pd.read_csv(OUT/"published_recipe_track/summary.csv")
assert published.complete_comparison.all()
display(published)
''')
    md('''## Optional conservative candidates

07 stays anchored to the winning published file and puts 25% weight on Ani's existing capped Phase III year-trend alternative. 08 blends 75% of that winning file with 25% of our linear component, now refitted on the same 9,050 labeled rows as Ani's final fit. Both were evaluated using matching component predictions on every saved split in both tracks. They are exploratory candidates with unknown Kaggle scores, not established improvements.

05 remains on hold because the new time-based comparisons favor Ani over our linear endpoint. Preserve the 0.58688 file until a new submitted score actually improves it. Full source recipes and hashes are in `submissions/ani_candidates_manifest.json`.
''')
    code('''for track in [OUT,OUT/"published_recipe_track"]:
    display(pd.read_csv(track/"derived_candidates/summary.csv"))
display(pd.read_csv(ROOT/"submissions/ani_candidates_index.csv"))
display(json.loads((ROOT/"submissions/ani_candidates_manifest.json").read_text()))
''')
    md('''## Repeatability checks and commands

The protocol tests check trial separation, test-set exclusion, repeated-CV coverage, chronological ordering, equal-phase loss, a known paired-bootstrap result and incomplete-run handling. Run the commands from the repository root:

```sh
make -C AML-Midterm/part2 benchmark-test
make -C AML-Midterm/part2 benchmark
make -C AML-Midterm/part2 benchmark-report
```

New code or package versions receive a separate result fingerprint. Dataset changes are rejected by the existing protocol. Final fitting and held-out benchmark scores remain separate. See `LOCAL_TESTING.md` for the full operating procedure and limitations.
''')
    code('''from benchmark_protocol import resolve
for split in protocol["splits"]:
    train,ev=resolve(universe,split)
    assert not set(train.trial_id)&set(ev.trial_id)
for path in (OUT/"models").glob("*/audit.json"):
    audit=json.loads(path.read_text())
    assert audit["trial_overlap"]==0
    assert all(f["trial_overlap"]==0 for f in audit["inner_folds"])
sample=pd.read_csv(ROOT/"part2_sample_submission.csv")
for row in pd.read_csv(ROOT/"submissions/ani_candidates_index.csv").itertuples():
    result=pd.read_csv(ROOT/"submissions"/row.submission)
    assert len(result)==1500 and result.ID.is_unique and result.ID.equals(sample.ID)
    assert result.label.between(0,1,inclusive="neither").all()
print("All ten split audits and both candidate CSVs pass.")
verification=ROOT/"experiments/local_benchmark/verification.json"
if verification.exists():display(json.loads(verification.read_text()))
''')
    nb=nbf.v4.new_notebook(cells=cells,metadata={'kernelspec':{'name':'python3','display_name':'Python 3','language':'python'},'language_info':{'name':'python','version':sys.version.split()[0]}})
    os.environ['PATH']=str(Path(sys.executable).parent)+os.pathsep+os.environ.get('PATH','')
    for key,folder in [('IPYTHONDIR','.ipython'),('JUPYTER_RUNTIME_DIR','.jupyter_runtime'),('MPLCONFIGDIR','.matplotlib')]:
        os.environ[key]=str(ROOT/'experiments'/folder);Path(os.environ[key]).mkdir(parents=True,exist_ok=True)
    NotebookClient(nb,timeout=600,kernel_name='python3',resources={'metadata':{'path':str(ROOT)}}).execute()
    nbf.write(nb,ROOT/'local_benchmark.ipynb')
    assert not any(o.output_type=='error' for c in nb.cells if c.cell_type=='code' for o in c.get('outputs',[]))
    print('Executed benchmark notebook:',ROOT/'local_benchmark.ipynb')

if __name__=='__main__':run()


Overwriting src/build_benchmark_notebook.py


## `src/build_improvement_report.py`

SHA-256: `3c7fb1c4b5f6d8b0a65b1bf41a1e11ae46b2a7f389eec720d0141f8b845e656f`

In [25]:
%%writefile src/build_improvement_report.py
"""Render a concise research report, figures, and a runnable reading notebook."""
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import nbformat
from experiments import ROOT
from improvement_report import current, REFERENCES


def table(df):
    if isinstance(df.index, pd.MultiIndex):
        reset = df.reset_index()
        headers = list(reset.columns)
        rows = [[f"{v:.6f}" if isinstance(v, (float, np.floating)) else str(v) for v in row]
                for row in reset.to_numpy()]
        return "\n".join(["| "+" | ".join(headers)+" |", "| "+" | ".join(["---"]*len(headers))+" |"]+
                         ["| "+" | ".join(row)+" |" for row in rows])
    rows = [[str(i)]+[f"{v:.6f}" if isinstance(v, (float, np.floating)) else str(v) for v in row]
            for i,row in zip(df.index, df.to_numpy())]
    headers = [df.index.name or "model"]+list(df.columns)
    return "\n".join(["| "+" | ".join(headers)+" |", "| "+" | ".join(["---"]*len(headers))+" |"]+
                     ["| "+" | ".join(row)+" |" for row in rows])


def build():
    out = current()
    selection = json.loads((out / "selection.json").read_text())
    verification = json.loads((out / "verification.json").read_text())
    manifest = json.loads((ROOT / "submissions/stack_improvements_manifest.json").read_text())
    summaries = {t: pd.read_csv(out / "text_followup" / t / "independent_summary.csv").set_index("model")
                 for t in ["strict", "published"]}
    pairs = pd.read_csv(out / "paired_comparisons.csv")
    chosen = selection["shortlist"]
    display_names = list(dict.fromkeys(REFERENCES+chosen+["ani_phase3_trend", "tau6_stack", "blend_winners50",
                    "recent_joint_stack", "recent_text_stack", "unweighted_text_stack", "unweighted_text_trend"]))
    figdir = out / "figures"
    figdir.mkdir(exist_ok=True)
    plt.rcParams.update({"font.size":10, "axes.spines.top":False, "axes.spines.right":False})
    fig, axes = plt.subplots(1, 2, figsize=(13,5), constrained_layout=True)
    for ax, (track, summary) in zip(axes, summaries.items()):
        ax.scatter(summary.cv, summary.temporal, color="#a4acb4", s=35, label="Proposals / controls")
        highlighted = [("ani_stack", "#0067b1"), ("ani_first", "#db7600")]+[(n,"#268046") for n in chosen]
        best_cv = summary.cv.idxmin()
        if best_cv not in REFERENCES+chosen:
            highlighted.append((best_cv, "#7851a9"))
        for name, color in highlighted:
            row = summary.loc[name]
            ax.scatter([row.cv], [row.temporal], color=color, s=90, label=name)
            ax.annotate(name, (row.cv,row.temporal), xytext=(5,5), textcoords="offset points", fontsize=8)
        ax.set(title=f"{track.capitalize()} preprocessing", xlabel="Grouped CV log loss (lower is better)",
               ylabel="Chronological mean log loss (lower is better)")
        ax.grid(alpha=.15)
    axes[0].legend(fontsize=8, loc="upper right")
    fig.savefig(figdir / "tradeoffs.png", dpi=180)
    plt.close(fig)
    scores = pd.read_csv(out / "text_followup/strict/independent_scores.csv")
    fig, ax = plt.subplots(figsize=(10,4), constrained_layout=True)
    ts = scores[scores.kind.eq("temporal")].pivot(index="split",columns="model",values="phase_mean_logloss")
    for name in list(dict.fromkeys(REFERENCES+chosen+["ani_phase3_trend"])):
        ax.plot(np.arange(len(ts)), ts[name], marker="o", label=name)
    ax.set_xticks(np.arange(len(ts)), [x.removeprefix("time_").replace("_to_"," → ").replace("_","–") for x in ts.index])
    ax.set(ylabel="Equal-phase log loss", title="Identical forward windows, strict preprocessing")
    ax.legend(fontsize=8)
    ax.grid(alpha=.2)
    fig.savefig(figdir / "chronological.png", dpi=180)
    plt.close(fig)
    root = ROOT / "experiments/stack_improvements"
    rel = f"runs/{out.name}"
    if chosen:
        recommendation = "The fixed screening rule shortlisted **"+", ".join(chosen)+"**. Their new CSVs are exploratory and unsubmitted; inspect the gapped-window checks below before using them."
    else:
        recommendation = "**Retain Ani's 0.58688 submission. No new recipe passed the rule fixed before testing, so no new numbered Kaggle submission was promoted or added.**"
    sections = ["# Part Two: both Ani winners and proposed improvements\n", recommendation,
        "Both original winners were reproduced: unweighted stack **0.58810** and recency-weighted stack **0.58688** Kaggle log loss. These scores are tied to the archived CSV hashes and Ani's commit messages, not a fresh leaderboard query. New local scores are not Kaggle results.",
        "## What was tested\n\n27 recipes on the identical ten saved splits in each of two preprocessing tracks: **540 held-out recipe/split evaluations**. These comprise both winners, the existing Phase III trend control, and 24 other variants. A 13-recipe first batch tests weighting, calibration, interactions and mixtures; an adaptive eight-recipe batch adds capped years; a final adaptive six-recipe batch adds joint clinical-text and supervised word-ratio classifiers. All attempts are retained. No further search followed the gapped checks.",
        "Training folds exclude each held-out trial and all its phase rows. The strict track refits vocabulary/IDF inside stacking folds; the separately labeled published track preserves Ani's use of all available covariates for text preprocessing. Neither uses evaluation outcomes for fitting. All final fits use the same 9,050 labeled rows.",
        "## Strict comparison\n\n"+table(summaries["strict"].loc[display_names,["official","temporal","worst","cv"]]),
        "## Published-recipe sensitivity comparison\n\n"+table(summaries["published"].loc[display_names,["official","temporal","worst","cv"]]),
        f"Full [strict table]({rel}/text_followup/strict/independent_summary.csv) and [published table]({rel}/text_followup/published/independent_summary.csv) include every attempted recipe. Lower is better. Temporal means give equal weight to windows and phases; CV pools predictions across folds within each repeat.",
        f"![Trade-offs]({rel}/figures/tradeoffs.png)\n\n![Chronological checks]({rel}/figures/chronological.png)",
        "## Selection and uncertainty\n\nThe predeclared rule requires a lower chronological mean than **both** references, CV within 0.001 of the better reference, and worst-window loss within 0.003 of the better reference's worst loss. It is intentionally conservative. Ani's recency winner and unweighted winner can trade places between chronological and random grouped CV, so one favorable metric is insufficient.",
        f"[Paired intervals against both references]({rel}/paired_comparisons.csv) resample whole trials and average repeated losses before resampling. They are descriptive and conditional, without correction for adaptive model selection. All supplied labels have been consulted; these are not untouched-test estimates."]
    strict = summaries["strict"]
    best_cv = strict.cv.idxmin()
    row = strict.loc[best_cv]
    sections.insert(4, f"The strongest strict grouped-CV result is **{best_cv}**, with loss **{row.cv:.6f}**, versus **{strict.loc['ani_first','cv']:.6f}** for Ani's unweighted stack. Its chronological mean is **{row.temporal:.6f}**, versus **{strict.loc['ani_stack','temporal']:.6f}** for Ani's recency winner. This is a useful CV gain, but it does not justify replacing the model with better time-based performance.")
    focus = pairs[(pairs.track=="strict") & pairs.model.isin(chosen or ["ani_phase3_trend", best_cv])]
    sections.append(table(focus.set_index(["model","reference","kind"])[["delta","lower","upper"]]))
    sections.append("## Larger time-gap robustness checks\n\nAfter freezing selection, fit through 2005/2007/2009 and evaluate 2008–2009/2010–2011/2012–2013, respectively. The additional gaps probe longer-range transport. Evaluation labels overlap prior checks; this is robustness evidence, not new independent data.")
    for track in ["strict","published"]:
        gap = pd.read_csv(out / "robustness" / track / "summary.csv").set_index("model")
        sections.append(f"### {track.capitalize()}\n\n"+table(gap[["temporal","worst"]]))
    sections.append("## Reproduction and deliverables\n\n"+f"All source/data hashes and fitting boundaries were checked. Maximum probability difference from the archived ten-split Ani predictions is {verification['historical_baseline_max_probability_error']:.2g}. Maximum differences from original Kaggle CSV probabilities: "+", ".join(f"{name} {v['max_probability_error']:.2g}" for name,v in verification["both_kaggle_sources_reproduced"].items())+". The final native model replay also verifies that changing IDs or evaluation labels cannot change predictions. All 13 protocol/scoring/model checks passed.")
    if manifest["submissions"]:
        sections.append("New optional CSVs: "+", ".join(f"[{v['file']}](../../submissions/{v['file']})" for v in manifest["submissions"])+". Their Kaggle scores are unknown.")
    sections.append("Reproduce with [IMPROVEMENT_PROTOCOL.md](../../IMPROVEMENT_PROTOCOL.md), pinned [requirements-benchmark.txt](../../requirements-benchmark.txt), and the [executed notebook](../../stack_improvements.ipynb). The original data, submissions, branch history, and ten split definitions are preserved.")
    (root / "RESULTS.md").write_text("\n\n".join(sections)+"\n")
    nb = nbformat.v4.new_notebook()
    nb.metadata.kernelspec = dict(display_name="Python (Part Two)", language="python", name="part2")
    nb.cells = [nbformat.v4.new_markdown_cell("# Part Two: paired comparison of both Ani winners\n\n"+recommendation+"\n\nRead `IMPROVEMENT_PROTOCOL.md` for the frozen rules and adaptive-batch history. This notebook reads the saved evaluations; it does not retrain models."),
        nbformat.v4.new_code_cell("from pathlib import Path\nimport json\nimport pandas as pd\nfrom IPython.display import display, Image\nroot = Path.cwd()\nif (root / 'part2').exists(): root /= 'part2'\nbase = root / 'experiments/stack_improvements'\nrun = base / 'runs' / json.loads((base / 'current_run.json').read_text())['run']\nprint('Run:', run.name)\ndisplay(json.loads((run / 'selection.json').read_text()))"),
        nbformat.v4.new_markdown_cell("## All recipes, identical splits\n\nSmaller log loss is better. Keep strict and published preprocessing policies separate."),
        nbformat.v4.new_code_cell("for track in ['strict', 'published']:\n    print(track)\n    display(pd.read_csv(run / 'text_followup' / track / 'independent_summary.csv').sort_values('temporal').round(6))\ndisplay(Image(filename=str(run / 'figures/tradeoffs.png')))"),
        nbformat.v4.new_markdown_cell("## Paired differences against both winners\n\nNegative favors the proposal; intervals crossing zero are inconclusive. Resampling keeps trial rows together and does not count repeat appearances as independent trials."),
        nbformat.v4.new_code_cell("pairs = pd.read_csv(run / 'paired_comparisons.csv')\nshortlist = json.loads((run / 'selection.json').read_text())['shortlist']\nfocus = shortlist or ['ani_phase3_trend', 'recent_text_stack']\ndisplay(pairs[pairs.model.isin(focus)].round(6))\ndisplay(Image(filename=str(run / 'figures/chronological.png')))"),
        nbformat.v4.new_markdown_cell("## Additional time gaps and verification\n\nThese reuse historical labels; they are robustness checks, not untouched validation. Only the frozen shortlist and both references are evaluated."),
        nbformat.v4.new_code_cell("for track in ['strict','published']:\n    print(track)\n    display(pd.read_csv(run / 'robustness' / track / 'summary.csv').round(6))\ndisplay(json.loads((run / 'verification.json').read_text()))\ndisplay(json.loads((root / 'submissions/stack_improvements_manifest.json').read_text()))")]
    nbformat.write(nb, ROOT / "stack_improvements.ipynb")
    print("Built", root / "RESULTS.md")


if __name__ == "__main__":
    build()


Overwriting src/build_improvement_report.py


## `src/build_notebook.py`

SHA-256: `e7299f0a327a79a249e958ee61b10f69a6a7150aa36ed23ef75ecacfbc6c1829`

In [26]:
%%writefile src/build_notebook.py
"""Create the self-contained, executed Part Two notebook."""
import json
import os
import sys
from pathlib import Path
import nbformat as nbf
from nbclient import NotebookClient

ROOT=Path(__file__).resolve().parent
MODULES=['experiments.py','boosted.py','refinements.py','recency_and_sparse_boost.py',
         'boosted_robustness.py','year_cap.py','ensemble_search.py','submission_models.py',
         'error_analysis.py','reporting.py','followup_ensembles.py']


def run():
    selected=json.loads((ROOT/'experiments/model_search/selected_ensembles.json').read_text())
    cells=[]
    def md(t):cells.append(nbf.v4.new_markdown_cell(t))
    def code(t,hidden=False):
        cell=nbf.v4.new_code_cell(t)
        if hidden:cell.metadata['jupyter']={'source_hidden':True}
        cells.append(cell)
    md('''# Part Two: clinical-trial outcomes

Predict the probability of success for Phases I, II and III. Kaggle evaluates log loss (lower is better).
The best reported reference score is **0.58688**. Our **03 scored 0.59562**, and **02 improved to 0.58993**. The local ranking did not transfer to Kaggle. Initial 01/04 remain unscored. Results are recorded against file hashes in `submissions/leaderboard_results.json`.

This notebook includes the complete experiment source, two local evaluation protocols, per-phase F1/ROC-AUC/average precision, the required stopping-reason analysis, and four checked submission files. It uses the supplied data and classical machine learning. IDs and Part One reason information never enter outcome prediction.

**Latest update: hold 05/06.** Ani's published model is reproduced, and the new ten-split benchmark favors Ani over our linear endpoint across time. See `local_benchmark.ipynb`, `LOCAL_TESTING.md` and `experiments/local_benchmark/RESULTS.md` before choosing another submission. This notebook preserves the initial search and original outcome-model fits.
''')
    code('''from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if (ROOT / "AML-Midterm/part2").is_dir(): ROOT = ROOT / "AML-Midterm/part2"
assert (ROOT / "outcome_train.csv").exists(), "Run in the repository or part2 directory"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["MPLCONFIGDIR"] = str(ROOT / "experiments/.matplotlib")
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
''')
    md('''## Complete implementation

These collapsed cells contain all experiment and inference code as executable notebook source. Running them materializes the modules in this directory. Search execution is controlled separately below.
''')
    for name in MODULES:code('%%writefile '+name+'\n'+(ROOT/name).read_text(),True)
    md('''## Reproduction and evaluation design

The default reuses completed experiments and does not retrain. Set `RUN_TRAINING=True` to repeat the searches and produce the four submissions.

The supplied validation set has 149 Phase I, 501 Phase II and 363 Phase III rows. Selection uses the **equal mean of per-phase log losses**, since test has 500 rows per phase. A second, train-only temporal comparison fits through 2008 and evaluates on 2009-2013 trials. Combined-phase rows stay together. Every learned feature transform is fitted only on the corresponding fitting population. Final models use all 8,037 training rows and no validation labels.

Both local evaluations are used in model selection. They are not untouched estimates of generalization; the latest test cohort (2014-2021) can behave differently.
''')
    code('''RUN_TRAINING = False
import subprocess
from experiments import OUT, load_data, metrics, audit_data
SELECTED = '''+repr(selected)+'''
if RUN_TRAINING:
    for script in ["experiments.py", "refinements.py", "recency_and_sparse_boost.py", "year_cap.py",
                   "boosted.py", "boosted_robustness.py", "ensemble_search.py"]:
        command = [sys.executable, script]
        if script == "ensemble_search.py": command.append("--select")
        subprocess.run(command, check=True)
    (OUT / "selected_ensembles.json").write_text(json.dumps(SELECTED, indent=2))
    for script in ["submission_models.py", "error_analysis.py", "followup_ensembles.py", "reporting.py"]:
        subprocess.run([sys.executable, script], check=True)
import numpy as np
import pandas as pd
from IPython.display import display, Image
data = load_data()
display(pd.DataFrame([{ "split":s, "rows":len(d), "trials":d.trial_id.nunique(),
    "first_year":d.start_year.min(), "last_year":d.start_year.max(),
    "matched_drug_fraction":d.matched_drugs.notna().mean()} for s,d in data.items()]))
audit = audit_data(data)
assert all(n == 0 for n in audit["overlaps"].values())
''')
    md('''## Model and feature comparisons

Compare eligibility word/bigram and character TF-IDF, binary words, diseases, drugs, ICD groups, protein targets/actions/pathways, Morgan fingerprints/descriptors, shared and separate phase effects, newer-trial weighting, raw/capped start year, and NB log-ratio features. Algorithms include logistic regression, CatBoost, histogram gradient boosting, sparse LightGBM and Extra Trees. All settings and predictions are saved under `experiments/model_search`.

The baseline uses eligibility TF-IDF (35,000 unigram/bigram terms, minimum document frequency 3, sublinear TF), phase indicators, and pooled L2 logistic regression with C=2. Molecular features do not consistently help. Raw year extrapolation is fragile in the temporal check. Annotation coverage declines in test, which favors retaining a strong text component. The selected mixes also test whether phase-specific model balance improves over one global blend.
''')
    code('''components = pd.read_csv(OUT / "all_component_scores.csv")
ensemble_scores = pd.read_csv(OUT / "ensemble_scores.csv")
print(f"{len(components)} matching component configurations; {len(ensemble_scores)} fixed ensemble configurations")
display(components[["name", "validation_phase_mean_logloss", "temporal_phase_mean_logloss", "joint_logloss"]].head(15))
display(ensemble_scores[["name", "validation_phase_mean_logloss", "temporal_phase_mean_logloss", "joint_logloss"]].head(12))
display(Image(filename=str(OUT / "figures/model_comparison.png")))
''')
    md('''## Four manual Kaggle submissions

03 was selected first because it had the lowest local joint loss, but scored **0.59562**. 02 improved to **0.58993**, still behind the **0.58688** reference. Less tree influence in Phases II/III improved this comparison, but the cause of the generalization gap remains uncertain. The original four CSVs are preserved. Their predictions use models fitted on the full training file, never the reduced temporal fit.
''')
    code('''index = pd.read_csv(ROOT / "submissions/index.csv")
display(index[["submission", "validation_logloss", "temporal_logloss", "mean_test_probability", "kaggle_score"]])
manifest = json.loads((ROOT / "submissions/manifest.json").read_text())
display(manifest["ensembles"])
test_predictions = pd.DataFrame({row.submission[:2]:pd.read_csv(ROOT/"submissions"/row.submission).label
                                 for row in index.itertuples()})
display(pd.DataFrame([[np.mean(np.abs(test_predictions[a]-test_predictions[b]))
                      for b in test_predictions] for a in test_predictions],
                     index=test_predictions.columns, columns=test_predictions.columns).rename_axis("Mean absolute test-probability difference"))
''')
    md('''## Prospective follow-ups after the leaderboard feedback

05 keeps Phase I exactly as in 02 and removes trees in Phases II/III. 06 keeps Phase I unchanged and uses 12.5% trees in II/III. Both reuse the original fitted components without refitting or new outcome labels. **These are now on hold after the Ani comparison.** Their selection used the reported leaderboard direction; the new fixed benchmark gives a more informative comparison. Use the separate benchmark notebook for current Ani-anchored candidates and their uncertainty.
''')
    code('''display(pd.read_csv(ROOT / "submissions/followup_index.csv"))
display(pd.read_csv(OUT / "followup_phase_metrics.csv"))
display(json.loads((ROOT / "submissions/followup_manifest.json").read_text())["ensembles"])
display(json.loads((ROOT / "submissions/leaderboard_results.json").read_text()))
''')
    md('''## Per-phase performance

Success (`label=1`) is the positive class. F1 uses threshold 0.5. ROC-AUC and PR-AUC use success probabilities; PR-AUC is average precision. Compare all four candidates with the baseline, including trade-offs when log loss improves without an F1 improvement.
''')
    code('''phase_metrics = pd.read_csv(OUT / "phase_metrics.csv")
display(phase_metrics)
display(Image(filename=str(OUT / "figures/phase_roc_pr.png")))
''')
    md('''## Error analysis using Part One reasons

For unsuccessful validation rows, join `trial_id` to Part One `ID`. Use true visible train/validation reason labels when available; otherwise use the frozen original 157-label DBSCAN/logistic model (0.5 cutoff and argmax fallback). Trials absent from Part One form a separate no-recorded-stopping-reason group. Reasons can overlap. Count each trial once within each phase.

The baseline predicts success for 8 of 9 Phase III efficacy-tagged failures, suggesting that eligibility-only text misses important failure mechanisms. All nine tags are predicted, making this a small, uncertain subgroup. Clinical context, phase effects and nonlinear models are tested to broaden the predictor, while the tables below show their actual improvements and regressions. No reason label, embedding, or Part One membership is an outcome-model input.
''')
    code('''reasons = pd.read_csv(OUT / "reason_labels_for_analysis.csv")
display(reasons.reason_source.value_counts().rename_axis("Part One label source").to_frame("trials"))
display(json.loads((OUT / "reason_model_provenance.json").read_text()))
comparison = pd.read_csv(OUT / "reason_error_comparison.csv")
display(comparison[["phase", "reason", "n", "false_success_baseline", "false_success_improved",
                    "false_success_rate_baseline", "false_success_rate_improved",
                    "true_reason_rows_baseline", "predicted_reason_rows_baseline"]])
display(Image(filename=str(OUT / "figures/reason_errors.png")))
''')
    md('''Funding has no unsuccessful Phase III examples in this validation set, so its rate is undefined. Small group sizes and uncertainty in predicted reason labels limit conclusions about differences between categories. Wilson intervals and the complete tables for every candidate are saved in CSV files. The paired trial-level bootstrap below is descriptive because the same validation set was used for selection.
''')
    code('''display(pd.read_csv(OUT / "paired_validation_bootstrap.csv"))
display(pd.read_csv(OUT / "03_phase_specialized_reason_errors.csv"))
''')
    md('''## Verification and saved artifacts

Independent linear refits and saved tree-model reloads must reproduce the stored validation and test probabilities. All CSVs have exactly 1,500 unique IDs in sample-submission order, columns `ID,label`, and finite probabilities strictly between zero and one. `index.csv` is bookkeeping and should not be submitted.

The complete fitted bundle is `experiments/model_search/submission_models.joblib`. Reload inference with `python submission_models.py --input outcome_test.csv --output replayed_predictions`. Library versions are pinned in `requirements.txt`. No Kaggle upload is performed here.
''')
    code('''checks = pd.read_csv(OUT / "model_replay_verification.csv")
assert checks.max_probability_difference.max() < 1e-8
display(checks)
sample = pd.read_csv(ROOT / "part2_sample_submission.csv")
all_files = pd.concat([index, pd.read_csv(ROOT / "submissions/followup_index.csv")], ignore_index=True)
for row in all_files.itertuples():
    result = pd.read_csv(ROOT / "submissions" / row.submission)
    assert result.columns.tolist() == ["ID", "label"]
    assert len(result) == 1500 and result.ID.nunique() == 1500
    assert result.ID.equals(sample.ID)
    assert np.isfinite(result.label).all() and result.label.between(0, 1, inclusive="neither").all()
print(f"All {len(all_files)} submissions pass the format, ID, and probability checks.")
display(json.loads((OUT / "versions.json").read_text()))
''')
    nb=nbf.v4.new_notebook(cells=cells,metadata={'kernelspec':{'name':'python3','display_name':'Python 3','language':'python'},
        'language_info':{'name':'python','version':sys.version.split()[0]}})
    nbf.write(nb,ROOT/'part2.ipynb')
    os.environ['PATH']=str(Path(sys.executable).parent)+os.pathsep+os.environ.get('PATH','')
    os.environ['IPYTHONDIR']=str(ROOT/'experiments/.ipython')
    os.environ['JUPYTER_RUNTIME_DIR']=str(ROOT/'experiments/.jupyter_runtime')
    os.environ['MPLCONFIGDIR']=str(ROOT/'experiments/.matplotlib')
    for key in ['IPYTHONDIR','JUPYTER_RUNTIME_DIR','MPLCONFIGDIR']:Path(os.environ[key]).mkdir(parents=True,exist_ok=True)
    client=NotebookClient(nb,timeout=600,kernel_name='python3',resources={'metadata':{'path':str(ROOT)}})
    client.execute()
    nbf.write(nb,ROOT/'part2.ipynb')
    errors=[output for cell in nb.cells if cell.cell_type=='code' for output in cell.get('outputs',[]) if output.output_type=='error']
    assert not errors
    print('Executed',len(nb.cells),'cells without errors:',ROOT/'part2.ipynb')

if __name__=='__main__':run()


Overwriting src/build_notebook.py


## `src/ensemble_search.py`

SHA-256: `7bcb0054fd1ddbc82629d59acb4dc0337838fa3ee1a1a8c3965e41bb57aee3b9`

In [27]:
%%writefile src/ensemble_search.py
"""Evaluate fixed-weight ensembles on both local evaluation populations."""
import json
import numpy as np
import pandas as pd
from experiments import OUT, load_data, split_data, metrics

FAMILIES=['linear','refined','extra','boosted','robust','yearcap']


def collect():
    pred={s:{} for s in ['validation','temporal']}
    tests={s:{} for s in pred}
    configs={}
    for family in FAMILIES:
        c=OUT/f'{family}_configs.json'
        if c.exists():configs.update(json.loads(c.read_text()))
        for split in pred:
            p=OUT/f'{family}_{split}_predictions.npz'
            t=OUT/f'{family}_{split}_test_predictions.npz'
            if p.exists() and t.exists():
                with np.load(p) as f:pred[split].update({k:f[k] for k in f.files})
                with np.load(t) as f:tests[split].update({k:f[k] for k in f.files})
    return pred,tests,configs


def weighted(bank,weights,phases=None):
    if any(isinstance(v,dict) for v in weights.values()):
        assert phases is not None
        phases=np.asarray(phases)
        for p in ['I','II','III']:assert abs(sum(w[p] for w in weights.values())-1)<1e-9
        return sum(np.array([weight[p] for p in phases])*bank[name] for name,weight in weights.items())
    assert abs(sum(weights.values())-1)<1e-9
    return sum(weight*bank[name] for name,weight in weights.items())


def combine(a,b,fraction):
    keys=set(a)|set(b)
    return {k:(1-fraction)*a.get(k,0)+fraction*b.get(k,0) for k in keys if (1-fraction)*a.get(k,0)+fraction*b.get(k,0)>0}


def average(names):return {n:1/len(names) for n in names}


def run():
    data=load_data();pred,tests,configs=collect()
    common=set(pred['validation'])&set(pred['temporal'])
    groups={
      'linear_wordwide':{'text_wordwide_C2':1.},
      'linear_baseline':{'lr_criteria_none_C2_phase0':1.},
      'linear_clinical_baseline':{'lr_clinical_none_C0.5_phase0':1.},
      'linear_yearcap':{'lr_criteria_yearcap_C2_phase0':1.},
      'linear_text':average(['text_wordwide_C2','recency_criteria_C2_half6']),
      'linear_clinical':{'recency_clinical_C0.5_half3':1.},
      'catboost_seeds':average(['robust_catboost_seed43_n1500','robust_catboost_seed44_n1500']),
      'catboost_seeds_early':average(['robust_catboost_seed43_n800','robust_catboost_seed44_n800']),
      'catboost_standard':average(['catboost_d4_n1000_none','catboost_d6_n1000_none']),
      'catboost_early':average(['catboost_d4_n600_none','catboost_d6_n600_none']),
      'catboost_deep':average(['catboost_d6_n1000_none','catboost_d8_n1000_none']),
      'catboost_recency':{'robust_catboost_recency6_n1500':1.},
      'catboost_recency_early':{'robust_catboost_recency6_n800':1.},
      'catboost_phasebalanced':{'robust_catboost_phasebalanced_n1500':1.},
      'catboost_year':average(['catboost_d4_n1000_year','catboost_d6_n1000_year']),
      'catboost_year_early':average(['catboost_d4_n600_year','catboost_d6_n600_year']),
      'histgb':{'histgb_leaves31_none':1.},
      'extratrees':{'extratrees_leaf5_none':1.},
      'extratrees_year':{'extratrees_leaf5_year':1.},
      'lightgbm':{'lightgbm_leaves7_n1000_none':1.},
    }
    groups={k:v for k,v in groups.items() if set(v)<=common}
    candidates={k:v for k,v in groups.items()}
    if 'linear_text' in groups and 'linear_clinical' in groups:
        for w in [.25,.5]:
            groups[f'linear_mix_clinical{w:g}']=combine(groups['linear_text'],groups['linear_clinical'],w)
    linear={k:v for k,v in groups.items() if k.startswith('linear')}
    trees={k:v for k,v in groups.items() if not k.startswith('linear')}
    if 'catboost_seeds' in groups and 'catboost_standard' in groups:
        trees['catboost_bagged']=combine(groups['catboost_standard'],groups['catboost_seeds'],.5)
    if 'catboost_seeds' in groups and 'catboost_recency' in groups:
        trees['catboost_recentmix']=combine(groups['catboost_seeds'],groups['catboost_recency'],.5)
    for ln,lw in linear.items():
        for tn,tw in trees.items():
            for alpha in [.25,.5,.75]:
                candidates[f'{ln}__{tn}__tree{alpha:g}']=combine(lw,tw,alpha)
    for ln in ['linear_text','linear_mix_clinical0.25','linear_mix_clinical0.5']:
        if ln not in linear:continue
        for tn in ['catboost_standard','catboost_deep','catboost_seeds_early','catboost_year','histgb']:
            if tn not in trees:continue
            for alphas in [(.25,.5,.75),(.25,.25,.75)]:
                lw,tw=linear[ln],trees[tn]
                weights={k:{p:(1-a)*lw.get(k,0)+a*tw.get(k,0) for p,a in zip(['I','II','III'],alphas)} for k in set(lw)|set(tw)}
                candidates[f'{ln}__{tn}__phases'+','.join(f'{a:g}' for a in alphas)]=weights
    rows=[]
    for name,w in candidates.items():
        row={'name':name,'n_components':len(w)}
        for split in pred:
            evaluation=split_data(data,split)[1]
            m=metrics(evaluation,weighted(pred[split],w,evaluation.phase))
            for key,value in m.items():row[split+'_'+key]=value
        row['joint_logloss']=.5*(row['validation_phase_mean_logloss']+row['temporal_phase_mean_logloss'])
        rows.append(row)
    scores=pd.DataFrame(rows).sort_values('joint_logloss')
    scores.to_csv(OUT/'ensemble_scores.csv',index=False)
    (OUT/'ensemble_configs.json').write_text(json.dumps(candidates,indent=2))
    component_rows=[]
    for name in common:
        row={'name':name,**configs.get(name,{})}
        for split in pred:
            m=metrics(split_data(data,split)[1],pred[split][name])
            row[split+'_phase_mean_logloss']=m['phase_mean_logloss']
        row['joint_logloss']=.5*(row['validation_phase_mean_logloss']+row['temporal_phase_mean_logloss'])
        component_rows.append(row)
    pd.DataFrame(component_rows).sort_values('joint_logloss').to_csv(OUT/'all_component_scores.csv',index=False)
    print(scores[['name','validation_phase_mean_logloss','temporal_phase_mean_logloss','joint_logloss']].head(30).to_string(index=False))


def choose_four():
    scores=pd.read_csv(OUT/'ensemble_scores.csv')
    no_cap=~scores.name.str.startswith('linear_yearcap')
    filters={
      '01_balanced_ensemble':no_cap & scores.name.str.endswith('__tree0.5'),
      '02_text_heavy':no_cap & scores.name.str.endswith('__tree0.25'),
      '03_phase_specialized':scores.name.str.contains('__phases'),
      '04_capped_year':scores.name.str.startswith('linear_yearcap__'),
    }
    selected={name:scores[mask].iloc[0]['name'] for name,mask in filters.items()}
    assert len(set(selected.values()))==4
    (OUT/'selected_ensembles.json').write_text(json.dumps(selected,indent=2))
    (OUT/'selection_policy.json').write_text(json.dumps({
        'ranking':'Mean of equal-phase validation and temporal log loss',
        'directions':{
            '01_balanced_ensemble':'Best 50% text / 50% tree mixture without linear year effects',
            '02_text_heavy':'Best 75% text / 25% tree mixture without linear year effects',
            '03_phase_specialized':'Best mixture allowing different fixed weights in the three phases',
            '04_capped_year':'Best mixture using a linear year effect capped at the latest training year'},
        'interpretation':'Distinct strong candidates for four manual submissions, rather than the four nearly identical minimum-loss rows.'},indent=2))
    print('Selected:',json.dumps(selected,indent=2))


if __name__=='__main__':
    import sys
    run()
    if '--select' in sys.argv:choose_four()


Overwriting src/ensemble_search.py


## `src/error_analysis.py`

SHA-256: `115b477403a1e8cb7e90f5ba139c9653a209f4a0180298f22ef59c1b750e8791`

In [28]:
%%writefile src/error_analysis.py
"""Part 1 reason labels are joined exclusively after outcome prediction."""
from pathlib import Path
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
from experiments import ROOT, OUT, REASONS, PHASES, metrics, load_data


def reason_labels():
    path=ROOT.parent/'part1/experiments/additional_methods/classifier_models/dbscan_m5_q25__logistic_C1000.joblib'
    model=joblib.load(path)
    frames=[]
    for split in ['train','val','test']:
        df=pd.read_csv(ROOT.parent/f'part1/data/part1_{split}.csv')
        x=model['scaler'].transform(df[model['features']].to_numpy())
        p=np.column_stack([m.predict_proba(x)[:,1] for m in model['estimators']])
        y=p>=.5
        empty=~y.any(axis=1)
        y[empty,p[empty].argmax(axis=1)]=True
        known=df[REASONS].ge(0).all(axis=1).to_numpy() if all(c in df for c in REASONS) else np.zeros(len(df),bool)
        y=y.astype(int)
        if known.any():y[known]=df.loc[known,REASONS].to_numpy()
        result=pd.DataFrame(y,columns=REASONS)
        result['trial_id']=df.ID.to_numpy()
        result['reason_source']=np.where(known,'true_'+split,'predicted_'+split)
        frames.append(result)
    labels=pd.concat(frames,ignore_index=True)
    assert not labels.trial_id.duplicated().any()
    labels.to_csv(OUT/'reason_labels_for_analysis.csv',index=False)
    (OUT/'reason_model_provenance.json').write_text(json.dumps({
        'path':str(path.relative_to(ROOT.parent)),
        'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),
        'policy':'Visible Part 1 train/val labels when available; otherwise frozen DBSCAN/logistic Part 1 model, cutoff 0.5, argmax fallback.',
        'usage':'Error analysis only; none of these columns or embeddings is an outcome predictor.'},indent=2))
    return labels


def errors_by_reason(df, probabilities, labels):
    d=df.merge(labels,on='trial_id',how='left',validate='many_to_one',sort=False)
    assert np.array_equal(d.ID,df.ID)
    d['probability']=probabilities
    rows=[]
    for phase in PHASES:
        for reason in REASONS+['No recorded stopping reason']:
            group=d.reason_source.isna() if reason.startswith('No recorded') else d[reason].eq(1)
            v=d[d.phase.eq(phase)&d.label.eq(0)&group]
            assert not v.trial_id.duplicated().any()
            n=len(v);k=int((v.probability>=.5).sum())
            # Wilson intervals indicate small groups; predictions also have label uncertainty.
            if n:
                phat=k/n;z=1.96;denom=1+z*z/n
                center=(phat+z*z/(2*n))/denom
                radius=z*np.sqrt(phat*(1-phat)/n+z*z/(4*n*n))/denom
                lo,hi=center-radius,center+radius
            else:phat=lo=hi=np.nan
            rows.append(dict(phase=phase,reason=reason,n=n,false_success=k,false_success_rate=phat,
                wilson_lower=lo,wilson_upper=hi,
                true_reason_rows=int(v.reason_source.fillna('').str.startswith('true').sum()),
                predicted_reason_rows=int(v.reason_source.fillna('').str.startswith('predicted').sum())))
    return pd.DataFrame(rows)


def run():
    labels=reason_labels();data=load_data()
    base=np.load(OUT/'linear_validation_predictions.npz')['lr_criteria_none_C2_phase0']
    baseline=errors_by_reason(data['val'],base,labels)
    baseline.to_csv(OUT/'baseline_reason_errors.csv',index=False)
    selected=np.load(OUT/'selected_validation_predictions.npz')
    performance=[]
    for name,p in [('baseline',base)]+[(k,selected[k]) for k in selected.files]:
        for phase in PHASES:
            m=metrics(data['val'],p)
            performance.append(dict(model=name,phase=phase,n=int(data['val'].phase.eq(phase).sum()),
                log_loss=m[phase+'_logloss'],f1=m[phase+'_f1'],roc_auc=m[phase+'_roc_auc'],pr_auc=m[phase+'_pr_auc']))
        table=errors_by_reason(data['val'],p,labels)
        table.to_csv(OUT/f'{name}_reason_errors.csv',index=False)
        if name=='03_phase_specialized':
            compare=baseline.merge(table,on=['phase','reason','n'],suffixes=('_baseline','_improved'))
            compare['false_success_rate_change']=compare.false_success_rate_improved-compare.false_success_rate_baseline
            compare.to_csv(OUT/'reason_error_comparison.csv',index=False)
    pd.DataFrame(performance).to_csv(OUT/'phase_metrics.csv',index=False)

if __name__=='__main__':run()


Overwriting src/error_analysis.py


## `src/experiments.py`

SHA-256: `7346197ed8466c54bc4caad0ad31f4570e8838a9aa81663eab01903f901e4a70`

In [29]:
%%writefile src/experiments.py
"""Reproducible Part 2 experiments. Only supplied pre-outcome features are used.

Run from any directory: ../../.venv/bin/python experiments.py --stage screen
All preprocessing is refitted on each experiment's fitting population.
"""
from __future__ import annotations

import argparse
import ast
import hashlib
import json
import os
import re
import time
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, log_loss, roc_auc_score
from sklearn.preprocessing import normalize

ROOT = Path(__file__).resolve().parent
OUT = ROOT / "experiments" / "model_search"
OUT.mkdir(parents=True, exist_ok=True)
PHASES = ("I", "II", "III")
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]


def load_data():
    return {s: pd.read_csv(ROOT / f"outcome_{s}.csv") for s in ("train", "val", "test")}


def audit_data(data=None):
    data=load_data() if data is None else data
    audit={'datasets':{},'overlaps':{},
        'excluded_predictors':['ID','trial_id','label','all Part 1 reason labels and embeddings'],
        'evaluation':'Equal mean of per-phase log losses; official validation and train-only chronological holdout at 2008'}
    for split,df in data.items():
        assert not df.ID.duplicated().any()
        if 'label' in df:
            assert set(df.label)=={0,1}
            assert df.groupby('trial_id').label.nunique().max()==1
        audit['datasets'][split]={'rows':len(df),'trials':df.trial_id.nunique(),
            'min_year':float(df.start_year.min()),'max_year':float(df.start_year.max()),
            'sha256':hashlib.sha256((ROOT/f'outcome_{split}.csv').read_bytes()).hexdigest(),
            'phases':df.phase.value_counts().to_dict(),
            'matched_drug_fraction':float(df.matched_drugs.notna().mean())}
    for a,b in [('train','val'),('train','test'),('val','test')]:
        n=len(set(data[a].trial_id)&set(data[b].trial_id));assert n==0
        audit['overlaps'][a+'_'+b]=n
    sample=pd.read_csv(ROOT/'part2_sample_submission.csv')
    assert set(sample.ID)==set(data['test'].ID)
    audit['sample_order_matches_test']=bool(sample.ID.equals(data['test'].ID))
    (OUT/'data_audit.json').write_text(json.dumps(audit,indent=2))
    return audit


def list_values(value):
    if pd.isna(value):
        return []
    try:
        x = ast.literal_eval(value)
        return [str(t).lower().strip() for t in x] if isinstance(x, list) else [str(x).lower()]
    except (ValueError, SyntaxError):
        return [str(value).lower()]


def metadata_dict(row):
    out = {"phase=" + row.phase: 1., "phase_detail=" + row.phase_detail: 1.}
    for col in ("diseases_cleaned", "drugs"):
        for token in set(list_values(row[col])):
            out[col + "=" + token] = 1.
    for code in set(re.findall(r"[A-Z][0-9][0-9](?:\.[0-9A-Z]+)?", str(row.icdcodes))):
        out["icd=" + code[:3]] = 1.
        out["icdgroup=" + code[0]] = 1.
    for col in ("matched_drugs", "drug_targets", "drug_actions", "drug_pathways"):
        vals = [] if pd.isna(row[col]) else str(row[col]).lower().split(";")
        for token in set(vals):
            out[col + "=" + token.strip()] = 1.
        out[col + "_missing"] = float(not vals)
    return out


class Features:
    """Independent TF-IDF blocks; no IDs, stopping labels or reason embeddings."""
    def fit(self, df):
        self.vecs = {}
        specs = {
            "criteria": dict(ngram_range=(1, 2), min_df=3, max_df=.99, max_features=35000),
            "disease": dict(ngram_range=(1, 2), min_df=2, max_features=8000),
            "drug": dict(ngram_range=(1, 2), min_df=2, max_features=8000),
            "char": dict(analyzer="char_wb", ngram_range=(3, 5), min_df=4, max_features=25000),
        }
        for name, kw in specs.items():
            self.vecs[name] = TfidfVectorizer(sublinear_tf=True, strip_accents="unicode", dtype=np.float64, **kw)
            self.vecs[name].fit(self.documents(df, name))
        self.dv = DictVectorizer()
        self.dv.fit([metadata_dict(r) for _, r in df.iterrows()])
        self.year_median = float(df.start_year.median())
        self.year_max = float(df.start_year.max())
        return self

    @staticmethod
    def documents(df, name):
        if name == "criteria":
            return df.criteria.fillna("").str.replace(r"\\[nr]", " ", regex=True)
        if name == "disease":
            return df.diseases_cleaned.fillna("")
        if name == "drug":
            return df.drugs.fillna("") + " " + df.matched_drugs.fillna("")
        return df.diseases_cleaned.fillna("") + " " + df.drugs.fillna("")

    def transform(self, df):
        b = {name: v.transform(self.documents(df, name)) for name, v in self.vecs.items()}
        b["knowledge"] = self.dv.transform([metadata_dict(r) for _, r in df.iterrows()])
        b["phase"] = sparse.csr_matrix(np.column_stack([(df.phase == p).astype(float) for p in PHASES]))
        year = df.start_year.fillna(self.year_median).to_numpy()
        b["year"] = sparse.csr_matrix(((year - 2007) / 5)[:, None])
        b["year_cap"] = sparse.csr_matrix(((np.minimum(year, self.year_max) - 2007) / 5)[:, None])
        return b


REPRESENTATIONS = {
    "criteria": {"criteria": 1., "phase": 1.},
    "clinical": {"criteria": 1., "disease": 1., "drug": .7, "phase": 1.},
    "knowledge": {"criteria": 1., "disease": 1., "drug": .7, "knowledge": .35, "phase": 1.},
    "rich": {"criteria": 1., "disease": .8, "drug": .5, "char": .5, "knowledge": .25, "phase": 1.},
    "metadata": {"disease": 1., "drug": .7, "knowledge": .35, "phase": 1.},
}


def matrix(blocks, representation, year="none"):
    weights = dict(REPRESENTATIONS[representation])
    if year != "none":
        weights[year] = 1.
    return sparse.hstack([blocks[k] * weight for k, weight in weights.items()], format="csr")


def metrics(df, p):
    p = np.clip(p, 1e-6, 1-1e-6)
    result = {"logloss": log_loss(df.label, p)}
    for phase in PHASES:
        m = df.phase.eq(phase).to_numpy()
        result[f"{phase}_logloss"] = log_loss(df.label[m], p[m])
        result[f"{phase}_f1"] = f1_score(df.label[m], p[m] >= .5)
        result[f"{phase}_roc_auc"] = roc_auc_score(df.label[m], p[m])
        result[f"{phase}_pr_auc"] = average_precision_score(df.label[m], p[m])
    result["phase_mean_logloss"] = np.mean([result[f"{p}_logloss"] for p in PHASES])
    return result


def fit_linear(x, df, C, separate):
    models = {}
    for phase in (PHASES if separate else ("all",)):
        idx = np.ones(len(df), dtype=bool) if phase == "all" else df.phase.eq(phase).to_numpy()
        model = LogisticRegression(C=C, solver="liblinear", max_iter=1000, tol=1e-5, random_state=42)
        model.fit(x[idx], df.label.to_numpy()[idx])
        models[phase] = model
    return models


def predict_linear(models, x, df):
    if "all" in models:
        return models["all"].predict_proba(x)[:, 1]
    p = np.zeros(len(df))
    for phase, model in models.items():
        idx = df.phase.eq(phase).to_numpy()
        p[idx] = model.predict_proba(x[idx])[:, 1]
    return p


def split_data(data, split):
    if split == "validation":
        return data["train"].copy(), data["val"].copy()
    # Year is trial-invariant: combined-phase rows cannot cross this boundary.
    tr = data["train"]
    return tr[tr.start_year.fillna(2007) <= 2008].copy(), tr[tr.start_year > 2008].copy()


def prepare(data, split):
    path = OUT / f"features_{split}.joblib"
    if path.exists():
        return joblib.load(path)
    tr, ev = split_data(data, split)
    assert not set(tr.trial_id) & set(ev.trial_id)
    f = Features().fit(tr)
    packet = (tr, ev, f, f.transform(tr), f.transform(ev), f.transform(data["test"]))
    joblib.dump(packet, path, compress=3)
    return packet


def screen():
    data = load_data()
    audit_data(data)
    for split in ("temporal", "validation"):
        tr, ev, features, bt, be, btest = prepare(data, split)
        scores, predictions, tests, configs = [], {}, {}, {}
        scorepath = OUT / f"linear_{split}_scores.csv"
        for rep in REPRESENTATIONS:
            for year in ("none", "year"):
                xt, xe, xx = [matrix(b, rep, year) for b in (bt, be, btest)]
                for C in (.1, .5, 2., 8.):
                    pair, pair_test = [], []
                    for separate in (False, True):
                        start = time.time()
                        models = fit_linear(xt, tr, C, separate)
                        p = predict_linear(models, xe, ev)
                        pt = predict_linear(models, xx, data["test"])
                        pair.append(p); pair_test.append(pt)
                        alpha = float(separate)
                        name = f"lr_{rep}_{year}_C{C:g}_phase{alpha:g}"
                        config = dict(family="lr", representation=rep, year=year, C=C, phase_weight=alpha)
                        scores.append({"name": name, **metrics(ev, p), "seconds": time.time()-start})
                        predictions[name], tests[name], configs[name] = p, pt, config
                    for alpha in (.25, .5, .75):
                        name = f"lr_{rep}_{year}_C{C:g}_phase{alpha:g}"
                        p = (1-alpha)*pair[0] + alpha*pair[1]
                        predictions[name] = p
                        tests[name] = (1-alpha)*pair_test[0] + alpha*pair_test[1]
                        configs[name] = dict(family="lr", representation=rep, year=year, C=C, phase_weight=alpha)
                        scores.append({"name": name, **metrics(ev, p), "seconds": 0.})
                    pd.DataFrame(scores).sort_values("phase_mean_logloss").to_csv(scorepath, index=False)
                    print(split, rep, year, C, 'best', pd.DataFrame(scores).phase_mean_logloss.min(), flush=True)
        np.savez_compressed(OUT / f"linear_{split}_predictions.npz", **predictions)
        np.savez_compressed(OUT / f"linear_{split}_test_predictions.npz", **tests)
        (OUT / "linear_configs.json").write_text(json.dumps(configs, indent=2))
        print(pd.DataFrame(scores).sort_values("phase_mean_logloss").head(12).to_string(index=False), flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--stage", choices=["screen"], default="screen")
    args = parser.parse_args()
    from experiments import screen
    screen()


Overwriting src/experiments.py


## `src/extension_common.py`

SHA-256: `762962c7790e2a22462015417c78b1416fcc2c677f3f91d675b76f2c0d9794b2`

In [30]:
%%writefile src/extension_common.py
"""Source-checked access to the fixed benchmark and previous base predictions."""
import hashlib
import importlib.metadata
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from benchmark_protocol import initialize, resolve, digest, DIRECTORY
from experiments import ROOT
from improvement_report import audit_track

PREVIOUS = ROOT / "experiments/stack_improvements/runs/097ac8161372126d"
CACHE = ROOT / "experiments/stack_improvements/cache/097ac8161372126d"
OUT = ROOT / "experiments/ani_extensions"
REFERENCES = ["ani_stack", "ani_first", "ani_phase3_trend"]


def register(batch, recipes, files):
    universe, protocol = initialize()
    old = json.loads((PREVIOUS / "manifest.json").read_text())
    for f, expected in old["source_sha256"].items():
        assert digest(ROOT / f) == expected, f
    manifest = dict(batch=batch, recipes=recipes, protocol_sha256=digest(DIRECTORY / "protocol.json"),
        design_sha256=digest(ROOT / "EXTENSION_PROTOCOL.md"),
        source_sha256={f: digest(ROOT / f) for f in files+["extension_common.py"]},
        previous_run_sha256=digest(PREVIOUS / "manifest.json"),
        versions={p:importlib.metadata.version(p) for p in ["numpy","pandas","scipy","scikit-learn","joblib"]})
    key = hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()[:16]
    path = OUT / batch / key
    path.mkdir(parents=True,exist_ok=True)
    (path / "manifest.json").write_text(json.dumps(manifest,indent=2))
    (OUT / f"{batch}_current.json").write_text(json.dumps(dict(directory=str(path.relative_to(ROOT))),indent=2))
    return universe, protocol, path


def current(batch):
    return ROOT / json.loads((OUT / f"{batch}_current.json").read_text())["directory"]


def previous_packet(track, split):
    name = split["name"] if isinstance(split,dict) else split
    if (CACHE / track / f"{name}.joblib").exists():
        return joblib.load(CACHE / track / f"{name}.joblib")
    path = PREVIOUS / track / "bases" / f"{name}.npz"
    arrays = np.load(path)
    audit = json.loads(path.with_name(name+"_audit.json").read_text())
    audit["bank"] = {k:{s:arrays[k+"_"+s] for s in ["oof","eval"]}
                     for k in ["unweighted","recent","moderate"]}
    audit["fold_ids"] = arrays["fold_ids"]
    return audit


def controls(track, split, ev):
    probabilities = {}
    for name in REFERENCES:
        path = PREVIOUS / track / "predictions" / split["name"] / f"{name}.csv"
        data = pd.read_csv(path)
        assert data.ID.tolist() == ev.ID.tolist()
        probabilities[name] = data.probability.to_numpy()
    return probabilities


def summary(folder, universe, protocol, names):
    table, scores, _ = audit_track(folder, universe, protocol, names=REFERENCES+list(names))
    baseline = table.loc["ani_stack"]
    table["temporal_delta"] = table.temporal-baseline.temporal
    table["cv_delta"] = table.cv-baseline.cv
    times = scores[scores.kind.eq("temporal")].pivot(index="split",columns="model",values="phase_mean_logloss")
    table["temporal_wins"] = (times.lt(times.ani_stack,axis=0)).sum(axis=0)
    table["eligible"] = ((table.temporal_delta <= -.001) & (table.temporal_wins >= 2)
        & (table.worst <= baseline.worst+.005) & (table.cv <= baseline.cv+.003))
    table.loc[REFERENCES,"eligible"] = False
    table.sort_values("temporal").to_csv(folder / "summary.csv")
    scores.to_csv(folder / "fold_scores.csv",index=False)
    print(table.sort_values("temporal").to_string(),flush=True)
    return table


Overwriting src/extension_common.py


## `src/extension_followthrough.py`

SHA-256: `06a5403476adc9191296b29362f74da150044fe30b99c30fc136982bfca50e0b`

In [31]:
%%writefile src/extension_followthrough.py
"""Freeze the extension shortlist, then check larger time gaps and final inference."""
import os
os.environ.setdefault("OMP_NUM_THREADS","2")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS","2")
os.environ.setdefault("OPENBLAS_NUM_THREADS","2")
import argparse
import json
import time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from ani_models import AniFeatures,BASES,predict_base
from benchmark_protocol import initialize,resolve,digest
from experiments import ROOT,load_data,metrics,PHASES
from improvement_models import fit_bank,fit_recipes as baseline_recipes,meta_inputs as baseline_inputs,RECIPES as BASE_RECIPES
from improvement_report import audit_track,paired_interval
from extension_common import OUT,PREVIOUS,CACHE,REFERENCES,current,previous_packet
from extension_meta import RECIPES as META_RECIPES,fit_metas,inputs as meta_inputs
from extension_neighbors import RECIPES as NN_RECIPES,fit_extended,fit_recipes as neighbor_recipes,neighbor_predictions,x_meta
from local_benchmark import predict_and_score

DECISION=OUT/"decision"


def select():
    DECISION.mkdir(parents=True,exist_ok=True)
    tables=[];inputs={};sources={}
    for batch in ["meta","neighbors"]:
        sources[batch]=str(current(batch).relative_to(ROOT))
        for track in ["strict","published"]:
            path=current(batch)/track/"summary.csv"
            data=pd.read_csv(path);assert data.completed_splits.eq(10).all()
            data["batch"]=batch;data["track"]=track;tables.append(data)
            inputs[batch+"/"+track]=digest(path)
    table=pd.concat(tables,ignore_index=True)
    table.to_csv(DECISION/"all_comparisons.csv",index=False)
    possible=table[table.track.eq("strict")&table.eligible].sort_values("temporal")
    shortlist=[]
    for row in possible.itertuples():
        published=table[(table.track=="published")&(table.model==row.model)&(table.batch==row.batch)].iloc[0]
        if published.temporal_delta>=0:continue
        shortlist.append(dict(model=row.model,batch=row.batch,
                              recipe=(META_RECIPES if row.batch=="meta" else NN_RECIPES)[row.model]))
        if len(shortlist)==2:break
    decision=dict(shortlist=shortlist,source_runs=sources,input_hashes=inputs,
                  policy_sha256=digest(ROOT/"EXTENSION_PROTOCOL.md"),
                  status="Frozen before candidate predictions on gapped windows and final fitting")
    path=DECISION/"selection.json"
    if path.exists():assert json.loads(path.read_text())==decision
    else:path.write_text(json.dumps(decision,indent=2))
    print(json.dumps(decision,indent=2),flush=True)


def selection():return json.loads((DECISION/"selection.json").read_text())


def merge_selected_predictions(track,split,ev):
    result={}
    for item in selection()["shortlist"]:
        path=ROOT/selection()["source_runs"][item["batch"]]/track/"predictions"/split["name"]/f"{item['model']}.csv"
        data=pd.read_csv(path);assert data.ID.tolist()==ev.ID.tolist()
        result[item["model"]]=data.probability.to_numpy()
    for name in REFERENCES:
        data=pd.read_csv(PREVIOUS/track/"predictions"/split["name"]/f"{name}.csv")
        assert data.ID.tolist()==ev.ID.tolist();result[name]=data.probability.to_numpy()
    return result


def compare():
    universe,protocol=initialize();names=REFERENCES+[x["model"] for x in selection()["shortlist"]]
    pairs=[]
    for track in ["strict","published"]:
        folder=DECISION/track;folder.mkdir(parents=True,exist_ok=True)
        for split in protocol["splits"]:
            _,ev=resolve(universe,split)
            predict_and_score(folder,split,ev,merge_selected_predictions(track,split,ev))
        table,scores,frames=audit_track(folder,universe,protocol,names=names)
        table.to_csv(folder/"summary.csv");scores.to_csv(folder/"fold_scores.csv",index=False)
        for item in selection()["shortlist"]:
            for reference in REFERENCES:
                for kind in ["temporal","group_cv"]:
                    pairs.append(dict(track=track,**paired_interval(frames,item["model"],reference,kind)))
    pd.DataFrame(pairs).to_csv(DECISION/"paired_comparisons.csv",index=False)
    labels=pd.read_csv(ROOT/"experiments/model_search/reason_labels_for_analysis.csv")
    from error_analysis import errors_by_reason
    rows=[];phase_rows=[]
    for track in ["strict","published"]:
        _,ev=resolve(universe,next(s for s in protocol["splits"] if s["name"]=="official"))
        for name in names:
            pred=pd.read_csv(DECISION/track/"predictions/official"/f"{name}.csv").probability.to_numpy()
            rows.append(errors_by_reason(ev,pred,labels).assign(track=track,model=name))
            m=metrics(ev,pred)
            for phase in PHASES:
                phase_rows.append(dict(track=track,model=name,phase=phase,
                    **{metric:m[phase+"_"+metric] for metric in ["logloss","f1","roc_auc","pr_auc"]}))
    pd.concat(rows,ignore_index=True).to_csv(DECISION/"reason_errors.csv",index=False)
    pd.DataFrame(phase_rows).to_csv(DECISION/"phase_metrics.csv",index=False)
    (DECISION/"reason_provenance.json").write_text(json.dumps(dict(
        source="experiments/model_search/reason_labels_for_analysis.csv",
        sha256=digest(ROOT/"experiments/model_search/reason_labels_for_analysis.csv"),
        use="Joined after predicting; no reason or embedding input was fitted"),indent=2))
    print("Saved paired comparisons and per-phase/reason diagnostics",flush=True)


def gaps(track):
    universe,_=initialize();decision=selection()
    previous=json.loads((PREVIOUS/"robustness/protocol.json").read_text())
    protocol=dict(splits=previous["splits"],selection_sha256=digest(DECISION/"selection.json"),
                  original_protocol_sha256=digest(PREVIOUS/"robustness/protocol.json"))
    root=DECISION/"gaps";root.mkdir(exist_ok=True)
    path=root/"protocol.json"
    if path.exists():assert json.loads(path.read_text())==protocol
    else:path.write_text(json.dumps(protocol,indent=2))
    folder=root/track;folder.mkdir(exist_ok=True)
    cache=OUT/"cache/decision"/track;cache.mkdir(parents=True,exist_ok=True)
    space=AniFeatures().fit(pd.concat(list(load_data().values()),ignore_index=True)) if track=="published" else None
    for split in protocol["splits"]:
        if (folder/f"scores_{split['name']}.csv").exists():continue
        start=time.time();train,ev=resolve(universe,split)
        old_path=CACHE/track/f"{split['name']}.joblib"
        old=joblib.load(old_path) if old_path.exists() else fit_bank(train,ev,features=space)
        assert old["training_ids"]==train.ID.tolist() and old["evaluation_ids"]==ev.ID.tolist()
        base,_=baseline_recipes(old,train,ev);predictions={k:base[k] for k in REFERENCES}
        extra=None
        if any(x["batch"]=="neighbors" for x in decision["shortlist"]):
            path=cache/f"{split['name']}.joblib"
            if path.exists():extra=joblib.load(path)
            else:
                extra=fit_extended(train,ev,old,space=space,
                    progress=lambda f:print(track,split["name"],"inner",f,round(time.time()-start,1),flush=True))
                joblib.dump(extra,path,compress=3)
            ps,_=neighbor_recipes(old,extra,train,ev,base)
            for item in decision["shortlist"]:
                if item["batch"]=="neighbors":predictions[item["model"]]=ps[item["model"]]
        if any(x["batch"]=="meta" for x in decision["shortlist"]):
            ps,_=fit_metas(old,train,ev,base)
            for item in decision["shortlist"]:
                if item["batch"]=="meta":predictions[item["model"]]=ps[item["model"]]
        predict_and_score(folder,split,ev,predictions)
        audits=folder/"audits";audits.mkdir(exist_ok=True)
        (audits/f"{split['name']}.json").write_text(json.dumps(dict(train_ids=train.ID.tolist(),
            eval_ids=ev.ID.tolist(),extra=extra["audit"] if extra is not None else None)))
    names=REFERENCES+[x["model"] for x in decision["shortlist"]]
    table,scores,frames=audit_track(folder,universe,protocol,names=names)
    table.to_csv(folder/"summary.csv");scores.to_csv(folder/"fold_scores.csv",index=False)
    pairs=[paired_interval(frames,item["model"],ref,"temporal") for item in decision["shortlist"] for ref in REFERENCES]
    pd.DataFrame(pairs).to_csv(folder/"paired_comparisons.csv",index=False)
    print(table.to_string(),flush=True)


def infer(bundle,df):
    old=bundle["old"]
    f=old["features"].transform(df)
    banks={k:{"eval":np.column_stack([predict_base(name,blocks,m,f,df)
        for (name,blocks,_),m in zip(BASES,models)])} for k,models in old["models"].items()}
    packet=dict(bank=banks,year_median=old["year_median"],year_cap=old["year_cap"])
    base={name:bundle["base_metas"][name].predict_proba(baseline_inputs(
        banks,"eval",df,BASE_RECIPES[name],old["year_median"],old["year_cap"]))[:,1]
        for name in REFERENCES}
    extra=bundle.get("extra");ps=dict(base)
    if extra is not None:
        fe=extra["features"].transform(df)
        updated=dict(fast={k:{"eval":np.column_stack([predict_base(name,blocks,m,fe,df)
            for (name,blocks,_),m in zip(BASES,models)])} for k,models in extra["models"].items()},
            neighbors={"eval":neighbor_predictions(extra["neighbor_train_features"],fe,extra["neighbor_train"],df)})
        for name,recipe in NN_RECIPES.items():
            if "blend" in recipe:ps[name]=sum(w*ps[k] for k,w in recipe["blend"].items())
            else:ps[name]=bundle["neighbor_metas"][name].predict_proba(x_meta(packet,updated,df,"eval",recipe))[:,1]
    if bundle.get("meta_metas"):
        for item in bundle["decision"]["shortlist"]:
            if item["batch"]=="meta":
                recipe=META_RECIPES[item["model"]]
                assert recipe["kind"]=="lr","Add explicit nonlinear replay before exporting this family"
                ps[item["model"]]=bundle["meta_metas"][item["model"]].predict_proba(meta_inputs(packet,df,"eval",recipe))[:,1]
    return {k:ps[k] for k in REFERENCES+[x["model"] for x in bundle["decision"]["shortlist"]]}


def final():
    decision=selection();universe,_=initialize();test=load_data()["test"]
    for track in ["strict","published"]:assert (DECISION/"gaps"/track/"summary.csv").exists()
    cache=OUT/"cache/decision/final";cache.mkdir(parents=True,exist_ok=True)
    path=CACHE/"published/final.joblib"
    if path.exists():old=joblib.load(path)
    else:
        features=AniFeatures().fit(pd.concat(list(load_data().values()),ignore_index=True))
        old=fit_bank(universe,test,features=features,retain_models=True)
    base,base_metas=baseline_recipes(old,universe,test)
    bundle=dict(old=old,base_metas=base_metas,decision=decision,extra=None,neighbor_metas=None,meta_metas=None)
    direct={k:base[k] for k in REFERENCES}
    if any(x["batch"]=="neighbors" for x in decision["shortlist"]):
        path=cache/"extra.joblib"
        if path.exists():extra=joblib.load(path)
        else:
            extra=fit_extended(universe,test,old,space=old["features"],retain=True,
                               progress=lambda f:print("final neighbor fold",f,flush=True))
            joblib.dump(extra,path,compress=3)
        ps,metas=neighbor_recipes(old,extra,universe,test,base)
        bundle.update(extra=extra,neighbor_metas=metas)
        for item in decision["shortlist"]:
            if item["batch"]=="neighbors":direct[item["model"]]=ps[item["model"]]
    if any(x["batch"]=="meta" for x in decision["shortlist"]):
        ps,metas=fit_metas(old,universe,test,base);bundle["meta_metas"]=metas
        for item in decision["shortlist"]:
            if item["batch"]=="meta":direct[item["model"]]=ps[item["model"]]
    bundle_path=cache/"submission_bundle.joblib";joblib.dump(bundle,bundle_path,compress=3)
    changed=test.copy();changed["ID"]="unused";changed["trial_id"]="unused";changed["label"]=0
    replay=infer(joblib.load(bundle_path),changed)
    errors={k:float(np.max(np.abs(v-replay[k]))) for k,v in direct.items()}
    assert max(errors.values())<1e-12,errors
    sample=pd.read_csv(ROOT/"part2_sample_submission.csv")
    records=[]
    for i,item in enumerate(decision["shortlist"],start=9):
        name=item["model"]
        frame=pd.DataFrame(dict(ID=test.ID,label=np.clip(direct[name],.001,.999))).set_index("ID").loc[sample.ID].reset_index()
        assert frame.ID.equals(sample.ID) and frame.ID.is_unique and len(frame)==1500
        assert np.isfinite(frame.label).all() and frame.label.between(0,1,inclusive="neither").all()
        dest=ROOT/"submissions"/f"{i:02d}_ani_{name}.csv"
        if dest.exists():assert np.allclose(pd.read_csv(dest).label,frame.label,rtol=0,atol=1e-14)
        else:frame.to_csv(dest,index=False)
        gap_deltas={t:float((lambda d:d.loc[name,"temporal"]-d.loc["ani_stack","temporal"])(
            pd.read_csv(DECISION/"gaps"/t/"summary.csv").set_index("model"))) for t in ["strict","published"]}
        records.append(dict(file=dest.name,model=name,sha256=digest(dest),kaggle_score=None,
            training_rows=9050,recipe=item,gapped_deltas=gap_deltas,
            status="Corroborated locally; Kaggle untested" if max(gap_deltas.values())<0 else "Mixed gapped evidence; exploratory only"))
    manifest=dict(submissions=records,selection_sha256=digest(DECISION/"selection.json"),
                  bundle_path=str(bundle_path.relative_to(ROOT)),bundle_sha256=digest(bundle_path),
                  native_replay_errors=errors,preprocessing="Published covariate-only transduction; supervised train+validation refit")
    (ROOT/"submissions/ani_extensions_manifest.json").write_text(json.dumps(manifest,indent=2))
    (DECISION/"final_manifest.json").write_text(json.dumps(manifest,indent=2))
    print(json.dumps(manifest,indent=2),flush=True)


if __name__=="__main__":
    parser=argparse.ArgumentParser();parser.add_argument("action",choices=["select","compare","gaps","final"])
    parser.add_argument("--track",choices=["strict","published"],default="strict")
    args=parser.parse_args()
    if args.action=="gaps":gaps(args.track)
    else:{"select":select,"compare":compare,"final":final}[args.action]()


Overwriting src/extension_followthrough.py


## `src/extension_meta.py`

SHA-256: `975585971a711aa2bc08d93c097cba2453c490bef8ca6593cdba3acf8b2009b5`

In [32]:
%%writefile src/extension_meta.py
"""Recent calibration and shallow nonlinear stacks on identical saved OOF bases."""
import os
os.environ.setdefault("OMP_NUM_THREADS","2")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS","2")
os.environ.setdefault("OPENBLAS_NUM_THREADS","2")
import argparse
import json
import numpy as np
from scipy.special import expit
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from ani_models import onehot
from improvement_models import recency_weights, meta_inputs
from extension_common import register, previous_packet, controls, summary, resolve
from local_benchmark import predict_and_score

RECIPES = {}
for tau in [1.,1.5,2.]:
    for trend in [False,True]:
        RECIPES[f"meta_tau{tau:g}"+("_trend" if trend else "")] = dict(kind="lr",tau=tau,trend=trend,C=1.)
for drop in [[1,2,4,6],[4],[1,2]]:
    RECIPES["reduced_"+"_".join(map(str,drop))] = dict(kind="lr",tau=3.,trend=False,C=1.,drop=drop)
RECIPES["nonnegative_stack"] = dict(kind="nonnegative",tau=3.,trend=False,C=1.)
RECIPES["nonnegative_trend"] = dict(kind="nonnegative",tau=3.,trend=True,C=1.)
for leaves in [3,7]:
    for year in [False,True]:
        name=f"nonlinear_{leaves}"+("_year" if year else "")
        RECIPES[name] = dict(kind="hgb",tau=3.,trend=False,leaves=leaves,year=year)
        RECIPES[name+"_half"] = dict(blend={"ani_stack":.5,name:.5})


def inputs(packet, df, side, recipe):
    x=meta_inputs(packet["bank"],side,df,dict(banks=["recent"],trend=recipe.get("trend",False)),
                  packet["year_median"],packet["year_cap"])
    if recipe.get("drop"):
        x=np.delete(x,recipe["drop"],axis=1)
    if recipe.get("year"):
        year=np.minimum(df.start_year.fillna(packet["year_median"]),packet["year_cap"]-2)
        x=np.column_stack([x,(year-2007)/3])
    return x


def fit_metas(packet, train, ev, baseline):
    predictions=dict(baseline);models={}
    for name, recipe in RECIPES.items():
        if "blend" in recipe:
            predictions[name]=sum(w*predictions[n] for n,w in recipe["blend"].items());continue
        xt,xe=[inputs(packet,df,side,recipe) for df,side in [(train,"oof"),(ev,"eval")]]
        w=recency_weights(train,recipe["tau"],packet["year_median"])
        if recipe["kind"] == "lr":
            model=LogisticRegression(C=recipe["C"],max_iter=3000).fit(xt,train.label,sample_weight=w)
            p=model.predict_proba(xe)[:,1]
        elif recipe["kind"] == "hgb":
            model=HistGradientBoostingClassifier(max_iter=150,learning_rate=.04,max_leaf_nodes=recipe["leaves"],
                min_samples_leaf=100,l2_regularization=10.,early_stopping=False,random_state=42).fit(xt,train.label,sample_weight=w)
            p=model.predict_proba(xe)[:,1]
        else:
            y=train.label.to_numpy();design=np.column_stack([xt,np.ones(len(xt))])
            def objective(beta):
                z=design@beta
                loss=(w*(np.logaddexp(0,z)-y*z)).sum()+.5*np.dot(beta[:-1],beta[:-1])/recipe["C"]
                grad=design.T@(w*(expit(z)-y));grad[:-1]+=beta[:-1]/recipe["C"]
                return loss,grad
            opt=minimize(objective,np.zeros(design.shape[1]),jac=True,method="L-BFGS-B",
                         bounds=[(0,None)]*8+[(None,None)]*(design.shape[1]-8),
                         options=dict(maxiter=3000,ftol=1e-12,gtol=1e-7))
            assert opt.success,opt.message
            model=opt.x;p=expit(np.column_stack([xe,np.ones(len(xe))])@model)
        predictions[name]=p;models[name]=model
    return predictions,models


def run(track):
    universe,protocol,out=register("meta",RECIPES,["extension_meta.py"])
    folder=out/track;folder.mkdir(exist_ok=True)
    for split in protocol["splits"]:
        if (folder/f"scores_{split['name']}.csv").exists():continue
        train,ev=resolve(universe,split);packet=previous_packet(track,split)
        assert packet["training_ids"]==train.ID.tolist()
        preds,_=fit_metas(packet,train,ev,controls(track,split,ev))
        predict_and_score(folder,split,ev,preds)
        print(track,split["name"],"complete",flush=True)
    summary(folder,universe,protocol,RECIPES)


if __name__=="__main__":
    parser=argparse.ArgumentParser();parser.add_argument("--track",choices=["strict","published"],default="strict")
    run(parser.parse_args().track)


Overwriting src/extension_meta.py


## `src/extension_neighbors.py`

SHA-256: `0a4b3a8b3fb3019144470188fb7e6b5b4208f18063332abe64cd44fca6abcd38`

In [33]:
%%writefile src/extension_neighbors.py
"""Faster recency in Ani's bases and fold-local recent/context neighbors."""
import os
os.environ.setdefault("OMP_NUM_THREADS","2")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS","2")
os.environ.setdefault("OPENBLAS_NUM_THREADS","2")
import argparse
import gc
import json
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from ani_models import AniFeatures, BASES, PHASES, fit_base, predict_base
from improvement_models import recency_weights, meta_inputs
from extension_common import OUT, register, previous_packet, controls, summary, resolve
from experiments import load_data
from local_benchmark import predict_and_score

FAST={"fast15":1.5,"fast2":2.}
NEIGHBORS=[dict(view="criteria",k=k,tau=None,power=1) for k in [10,50,200]]+[
    dict(view="criteria",k=50,tau=3.,power=1),dict(view="criteria",k=50,tau=2.,power=1),
    dict(view="criteria",k=50,tau=3.,power=2),dict(view="context",k=50,tau=3.,power=1),
    dict(view="context",k=200,tau=3.,power=1)]
RECIPES={}
for bank,tau in FAST.items():
    for trend in [False,True]:
        name=bank+("_trend" if trend else "")
        RECIPES[name]=dict(bank=bank,tau=tau,trend=trend,C=1.)
        RECIPES[name+"_half"]=dict(blend={"ani_stack":.5,name:.5})
for ix in [1,3,4,5,6]:
    RECIPES[f"replace_knn{ix}"]=dict(bank="recent",tau=3.,trend=False,C=1.,replace=ix)
for bank,tau in [("recent",3.),("unweighted",None),("fast2",2.)]:
    name=f"{bank}_neighbors"
    RECIPES[name]=dict(bank=bank,tau=tau,trend=False,C=.3,neighbors=True)
    RECIPES[name+"_trend"]=dict(bank=bank,tau=tau,trend=True,C=.3,neighbors=True)
    if bank!="unweighted":
        RECIPES[name+"_half"]=dict(blend={"ani_stack":.5,name:.5})


def neighbor_predictions(fa,fb,train,ev):
    predictions=np.zeros((len(ev),len(NEIGHBORS)))
    y=train.label.to_numpy();years=train.start_year.fillna(train.start_year.median()).to_numpy()
    for phase in PHASES:
        a=np.flatnonzero(train.phase.eq(phase));b=np.flatnonzero(ev.phase.eq(phase))
        if not len(b):continue
        assert len(a)>0
        criterion=(fb["knn"][b]@fa["knn"][a].T).toarray()
        context=.5*criterion
        for field,scale in [("disease",.2),("drug",.2),("icd",.1)]:
            context+=scale*(fb[field][b]@fa[field][a].T).toarray()
        for j,spec in enumerate(NEIGHBORS):
            sims=criterion if spec["view"]=="criteria" else context
            k=min(spec["k"],len(a))
            indices=np.argpartition(-sims,k-1,axis=1)[:,:k]
            sim=np.take_along_axis(sims,indices,axis=1)
            if spec["power"]>1:
                sim=(sim/np.maximum(sim.max(axis=1,keepdims=True),1e-12))**spec["power"]
            w=np.ones(len(a)) if spec["tau"] is None else np.exp((years[a]-years[a].max())/spec["tau"])
            w=w/w.mean()
            prior=(w@y[a]+1)/(w.sum()+2)
            weights=sim*w[indices]
            predictions[b,j]=((weights*y[a][indices]).sum(axis=1)+2*prior)/(weights.sum(axis=1)+2)
    assert np.isfinite(predictions).all() and ((predictions>0)&(predictions<1)).all()
    return predictions


def fit_extended(train,ev,old,space=None,retain=False,progress=None):
    strict=space is None
    outer=AniFeatures().fit(train) if strict else space
    w={k:recency_weights(train,tau,outer.year_median) for k,tau in FAST.items()}
    fast={k:dict(oof=np.zeros((len(train),8)),eval=np.zeros((len(ev),8))) for k in FAST}
    neighbors=dict(oof=np.zeros((len(train),len(NEIGHBORS))),eval=np.zeros((len(ev),len(NEIGHBORS))))
    audits=[]
    for fold,(a,b) in enumerate(GroupKFold(5).split(train,groups=train.trial_id)):
        assert not set(train.trial_id.iloc[a])&set(train.trial_id.iloc[b])
        features=AniFeatures().fit(train.iloc[a]) if strict else outer
        if strict:
            assert set(features.fit_ids)==set(train.ID.iloc[a])
            assert not set(features.fit_ids)&(set(ev.ID)|set(train.ID.iloc[b]))
        fa,fb=features.transform(train.iloc[a]),features.transform(train.iloc[b])
        for j,(name,blocks,C) in enumerate(BASES):
            for key in FAST:
                if name=="criteria kNN":
                    fast[key]["oof"][b,j]=old["bank"]["recent"]["oof"][b,j];continue
                m=fit_base(name,blocks,C,fa,train.iloc[a],w[key][a])
                fast[key]["oof"][b,j]=predict_base(name,blocks,m,fb,train.iloc[b])
        neighbors["oof"][b]=neighbor_predictions(fa,fb,train.iloc[a],train.iloc[b])
        audits.append(dict(fold=fold,fit_ids=train.ID.iloc[a].tolist(),eval_ids=train.ID.iloc[b].tolist(),
                           preprocessing_fit_ids=features.fit_ids))
        del features,fa,fb,m;gc.collect()
        if progress:progress(fold+1)
    fa,fb=outer.transform(train),outer.transform(ev)
    models={k:[] for k in FAST}
    for j,(name,blocks,C) in enumerate(BASES):
        for key in FAST:
            if name=="criteria kNN":
                fast[key]["eval"][:,j]=old["bank"]["recent"]["eval"][:,j]
                if retain:models[key].append(fit_base(name,blocks,C,fa,train,w[key]))
                continue
            m=fit_base(name,blocks,C,fa,train,w[key]);fast[key]["eval"][:,j]=predict_base(name,blocks,m,fb,ev)
            if retain:models[key].append(m)
    neighbors["eval"]=neighbor_predictions(fa,fb,train,ev)
    packet=dict(fast=fast,neighbors=neighbors,audit=audits,train_ids=train.ID.tolist(),eval_ids=ev.ID.tolist())
    if retain:packet.update(features=outer,models=models,neighbor_train_features=fa,neighbor_train=train)
    return packet


def x_meta(old,extra,df,side,recipe):
    bank=dict(old["bank"]);bank.update(extra["fast"])
    if "replace" in recipe:
        bank=dict(bank);bank[recipe["bank"]]={k:v.copy() for k,v in bank[recipe["bank"]].items()}
        bank[recipe["bank"]][side][:,7]=extra["neighbors"][side][:,recipe["replace"]]
    x=meta_inputs(bank,side,df,dict(banks=[recipe["bank"]],trend=recipe["trend"]),old["year_median"],old["year_cap"])
    if recipe.get("neighbors"):
        z=np.clip(extra["neighbors"][side],1e-4,1-1e-4);x=np.column_stack([x,np.log(z/(1-z))])
    return x


def fit_recipes(old,extra,train,ev,baseline):
    predictions=dict(baseline);models={}
    for name,recipe in RECIPES.items():
        if "blend" in recipe:
            predictions[name]=sum(w*predictions[k] for k,w in recipe["blend"].items());continue
        xt,xe=[x_meta(old,extra,d,s,recipe) for d,s in [(train,"oof"),(ev,"eval")]]
        w=recency_weights(train,recipe["tau"],old["year_median"])
        model=LogisticRegression(C=recipe["C"],max_iter=3000).fit(xt,train.label,sample_weight=w)
        predictions[name]=model.predict_proba(xe)[:,1];models[name]=model
    return predictions,models


def run(track):
    universe,protocol,out=register("neighbors",dict(models=RECIPES,neighbors=NEIGHBORS,fast=FAST),["extension_neighbors.py"])
    folder=out/track;folder.mkdir(exist_ok=True)
    cache=OUT/"cache"/out.name/track;cache.mkdir(parents=True,exist_ok=True)
    space=AniFeatures().fit(pd.concat(list(load_data().values()),ignore_index=True)) if track=="published" else None
    for split in protocol["splits"]:
        if (folder/f"scores_{split['name']}.csv").exists():continue
        start=time.time();train,ev=resolve(universe,split);old=previous_packet(track,split)
        assert old["training_ids"]==train.ID.tolist()
        path=cache/f"{split['name']}.joblib"
        if path.exists():extra=joblib.load(path)
        else:
            extra=fit_extended(train,ev,old,space=space,
                progress=lambda f:print(track,split["name"],"inner",f,round(time.time()-start,1),flush=True))
            joblib.dump(extra,path,compress=3)
        predictions,_=fit_recipes(old,extra,train,ev,controls(track,split,ev))
        predict_and_score(folder,split,ev,predictions)
        arrays=folder/"bases";arrays.mkdir(exist_ok=True)
        np.savez_compressed(arrays/f"{split['name']}.npz",**{k+"_"+s:v for k,p in extra["fast"].items() for s,v in p.items()},
            **{"neighbors_"+s:v for s,v in extra["neighbors"].items()})
        (arrays/f"{split['name']}_audit.json").write_text(json.dumps(extra["audit"]))
        print(track,split["name"],"complete",round(time.time()-start,1),flush=True)
    summary(folder,universe,protocol,RECIPES)


if __name__=="__main__":
    parser=argparse.ArgumentParser();parser.add_argument("--track",choices=["strict","published"],default="strict")
    run(parser.parse_args().track)


Overwriting src/extension_neighbors.py


## `src/extension_predict.py`

SHA-256: `fef63f8b77716e19a6a76e96e1faa196cb8e6cd4ad8bfac9ffad12f263472c5b`

In [34]:
%%writefile src/extension_predict.py
"""Replay a frozen extension bundle on an outcome-format CSV."""
import argparse
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from experiments import ROOT
from extension_followthrough import infer


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument("--input",type=Path,required=True)
    parser.add_argument("--output",type=Path,required=True)
    parser.add_argument("--model",default="fast2_neighbors_half")
    parser.add_argument("--bundle",type=Path)
    args=parser.parse_args()
    if args.bundle is None:
        manifest=json.loads((ROOT/"submissions/ani_extensions_manifest.json").read_text())
        args.bundle=ROOT/manifest["bundle_path"]
    if not args.bundle.exists():
        raise FileNotFoundError("Rebuild the native bundle with extension_followthrough.py final first")
    data=pd.read_csv(args.input)
    assert data.ID.is_unique and set(data.phase).issubset({"I","II","III"})
    predictions=infer(joblib.load(args.bundle),data)
    assert args.model in predictions,f"Available models: {list(predictions)}"
    result=pd.DataFrame(dict(ID=data.ID,label=np.clip(predictions[args.model],.001,.999)))
    assert np.isfinite(result.label).all()
    result.to_csv(args.output,index=False)
    print("Wrote",len(result),"predictions to",args.output)


if __name__=="__main__":main()


Overwriting src/extension_predict.py


## `src/extension_report.py`

SHA-256: `6cbc4a17afb5d78deb3cc2710bce5edc0a2d56e294603c0eaa4c5507327a6286`

In [35]:
%%writefile src/extension_report.py
"""Write the extension findings and an executed-notebook source bundle."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import nbformat
from experiments import ROOT
from extension_common import OUT,REFERENCES,current
from extension_followthrough import DECISION,selection
from extension_meta import RECIPES as META_RECIPES
from extension_neighbors import RECIPES as NN_RECIPES


def table(df):
    headers=list(df.columns)
    rows=[[f"{v:.6f}" if isinstance(v,(float,np.floating)) else str(v) for v in row] for row in df.to_numpy()]
    return "\n".join(["| "+" | ".join(headers)+" |","| "+" | ".join(["---"]*len(headers))+" |"]+
                     ["| "+" | ".join(row)+" |" for row in rows])


def build():
    decision=selection();names=REFERENCES+[x["model"] for x in decision["shortlist"]]
    manifest=json.loads((DECISION/"final_manifest.json").read_text())
    verification=json.loads((DECISION/"verification.json").read_text())
    ledger=json.loads((ROOT/"submissions/leaderboard_results.json").read_text())
    score_labels={}
    for item in manifest["submissions"]:
        record=ledger["submissions"].get(item["file"])
        if record: assert record["sha256"]==item["sha256"]
        score_labels[item["file"]]=f"{record['kaggle_score']:.5f} (user-reported)" if record else "unknown"
    score_note=("**File 10 Kaggle result: "+score_labels["10_ani_fast2_neighbors_half.csv"]+
        ".** File 09's Kaggle score is "+score_labels["09_ani_replace_knn5.csv"]+
        ". The current ledger records results reported after the frozen export manifests.")
    primary="fast2_neighbors_half";sharp="replace_knn5"
    sources={t:pd.read_csv(DECISION/t/"summary.csv").set_index("model") for t in ["strict","published"]}
    gaps={t:pd.read_csv(DECISION/"gaps"/t/"summary.csv").set_index("model") for t in sources}
    paired=pd.read_csv(DECISION/"paired_comparisons.csv")
    figures=DECISION/"figures";figures.mkdir(exist_ok=True)
    plt.rcParams.update({"font.size":10,"axes.spines.top":False,"axes.spines.right":False})
    fig,axes=plt.subplots(1,2,figsize=(13,4.5),constrained_layout=True)
    for ax,track in zip(axes,sources):
        normal=pd.read_csv(DECISION/track/"fold_scores.csv")
        normal=normal[normal.kind.eq("temporal")].pivot(index="split",columns="model",values="phase_mean_logloss")
        gap=pd.read_csv(DECISION/"gaps"/track/"fold_scores.csv").pivot(index="split",columns="model",values="phase_mean_logloss")
        d=pd.concat([normal,gap])
        for name,color in [(primary,"#12775d"),(sharp,"#7851a9")]:
            ax.plot(np.arange(len(d)),d[name]-d.ani_stack,marker="o",label=name,color=color)
        ax.axhline(0,color="#666666",linewidth=1)
        ax.axvline(2.5,color="#cccccc",linestyle="--")
        ax.set_xticks(np.arange(6),["08–09","10–11","12–13","Gap 08–09","Gap 10–11","Gap 12–13"],rotation=20)
        ax.set(title=track.capitalize()+" preprocessing",ylabel="Log-loss difference from Ani (negative is better)")
        ax.grid(alpha=.15)
    axes[0].legend(fontsize=8)
    fig.savefig(figures/"chronological_differences.png",dpi=180);plt.close(fig)
    sections=["# New Part Two candidates: recent, sharper neighbors",
        score_note,
        "File 10 improves all six chronological windows under both preprocessing policies, with a small grouped-CV improvement over Ani's 0.58688 model. Its reported Kaggle score of 0.58609 improves that reference by 0.00079. At that stage, the user reported second place behind 0.58526, a gap of 0.00083. File 10 became the frozen reference for the subsequent sprint; see the current Part Two README and ledger for later results. File 09 has the strongest improvement on the original three forward windows, but a small grouped-CV regression and no reported Kaggle score.",
        "## What the earlier search missed\n\nThe first round tested slower recency and many calibration changes while retaining Ani's nearest-neighbor estimator. That estimator ignores recency, uses linear text similarities, and shrinks toward a fixed 50% success rate. The previous promotion rule also demanded near-best random CV from the unweighted model, despite its weaker chronological and reported Kaggle results. This round preserves that history and uses a separately declared rule that prioritizes later-cohort prediction. No old result was relabeled as new evidence.",
        "## Changes that helped\n\n**File 09** retains Ani's seven other base estimators and replaces only the kNN input before refitting his logistic stack. For the 50 most similar trials in the same phase, it squares similarity relative to the closest match, weights recent trials more strongly (exponential tau=3), and shrinks toward a smoothed training-only success prior for that phase. All neighbor labels, priors, vocabularies and IDF values are rebuilt inside each grouped inner fold.\n\n**File 10** averages Ani's original stack with a tau=2 stack that also receives eight neighborhood estimates: several neighborhood sizes, recent weighting, sharper text similarity, and combined criteria/disease/drug/ICD similarity. The probability blend is fixed at 50/50. Its OOF meta-model is regularized with C=0.3.",
        "For file 09, neighbor weights are `(similarity / best_similarity)^2 × recency_weight`. Recency weights have mean one within the fitting phase. The phase prior is `(weighted_successes + 1) / (total_weight + 2)`, and the neighbor probability is `(weighted_neighbor_successes + 2 × prior) / (neighbor_weight + 2)`. With no overlapping text, prediction falls back to the fitted phase prior.",
        "## Identical-split comparison\n\nLower log loss is better. All tables score the exact same held-out trials, with equal phase weighting. Random grouped CV is pooled within each repeat. `official` is the supplied validation set; `temporal` averages the three original forward windows; `worst` is the worst forward window."]
    for track in sources:
        sections.append("### "+track.capitalize()+"\n\n"+table(sources[track].loc[names,["official","temporal","worst","cv"]].reset_index()))
    sections.append("Both proposals improve the original forward windows against the recency winner in all three periods. The supplied validation score worsens, and file 09's CV worsens. Those trade-offs are retained in the table rather than hidden. Ani's unweighted winner still has lower grouped-CV loss than these candidates; its chronological loss is higher.")
    sections.append("## Frozen-shortlist checks at larger time gaps\n\nThe two recipes were frozen before scoring their predictions on the three larger-gap tests: train through 2005/evaluate 2008–2009, through 2007/evaluate 2010–2011, and through 2009/evaluate 2012–2013. These labels were used in earlier research; the checks are robustness evidence, not an untouched independent test.")
    for track in gaps:
        sections.append("### "+track.capitalize()+"\n\n"+table(gaps[track].loc[names,["temporal","worst"]].reset_index()))
    sections.append("File 10 wins all three additional windows in both tracks. File 09 improves their average but slightly worsens the first gap. Ani's unsubmitted Phase III trend control remains better on the *average of these longer-gap tests*; the new candidates are better on the original forward windows. This prevents claiming dominance over every alternative.")
    sections.append("![Window-by-window differences](decision/figures/chronological_differences.png)")
    sections.append("## Uncertainty\n\nPaired trial bootstraps share weights across phase rows and average repeated-CV losses before resampling. The chronological confidence intervals versus Ani include zero. The local intervals alone do not establish a Kaggle improvement; the later reported score above is separate evidence. Selection across this and earlier research is adaptive, and the intervals do not correct for it.\n\n"+table(paired[(paired.track=="strict")&(paired.reference=="ani_stack")][["model","kind","delta","lower","upper"]]))
    sections.append("All paired comparisons, including against the earlier 0.58810 winner, are in [paired_comparisons.csv](decision/paired_comparisons.csv). Larger-gap intervals are stored separately under each track's `gaps` directory.")
    phases=pd.read_csv(DECISION/"phase_metrics.csv")
    sections.append("## Required per-phase and stopping-reason diagnostics\n\nSupplied-validation metrics for the recommended candidate and Ani's recency model, under strict preprocessing:\n\n"+
        table(phases[(phases.track=="strict")&phases.model.isin(["ani_stack",primary])].drop(columns="track")))
    reasons=pd.read_csv(DECISION/"reason_errors.csv")
    a=reasons[(reasons.track=="strict")&(reasons.model=="ani_stack")]
    b=reasons[(reasons.track=="strict")&(reasons.model==primary)]
    keys=["phase","reason","n","true_reason_rows","predicted_reason_rows"]
    combined=a[keys+["false_success"]].merge(b[keys+["false_success"]],on=keys,suffixes=("_ani","_candidate"))
    sections.append("Counts of unsuccessful validation trials incorrectly predicted successful at cutoff 0.5:\n\n"+table(combined))
    sections.append("Reason tags are joined only after predicting. Visible true tags and frozen Part One predictions are distinguished; categories overlap and small groups are uncertain. Full rates and Wilson intervals are in [reason_errors.csv](decision/reason_errors.csv). No identifiers, stopping-reason labels or reason embeddings enter a predictor.")
    total=len(META_RECIPES)+len(NN_RECIPES)
    sections.append(f"## Reproduction and complete record\n\nThis round tested **{total} extension recipes**, each on ten frozen splits under two preprocessing policies: **{verification['new_recipe_evaluations']} new recipe evaluations**, plus unchanged controls. All proposals, including failures, are in [all_comparisons.csv](decision/all_comparisons.csv). The shallow nonlinear stacks did not improve temporal loss; faster recency alone also produced mixed results. The neighbor changes supply the useful gain.\n\n[Selection and source manifests](decision/selection.json) preserve recipe definitions, hashes and source runs. [Verification](decision/verification.json) checks fitting boundaries, all saved scores, and unchanged controls. Native model replay has maximum probability difference {verification['native_replay_max_error']:.2g}; changing evaluation labels/IDs does not alter inference. All 16 unit checks passed.")
    sections.append("Final fits use all 9,050 labeled rows and the same feature-only transductive preprocessing as Ani. Each CSV has exactly 1,500 unique IDs in sample-submission order, finite probabilities, and columns `ID,label`. The original Ani files remain intact.\n\n"+
        "\n".join(f"- [{r['file']}](../../submissions/{r['file']}): Kaggle score {score_labels[r['file']]}; SHA-256 `{r['sha256']}`." for r in manifest["submissions"]))
    commands="""```sh
# From repository root, with part2/requirements-benchmark.txt installed in .venv:
.venv/bin/python part2/extension_meta.py --track strict
.venv/bin/python part2/extension_meta.py --track published
.venv/bin/python part2/extension_neighbors.py --track strict
.venv/bin/python part2/extension_neighbors.py --track published
.venv/bin/python part2/extension_followthrough.py select
.venv/bin/python part2/extension_followthrough.py compare
.venv/bin/python part2/extension_followthrough.py gaps --track strict
.venv/bin/python part2/extension_followthrough.py gaps --track published
.venv/bin/python part2/extension_followthrough.py final
.venv/bin/python part2/extension_verify.py
.venv/bin/python -m unittest discover -s part2 -p 'test_*.py'
.venv/bin/python part2/extension_predict.py --input part2/outcome_test.csv --output /tmp/replayed.csv
```"""
    sections.append(commands+"\n\nNative caches are regenerable and ignored by Git. Existing OOF arrays and fitting IDs permit replay without retraining the original base models. If the old final native cache is missing, final fitting rebuilds it. [Notebook with embedded implementation source](../../ani_extensions.ipynb).")
    (OUT/"RESULTS.md").write_text("\n\n".join(sections)+"\n")
    notebook=nbformat.v4.new_notebook()
    notebook.metadata.kernelspec=dict(name="part2",display_name="Python (Part Two)",language="python")
    notebook.cells=[nbformat.v4.new_markdown_cell("# Extending Ani's outcome stack\n\n"+score_note+" Local chronological intervals include zero. Training code is embedded below; default execution reads the saved results."),
        nbformat.v4.new_code_cell("from pathlib import Path\nimport json\nimport pandas as pd\nfrom IPython.display import display, Image\nroot = Path.cwd()\nif (root / 'part2').exists(): root /= 'part2'\nout = root / 'experiments/ani_extensions'\ndecision = out / 'decision'\ndisplay(json.loads((decision / 'selection.json').read_text()))"),
        nbformat.v4.new_code_cell("for track in ['strict','published']:\n    print(track)\n    display(pd.read_csv(decision / track / 'summary.csv').round(6))\n    print('Larger time gaps')\n    display(pd.read_csv(decision / 'gaps' / track / 'summary.csv').round(6))\ndisplay(Image(filename=str(decision / 'figures/chronological_differences.png')))"),
        nbformat.v4.new_code_cell("pairs=pd.read_csv(decision / 'paired_comparisons.csv')\ndisplay(pairs[(pairs.track=='strict') & (pairs.reference=='ani_stack')].round(6))\ndisplay(pd.read_csv(decision / 'phase_metrics.csv').round(6))\ndisplay(pd.read_csv(decision / 'reason_errors.csv').round(4))\ndisplay(json.loads((decision / 'verification.json').read_text()))\ndisplay(json.loads((decision / 'final_manifest.json').read_text()))"),
        nbformat.v4.new_markdown_cell("## Full implementation source\n\nThe source bundle below includes the estimators, feature extraction, benchmark orchestration, inference and verification code. It is stored as strings so normal execution does not launch training. The repository supplies CSVs, frozen splits and source-checksummed previous OOF arrays. No external labeled data is retrieved."),
        nbformat.v4.new_code_cell("SOURCES = {}")]
    files=["ani_models.py","experiments.py","benchmark_protocol.py","benchmark_models.py","local_benchmark.py","improvement_models.py","improvement_benchmark.py",
           "improvement_report.py","error_analysis.py","extension_common.py","extension_meta.py","extension_neighbors.py",
           "extension_followthrough.py","extension_predict.py","extension_verify.py","test_extensions.py","extension_report.py"]
    for filename in files:
        source=(ROOT/filename).read_text()
        if "'''" not in source:
            code=f"# {filename}\nSOURCES[{filename!r}] = r'''"+source+"'''"
        else:
            code=f"# {filename}\nSOURCES[{filename!r}] = (\n"+"\n".join(json.dumps(line,ensure_ascii=False) for line in source.splitlines(keepends=True))+"\n)"
        notebook.cells.append(nbformat.v4.new_code_cell(code))
    notebook.cells.append(nbformat.v4.new_code_cell("for filename, source in SOURCES.items():\n    assert source == (root / filename).read_text(), filename\n    compile(source, filename, 'exec')\nprint('Embedded source files verified:', len(SOURCES))"))
    notebook.cells.append(nbformat.v4.new_markdown_cell("## Re-run experiments\n\n"+commands+"\n\nThe original comparison rule is preserved in `IMPROVEMENT_PROTOCOL.md`; this round's decision policy is separately recorded in `EXTENSION_PROTOCOL.md`. The changed policy is not evidence of better performance by itself; compare the actual fixed-split and gapped predictions."))
    nbformat.write(notebook,ROOT/"ani_extensions.ipynb")
    print("Wrote",OUT/"RESULTS.md",flush=True)


if __name__=="__main__":build()


Overwriting src/extension_report.py


## `src/extension_verify.py`

SHA-256: `e86a721d0d02993a47cf72844247de005678a141a6e46518b35ad9fc90bf18ff`

In [36]:
%%writefile src/extension_verify.py
"""Independently check all scored extensions, fitting boundaries and exports."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
from benchmark_protocol import initialize,resolve,digest
from experiments import ROOT
from improvement_report import audit_track
from extension_common import current,OUT,PREVIOUS,REFERENCES,previous_packet
from extension_meta import RECIPES as META_RECIPES
from extension_neighbors import RECIPES as NN_RECIPES
from extension_followthrough import DECISION,selection


def verify():
    universe,protocol=initialize();decision=selection()
    assert digest(ROOT/"EXTENSION_PROTOCOL.md")==decision["policy_sha256"]
    checked=0
    for batch,registry in [("meta",META_RECIPES),("neighbors",NN_RECIPES)]:
        root=current(batch);manifest=json.loads((root/"manifest.json").read_text())
        for source,expected in manifest["source_sha256"].items():assert digest(ROOT/source)==expected,source
        for track in ["strict","published"]:
            assert digest(root/track/"summary.csv")==decision["input_hashes"][batch+"/"+track]
            audit_track(root/track,universe,protocol,names=REFERENCES+list(registry))
            checked+=len(protocol["splits"])*len(registry)
            for split in protocol["splits"]:
                train,ev=resolve(universe,split)
                for name in REFERENCES:
                    original=pd.read_csv(PREVIOUS/track/"predictions"/split["name"]/f"{name}.csv")
                    new=pd.read_csv(root/track/"predictions"/split["name"]/f"{name}.csv")
                    assert original.ID.equals(new.ID)
                    np.testing.assert_allclose(original.probability,new.probability,rtol=0,atol=1e-14)
                if batch=="neighbors":
                    audits=json.loads((root/track/"bases"/f"{split['name']}_audit.json").read_text())
                    old=previous_packet(track,split)
                    for fold in audits:
                        ix=np.flatnonzero(old["fold_ids"]==fold["fold"])
                        assert fold["eval_ids"]==train.ID.iloc[ix].tolist()
                        assert not set(fold["fit_ids"])&set(fold["eval_ids"])
                        held_trials=set(train.set_index("ID").loc[fold["eval_ids"],"trial_id"])
                        fit_trials=set(train.set_index("ID").loc[fold["fit_ids"],"trial_id"])
                        assert not held_trials&fit_trials
                        if track=="strict":
                            assert set(fold["preprocessing_fit_ids"])==set(fold["fit_ids"])
                            assert not set(fold["preprocessing_fit_ids"])&set(ev.ID)
    names=REFERENCES+[x["model"] for x in decision["shortlist"]]
    gap_protocol=json.loads((DECISION/"gaps/protocol.json").read_text())
    assert gap_protocol["selection_sha256"]==digest(DECISION/"selection.json")
    for track in ["strict","published"]:
        audit_track(DECISION/track,universe,protocol,names=names)
        audit_track(DECISION/"gaps"/track,universe,gap_protocol,names=names)
    manifest=json.loads((DECISION/"final_manifest.json").read_text())
    assert digest(ROOT/manifest["bundle_path"])==manifest["bundle_sha256"]
    assert max(manifest["native_replay_errors"].values())<1e-12
    sample=pd.read_csv(ROOT/"part2_sample_submission.csv")
    ledger=json.loads((ROOT/"submissions/leaderboard_results.json").read_text())
    scores={}
    for item in manifest["submissions"]:
        path=ROOT/"submissions"/item["file"]
        # Export manifests are frozen before any later leaderboard results.
        assert digest(path)==item["sha256"] and item["kaggle_score"] is None
        record=ledger["submissions"].get(item["file"])
        if record is not None: assert record["sha256"]==item["sha256"]
        scores[item["file"]]=record["kaggle_score"] if record else None
        data=pd.read_csv(path)
        assert data.ID.equals(sample.ID) and data.ID.is_unique and len(data)==1500
        assert data.label.between(0,1,inclusive="neither").all()
    result=dict(extension_recipes=len(META_RECIPES)+len(NN_RECIPES),new_recipe_evaluations=checked,
        original_splits_per_track=10,preprocessing_tracks=2,controls_reproduce=True,
        source_and_input_hashes_match=True,grouped_inner_preprocessing_verified=True,
        independently_rescored=True,gapped_splits_per_track=3,exported_submissions=len(manifest["submissions"]),
        native_replay_max_error=max(manifest["native_replay_errors"].values()),
        labels_and_identifiers_do_not_change_inference=True,
        kaggle_scores_unknown=all(value is None for value in scores.values()),
        reported_kaggle_scores=scores,leaderboard_sha256=digest(ROOT/"submissions/leaderboard_results.json"))
    (DECISION/"verification.json").write_text(json.dumps(result,indent=2))
    print(json.dumps(result,indent=2))


if __name__=="__main__":verify()


Overwriting src/extension_verify.py


## `src/followup_ensembles.py`

SHA-256: `822ea9b6fd39d4c4e84e9001eb37126ad1bdc6792291a10351f83fd0249c4453`

In [37]:
%%writefile src/followup_ensembles.py
"""Two controlled prospective follow-ups after the first two Kaggle results.

Reuse the original fitted models without retraining or changing earlier CSVs.
Phase I stays exactly as in 02; change only tree fractions in Phases II/III.
"""
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
from experiments import ROOT,OUT,PHASES,load_data,split_data,metrics
from ensemble_search import collect,weighted
from submission_models import predict,reported_kaggle_score


def run():
    data=load_data();pred,tests,_=collect()
    original=joblib.load(OUT/'submission_models.joblib')
    prior_hashes={f.name:hashlib.sha256(f.read_bytes()).hexdigest() for f in (ROOT/'submissions').glob('0[1-4]_*.csv')}
    linear={'recency_clinical_C0.5_half3':.25,'recency_criteria_C2_half6':.375,'text_wordwide_C2':.375}
    tree={'catboost_d6_n1000_none':.5,'catboost_d8_n1000_none':.5}
    policies={'05_text_later_phases':(.25,0.,0.),'06_lighter_tree_blend':(.25,.125,.125)}
    ensembles={}
    for name,alphas in policies.items():
        ensembles[name]={k:{phase:(1-a)*linear.get(k,0)+a*tree.get(k,0) for phase,a in zip(PHASES,alphas)} for k in set(linear)|set(tree)}
    followup=dict(original,ensembles=ensembles)
    followup['components']={k:v for k,v in original['components'].items() if k in set(linear)|set(tree)}
    joblib.dump(followup,OUT/'followup_models.joblib',compress=3)
    reloaded=joblib.load(OUT/'followup_models.joblib')
    vp=predict(reloaded,data['val']);tp=predict(reloaded,data['test'])
    sample=pd.read_csv(ROOT/'part2_sample_submission.csv')
    previous=pd.read_csv(ROOT/'submissions/02_text_heavy.csv')
    assert previous.ID.equals(data['test'].ID)
    temporal=split_data(data,'temporal')[1]
    scores=[];details=[];hist={}
    for name,weights in ensembles.items():
        for split,df,new,bank in [('validation',data['val'],vp[name],pred['validation']),('test',data['test'],tp[name],tests['validation'])]:
            expected=weighted(bank,weights,df.phase)
            assert np.max(np.abs(new-expected))<1e-8
        p=tp[name]
        assert np.isfinite(p).all() and ((p>0)&(p<1)).all()
        assert np.max(np.abs(p[data['test'].phase.eq('I')]-previous.label[data['test'].phase.eq('I')]))<5.1e-11
        output=sample[['ID']].merge(pd.DataFrame({'ID':data['test'].ID,'label':p}),on='ID',validate='one_to_one')
        assert len(output)==1500 and output.ID.equals(sample.ID) and output.ID.is_unique
        path=ROOT/'submissions'/f'{name}.csv';output.to_csv(path,index=False,float_format='%.10f')
        disk=pd.read_csv(path);assert disk.columns.tolist()==['ID','label']
        assert disk.label.between(0,1,inclusive='neither').all() and np.max(abs(disk.label-p))<5.1e-11
        h=weighted(pred['temporal'],weights,temporal.phase);hist[name]=h
        a,b=metrics(data['val'],vp[name]),metrics(temporal,h)
        sha=hashlib.sha256(path.read_bytes()).hexdigest()
        scores.append(dict(submission=path.name,validation_logloss=a['phase_mean_logloss'],temporal_logloss=b['phase_mean_logloss'],
            mean_test_probability=float(p.mean()),sha256=sha,kaggle_score=reported_kaggle_score(path.name,sha)))
        for phase in PHASES:details.append(dict(submission=path.name,phase=phase,tree_fraction=policies[name][PHASES.index(phase)],
            validation_logloss=a[phase+'_logloss'],temporal_logloss=b[phase+'_logloss']))
    for name,sha in prior_hashes.items():assert hashlib.sha256((ROOT/'submissions'/name).read_bytes()).hexdigest()==sha
    np.savez_compressed(OUT/'followup_validation_predictions.npz',**vp)
    np.savez_compressed(OUT/'followup_temporal_predictions.npz',**hist)
    table=pd.DataFrame(scores);table.to_csv(ROOT/'submissions/followup_index.csv',index=False)
    pd.DataFrame(details).to_csv(OUT/'followup_phase_metrics.csv',index=False)
    (ROOT/'submissions/followup_manifest.json').write_text(json.dumps({
        'motivation':'03 Kaggle 0.59562; 02 Kaggle 0.58993; reference 0.58688. Test less tree influence in later phases; no guarantee of improvement.',
        'training':'Reuse the original models fitted on outcome_train.csv only; no refitting or new label inputs.',
        'first_comparison':'05_text_later_phases.csv; reserve 06 until 05 is scored.',
        'ensembles':ensembles,'files':scores,'original_four_preserved':True},indent=2))
    print(table.to_string(index=False))

if __name__=='__main__':run()


Overwriting src/followup_ensembles.py


## `src/improvement_benchmark.py`

SHA-256: `b25a472ef73f8c82007bc79997b4a05dadfa357dc17388d941f86eb2b284e88d`

In [38]:
%%writefile src/improvement_benchmark.py
"""Both Ani winners and a frozen, small proposal batch on the existing protocol."""
from __future__ import annotations
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "2")
import argparse
import hashlib
import importlib.metadata
import json
import time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from ani_models import AniFeatures
from benchmark_protocol import initialize, resolve, digest, DIRECTORY
from experiments import ROOT, load_data
from improvement_models import RECIPES, TAUS, fit_bank, fit_recipes, replay
from local_benchmark import predict_and_score, summarize

OUT = ROOT / "experiments/stack_improvements"


def setup():
    universe, protocol = initialize()
    manifest = dict(protocol_sha256=digest(DIRECTORY / "protocol.json"), recipes=RECIPES,
                    banks=TAUS, versions={p: importlib.metadata.version(p) for p in
                    ["numpy", "pandas", "scipy", "scikit-learn", "joblib"]},
                    source_sha256={f: digest(ROOT / f) for f in ["improvement_models.py",
                    "improvement_benchmark.py", "ani_models.py", "benchmark_protocol.py",
                    "local_benchmark.py", "experiments.py"]},
                    references={"ani_first": {"kaggle": .58810, "commit": "628088a3be8050b0a20d8719991ec8f517ad8649"},
                                "ani_stack": {"kaggle": .58688, "commit": "1b8bd11c47428fb5453e53cdc275542b92a3b347"}},
                    selection="Freeze all recipes, evaluate all ten saved splits. Prioritize chronological mean and worst window; require corroboration in pooled grouped CV and compare both winners. Bootstrap intervals are conditional exploratory evidence.")
    key = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:16]
    out = OUT / "runs" / key
    out.mkdir(parents=True, exist_ok=True)
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2))
    (OUT / "current_run.json").write_text(json.dumps(dict(run=key), indent=2))
    return universe, protocol, out


def packet_path(out, track, split):
    cache = OUT / "cache" / out.name / track
    cache.mkdir(parents=True, exist_ok=True)
    return cache / f"{split}.joblib"


def run(track, names=None):
    universe, protocol, run_out = setup()
    out = run_out / track
    out.mkdir(exist_ok=True)
    features = None
    if track == "published":
        data = load_data()
        features = AniFeatures().fit(pd.concat(list(data.values()), ignore_index=True))
    for split in protocol["splits"]:
        if names and split["name"] not in names:
            continue
        target = out / f"scores_{split['name']}.csv"
        if target.exists() and set(pd.read_csv(target).model) == set(RECIPES):
            continue
        start = time.time()
        train, ev = resolve(universe, split)
        cache = packet_path(run_out, track, split["name"])
        if cache.exists():
            packet = joblib.load(cache)
        else:
            print(track, split["name"], "fitting", flush=True)
            packet = fit_bank(train, ev, features=features,
                progress=lambda s: print(track, split["name"], s, round(time.time()-start, 1), "s", flush=True))
            joblib.dump(packet, cache, compress=3)
        assert packet["training_ids"] == train.ID.tolist()
        assert packet["evaluation_ids"] == ev.ID.tolist()
        probabilities, _ = fit_recipes(packet, train, ev)
        predict_and_score(out, split, ev, probabilities)
        bank_dir = out / "bases"
        bank_dir.mkdir(exist_ok=True)
        np.savez_compressed(bank_dir / f"{split['name']}.npz", fold_ids=packet["fold_ids"],
                            **{k+"_"+side: v for k, values in packet["bank"].items() for side, v in values.items()})
        audit = {k: v for k, v in packet.items() if k not in ["bank", "fold_ids"]}
        (bank_dir / f"{split['name']}_audit.json").write_text(json.dumps(audit, indent=2))
        print(track, split["name"], "complete", round(time.time()-start, 1), "s", flush=True)
        print(pd.read_csv(target)[["model", "phase_mean_logloss"]].to_string(index=False), flush=True)
    summarize(out, protocol, bootstrap=names is None)


def final_fit():
    universe, _, run_out = setup()
    data = load_data()
    folder = run_out / "final"
    folder.mkdir(exist_ok=True)
    cache = packet_path(run_out, "published", "final")
    if cache.exists():
        packet = joblib.load(cache)
    else:
        space = AniFeatures().fit(pd.concat(list(data.values()), ignore_index=True))
        packet = fit_bank(universe, data["test"], features=space, retain_models=True,
                          progress=lambda s: print("final", s, flush=True))
        joblib.dump(packet, cache, compress=3)
    predictions, metas = fit_recipes(packet, universe, data["test"])
    meta_path = cache.with_name("final_metas.joblib")
    joblib.dump(metas, meta_path, compress=3)
    changed = data["test"].copy()
    changed["ID"] = "unused"
    changed["trial_id"] = "unused"
    changed["label"] = 0
    other = replay(joblib.load(cache), joblib.load(meta_path), changed)
    parity = {}
    for name, p in predictions.items():
        assert np.allclose(other[name], p, rtol=0, atol=1e-12)
        pd.DataFrame({"ID": data["test"].ID, "label": np.clip(p, .001, .999)}).to_csv(folder / f"{name}.csv", index=False)
        if name in ["ani_first", "ani_stack"]:
            file = "part2_first_submission.csv" if name == "ani_first" else "part2_best_submission.csv"
            reference = pd.read_csv(ROOT / "experiments/ani_comparison/source" / file)
            assert reference.ID.equals(data["test"].ID)
            parity[name] = dict(source_sha256=digest(ROOT / "experiments/ani_comparison/source" / file),
                                max_probability_error=float(np.max(np.abs(reference.label - np.clip(p, .001, .999)))))
    (folder / "parity.json").write_text(json.dumps(parity, indent=2))
    print(json.dumps(parity, indent=2), flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--track", choices=["strict", "published"], default="strict")
    parser.add_argument("--splits", nargs="+")
    parser.add_argument("--final", action="store_true")
    args = parser.parse_args()
    final_fit() if args.final else run(args.track, args.splits)


Overwriting src/improvement_benchmark.py


## `src/improvement_models.py`

SHA-256: `79c482fa2a2fcb4327df03b44e484822f505f4500a76db31e95b919661790939`

In [39]:
%%writefile src/improvement_models.py
"""Fixed proposals around both published Ani winners; no evaluation labels fit models."""
from __future__ import annotations

import gc
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from ani_models import AniFeatures, BASES, fit_base, predict_base, onehot

TAUS = {"unweighted": None, "recent": 3., "moderate": 6.}
# Freeze this registry before scoring. Do not silently add a post-hoc winner.
RECIPES = {
    "ani_stack": dict(banks=["recent"], tau=3., C=1.),
    "ani_first": dict(banks=["unweighted"], tau=None, C=1.),
    "tau6_stack": dict(banks=["moderate"], tau=6., C=1.),
    "recent_meta_tau6": dict(banks=["recent"], tau=6., C=1.),
    "recent_equal_phase": dict(banks=["recent"], tau=3., C=1., balance=True),
    "recent_phase_shrink": dict(banks=["recent"], tau=3., C=.3, interactions=True),
    "recent_phase_C1": dict(banks=["recent"], tau=3., C=1., interactions=True),
    "tau6_phase_shrink": dict(banks=["moderate"], tau=6., C=.3, interactions=True),
    "recent_without_nb": dict(banks=["recent"], tau=3., C=1., drop_nb=True),
    "multiscale_stack": dict(banks=["recent", "unweighted"], tau=3., C=.3),
    "multiscale_phase": dict(banks=["recent", "unweighted"], tau=3., C=.3, interactions=True),
    "ani_phase3_trend": dict(banks=["recent"], tau=3., C=1., trend=True),
    "blend_winners50": dict(blend={"ani_stack": .5, "ani_first": .5}),
}


def recency_weights(df, tau, median):
    year = df.start_year.fillna(median).to_numpy()
    w = np.ones(len(df)) if tau is None else np.exp((year - year.max()) / tau)
    return w / w.mean()


def fit_bank(train, ev, features=None, retain_models=False, progress=None):
    """Share only unsupervised transforms across fixed recency settings.

    Strict mode refits preprocessing inside each grouped inner fold. Passing a
    feature space explicitly selects the separately reported transductive track.
    No labels from ev are read. Inner weights match Ani's outer-normalized recipe.
    """
    train = train.reset_index(drop=True)
    strict = features is None
    features = AniFeatures().fit(train) if strict else features
    weights = {k: recency_weights(train, tau, features.year_median) for k, tau in TAUS.items()}
    bank = {k: {"oof": np.zeros((len(train), len(BASES))),
                "eval": np.zeros((len(ev), len(BASES)))} for k in TAUS}
    fold_ids = np.full(len(train), -1)
    audit = []
    for fold, (a, b) in enumerate(GroupKFold(5).split(train, groups=train.trial_id)):
        assert not set(train.trial_id.iloc[a]) & set(train.trial_id.iloc[b])
        space = AniFeatures().fit(train.iloc[a]) if strict else features
        fa, fb = space.transform(train.iloc[a]), space.transform(train.iloc[b])
        if strict:
            assert set(space.fit_ids) == set(train.ID.iloc[a])
            assert not set(space.fit_ids) & (set(train.ID.iloc[b]) | set(ev.ID))
        for j, (name, blocks, C) in enumerate(BASES):
            for key in TAUS:
                if name == "criteria kNN" and key != "unweighted":
                    bank[key]["oof"][b, j] = bank["unweighted"]["oof"][b, j]
                    continue
                m = fit_base(name, blocks, C, fa, train.iloc[a], weights[key][a])
                bank[key]["oof"][b, j] = predict_base(name, blocks, m, fb, train.iloc[b])
        fold_ids[b] = fold
        audit.append(dict(fold=fold, fit_ids=train.ID.iloc[a].tolist(),
                          heldout_ids=train.ID.iloc[b].tolist(),
                          preprocessing_fit_ids=space.fit_ids, trial_overlap=0))
        del space, fa, fb, m
        gc.collect()
        if progress:
            progress(f"inner {fold + 1}/5")
    assert (fold_ids >= 0).all()
    ft, fe = features.transform(train), features.transform(ev)
    models = {k: [] for k in TAUS}
    for j, (name, blocks, C) in enumerate(BASES):
        for key in TAUS:
            if name == "criteria kNN" and key != "unweighted":
                bank[key]["eval"][:, j] = bank["unweighted"]["eval"][:, j]
                if retain_models:
                    models[key].append(models["unweighted"][-1])
                continue
            m = fit_base(name, blocks, C, ft, train, weights[key])
            bank[key]["eval"][:, j] = predict_base(name, blocks, m, fe, ev)
            if retain_models:
                models[key].append(m)
    for values in bank.values():
        assert all(np.isfinite(v).all() for v in values.values())
    packet = dict(bank=bank, fold_ids=fold_ids, audit=audit,
                  year_median=features.year_median, year_cap=float(train.start_year.max() + 2),
                  feature_fit_ids=features.fit_ids, training_ids=train.ID.tolist(),
                  evaluation_ids=ev.ID.tolist())
    if retain_models:
        packet.update(features=features, models=models)
    return packet


def meta_inputs(bank, side, df, recipe, median, cap):
    columns = []
    for key in recipe["banks"]:
        z = bank[key][side]
        if recipe.get("drop_nb"):
            z = z[:, [i for i, (name, _, _) in enumerate(BASES) if name != "criteria MNB"]]
        z = np.clip(z, 1e-4, 1 - 1e-4)
        columns.append(np.log(z / (1 - z)))
    logits = np.hstack(columns)
    phase = onehot(df)
    parts = [logits, phase]
    if recipe.get("interactions"):
        # Shared coefficients plus shrunk phase-specific deviations.
        parts.extend([logits * phase[:, i, None] for i in range(3)])
    if recipe.get("trend"):
        year = df.start_year.fillna(median).to_numpy()
        parts.append((phase[:, 2] * (np.minimum(year, cap) - 2007) / 3)[:, None])
    return np.hstack(parts)


def fit_recipes(packet, train, ev):
    predictions, meta_models = {}, {}
    for name, recipe in RECIPES.items():
        if "blend" in recipe:
            predictions[name] = sum(w * predictions[k] for k, w in recipe["blend"].items())
            continue
        w = recency_weights(train, recipe["tau"], packet["year_median"])
        if recipe.get("balance"):
            for phase in ("I", "II", "III"):
                mask = train.phase.eq(phase).to_numpy()
                w[mask] *= len(train) / (3 * w[mask].sum())
        args = (packet["year_median"], packet["year_cap"])
        xt = meta_inputs(packet["bank"], "oof", train, recipe, *args)
        xe = meta_inputs(packet["bank"], "eval", ev, recipe, *args)
        model = LogisticRegression(C=recipe["C"], max_iter=3000).fit(xt, train.label, sample_weight=w)
        predictions[name] = model.predict_proba(xe)[:, 1]
        meta_models[name] = model
    return predictions, meta_models


def replay(packet, meta_models, df):
    f = packet["features"].transform(df)
    bank = {key: {"eval": np.column_stack([
        predict_base(name, blocks, model, f, df)
        for (name, blocks, _), model in zip(BASES, packet["models"][key])])} for key in TAUS}
    out = {}
    for name, recipe in RECIPES.items():
        if "blend" in recipe:
            out[name] = sum(w * out[k] for k, w in recipe["blend"].items())
        else:
            x = meta_inputs(bank, "eval", df, recipe, packet["year_median"], packet["year_cap"])
            out[name] = meta_models[name].predict_proba(x)[:, 1]
    return out


Overwriting src/improvement_models.py


## `src/improvement_report.py`

SHA-256: `b84ba427e78b93fc21ec5ed73fdf8eba8fde0933a6a05ee85aa172141a4c99fa`

In [40]:
%%writefile src/improvement_report.py
"""Audit saved predictions, compare BOTH winners, freeze selection, and export."""
from __future__ import annotations
import argparse
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
from benchmark_protocol import initialize, resolve, digest
from experiments import ROOT, metrics, load_data
from improvement_models import RECIPES

OUT = ROOT / "experiments/stack_improvements"
REFERENCES = ["ani_stack", "ani_first"]


def current():
    return OUT / "runs" / json.loads((OUT / "current_run.json").read_text())["run"]


def row_loss(frame):
    p = np.clip(frame.probability.to_numpy(), 1e-6, 1 - 1e-6)
    y = frame.label.to_numpy()
    return -(y * np.log(p) + (1-y) * np.log1p(-p))


def paired_interval(frames, model, reference, kind, repetitions=2000):
    pieces = []
    for name, data in frames.items():
        if data["split"]["kind"] != kind:
            continue
        a, b = data[model], data[reference]
        assert a[["ID", "trial_id", "phase", "label"]].equals(b[["ID", "trial_id", "phase", "label"]])
        d = a[["ID", "trial_id", "phase"]].copy()
        d["period"] = name if kind == "temporal" else "cv"
        d["delta"] = row_loss(a) - row_loss(b)
        pieces.append(d)
    d = pd.concat(pieces, ignore_index=True)
    d = d.groupby(["ID", "trial_id", "phase", "period"], as_index=False).delta.mean()
    trials, indices = np.unique(d.trial_id, return_inverse=True)
    rng = np.random.default_rng(20261106)
    # Chunking avoids allocating a repetitions-by-row dense array.
    samples = []
    for start in range(0, repetitions, 100):
        weights = rng.poisson(1, (min(100, repetitions-start), len(trials)))
        group_scores = []
        for _, group in d.groupby(["period", "phase"]):
            w = weights[:, indices[group.index]]
            denom = w.sum(axis=1)
            group_scores.append(np.divide(w @ group.delta.to_numpy(), denom,
                out=np.full(len(w), np.nan), where=denom > 0))
        samples.extend(np.nanmean(group_scores, axis=0))
    point = d.groupby(["period", "phase"]).delta.mean().mean()
    return dict(model=model, reference=reference, kind=kind, delta=float(point),
                lower=float(np.quantile(samples, .025)), upper=float(np.quantile(samples, .975)),
                trials=len(trials), bootstrap_repetitions=repetitions)


def audit_track(folder, universe, protocol, names=None):
    names = list(RECIPES) if names is None else names
    frames, rows = {}, []
    for split in protocol["splits"]:
        train, ev = resolve(universe, split)
        group = {"split": split}
        for name in names:
            file = folder / "predictions" / split["name"] / f"{name}.csv"
            d = pd.read_csv(file)
            assert d.ID.is_unique and d.ID.tolist() == ev.ID.tolist(), str(file)
            for col in ["trial_id", "phase", "label"]:
                assert d[col].tolist() == ev[col].tolist(), (file, col)
            assert np.isfinite(d.probability).all() and d.probability.between(0, 1, inclusive="neither").all()
            assert not set(d.trial_id) & set(train.trial_id)
            group[name] = d
            loss = row_loss(d)
            row = dict(model=name, split=split["name"], kind=split["kind"], repeat=split.get("repeat"))
            for phase in ["I", "II", "III"]:
                row[phase+"_logloss"] = float(loss[d.phase.eq(phase)].mean())
            row["phase_mean_logloss"] = np.mean([row[p+"_logloss"] for p in ["I", "II", "III"]])
            saved = pd.read_csv(folder / f"scores_{split['name']}.csv").set_index("model").loc[name]
            assert abs(row["phase_mean_logloss"] - saved.phase_mean_logloss) < 1e-12
            rows.append(row)
        frames[split["name"]] = group
    scores = pd.DataFrame(rows)
    summaries = []
    for name, g in scores.groupby("model"):
        temporal = g[g.kind.eq("temporal")]
        official = g[g.kind.eq("official")]
        cv = []
        for repeat in sorted({s.get("repeat") for s in protocol["splits"] if s["kind"] == "group_cv"}):
            d = pd.concat([frames[s["name"]][name] for s in protocol["splits"]
                           if s["kind"] == "group_cv" and s["repeat"] == repeat])
            assert d.ID.is_unique
            cv.append(float(pd.Series(row_loss(d)).groupby(d.phase.to_numpy()).mean().mean()))
        summaries.append(dict(model=name, completed_splits=len(g),
                              official=float(official.phase_mean_logloss.iloc[0]) if len(official) else None,
                              temporal=float(temporal.phase_mean_logloss.mean()),
                              worst=float(temporal.phase_mean_logloss.max()), cv=float(np.mean(cv)) if cv else None))
    return pd.DataFrame(summaries).set_index("model"), scores, frames


def eligible(summary):
    assert (summary.completed_splits == 10).all()
    refs = summary.loc[REFERENCES]
    return [name for name, row in summary.iterrows() if name not in REFERENCES
            and row.temporal < refs.temporal.min()
            and row.cv <= refs.cv.min() + .001
            and row.worst <= refs.worst.min() + .003]


def select():
    out = current()
    universe, protocol = initialize()
    summaries = {}
    for track in ["strict", "published"]:
        summary, scores, _ = audit_track(out / track, universe, protocol)
        summary.to_csv(out / track / "independent_summary.csv")
        scores.to_csv(out / track / "independent_scores.csv", index=False)
        summaries[track] = summary
    candidates = eligible(summaries["strict"])
    shortlist = summaries["strict"].loc[candidates].sort_values("temporal").head(2).index.tolist()
    decision = dict(shortlist=shortlist, selection_track="strict", strict_eligible=candidates,
                    published_eligible=eligible(summaries["published"]),
                    protocol_document_sha256=digest(ROOT / "IMPROVEMENT_PROTOCOL.md"),
                    selection_code_sha256=digest(Path(__file__)),
                    input_sha256={track: digest(out / track / "independent_summary.csv") for track in summaries},
                    status="Frozen before gapped-window checks and final fitting",
                    known_kaggle_best=dict(model="ani_stack", score=.58688))
    path = out / "selection.json"
    if path.exists():
        old = json.loads(path.read_text())
        assert old["shortlist"] == shortlist and old["input_sha256"] == decision["input_sha256"]
    else:
        path.write_text(json.dumps(decision, indent=2))
    print(json.dumps(decision, indent=2))
    for track, summary in summaries.items():
        print(track, summary.sort_values("temporal").to_string())


def comparisons():
    from improvement_trends import TRENDS
    from improvement_text import TEXT_RECIPES
    out = current()
    universe, protocol = initialize()
    all_intervals = []
    combined = (out / "trend_followup/strict/independent_summary.csv").exists()
    names = list(RECIPES) + list(TRENDS) if combined else list(RECIPES)
    root = out / "trend_followup" if combined else out
    if (out / "text_followup/strict/independent_summary.csv").exists():
        names += list(TEXT_RECIPES)
        root = out / "text_followup"
    for track in ["strict", "published"]:
        _, _, frames = audit_track(root / track, universe, protocol, names=names)
        for name in names:
            for ref in REFERENCES:
                if name == ref:
                    continue
                for kind in ["temporal", "group_cv"]:
                    row = paired_interval(frames, name, ref, kind)
                    all_intervals.append(dict(track=track, **row))
    pd.DataFrame(all_intervals).to_csv(out / "paired_comparisons.csv", index=False)
    print("Saved paired intervals against BOTH Ani winners:", len(all_intervals))


def export():
    from improvement_trends import TRENDS
    from improvement_text import TEXT_RECIPES
    out = current()
    selection = json.loads((out / "selection.json").read_text())
    assert (out / "robustness/verification.json").exists(), "Complete the predeclared robustness check first"
    parity = json.loads((out / "final/parity.json").read_text())
    assert set(parity) == set(REFERENCES)
    assert all(v["max_probability_error"] < 1e-6 for v in parity.values())
    sample = pd.read_csv(ROOT / "part2_sample_submission.csv")
    records = []
    for i, name in enumerate(selection["shortlist"], start=9):
        source = out / "final" / f"{name}.csv"
        df = pd.read_csv(source).set_index("ID").loc[sample.ID].reset_index()
        assert list(df.columns) == ["ID", "label"] and len(df) == 1500
        assert df.ID.is_unique and df.ID.equals(sample.ID)
        assert np.isfinite(df.label).all() and df.label.between(0, 1, inclusive="neither").all()
        path = ROOT / "submissions" / f"{i:02d}_{name}.csv"
        if path.exists():
            assert np.allclose(pd.read_csv(path).label, df.label, rtol=0, atol=1e-14)
        else:
            df.to_csv(path, index=False)
        records.append(dict(model=name, file=path.name, sha256=digest(path), rows=len(df), kaggle_score=None,
                            recipe=(RECIPES | TRENDS | TEXT_RECIPES)[name], training_rows=9050,
                            preprocessing="Published track: all supplied covariates, training labels only",
                            benchmark_run=out.name, selection_sha256=digest(out / "selection.json"),
                            robustness_sha256=digest(out / "robustness/verification.json")))
    status = ("Exploratory, unsubmitted; Ani 0.58688 remains the confirmed Kaggle best" if records
              else "No proposal passed the predeclared rule; no new numbered submission exported. Retain Ani 0.58688.")
    manifest = dict(status=status,
                    submissions=records, source_parity=parity)
    (ROOT / "submissions/stack_improvements_manifest.json").write_text(json.dumps(manifest, indent=2))
    print(json.dumps(manifest, indent=2))


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("action", choices=["select", "compare", "export"])
    args = parser.parse_args()
    {"select": select, "compare": comparisons, "export": export}[args.action]()


Overwriting src/improvement_report.py


## `src/improvement_robustness.py`

SHA-256: `b2261ae031933f7def8885272f8bcb214634d71dc29309790b4015cb0f5352f0`

In [41]:
%%writefile src/improvement_robustness.py
"""Predeclared gapped forward checks of frozen shortlist and both Ani references."""
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "2")
import argparse
import json
import time
import joblib
import numpy as np
import pandas as pd
from ani_models import AniFeatures
from benchmark_protocol import initialize, resolve, digest
from experiments import load_data, ROOT
from improvement_benchmark import packet_path
from improvement_models import fit_bank, fit_recipes
from improvement_trends import fit_trends
from improvement_text import TEXT_RECIPES, fit_extra_bank, fit_text
from improvement_report import current, audit_track, paired_interval, REFERENCES
from local_benchmark import predict_and_score


def run(track):
    out = current()
    universe, _ = initialize()
    selection = json.loads((out / "selection.json").read_text())
    names = REFERENCES + selection["shortlist"]
    root = out / "robustness"
    root.mkdir(exist_ok=True)
    splits = []
    for cut, start, end in [(2005, 2008, 2009), (2007, 2010, 2011), (2009, 2012, 2013)]:
        year = universe.start_year.fillna(2007)
        train, ev = universe[year <= cut], universe[year.between(start, end)]
        splits.append(dict(name=f"gap_{cut}_to_{start}_{end}", kind="temporal", repeat=None,
                           fit_ids=train.ID.tolist(), eval_ids=ev.ID.tolist(), fit_rows=len(train), eval_rows=len(ev)))
    protocol = dict(splits=splits, selection_sha256=digest(out / "selection.json"),
                    note="Previously consulted labels; larger forecast gaps, not an independent holdout.")
    path = root / "protocol.json"
    if path.exists():
        assert json.loads(path.read_text()) == protocol
    else:
        path.write_text(json.dumps(protocol, indent=2))
    folder = root / track
    folder.mkdir(exist_ok=True)
    space = None
    if track == "published":
        space = AniFeatures().fit(pd.concat(list(load_data().values()), ignore_index=True))
    for split in splits:
        if (folder / f"scores_{split['name']}.csv").exists():
            continue
        train, ev = resolve(universe, split)
        cache = packet_path(out, track, split["name"])
        start = time.time()
        if cache.exists():
            packet = joblib.load(cache)
        else:
            packet = fit_bank(train, ev, features=space,
                progress=lambda s: print(track, split["name"], s, round(time.time()-start, 1), flush=True))
            joblib.dump(packet, cache, compress=3)
        predictions, _ = fit_recipes(packet, train, ev)
        trend_predictions, _ = fit_trends(packet, train, ev)
        predictions.update(trend_predictions)
        if any(name in TEXT_RECIPES for name in names):
            extra_cache = packet_path(out, track, "text_"+split["name"])
            if extra_cache.exists():
                extra = joblib.load(extra_cache)
            else:
                extra = fit_extra_bank(train, ev, space=space)
                joblib.dump(extra, extra_cache, compress=3)
            text_predictions, _ = fit_text(packet, extra, train, ev)
            predictions.update(text_predictions)
        predict_and_score(folder, split, ev, {k: predictions[k] for k in names})
        audit_dir = folder / "audits"
        audit_dir.mkdir(exist_ok=True)
        (audit_dir / f"{split['name']}.json").write_text(json.dumps({k: v for k, v in packet.items()
            if k not in ["bank", "fold_ids"]}, indent=2))
    summary, scores, frames = audit_track(folder, universe, protocol, names=names)
    summary.to_csv(folder / "summary.csv")
    scores.to_csv(folder / "fold_scores.csv", index=False)
    pairs = [paired_interval(frames, name, ref, "temporal") for name in names for ref in REFERENCES if name != ref]
    pd.DataFrame(pairs).to_csv(folder / "paired_comparisons.csv", index=False)
    print(summary.to_string(), flush=True)
    if all((root / t / "summary.csv").exists() for t in ["strict", "published"]):
        (root / "verification.json").write_text(json.dumps(dict(tracks=2, splits_per_track=3, models=names,
            protocol_sha256=digest(path), selection_sha256=digest(out / "selection.json"),
            source_sha256=digest(__file__), complete=True), indent=2))


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--track", choices=["strict", "published"], default="strict")
    run(parser.parse_args().track)


Overwriting src/improvement_robustness.py


## `src/improvement_text.py`

SHA-256: `383e85858a3990d07af6f96c2c5922005b1fd33a66c4724198dfd27f1c973307`

In [42]:
%%writefile src/improvement_text.py
"""Final adaptive batch: add two genuinely different text estimators to Ani OOF.

Declared after the two calibration batches. Uses all original splits, the same
feature boundaries, and the unchanged selection rule. No gapped results seen.
"""
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "2")
import argparse
import gc
import json
import time
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from ani_models import AniFeatures, matrix
from benchmark_protocol import initialize, resolve, digest
from experiments import ROOT, load_data
from improvement_models import RECIPES, recency_weights, meta_inputs
from improvement_report import current, audit_track, eligible
from improvement_trends import TRENDS
from improvement_benchmark import packet_path
from local_benchmark import predict_and_score

TEXT_RECIPES = {
    "recent_joint_stack": dict(bank="recent", tau=3., extra=[0], trend=False),
    "recent_ratio_stack": dict(bank="recent", tau=3., extra=[1], trend=False),
    "recent_text_stack": dict(bank="recent", tau=3., extra=[0, 1], trend=False),
    "recent_text_trend": dict(bank="recent", tau=3., extra=[0, 1], trend=True),
    "unweighted_text_stack": dict(bank="unweighted", tau=None, extra=[0, 1], trend=False),
    "unweighted_text_trend": dict(bank="unweighted", tau=None, extra=[0, 1], trend=True),
}
EXTRA_TAUS = {"unweighted": None, "recent": 3.}


def fit_extra(f, df, w, kind):
    if kind == 0:
        x = matrix(f, ["criteria", "disease", "icd", "drug", "targets", "pathways", "actions", "meta"])
        model = LogisticRegression(C=1., max_iter=3000).fit(x, df.label, sample_weight=w)
        return (model, None)
    x = f["crit_counts"].astype(float).copy()
    x.data[:] = 1
    y = df.label.to_numpy()
    positive = 1 + np.asarray(x.T @ (w*y)).ravel()
    negative = 1 + np.asarray(x.T @ (w*(1-y))).ravel()
    ratio = np.log(positive / positive.sum()) - np.log(negative / negative.sum())
    xx = sparse.hstack([x.multiply(ratio), f["phase"]], format="csr")
    model = LogisticRegression(C=.1, solver="liblinear", max_iter=3000, random_state=42).fit(xx, y, sample_weight=w)
    return (model, ratio)


def predict_extra(f, model, kind):
    lr, ratio = model
    if kind == 0:
        x = matrix(f, ["criteria", "disease", "icd", "drug", "targets", "pathways", "actions", "meta"])
    else:
        counts = f["crit_counts"].astype(float).copy()
        counts.data[:] = 1
        x = sparse.hstack([counts.multiply(ratio), f["phase"]], format="csr")
    return lr.predict_proba(x)[:, 1]


def fit_extra_bank(train, ev, space=None, retain=False, progress=None):
    strict = space is None
    outer = AniFeatures().fit(train) if strict else space
    weights = {k: recency_weights(train, tau, outer.year_median) for k, tau in EXTRA_TAUS.items()}
    bank = {k: dict(oof=np.zeros((len(train), 2)), eval=np.zeros((len(ev), 2))) for k in EXTRA_TAUS}
    audit = []
    for fold, (a, b) in enumerate(GroupKFold(5).split(train, groups=train.trial_id)):
        assert not set(train.trial_id.iloc[a]) & set(train.trial_id.iloc[b])
        features = AniFeatures().fit(train.iloc[a]) if strict else outer
        if strict:
            assert set(features.fit_ids) == set(train.ID.iloc[a])
            assert not set(features.fit_ids) & (set(train.ID.iloc[b]) | set(ev.ID))
        fa, fb = features.transform(train.iloc[a]), features.transform(train.iloc[b])
        for key in EXTRA_TAUS:
            for kind in range(2):
                model = fit_extra(fa, train.iloc[a], weights[key][a], kind)
                bank[key]["oof"][b, kind] = predict_extra(fb, model, kind)
        audit.append(dict(fold=fold, fit_ids=train.ID.iloc[a].tolist(), eval_ids=train.ID.iloc[b].tolist(),
                          feature_fit_ids=features.fit_ids, outcome_ratio_fit_ids=train.ID.iloc[a].tolist()))
        del features, fa, fb, model
        gc.collect()
        if progress:
            progress(fold + 1)
    fa, fb = outer.transform(train), outer.transform(ev)
    models = {k: [] for k in EXTRA_TAUS}
    for key in EXTRA_TAUS:
        for kind in range(2):
            model = fit_extra(fa, train, weights[key], kind)
            bank[key]["eval"][:, kind] = predict_extra(fb, model, kind)
            if retain:
                models[key].append(model)
    packet = dict(bank=bank, audit=audit, train_ids=train.ID.tolist(), eval_ids=ev.ID.tolist())
    if retain:
        packet.update(features=outer, models=models)
    return packet


def meta_x(packet, extra, side, df, recipe):
    base = meta_inputs(packet["bank"], side, df, dict(banks=[recipe["bank"]], trend=recipe["trend"]),
                       packet["year_median"], packet["year_cap"])
    z = np.clip(extra["bank"][recipe["bank"]][side][:, recipe["extra"]], 1e-4, 1-1e-4)
    return np.hstack([base, np.log(z / (1-z))])


def fit_text(packet, extra, train, ev):
    predictions, models = {}, {}
    for name, recipe in TEXT_RECIPES.items():
        xt, xe = [meta_x(packet, extra, side, frame, recipe) for side, frame in [("oof", train), ("eval", ev)]]
        w = recency_weights(train, recipe["tau"], packet["year_median"])
        model = LogisticRegression(C=1., max_iter=3000).fit(xt, train.label, sample_weight=w)
        predictions[name], models[name] = model.predict_proba(xe)[:, 1], model
    return predictions, models


def declare():
    out = current() / "text_followup"
    out.mkdir(exist_ok=True)
    manifest = dict(recipes=TEXT_RECIPES, source_sha256=digest(__file__),
        joint_model="L2 logistic C1 on all eight feature blocks plus phase",
        ratio_model="Binary criteria counts times smoothed, class-normalized log-count ratio; logistic C0.1; ratios fit inside each inner fold",
        context="Final adaptive batch after two failed calibration batches, before any gapped-window scores; no further sweep")
    path = out / "manifest.json"
    if path.exists():
        assert json.loads(path.read_text()) == manifest
    else:
        path.write_text(json.dumps(manifest, indent=2))
    return out


def run(track):
    out = declare() / track
    out.mkdir(exist_ok=True)
    universe, protocol = initialize()
    space = None
    if track == "published":
        space = AniFeatures().fit(pd.concat(list(load_data().values()), ignore_index=True))
    for split in protocol["splits"]:
        if (out / f"scores_{split['name']}.csv").exists():
            continue
        start = time.time()
        train, ev = resolve(universe, split)
        packet = joblib.load(packet_path(current(), track, split["name"]))
        cache = packet_path(current(), track, "text_"+split["name"])
        if cache.exists():
            extra = joblib.load(cache)
        else:
            extra = fit_extra_bank(train, ev, space=space,
                progress=lambda f: print(track, split["name"], f, round(time.time()-start, 1), flush=True))
            joblib.dump(extra, cache, compress=3)
        assert extra["train_ids"] == packet["training_ids"]
        assert extra["eval_ids"] == ev.ID.tolist()
        predictions, _ = fit_text(packet, extra, train, ev)
        for name in list(RECIPES)+list(TRENDS):
            d = pd.read_csv(current() / "trend_followup" / track / "predictions" / split["name"] / f"{name}.csv")
            assert d.ID.tolist() == ev.ID.tolist()
            predictions[name] = d.probability.to_numpy()
        predict_and_score(out, split, ev, predictions)
        folder = out / "bases"
        folder.mkdir(exist_ok=True)
        np.savez_compressed(folder / f"{split['name']}.npz", **{k+"_"+s: v for k, values in extra["bank"].items() for s,v in values.items()})
        (folder / f"{split['name']}_audit.json").write_text(json.dumps(extra["audit"], indent=2))
        print(track, split["name"], "complete", round(time.time()-start, 1), flush=True)
    names = list(RECIPES)+list(TRENDS)+list(TEXT_RECIPES)
    summary, scores, _ = audit_track(out, universe, protocol, names=names)
    summary.to_csv(out / "independent_summary.csv")
    scores.to_csv(out / "independent_scores.csv", index=False)
    print(summary.sort_values("temporal").to_string(), flush=True)


def select():
    out = current()
    folder = declare()
    summaries = {t: pd.read_csv(folder / t / "independent_summary.csv").set_index("model") for t in ["strict", "published"]}
    possible = eligible(summaries["strict"])
    shortlist = summaries["strict"].loc[possible].sort_values("temporal").head(2).index.tolist()
    decision = dict(shortlist=shortlist, strict_eligible=possible,
                    published_eligible=eligible(summaries["published"]), selection_track="strict",
                    protocol_document_sha256=digest(ROOT / "IMPROVEMENT_PROTOCOL.md"),
                    selection_code_sha256=digest(__file__),
                    input_sha256={t:digest(folder / t / "independent_summary.csv") for t in summaries},
                    status="Frozen across three declared batches before gapped tests and final fitting",
                    known_kaggle_best=dict(model="ani_stack", score=.58688))
    path = out / "selection.json"
    if path.exists():
        assert json.loads(path.read_text()) == decision
    else:
        path.write_text(json.dumps(decision, indent=2))
    print(json.dumps(decision, indent=2))


def final():
    out = current()
    universe, _ = initialize()
    test = load_data()["test"]
    packet = joblib.load(packet_path(out, "published", "final"))
    cache = packet_path(out, "published", "text_final")
    if cache.exists():
        extra = joblib.load(cache)
    else:
        extra = fit_extra_bank(universe, test, space=packet["features"], retain=True,
                               progress=lambda f: print("final text fold", f, flush=True))
        joblib.dump(extra, cache, compress=3)
    predictions, models = fit_text(packet, extra, universe, test)
    meta_path = packet_path(out, "published", "text_metas")
    joblib.dump(models, meta_path, compress=3)
    native = joblib.load(cache)
    models = joblib.load(meta_path)
    changed = test.copy()
    changed["ID"], changed["trial_id"], changed["label"] = "unused", "unused", 0
    features = native["features"].transform(changed)
    regenerated = dict(bank={k: dict(eval=np.column_stack([predict_extra(features, m, i)
        for i, m in enumerate(native["models"][k])])) for k in EXTRA_TAUS})
    for name, p in predictions.items():
        x = meta_x(packet, regenerated, "eval", changed, TEXT_RECIPES[name])
        assert np.allclose(models[name].predict_proba(x)[:,1], p, rtol=0, atol=1e-12)
        pd.DataFrame(dict(ID=test.ID, label=np.clip(p, .001, .999))).to_csv(out / "final" / f"{name}.csv", index=False)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("action", choices=["declare", "run", "select", "final"])
    parser.add_argument("--track", choices=["strict", "published"], default="strict")
    args = parser.parse_args()
    if args.action == "run":
        run(args.track)
    else:
        {"declare":declare, "select":select, "final":final}[args.action]()


Overwriting src/improvement_text.py


## `src/improvement_trends.py`

SHA-256: `0cfba4370fea616849d9f1945ccd2caa54a16b01cf3ca88862a9c8608d78a603`

In [43]:
%%writefile src/improvement_trends.py
"""Declared follow-up: explicit capped year effects across recency scales.

This registry was fixed after the first batch's chronological results and first
group-CV folds were inspected. It is an adaptive second batch, not a blind test.
All eight additions reuse the same base predictions and all ten saved splits.
"""
import argparse
import json
import shutil
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from ani_models import onehot
from benchmark_protocol import initialize, resolve, digest
from experiments import ROOT, load_data
from improvement_models import RECIPES, meta_inputs, recency_weights
from improvement_report import current, audit_track, eligible, REFERENCES
from local_benchmark import predict_and_score

TRENDS = {
    "unweighted_phase3_year": dict(bank="unweighted", tau=None, phases="III", scale=1.),
    "unweighted_all_years": dict(bank="unweighted", tau=None, phases="all", scale=1.),
    "unweighted_shrunk_years": dict(bank="unweighted", tau=None, phases="all", scale=.25),
    "tau6_phase3_year": dict(bank="moderate", tau=6., phases="III", scale=1.),
    "tau6_all_years": dict(bank="moderate", tau=6., phases="all", scale=1.),
    "tau6_shrunk_years": dict(bank="moderate", tau=6., phases="all", scale=.25),
    "recent_all_years": dict(bank="recent", tau=3., phases="all", scale=1.),
    "recent_shrunk_years": dict(bank="recent", tau=3., phases="all", scale=.25),
}


def inputs(bank, side, df, recipe, median, cap):
    x = meta_inputs(bank, side, df, dict(banks=[recipe["bank"]]), median, cap)
    # Exactly the original Ani cap and time units, with all phases supported.
    year = (np.minimum(df.start_year.fillna(median).to_numpy(), cap)-2007)/3
    trend = onehot(df) * year[:, None] * recipe["scale"]
    if recipe["phases"] == "III":
        trend = trend[:, 2, None]
    return np.hstack([x, trend])


def fit_trends(packet, train, ev):
    predictions, models = {}, {}
    for name, recipe in TRENDS.items():
        xt = inputs(packet["bank"], "oof", train, recipe, packet["year_median"], packet["year_cap"])
        xe = inputs(packet["bank"], "eval", ev, recipe, packet["year_median"], packet["year_cap"])
        w = recency_weights(train, recipe["tau"], packet["year_median"])
        model = LogisticRegression(C=1., max_iter=3000).fit(xt, train.label, sample_weight=w)
        predictions[name], models[name] = model.predict_proba(xe)[:, 1], model
    return predictions, models


def declare():
    folder = current() / "trend_followup"
    folder.mkdir(exist_ok=True)
    manifest = dict(recipes=TRENDS, source_sha256=digest(__file__),
                    context="Adaptive batch fixed after first-batch chronological results and early CV folds; same selection rule, no new boundaries.")
    path = folder / "manifest.json"
    if path.exists():
        assert json.loads(path.read_text()) == manifest, "Follow-up is frozen; version changes separately"
    else:
        path.write_text(json.dumps(manifest, indent=2))
    return folder


def run():
    folder = declare()
    universe, protocol = initialize()
    for track in ["strict", "published"]:
        dest = folder / track
        dest.mkdir(exist_ok=True)
        for split in protocol["splits"]:
            source = current() / track
            arrays = np.load(source / "bases" / f"{split['name']}.npz")
            audit = json.loads((source / "bases" / f"{split['name']}_audit.json").read_text())
            packet = dict(year_median=audit["year_median"], year_cap=audit["year_cap"],
                          bank={k: {s: arrays[k+"_"+s] for s in ["oof", "eval"]}
                                for k in ["unweighted", "moderate", "recent"]})
            train, ev = resolve(universe, split)
            probabilities, _ = fit_trends(packet, train, ev)
            for name in RECIPES:
                d = pd.read_csv(source / "predictions" / split["name"] / f"{name}.csv")
                assert d.ID.tolist() == ev.ID.tolist()
                probabilities[name] = d.probability.to_numpy()
            predict_and_score(dest, split, ev, probabilities)
        summary, scores, _ = audit_track(dest, universe, protocol, names=list(RECIPES)+list(TRENDS))
        summary.to_csv(dest / "independent_summary.csv")
        scores.to_csv(dest / "independent_scores.csv", index=False)
        print(track, summary.sort_values("temporal").to_string(), flush=True)


def select():
    out = current()
    folder = declare()
    summaries = {t: pd.read_csv(folder / t / "independent_summary.csv").set_index("model")
                 for t in ["strict", "published"]}
    eligible_names = eligible(summaries["strict"])
    shortlist = summaries["strict"].loc[eligible_names].sort_values("temporal").head(2).index.tolist()
    decision = dict(shortlist=shortlist, selection_track="strict", strict_eligible=eligible_names,
                    published_eligible=eligible(summaries["published"]),
                    protocol_document_sha256=digest(ROOT / "IMPROVEMENT_PROTOCOL.md"),
                    selection_code_sha256=digest(__file__),
                    input_sha256={t: digest(folder / t / "independent_summary.csv") for t in summaries},
                    followup_manifest_sha256=digest(folder / "manifest.json"),
                    status="Frozen across both batches before gapped-window checks and final fitting",
                    known_kaggle_best=dict(model="ani_stack", score=.58688))
    path = out / "selection.json"
    if path.exists():
        assert json.loads(path.read_text()) == decision
    else:
        path.write_text(json.dumps(decision, indent=2))
    print(json.dumps(decision, indent=2))


def final():
    import joblib
    from improvement_benchmark import packet_path
    out = current()
    universe, _ = initialize()
    test = load_data()["test"]
    packet = joblib.load(packet_path(out, "published", "final"))
    predictions, models = fit_trends(packet, universe, test)
    changed = test.copy()
    changed["label"] = 1
    changed["ID"] = "unused"
    changed["trial_id"] = "unused"
    other, _ = fit_trends(packet, universe, changed)
    for name, p in predictions.items():
        assert np.array_equal(other[name], p)
        pd.DataFrame(dict(ID=test.ID, label=np.clip(p, .001, .999))).to_csv(out / "final" / f"{name}.csv", index=False)
    joblib.dump(models, packet_path(out, "published", "trend_metas"), compress=3)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("action", choices=["declare", "run", "select", "final"])
    args = parser.parse_args()
    {"declare": declare, "run": run, "select": select, "final": final}[args.action]()


Overwriting src/improvement_trends.py


## `src/local_benchmark.py`

SHA-256: `67ab861407c66f3572d8916ed933f04ce37a55c462ce00ebfecdf02a5bb4deae`

In [44]:
%%writefile src/local_benchmark.py
"""Run the frozen, grouped and chronological model-comparison protocol.

python local_benchmark.py --suite full
python local_benchmark.py --suite chronological
python local_benchmark.py --summarize

Results are cached under a code/version fingerprint. No Kaggle files are
submitted. This runner never fits on an evaluation trial or its text.
"""
from __future__ import annotations
import os
os.environ.setdefault('OMP_NUM_THREADS','4')
os.environ.setdefault('OPENBLAS_NUM_THREADS','4')
os.environ.setdefault('MKL_NUM_THREADS','4')
import argparse,copy,hashlib,importlib.metadata,json,time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from ani_models import AniStack
from benchmark_models import OurLinear
from benchmark_protocol import DIRECTORY,initialize,resolve,digest
from experiments import ROOT,PHASES,metrics

MODELS=['ani_stack','ani_meta_C0.3','ani_phase3_trend','our_linear','blend_ani75_linear25','blend_ani50_linear50']


def run_directory(protocol):
    code={f:digest(ROOT/f) for f in ['ani_models.py','benchmark_models.py','benchmark_protocol.py','experiments.py','local_benchmark.py']}
    versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','joblib']}
    manifest={'protocol_sha256':digest(DIRECTORY/'protocol.json'),'source_sha256':code,'versions':versions,
        'models':MODELS,'baseline':'ani_stack','ani_stack':{'tau':3,'meta_C':1,'inner_preprocessing':'inner-fold training only'},
        'model_selection':'Equal-phase log loss; prioritize mean and worst chronological windows, corroborate with repeated group CV; inspect paired uncertainty, not just validation F1.'}
    key=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()[:16]
    out=DIRECTORY/'runs'/key;out.mkdir(parents=True,exist_ok=True)
    (out/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
    (DIRECTORY/'current_run.json').write_text(json.dumps({'run':key,'directory':str(out.relative_to(ROOT))},indent=2))
    return out


def predict_and_score(out,split,df,probabilities):
    folder=out/'predictions'/split['name'];folder.mkdir(parents=True,exist_ok=True)
    scores=[]
    for name,p in probabilities.items():
        assert len(p)==len(df) and np.isfinite(p).all() and ((p>0)&(p<1)).all()
        frame=df[['ID','trial_id','phase','start_year','label']].copy();frame['probability']=p
        frame.to_csv(folder/f'{name}.csv',index=False,float_format='%.15g')
        row={'model':name,'split':split['name'],'kind':split['kind'],'repeat':split['repeat'],**metrics(df,p)}
        row['brier']=brier_score_loss(df.label,p)
        for phase in PHASES:
            m=df.phase.eq(phase);row[phase+'_brier']=brier_score_loss(df.label[m],p[m])
            row[phase+'_n']=int(m.sum());row[phase+'_predicted_mean']=float(p[m].mean());row[phase+'_observed_mean']=float(df.label[m].mean())
        scores.append(row)
    pd.DataFrame(scores).to_csv(out/f"scores_{split['name']}.csv",index=False)


def fit_split(out,split,train,ev):
    folder=out/'models'/split['name'];folder.mkdir(parents=True,exist_ok=True)
    start=time.time();fitted={}
    for name,factory in [('ani_stack',lambda:AniStack(tau=3,inner_refit=True)),('our_linear',OurLinear)]:
        path=folder/f'{name}.joblib'
        if path.exists():model=joblib.load(path)
        else:
            model=factory().fit(train)
            assert model.training_ids==train.ID.tolist()
            joblib.dump(model,path,compress=3)
        fitted[name]=model
    baseline=fitted['ani_stack']
    assert set(baseline.features.fit_ids)==set(train.ID)
    assert not set(baseline.features.fit_ids)&set(ev.ID)
    assert all(x['trial_overlap']==0 for x in baseline.fold_audit)
    z=baseline.predict_base_matrix(ev)
    probabilities={'ani_stack':baseline.meta.predict_proba(baseline.meta_matrix(z,ev))[:,1],
                   'our_linear':fitted['our_linear'].predict_proba(ev)}
    # These two variants change only the meta-model; all base fits remain identical.
    for name,C,trend in [('ani_meta_C0.3',.3,False),('ani_phase3_trend',1.,True)]:
        candidate=copy.copy(baseline);candidate.meta_C=C;candidate.phase3_year=trend
        year=train.start_year.fillna(candidate.features.year_median).to_numpy()
        w=np.exp((year-year.max())/3);w=w/w.mean()
        candidate.meta=LogisticRegression(C=C,max_iter=3000).fit(candidate.meta_matrix(candidate.oof_base_predictions,train),train.label,sample_weight=w)
        probabilities[name]=candidate.meta.predict_proba(candidate.meta_matrix(z,ev))[:,1]
        joblib.dump(candidate,folder/f'{name}.joblib',compress=3)
    probabilities['blend_ani75_linear25']=.75*probabilities['ani_stack']+.25*probabilities['our_linear']
    probabilities['blend_ani50_linear50']=.5*probabilities['ani_stack']+.5*probabilities['our_linear']
    if split['kind']=='official':
        changed=ev.copy();changed['ID']='unused';changed['trial_id']='unused';changed['label']=1-ev.label
        for name in ['ani_stack','our_linear']:
            assert np.allclose(fitted[name].predict_proba(changed),probabilities[name],atol=1e-12,rtol=0)
    predict_and_score(out,split,ev,probabilities)
    audit={'split':split['name'],'fit_rows':len(train),'eval_rows':len(ev),'trial_overlap':0,
        'fit_ID_sha256':hashlib.sha256('\n'.join(train.ID).encode()).hexdigest(),
        'eval_ID_sha256':hashlib.sha256('\n'.join(ev.ID).encode()).hexdigest(),
        'preprocessing_population':'training rows only, both outer and stacking-inner folds',
        'inner_folds':baseline.fold_audit,'seconds':time.time()-start}
    (folder/'audit.json').write_text(json.dumps(audit,indent=2))
    print(split['name'],'completed',round(time.time()-start,1),'seconds',
          {k:round(metrics(ev,p)['phase_mean_logloss'],5) for k,p in probabilities.items()},flush=True)


def loss_rows(df):
    p=np.clip(df.probability.to_numpy(),1e-6,1-1e-6);y=df.label.to_numpy()
    return -(y*np.log(p)+(1-y)*np.log1p(-p))


def paired_bootstrap(out,protocol,model,kind,repetitions=1200):
    frames=[]
    for split in protocol['splits']:
        if split['kind']!=kind:continue
        folder=out/'predictions'/split['name']
        if not (folder/f'{model}.csv').exists() or not (folder/'ani_stack.csv').exists():continue
        a,b=pd.read_csv(folder/f'{model}.csv'),pd.read_csv(folder/'ani_stack.csv')
        assert a.ID.equals(b.ID)
        a['delta']=loss_rows(a)-loss_rows(b)
        a['period']=split['name'] if kind=='temporal' else 'group_cv'
        frames.append(a[['ID','trial_id','phase','period','delta']])
    if not frames:return {}
    d=pd.concat(frames,ignore_index=True)
    # Each trial's repeated-CV losses are averaged before resampling.
    d=d.groupby(['ID','trial_id','phase','period'],as_index=False).delta.mean()
    trials,index=np.unique(d.trial_id,return_inverse=True)
    rng=np.random.default_rng(42);weights=rng.poisson(1,size=(repetitions,len(trials))).astype(np.float32)[:,index]
    boot=[];points=[]
    for (_,phase),group in d.groupby(['period','phase']):
        ix=group.index.to_numpy();points.append(group.delta.mean())
        boot.append((weights[:,ix]@group.delta.to_numpy())/weights[:,ix].sum(axis=1))
    values=np.mean(boot,axis=0)
    return {kind+'_delta':float(np.mean(points)),kind+'_delta_lower':float(np.quantile(values,.025)),
            kind+'_delta_upper':float(np.quantile(values,.975))}


def summarize(out,protocol,bootstrap=True):
    files=list(out.glob('scores_*.csv'))
    if not files:return
    scores=pd.concat([pd.read_csv(f) for f in files],ignore_index=True)
    scores.to_csv(out/'fold_scores.csv',index=False)
    rows=[]
    expected_cv=sum(s['kind']=='group_cv' for s in protocol['splits'])
    for model,g in scores.groupby('model'):
        row={'model':model,'completed_splits':len(g),'expected_splits':len(protocol['splits'])}
        official=g[g.kind.eq('official')]
        row['official_logloss']=float(official.phase_mean_logloss.iloc[0]) if len(official) else np.nan
        times=g[g.kind.eq('temporal')]
        row['temporal_windows']=len(times);row['temporal_mean_logloss']=times.phase_mean_logloss.mean();row['worst_temporal_logloss']=times.phase_mean_logloss.max()
        repeat_scores=[]
        for repeat in [0,1]:
            splits=[s for s in protocol['splits'] if s['kind']=='group_cv' and s['repeat']==repeat]
            paths=[out/'predictions'/s['name']/f'{model}.csv' for s in splits]
            if all(p.exists() for p in paths):
                d=pd.concat([pd.read_csv(p) for p in paths],ignore_index=True)
                assert d.ID.is_unique
                repeat_scores.append(metrics(d,d.probability.to_numpy())['phase_mean_logloss'])
        row['cv_completed_repeats']=len(repeat_scores)
        row['cv_mean_logloss']=float(np.mean(repeat_scores)) if repeat_scores else np.nan
        row['cv_repeat_std']=float(np.std(repeat_scores)) if len(repeat_scores)>1 else np.nan
        if bootstrap and model!='ani_stack':
            if len(times)==3:row.update(paired_bootstrap(out,protocol,model,'temporal'))
            if len(g[g.kind.eq('group_cv')])==expected_cv:row.update(paired_bootstrap(out,protocol,model,'group_cv'))
        rows.append(row)
    summary=pd.DataFrame(rows).sort_values('temporal_mean_logloss')
    baseline=summary[summary.model.eq('ani_stack')].iloc[0]
    summary['temporal_mean_delta_vs_ani']=summary.temporal_mean_logloss-baseline.temporal_mean_logloss
    summary['cv_mean_delta_vs_ani']=summary.cv_mean_logloss-baseline.cv_mean_logloss
    # A model cannot gain a complete-comparison status from a favorable subset.
    summary['complete_comparison']=summary.completed_splits.eq(summary.expected_splits)
    summary.to_csv(out/'summary.csv',index=False)
    print(summary.to_string(index=False),flush=True)


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--suite',choices=['full','chronological','official'],default='full')
    parser.add_argument('--summarize',action='store_true')
    parser.add_argument('--force',action='store_true',help='Recompute predictions using cached fitted models')
    args=parser.parse_args()
    universe,protocol=initialize();out=run_directory(protocol)
    if not args.summarize:
        for split in protocol['splits']:
            if args.suite=='chronological' and split['kind']=='group_cv':continue
            if args.suite=='official' and split['kind']!='official':continue
            score=out/f"scores_{split['name']}.csv"
            if score.exists() and not args.force:
                saved=pd.read_csv(score)
                if set(saved.model)==set(MODELS):continue
            train,ev=resolve(universe,split);fit_split(out,split,train,ev)
            summarize(out,protocol,bootstrap=False)
    summarize(out,protocol)
    print('Results:',out,flush=True)

if __name__=='__main__':main()


Overwriting src/local_benchmark.py


## `src/recency_and_sparse_boost.py`

SHA-256: `1f4f24e03547efe7eeaa383ab1faa0d5c69516dc32ac434823c855f6ff4a7c8c`

In [45]:
%%writefile src/recency_and_sparse_boost.py
"""Historical weighting sensitivity and sparse LightGBM comparisons."""
import os
os.environ.setdefault('OMP_NUM_THREADS','4')
os.environ.setdefault('OPENBLAS_NUM_THREADS','4')
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from experiments import OUT, load_data, prepare, matrix, metrics


def run():
    data=load_data()
    for split in ['temporal','validation']:
        tr,ev,f,bt,be,bx=prepare(data,split)
        scores=[];pred={};tests={};configs={}
        def record(name,p,pt,config):
            pred[name]=p;tests[name]=pt;configs[name]=config
            scores.append(dict(name=name,**metrics(ev,p)))
            pd.DataFrame(scores).sort_values('phase_mean_logloss').to_csv(OUT/f'extra_{split}_scores.csv',index=False)
            print(split,name,metrics(ev,p)['phase_mean_logloss'],flush=True)
        for rep in ['criteria','clinical']:
            xs=[matrix(b,rep) for b in [bt,be,bx]]
            for half_life in [3.,6.,12.]:
                w=2**((tr.start_year.fillna(tr.start_year.median()).to_numpy()-tr.start_year.max())/half_life)
                w=w/w.mean()
                for C in [.5,2.]:
                    model=LogisticRegression(C=C,solver='liblinear',max_iter=1000,tol=1e-5,random_state=42)
                    model.fit(xs[0],tr.label,sample_weight=w)
                    name=f'recency_{rep}_C{C:g}_half{half_life:g}'
                    record(name,model.predict_proba(xs[1])[:,1],model.predict_proba(xs[2])[:,1],
                           dict(family='recency',representation=rep,C=C,half_life=half_life))
        for year in ['none','year']:
            xs=[matrix(b,'knowledge',year) for b in [bt,be,bx]]
            for leaves in [7,15,31]:
                model=LGBMClassifier(n_estimators=1000,learning_rate=.025,num_leaves=leaves,
                    min_child_samples=40,reg_lambda=10,max_bin=31,colsample_bytree=.8,
                    subsample=.8,subsample_freq=1,n_jobs=4,verbosity=-1,random_state=42)
                model.fit(xs[0],tr.label)
                joblib.dump(model,OUT/f'lightgbm_{split}_leaves{leaves}_{year}.joblib',compress=3)
                for n in [400,1000]:
                    name=f'lightgbm_leaves{leaves}_n{n}_{year}'
                    p=model.predict_proba(xs[1],num_iteration=n)[:,1]
                    pt=model.predict_proba(xs[2],num_iteration=n)[:,1]
                    record(name,p,pt,dict(family='lightgbm',year=year,leaves=leaves,iterations=n))
        np.savez_compressed(OUT/f'extra_{split}_predictions.npz',**pred)
        np.savez_compressed(OUT/f'extra_{split}_test_predictions.npz',**tests)
        (OUT/'extra_configs.json').write_text(json.dumps(configs,indent=2))

if __name__=='__main__':run()


Overwriting src/recency_and_sparse_boost.py


## `src/refinements.py`

SHA-256: `6d5bfc2f735ce7ef61015cb52a94274323c4b89ab11d0614407543e2f1e01d41`

In [46]:
%%writefile src/refinements.py
"""Text ablations, molecular features, shared phase effects, and NB-log ratios."""
import os
os.environ.setdefault('OMP_NUM_THREADS','4')
os.environ.setdefault('OPENBLAS_NUM_THREADS','4')
import json
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.naive_bayes import MultinomialNB
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from experiments import ROOT, OUT, PHASES, load_data, prepare, matrix, metrics

RDLogger.DisableLog('rdApp.*')


def molecule_bank(data):
    path=OUT/'molecules.joblib'
    if path.exists(): return joblib.load(path)
    gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)
    descs=[Descriptors.MolWt,Descriptors.MolLogP,Descriptors.TPSA,Descriptors.NumHDonors,
           Descriptors.NumHAcceptors,Descriptors.NumRotatableBonds,Descriptors.RingCount,
           Descriptors.FractionCSP3,Descriptors.HeavyAtomCount]
    bank={}
    for text in pd.concat(list(data.values())).drug_smiles.fillna('').unique():
        molecules=[Chem.MolFromSmiles(t) for t in text.split(';') if t]
        molecules=[m for m in molecules if m is not None]
        if molecules:
            fp=np.max([gen.GetFingerprintAsNumPy(m) for m in molecules],axis=0)
            ds=np.array([[f(m) for f in descs] for m in molecules])
            d=np.r_[ds.mean(axis=0),ds.max(axis=0),len(molecules),0.]
        else: fp=np.zeros(1024); d=np.r_[np.zeros(19),1.]
        bank[text]=(fp,d)
    joblib.dump(bank,path,compress=3)
    return bank


def molmat(df,bank):
    values=[bank[t] for t in df.drug_smiles.fillna('')]
    return sparse.csr_matrix(normalize(np.array([x[0] for x in values]))),np.array([x[1] for x in values])


def interactions(x, df, weight):
    return sparse.hstack([x]+[x.multiply(df.phase.eq(p).to_numpy()[:,None])*weight for p in PHASES],format='csr')


def run():
    data=load_data(); bank=molecule_bank(data)
    for split in ['temporal','validation']:
        tr, ev, f, bt, be, bx=prepare(data,split)
        dfs=[tr,ev,data['test']]; blocks=[bt,be,bx]
        pred={}; tests={}; scores=[]; configs={}

        def record(name,p,pt,config):
            pred[name]=p; tests[name]=pt; configs[name]=config
            scores.append(dict(name=name,**metrics(ev,p)))
            pd.DataFrame(scores).sort_values('phase_mean_logloss').to_csv(OUT/f'refined_{split}_scores.csv',index=False)
            print(split,name,metrics(ev,p)['phase_mean_logloss'],flush=True)

        def linear(name,xs,C,config,weights=None):
            model=LogisticRegression(C=C,solver='liblinear',max_iter=1000,tol=1e-5,random_state=42)
            model.fit(xs[0],tr.label,sample_weight=weights)
            record(name,model.predict_proba(xs[1])[:,1],model.predict_proba(xs[2])[:,1],config)

        # Phase-dependent coefficients share a common set of coefficients and shrink toward it.
        for rep in ['criteria','clinical']:
            for C in [.5,2.,8.]:
                xs=[matrix(b,rep,'none') for b in blocks]
                for weight in [.3,1.]:
                    xx=[interactions(x,d,weight) for x,d in zip(xs,dfs)]
                    name=f'interactions_{rep}_C{C:g}_weight{weight:g}'
                    linear(name,xx,C,dict(family='interactions',representation=rep,C=C,weight=weight))
                # Equal phase weights match the competition's phase-balanced objective.
                w=tr.phase.map({p:len(tr)/(3*sum(tr.phase==p)) for p in PHASES}).to_numpy()
                name=f'balanced_{rep}_C{C:g}'
                linear(name,xs,C,dict(family='balanced',representation=rep,C=C),w)

        # Molecular structures are supplied inputs, not external outcome information.
        fps,ds=zip(*(molmat(d,bank) for d in dfs))
        scaler=StandardScaler().fit(ds[0]); ds=[scaler.transform(x) for x in ds]
        for weight in [.25,1.]:
            xs=[sparse.hstack([matrix(b,'clinical'),fp*weight,sparse.csr_matrix(np.clip(d,-5,5))*.05],format='csr')
                for b,fp,d in zip(blocks,fps,ds)]
            for C in [.5,2.,8.]:
                name=f'molecular_C{C:g}_weight{weight:g}'
                linear(name,xs,C,dict(family='molecular',C=C,weight=weight))

        # Alternate eligibility text representations.
        for rep,options in [
            ('binary',dict(binary=True,use_idf=False,ngram_range=(1,2),min_df=3,max_features=60000)),
            ('wordwide',dict(ngram_range=(1,2),min_df=2,max_features=80000,sublinear_tf=True)),
            ('charcriteria',dict(analyzer='char_wb',ngram_range=(3,5),min_df=5,max_features=40000,sublinear_tf=True)),
            ('stopwords',dict(ngram_range=(1,2),min_df=3,max_features=35000,sublinear_tf=True,stop_words='english')),
        ]:
            vec=TfidfVectorizer(dtype=np.float64,strip_accents='unicode',**options).fit(f.documents(tr,'criteria'))
            texts=[vec.transform(f.documents(d,'criteria')) for d in dfs]
            for C in [.5,2.,8.]:
                if rep=='charcriteria':
                    xs=[sparse.hstack([b['criteria'],t*.7,b['phase']],format='csr') for t,b in zip(texts,blocks)]
                else: xs=[sparse.hstack([t,b['phase']],format='csr') for t,b in zip(texts,blocks)]
                name=f'text_{rep}_C{C:g}'
                linear(name,xs,C,dict(family='text',representation=rep,C=C))
            joblib.dump(vec,OUT/f'text_{rep}_{split}.joblib',compress=3)

        # NB feature ratios are estimated from fitting labels only.
        nb=MultinomialNB(alpha=1.).fit(bt['criteria'],tr.label)
        ratio=nb.feature_log_prob_[1]-nb.feature_log_prob_[0]
        for C in [.5,2.,8.,32.]:
            xs=[sparse.hstack([b['criteria'].multiply(ratio),b['phase']],format='csr') for b in blocks]
            name=f'nb_logratio_C{C:g}'
            linear(name,xs,C,dict(family='nb_logratio',C=C))
        np.savez_compressed(OUT/f'refined_{split}_predictions.npz',**pred)
        np.savez_compressed(OUT/f'refined_{split}_test_predictions.npz',**tests)
        (OUT/'refined_configs.json').write_text(json.dumps(configs,indent=2))

if __name__=='__main__': run()


Overwriting src/refinements.py


## `src/reporting.py`

SHA-256: `3aafb5adf464a73f88c853a162951e0b6cf03b545f62a4299b5367a28cce90f2`

In [47]:
%%writefile src/reporting.py
"""Export the assignment's per-phase comparisons and report-ready figures."""
import os
os.environ.setdefault('MPLCONFIGDIR',str(__import__('pathlib').Path(__file__).resolve().parent/'experiments/.matplotlib'))
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
from experiments import ROOT,OUT,PHASES,load_data

FIGURES=OUT/'figures'


def mdtable(df):
    rows=['| '+' | '.join(df.columns)+' |','| '+' | '.join(['---']*len(df.columns))+' |']
    for row in df.itertuples(index=False,name=None):
        rows.append('| '+' | '.join('' if pd.isna(v) else f'{v:.5f}' if isinstance(v,float) else str(v) for v in row)+' |')
    return '\n'.join(rows)


def run():
    FIGURES.mkdir(exist_ok=True)
    plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':140})
    data=load_data();d=data['val'];selected=np.load(OUT/'selected_validation_predictions.npz')
    baseline=np.load(OUT/'linear_validation_predictions.npz')['lr_criteria_none_C2_phase0']
    improved=selected['03_phase_specialized']
    fig,axes=plt.subplots(2,3,figsize=(12,7))
    for j,phase in enumerate(PHASES):
        m=d.phase.eq(phase)
        for name,p,color in [('Text baseline',baseline,'#8495a6'),('Selected ensemble',improved,'#087f8c')]:
            RocCurveDisplay.from_predictions(d.label[m],p[m],ax=axes[0,j],name=name,curve_kwargs={'color':color})
            PrecisionRecallDisplay.from_predictions(d.label[m],p[m],ax=axes[1,j],name=name,curve_kwargs={'color':color})
        axes[0,j].set_title(f'Phase {phase} (n={m.sum()})')
        axes[0,j].plot([0,1],[0,1],ls=':',c='.6')
        axes[0,j].set_xlabel('False positive rate');axes[0,j].set_ylabel('True positive rate')
        axes[1,j].set_xlabel('Recall');axes[1,j].set_ylabel('Precision')
        axes[0,j].legend(fontsize=8,loc='lower right')
        axes[1,j].legend(fontsize=8,loc='lower left')
    fig.tight_layout();fig.savefig(FIGURES/'phase_roc_pr.png');plt.close(fig)
    comparison=pd.read_csv(OUT/'reason_error_comparison.csv')
    fig,axes=plt.subplots(1,3,figsize=(13,4.7),sharey=True)
    for ax,phase in zip(axes,PHASES):
        t=comparison[comparison.phase.eq(phase)].copy();x=np.arange(len(t))
        ax.bar(x-.18,t.false_success_rate_baseline,width=.36,color='#8495a6',label='Baseline')
        ax.bar(x+.18,t.false_success_rate_improved,width=.36,color='#087f8c',label='Ensemble')
        labels=[('No reason' if r.startswith('No recorded') else r)+f'\nn={n}' for r,n in zip(t.reason,t.n)]
        ax.set_xticks(x,labels,rotation=50,ha='right');ax.set_ylim(0,1.05);ax.set_title(f'Phase {phase}')
    axes[0].set_ylabel('Fraction of unsuccessful trials predicted successful')
    axes[-1].legend(loc='upper left',fontsize=8)
    fig.tight_layout();fig.savefig(FIGURES/'reason_errors.png');plt.close(fig)
    components=pd.read_csv(OUT/'all_component_scores.csv')
    manifest=pd.read_csv(ROOT/'submissions/index.csv')
    fig,ax=plt.subplots(figsize=(7.5,5))
    ax.scatter(components.validation_phase_mean_logloss,components.temporal_phase_mean_logloss,
        s=12,alpha=.22,c='#8495a6',label='Individual configurations')
    for row in manifest.itertuples():
        label=row.submission.removesuffix('.csv').replace('_',' ')
        ax.scatter(row.validation_logloss,row.temporal_logloss,s=110,marker='*',label=label)
    ax.set_xlabel('Supplied validation: phase-mean log loss (lower is better)')
    ax.set_ylabel('Later-year holdout: phase-mean log loss')
    ax.set_xlim(.54,.63);ax.set_ylim(.54,.64);ax.legend(loc='upper left',fontsize=8)
    fig.tight_layout();fig.savefig(FIGURES/'model_comparison.png');plt.close(fig)
    phases=pd.read_csv(OUT/'phase_metrics.csv')
    comparison_table=phases[phases.model.isin(['baseline','03_phase_specialized'])]
    category=comparison[['phase','reason','n','false_success_baseline','false_success_improved',
                         'true_reason_rows_baseline','predicted_reason_rows_baseline']]
    category.columns=['phase','reason','n','baseline false-success','ensemble false-success','true reason rows','predicted reason rows']
    total_models=len(components)
    total_ensembles=len(pd.read_csv(OUT/'ensemble_scores.csv'))
    best=manifest.iloc[0]
    report=f'''# Part Two: clinical-trial outcome prediction

**Latest recommendation: hold 05/06.** Ani's published 0.58688 model has now been reproduced. Use the [repeatable local benchmark](LOCAL_TESTING.md), [executed benchmark notebook](local_benchmark.ipynb), and [complete results](experiments/local_benchmark/RESULTS.md) for new comparisons. The original selection below is historical. Optional Ani-anchored candidates are documented in [ani_candidates_manifest.json](submissions/ani_candidates_manifest.json); their Kaggle scores remain unknown.

Prepared four initial manual Kaggle submissions and two follow-ups. **Lower log loss is better.** The best reported reference score is **0.58688**. Our `03_phase_specialized.csv` scored **0.59562**; `02_text_heavy.csv` improved to **0.58993**, still 0.00305 behind the reference. Initial candidates 01 and 04 have unknown Kaggle scores. Results are recorded against exact file hashes in [submissions/leaderboard_results.json](submissions/leaderboard_results.json).

## Evaluation protocol

Train: 8,037 rows / 7,535 trials. Supplied validation: 1,013 rows / 901 trials. Test: 1,500 rows, exactly 500 per phase. No trial crosses these files. Combined-phase rows stay together.

The chronological check trains on 5,660 rows starting through 2008 and evaluates on 2,377 training-file rows starting in 2009-2013. Every TF-IDF vocabulary, IDF, SVD, and metadata vocabulary is fitted only on that split's training population. Final models use **outcome_train.csv only**. No validation labels enter their fitting.

Selection compares the equal mean of the three per-phase log losses, matching Kaggle's phase proportions. The supplied validation set itself has unequal phase counts, so its ordinary row-weighted log loss is also saved but is not the selection objective. Both local splits are consulted for selection; neither is an untouched final assessment. Model selection across many settings can overfit these evaluations.

## Methods and comparisons

Evaluated {total_models} matching individual configurations (including pooled/separate-model blends) across both splits and {total_ensembles} fixed-weight ensemble configurations. Compare word and character TF-IDF, binary word features, clinical metadata, drug targets/actions/pathways, Morgan fingerprints and molecular descriptors, pooled and separate phase models, shared phase interactions, phase balancing, newer-trial weighting, raw/capped year effects, and NB log-ratio features. Classifiers include L2 logistic regression, CatBoost, histogram gradient boosting, sparse LightGBM, and Extra Trees. No deep learning or external outcome data is used.

The baseline is eligibility-criteria TF-IDF (unigrams/bigrams, at most 35,000 terms, minimum document frequency 3, sublinear TF), three phase indicators, and pooled L2 logistic regression with C=2. All F1 values below use cutoff 0.5. ROC-AUC and PR-AUC use probabilities, with success (label=1) as positive. PR-AUC means average precision.

Text models share signal across phases. Separate phase fits and phase interactions can overfit the smaller Phase I population. Molecular fingerprints do not consistently improve the text/clinical control. Raw year extrapolation is sensitive to the historical success trend; the capped-year alternative stops its effect at the latest observed training year. Drug annotations are less common in test (39.3%) than training (45.7%), so relying heavily on them risks a coverage shift. These patterns motivate mixed text/tree candidates and an explicit temporal comparison.

## Initial four submission candidates

{mdtable(manifest[['submission','validation_logloss','temporal_logloss','mean_test_probability','kaggle_score']])}

`03_phase_specialized.csv` was the lowest joint-loss candidate before Kaggle evaluation, but its local advantage did not transfer. 02 improved the leaderboard score by reducing tree influence in Phases II/III. That motivated diagnostic files 05/06, now **on hold**: our linear endpoint is worse than Ani in the new chronological comparisons. Their historical recipes remain in [submissions/followup_manifest.json](submissions/followup_manifest.json). The new benchmark compares strict fold-local preprocessing and Ani's faithful transductive recipe on the same ten saved splits. Optional 07 is a mild interpolation toward Ani's Phase III trend variant; 08 is an Ani-heavy linear blend. Read their paired uncertainty before submitting. Original files are preserved. Initial 01–06 use 8,037 supervised training rows; Ani's reference and the new final linear component use 9,050 train+validation rows after separate held-out evaluation. Reduced temporal fits are never submitted.

## Per-phase baseline and modification

{mdtable(comparison_table)}

![Per-phase ROC and precision-recall curves](experiments/model_search/figures/phase_roc_pr.png)

## Stopping-reason error analysis

Only unsuccessful validation rows are included. Their `trial_id` is matched to Part One's `ID` after outcome prediction. Use visible Part One train/validation labels when available; otherwise use the frozen original DBSCAN/logistic Part One model (157 verified training labels, C=1000, cutoff 0.5 with argmax fallback). The exact source model and checksum are recorded in `reason_model_provenance.json`. **Reason embeddings, reason labels, and membership in Part One never enter an outcome classifier.**

Rows absent from Part One form the separate no-recorded-stopping-reason group. Reasons overlap, so group counts need not sum to the number of failures. Each trial is counted at most once within a phase. Funding has no Phase III examples; its rate is undefined, not zero.

The baseline incorrectly predicts success for 8 of 9 Phase III efficacy-tagged failures. All nine tags are model predictions, so this is an uncertain subgroup signal. To address the baseline's restricted eligibility-only view, the comparisons add clinical and molecular context and nonlinear combinations, then blend probabilities to retain text generalization. The final table shows the modification's actual trade-offs; improvement in overall log loss need not improve every reason category or F1 at a fixed threshold.

{mdtable(category)}

![Baseline and modified false-success rates](experiments/model_search/figures/reason_errors.png)

Per-category rates, Wilson intervals, known/predicted-label counts, and all four candidates are saved in `experiments/model_search/*reason_errors.csv`. Small groups and predicted reason-label errors limit category-level conclusions. The paired trial-level bootstrap is descriptive and uses the same validation population used for selection.

## Reproduce

Open [part2.ipynb](part2.ipynb). It contains the complete module source plus the executed evaluation and figures. Set `RUN_TRAINING=True` to rerun all searches; the default reads the saved results. Install the pinned [requirements.txt](requirements.txt) in a virtual environment.

Run each search script, then `ensemble_search.py`, preserve the selected mapping in `experiments/model_search/selected_ensembles.json`, run `submission_models.py`, `error_analysis.py`, and `reporting.py`. `model_replay_verification.csv` checks independent linear refits and saved-model reloads against both stored validation and test probabilities. All exported files are checked for 1,500 unique IDs in sample-submission order, finite probabilities strictly between 0 and 1, exact column names, and CSV round-trip agreement.

For inference from the saved bundle: `python submission_models.py --input outcome_test.csv --output replayed_predictions`.

This directory supplies the Part Two implementation and report material. Integrate it with the team's Part One results in the required project report (maximum five pages, excluding references).
'''
    (ROOT/'README.md').write_text(report)
    short=manifest[['submission','validation_logloss','temporal_logloss','kaggle_score']]
    followup_path=ROOT/'submissions/followup_index.csv'
    followup_text='\n\n## Historical follow-ups — on hold\n\nHold **05/06** after the Ani comparison. The new chronological benchmark favors Ani over our linear endpoint.\n\n'+mdtable(pd.read_csv(followup_path)[['submission','validation_logloss','temporal_logloss','kaggle_score']]) if followup_path.exists() else ''
    ani_path=ROOT/'submissions/ani_candidates_index.csv'
    ani_text='\n\n## Optional Ani-anchored candidates\n\n07 blends 75% of the known 0.58688 reference with 25% of Ani\'s capped Phase III trend alternative. It has small favorable point differences across both benchmark tracks, but temporal intervals include zero. 08 is an Ani-heavy linear blend: it improves grouped CV but is less consistent across time. Neither has a known Kaggle score. Prefer 07 if spending another exploratory submission; retain Ani as the confirmed best. See [the benchmark report](../experiments/local_benchmark/RESULTS.md).\n\n'+mdtable(pd.read_csv(ani_path)[['submission','mean_abs_change_from_ani','kaggle_score']]) if ani_path.exists() else ''
    (ROOT/'submissions/README.md').write_text('# Part Two manual Kaggle submissions\n\nLower log loss is better. Best reported reference: **0.58688**. Our 03 scored **0.59562**, then 02 improved to **0.58993**. Initial 01/04 remain unscored. **Hold 05/06; use the new fixed benchmark for future comparisons.**\n\n'+mdtable(short)+followup_text+ani_text+'\n\nAll numbered files have exactly 1,500 rows and columns `ID,label`. Index CSVs are bookkeeping and should not be uploaded. Leaderboard results are tied to file hashes in `leaderboard_results.json`; exact recipes are in the corresponding manifest JSONs.\n')
    print('Wrote report and',len(list(FIGURES.glob('*.png'))),'figures')

if __name__=='__main__':run()


Overwriting src/reporting.py


## `src/reproduce_ani.py`

SHA-256: `935de49af0c0bca694778ed0a86b737bd6fb32e300fd8b40de16fc2b035d480e`

In [48]:
%%writefile src/reproduce_ani.py
"""Reproduce Ani's validation model and exact train+validation submission fit."""
import os
os.environ.setdefault('OMP_NUM_THREADS','4')
os.environ.setdefault('OPENBLAS_NUM_THREADS','4')
import json,time
import joblib
import numpy as np
import pandas as pd
from experiments import ROOT,load_data,metrics
from ani_models import AniFeatures,AniStack,BASES

OUT=ROOT/'experiments/ani_comparison'


def run():
    data=load_data();all_df=pd.concat(list(data.values()),ignore_index=True)
    path=OUT/'transductive_features.joblib'
    if path.exists():space=joblib.load(path)
    else:
        space=AniFeatures().fit(all_df);joblib.dump(space,path,compress=3)
    rows=[]
    for population in ['train','trainval']:
        path=OUT/f'ani_exact_{population}.joblib'
        fit=data['train'] if population=='train' else pd.concat([data['train'],data['val']],ignore_index=True)
        start=time.time()
        if path.exists():model=joblib.load(path)
        else:
            model=AniStack(tau=3).fit(fit,features=space)
            joblib.dump(model,path,compress=3)
        target=data['val'] if population=='train' else data['test']
        p=model.predict_proba(target)
        pd.DataFrame({'ID':target.ID,'probability':p}).to_csv(OUT/f'ani_exact_{population}_predictions.csv',index=False)
        np.savez_compressed(OUT/f'ani_exact_{population}_bases.npz',oof=model.oof_base_predictions,pred=model.predict_base_matrix(target))
        row={'model':f'ani_exact_{population}','training_rows':len(fit),'seconds':time.time()-start}
        if population=='train':row.update(metrics(target,p))
        else:
            reference=pd.read_csv(OUT/'source/part2_best_submission.csv')
            expected=reference.set_index('ID').loc[target.ID,'label'].to_numpy()
            difference=np.abs(np.clip(p,.001,.999)-expected)
            row.update(max_submission_probability_difference=float(difference.max()),mean_submission_probability_difference=float(difference.mean()))
        rows.append(row)
        print(row,flush=True)
        print('Meta coefficients',dict(zip([x[0] for x in BASES]+['I','II','III'],model.meta.coef_[0])),flush=True)
        pd.DataFrame(rows).to_csv(OUT/'exact_reproduction.csv',index=False)
    (OUT/'exact_reproduction.json').write_text(json.dumps(rows,indent=2))

if __name__=='__main__':run()


Overwriting src/reproduce_ani.py


## `src/sprint_benchmark.py`

SHA-256: `123448b2a5533adaa040346a81c5ceff7f1a8ec99c16fdc0cacf79c577324bab`

In [49]:
%%writefile src/sprint_benchmark.py
"""Score the frozen sprint batch using verified existing base-model packets."""
import os
for key in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ.setdefault(key, "2")
import argparse
import hashlib
import importlib.metadata
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from benchmark_protocol import initialize, resolve, digest, DIRECTORY
from experiments import ROOT
from extension_common import previous_packet, current as extension_current
from extension_followthrough import DECISION
from extension_neighbors import fit_recipes as fit_neighbors
from improvement_models import fit_recipes as fit_original
from improvement_report import audit_track
from local_benchmark import predict_and_score
from sprint_models import RECIPES, BASELINE, predict_recipes

OUT = ROOT / "experiments/leaderboard_sprint"
OLD_ALL = ROOT / "experiments/all_submissions/runs/8f68385e25e07615"
CONTROLS = [BASELINE, "ani_stack", "ani_first", "sharp", "trend", "legacy01", "legacy03"]


def register():
    universe, protocol = initialize()
    gaps = json.loads((DECISION / "gaps/protocol.json").read_text())
    # Pin original implementations and old packets, without rewriting their runs.
    old = json.loads((ROOT / "experiments/stack_improvements/runs/097ac8161372126d/manifest.json").read_text())
    for name, expected in old["source_sha256"].items(): assert digest(ROOT/name) == expected, name
    for name, expected in json.loads((extension_current("neighbors") / "manifest.json").read_text())["source_sha256"].items():
        assert digest(ROOT/name) == expected, name
    manifest = dict(recipes=RECIPES, baseline="10_ani_fast2_neighbors_half.csv", baseline_score=.58609,
        baseline_sha256=digest(ROOT / "submissions/10_ani_fast2_neighbors_half.csv"),
        policy_sha256=digest(ROOT/"SPRINT_PROTOCOL.md"), protocol_sha256=digest(DIRECTORY/"protocol.json"),
        gaps_sha256=digest(DECISION/"gaps/protocol.json"),
        source_sha256={name:digest(ROOT/name) for name in ["sprint_models.py", "sprint_benchmark.py"]},
        versions={p:importlib.metadata.version(p) for p in ["numpy","pandas","scipy","scikit-learn","joblib"]})
    key = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:16]
    out = OUT / "runs" / key; out.mkdir(parents=True, exist_ok=True)
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2)+"\n")
    (OUT / "current_run.json").write_text(json.dumps(dict(run=key), indent=2)+"\n")
    return universe, protocol, gaps, out


def current():
    return OUT / "runs" / json.loads((OUT / "current_run.json").read_text())["run"]


def packets(track, split, is_gap):
    old = previous_packet(track, split)
    root = ROOT / "experiments/ani_extensions/cache"
    extra_path = root / "decision" / track / f"{split['name']}.joblib" if is_gap else root / extension_current("neighbors").name / track / f"{split['name']}.joblib"
    return old, joblib.load(extra_path), extra_path


def components(old, extra, train, ev, track, split, is_gap):
    base, _ = fit_original(old, train, ev)
    neighbors, _ = fit_neighbors(old, extra, train, ev, base)
    p = dict(file10=neighbors["fast2_neighbors_half"], ani_stack=base["ani_stack"], ani_first=base["ani_first"],
             sharp=neighbors["replace_knn5"], trend=base["ani_phase3_trend"],
             fast2_neighbors=neighbors["fast2_neighbors"], recent_half=neighbors["recent_neighbors_half"],
             unweighted_nn=neighbors["unweighted_neighbors"])
    folder = OLD_ALL / "gaps" / track if is_gap else OLD_ALL / track
    for name, alias in [("legacy01", "01_balanced_ensemble"), ("legacy03", "03_phase_specialized")]:
        d = pd.read_csv(folder / "predictions" / split["name"] / f"{alias}.csv")
        assert d.ID.tolist() == ev.ID.tolist()
        p[name] = d.probability.to_numpy()
    for name, alias in [("file10", "10_ani_fast2_neighbors_half"), ("ani_stack", "ani_stack"), ("sharp", "09_ani_replace_knn5")]:
        d = pd.read_csv(folder / "predictions" / split["name"] / f"{alias}.csv")
        assert d.ID.tolist() == ev.ID.tolist()
        assert np.max(np.abs(d.probability-p[name])) < 1e-10, (track, split["name"], name)
    return p


def summarize(out, universe, protocol, gaps):
    for track in ["strict", "published"]:
        main, scores, _ = audit_track(out/track, universe, protocol, names=CONTROLS+list(RECIPES))
        gap, gap_scores, _ = audit_track(out/"gaps"/track, universe, gaps, names=CONTROLS+list(RECIPES))
        table = main.join(gap[["temporal","worst"]].rename(columns=dict(temporal="gap_mean",worst="gap_worst")))
        baseline = table.loc[BASELINE]
        for key in ["official","temporal","worst","cv","gap_mean","gap_worst"]:
            table[key+"_delta"] = table[key]-baseline[key]
        table["time_average_delta"] = .5*(table.temporal_delta+table.gap_mean_delta)
        table["eligible"] = (table.temporal_delta.lt(0) & table.gap_mean_delta.lt(0) &
            (table.temporal_delta.le(-.0002) | table.gap_mean_delta.le(-.0002)) & table.cv_delta.le(.001))
        table.to_csv(out / f"summary_{track}.csv")
        scores.to_csv(out/track/"fold_scores.csv",index=False)
        gap_scores.to_csv(out/"gaps"/track/"fold_scores.csv",index=False)
        print(track,table.sort_values("time_average_delta")[["temporal","gap_mean","cv","time_average_delta","eligible"]].head(18).to_string(),flush=True)


def main():
    parser=argparse.ArgumentParser();parser.add_argument("--track",choices=["strict","published"],required=True)
    parser.add_argument("--summarize",action="store_true");args=parser.parse_args()
    universe, protocol, gaps, out = register()
    if args.summarize:
        summarize(out,universe,protocol,gaps);return
    for is_gap, spec in [(False,protocol),(True,gaps)]:
        folder=out/"gaps"/args.track if is_gap else out/args.track
        for split in spec["splits"]:
            if (folder/f"scores_{split['name']}.csv").exists():continue
            train,ev=resolve(universe,split);old,extra,extra_path=packets(args.track,split,is_gap)
            assert old["training_ids"]==extra["train_ids"]==train.ID.tolist()
            assert old["evaluation_ids"]==extra["eval_ids"]==ev.ID.tolist()
            p=components(old,extra,train,ev,args.track,split,is_gap)
            candidate,_=predict_recipes(old,extra,train,ev,p)
            predict_and_score(folder,split,ev,{**{k:p[k] for k in CONTROLS},**candidate})
            audits=folder/"audits";audits.mkdir(exist_ok=True)
            (audits/f"{split['name']}.json").write_text(json.dumps(dict(train_ids=train.ID.tolist(),eval_ids=ev.ID.tolist(),
                extra_packet_sha256=digest(extra_path),old_fold_audit=old["audit"],extra_fold_audit=extra["audit"])))
            print(args.track,split["name"],"complete",flush=True)
    print("Finished",args.track,out,flush=True)


if __name__=="__main__":main()


Overwriting src/sprint_benchmark.py


## `src/sprint_export.py`

SHA-256: `ac57ad77a6e1a3ebd5e711509414b678fb642bd310ae2b320dd158669469a0fa`

In [50]:
%%writefile src/sprint_export.py
"""Export an evaluated sprint recipe and independently replay native inference."""
import os
for key in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ.setdefault(key, "2")
import argparse
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from ani_models import BASES, predict_base
from benchmark_protocol import initialize, digest
from experiments import ROOT, load_data
from extension_neighbors import RECIPES as NEIGHBOR_RECIPES, neighbor_predictions, x_meta, fit_recipes as fit_neighbors
from improvement_models import RECIPES as ORIGINAL_RECIPES, fit_recipes as fit_original, meta_inputs, recency_weights
from sprint_models import RECIPES, design
from sprint_benchmark import current, OUT

BUNDLE = ROOT / "experiments/ani_extensions/cache/decision/final/submission_bundle.joblib"
LEGACY = OUT / "cache/legacy_full.joblib"


def original_components(bundle, old, extra, df):
    p = {name: bundle["base_metas"][name].predict_proba(meta_inputs(old["bank"], "eval", df,
            ORIGINAL_RECIPES[name], old["year_median"], old["year_cap"]))[:, 1]
         for name in ["ani_stack", "ani_first", "ani_phase3_trend"]}
    for name, recipe in NEIGHBOR_RECIPES.items():
        if "blend" in recipe: p[name] = sum(w*p[k] for k,w in recipe["blend"].items())
        else: p[name] = bundle["neighbor_metas"][name].predict_proba(x_meta(old, extra, df, "eval", recipe))[:, 1]
    return dict(file10=p["fast2_neighbors_half"], ani_stack=p["ani_stack"], ani_first=p["ani_first"],
        sharp=p["replace_knn5"], trend=p["ani_phase3_trend"], fast2_neighbors=p["fast2_neighbors"],
        recent_half=p["recent_neighbors_half"], unweighted_nn=p["unweighted_neighbors"])


def native_packets(bundle, df):
    source=bundle["old"]; features=source["features"].transform(df)
    banks={key:{"eval":np.column_stack([predict_base(name, blocks, model, features, df)
        for (name,blocks,_),model in zip(BASES,models)])} for key,models in source["models"].items()}
    old=dict(bank=banks,year_median=source["year_median"],year_cap=source["year_cap"])
    extra=bundle["extra"]; fe=extra["features"].transform(df)
    new=dict(fast={key:{"eval":np.column_stack([predict_base(name,blocks,model,fe,df)
        for (name,blocks,_),model in zip(BASES,models)])} for key,models in extra["models"].items()},
        neighbors={"eval":neighbor_predictions(extra["neighbor_train_features"],fe,extra["neighbor_train"],df)})
    return old,new


def infer(packet, df, native=True):
    bundle=joblib.load(BUNDLE)
    old,extra=native_packets(bundle,df) if native else (bundle["old"],bundle["extra"])
    p=original_components(bundle,old,extra,df)
    recipe=packet["recipe"]
    if any(k.startswith("legacy") for k in recipe.get("blend",{})):
        saved=joblib.load(LEGACY)
        for name,sha in saved["source_sha256"].items(): assert digest(ROOT/name)==sha
        legacy=saved["model"].predict(df)
        p.update(legacy01=legacy["01_balanced_ensemble"],legacy03=legacy["03_phase_specialized"])
    if "blend" in recipe: result=sum(w*p[key] for key,w in recipe["blend"].items())
    else: result=.5*p["ani_stack"]+.5*packet["meta"].predict_proba(design(old,extra,df,"eval",recipe))[:,1]
    return np.clip(result,.001,.999)


def export(name, filename):
    out=current(); train,_=initialize(); test=load_data()["test"]
    recipe=RECIPES[name]
    tables={t:pd.read_csv(out/f"summary_{t}.csv").set_index("model") for t in ["strict","published"]}
    assert name in tables["strict"].index
    bundle=joblib.load(BUNDLE)
    assert bundle["old"]["training_ids"]==bundle["extra"]["train_ids"]==train.ID.tolist()
    packet=dict(name=name,recipe=recipe,training_ids=train.ID.tolist(),meta=None)
    if "blend" not in recipe:
        x=design(bundle["old"],bundle["extra"],train,"oof",recipe)
        packet["meta"]=LogisticRegression(C=recipe["C"],max_iter=3000).fit(x,train.label,
            sample_weight=recency_weights(train,recipe["tau"],bundle["old"]["year_median"]))
    if any(k.startswith("legacy") for k in recipe.get("blend",{})):
        legacy=joblib.load(LEGACY)
        assert legacy["model"].training_ids==train.ID.tolist()
    cache=OUT/"cache"/out.name;cache.mkdir(parents=True,exist_ok=True)
    model_path=cache/f"{name}.joblib";joblib.dump(packet,model_path,compress=3)
    direct=infer(packet,test,native=False)
    changed=test.copy();changed["label"]=0;changed["ID"]="unused";changed["trial_id"]="unused"
    replay=infer(joblib.load(model_path),changed,native=True)
    error=float(np.max(np.abs(direct-replay)));assert error<1e-12,error
    # Confirm the winner remains the same input artifact.
    control=original_components(bundle,bundle["old"],bundle["extra"],test)["file10"]
    saved=pd.read_csv(ROOT/"submissions/10_ani_fast2_neighbors_half.csv")
    assert saved.ID.tolist()==test.ID.tolist()
    assert np.max(np.abs(saved.label-np.clip(control,.001,.999)))<1e-14
    sample=pd.read_csv(ROOT/"part2_sample_submission.csv")
    frame=pd.DataFrame(dict(ID=test.ID,label=direct)).set_index("ID").loc[sample.ID].reset_index()
    assert frame.ID.equals(sample.ID) and frame.ID.is_unique and len(frame)==1500
    assert np.isfinite(frame.label).all() and frame.label.between(0,1,inclusive="neither").all()
    dest=ROOT/"submissions"/filename
    if dest.exists(): assert np.max(np.abs(pd.read_csv(dest).label-frame.label))<1e-14
    else: frame.to_csv(dest,index=False)
    record=dict(file=dest.name,model=name,recipe=recipe,sha256=digest(dest),kaggle_score=None,
        training_rows=len(train),native_replay_max_error=error,baseline_sha256=digest(ROOT/"submissions/10_ani_fast2_neighbors_half.csv"),
        original_bundle_sha256=digest(BUNDLE),model_bundle_path=str(model_path.relative_to(ROOT)),model_bundle_sha256=digest(model_path),
        source_sha256={f:digest(ROOT/f) for f in ["sprint_export.py","sprint_models.py"]},
        benchmark_manifest_sha256=digest(out/"manifest.json"),metrics={t:tables[t].loc[name].to_dict() for t in tables})
    if any(k.startswith("legacy") for k in recipe.get("blend",{})): record["legacy_bundle_sha256"]=digest(LEGACY)
    folder=out/"exports";folder.mkdir(exist_ok=True)
    (folder/(dest.stem+".json")).write_text(json.dumps(record,indent=2)+"\n")
    print(json.dumps(record,indent=2),flush=True)


if __name__=="__main__":
    parser=argparse.ArgumentParser();parser.add_argument("model",choices=list(RECIPES));parser.add_argument("filename")
    args=parser.parse_args();export(args.model,args.filename)


Overwriting src/sprint_export.py


## `src/sprint_finish.py`

SHA-256: `f96af64a249bcb7d99828d409697d75e3c8b04bc446004741beca2cc75f49b0c`

In [51]:
%%writefile src/sprint_finish.py
"""Nine bounded final-slot variations, frozen before file 15's Kaggle result."""
import argparse
import hashlib
import json
import numpy as np
import pandas as pd
from benchmark_protocol import initialize,resolve,digest,DIRECTORY
from experiments import ROOT,load_data
from improvement_report import audit_track,paired_interval
from local_benchmark import predict_and_score
from sprint_refine import load_components,native_components,blend,DEPENDENCIES,current as refine_current
from sprint_benchmark import OUT,DECISION
from sprint_export import BUNDLE,LEGACY
from sprint_tree_export import current as tree_current

CONTROLS={"file11":dict(legacy=.2),"file13":dict(legacy=.3),"file15":dict(legacy=.2,tree=.1,trend=.1)}
RECIPES={f"path{a:g}":dict(legacy=.3-.1*a,tree=.1*a,trend=.1*a) for a in [.25,.5,.75,1.25,1.5]}
for t in [.05,.15]:RECIPES[f"trend{t:g}"]=dict(legacy=.2,tree=.1,trend=t)
for t in [.075,.125]:RECIPES[f"tree{t:g}"]=dict(legacy=.2,tree=t,trend=.1)
NAMES=list(CONTROLS)+list(RECIPES)


def register():
    universe,protocol=initialize();gaps=json.loads((DECISION/"gaps/protocol.json").read_text())
    previous=refine_current();old=json.loads((previous/"manifest.json").read_text())
    for file,sha in old["source_sha256"].items():assert digest(ROOT/file)==sha
    manifest=dict(recipes=RECIPES,controls=CONTROLS,source_sha256={f:digest(ROOT/f) for f in DEPENDENCIES+["sprint_finish.py"]},
        policy_sha256=digest(ROOT/"FINISH_PROTOCOL.md"),protocol_sha256=digest(DIRECTORY/"protocol.json"),
        gaps_sha256=digest(DECISION/"gaps/protocol.json"),parent=str(previous.relative_to(ROOT)),
        parent_manifest_sha256=digest(previous/"manifest.json"),file15_score_at_declaration=None,
        file15_score_before_evaluation=.58492,
        controls_sha256={name:digest(ROOT/"submissions"/file) for name,file in [
            ("file11","11_file10_legacy20.csv"),("file13","13_file10_legacy30.csv"),("file15","15_file10_legacy20_tree10_trend10.csv")]})
    key=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()[:16]
    out=OUT/"finish"/key;out.mkdir(parents=True,exist_ok=True)
    path=out/"manifest.json"
    if path.exists():assert json.loads(path.read_text())==manifest
    else:path.write_text(json.dumps(manifest,indent=2)+"\n")
    (OUT/"finish_current.json").write_text(json.dumps(dict(run=key),indent=2)+"\n")
    return universe,protocol,gaps,out


def current():return OUT/"finish"/json.loads((OUT/"finish_current.json").read_text())["run"]


def benchmark():
    universe,protocol,gaps,out=register()
    for track in ["strict","published"]:
        all_scores=[];tables=[]
        for is_gap,spec in [(False,protocol),(True,gaps)]:
            folder=out/"gaps"/track if is_gap else out/track
            for split in spec["splits"]:
                train,ev=resolve(universe,split);p,hashes=load_components(track,split,is_gap,ev)
                predictions={name:blend(p,r) for name,r in (CONTROLS|RECIPES).items()}
                parent=refine_current()/"gaps"/track if is_gap else refine_current()/track
                for name,alias in [("file13","file13"),("file15","legacy0.2_tree0.1_trend0.1")]:
                    frame=pd.read_csv(parent/"predictions"/split["name"]/f"{alias}.csv")
                    assert frame.ID.tolist()==ev.ID.tolist()
                    assert np.max(np.abs(frame.probability-predictions[name]))<4e-15
                predict_and_score(folder,split,ev,predictions)
                audits=folder/"audits";audits.mkdir(exist_ok=True)
                (audits/f"{split['name']}.json").write_text(json.dumps(dict(train_ids=train.ID.tolist(),eval_ids=ev.ID.tolist(),component_sha256=hashes))+"\n")
            summary,scores,_=audit_track(folder,universe,spec,names=NAMES)
            scores.to_csv(folder/"fold_scores.csv",index=False);all_scores.append(scores);tables.append(summary)
        table=tables[0].join(tables[1][["temporal","worst"]].rename(columns=dict(temporal="gap_mean",worst="gap_worst")))
        latest=all_scores[0][all_scores[0].split.eq("time_2011_to_2012_2013")].set_index("model").phase_mean_logloss
        table["latest"]=latest
        for ref in ["file13","file15"]:
            for key in ["official","temporal","gap_mean","gap_worst","cv","latest"]:
                table[key+"_vs_"+ref]=table[key]-table.loc[ref,key]
        table.to_csv(out/f"summary_{track}.csv")
        print(track,table[["temporal","gap_mean","cv","temporal_vs_file15","gap_mean_vs_file15","gap_worst_vs_file15"]].to_string(),flush=True)
    print(out,flush=True)


if __name__=="__main__":
    parser=argparse.ArgumentParser();parser.add_argument("action",choices=["benchmark"])
    args=parser.parse_args();benchmark()


Overwriting src/sprint_finish.py


## `src/sprint_finish_report.py`

SHA-256: `b18cccd4baad33a4f07fa69fd2b1caa2aab981071f866b2db6efcf4ed95edc53`

In [52]:
%%writefile src/sprint_finish_report.py
"""Verify and retain all final-slot alternatives, including rejected directions."""
import json
import numpy as np
import pandas as pd
from benchmark_protocol import initialize,resolve,digest,DIRECTORY
from experiments import ROOT
from improvement_report import audit_track
from sprint_finish import current,NAMES,CONTROLS,RECIPES
from sprint_benchmark import DECISION
from sprint_report import table


def run():
    out=current();universe,protocol=initialize();gaps=json.loads((DECISION/"gaps/protocol.json").read_text())
    manifest=json.loads((out/"manifest.json").read_text())
    for file,sha in manifest["source_sha256"].items():assert digest(ROOT/file)==sha
    assert digest(ROOT/"FINISH_PROTOCOL.md")==manifest["policy_sha256"]
    assert digest(DIRECTORY/"protocol.json")==manifest["protocol_sha256"]
    assert digest(DECISION/"gaps/protocol.json")==manifest["gaps_sha256"]
    assert digest(ROOT/manifest["parent"]/"manifest.json")==manifest["parent_manifest_sha256"]
    scores_all=[];count=0
    aliases={"file10":"file10","legacy01":"legacy","sharp":"sharp","trend":"trend","recent_tree":"tree"}
    for track in ["strict","published"]:
        summaries=[]
        for is_gap,spec in [(False,protocol),(True,gaps)]:
            folder=out/"gaps"/track if is_gap else out/track
            summary,scores,frames=audit_track(folder,universe,spec,names=NAMES)
            summaries.append(summary);count+=len(scores)
            scores_all.append(scores.assign(track=track,suite="gaps" if is_gap else "main"))
            for split in spec["splits"]:
                train,ev=resolve(universe,split)
                a=json.loads((folder/"audits"/f"{split['name']}.json").read_text())
                assert a["train_ids"]==train.ID.tolist() and a["eval_ids"]==ev.ID.tolist()
                assert not set(train.trial_id)&set(ev.trial_id)
                components={}
                for file,sha in a["component_sha256"].items():
                    path=ROOT/file;assert digest(path)==sha
                    d=pd.read_csv(path)
                    for col in ["ID","trial_id","phase","label"]:assert d[col].tolist()==ev[col].tolist()
                    components[aliases[path.stem]]=d.probability.to_numpy()
                for name,recipe in (CONTROLS|RECIPES).items():
                    expected=(1-sum(recipe.values()))*components["file10"].copy()
                    for key,w in recipe.items():expected+=w*components[key]
                    assert np.max(np.abs(expected-frames[split["name"]][name].probability))<4e-15
        combined=summaries[0].join(summaries[1][["temporal","worst"]].rename(columns=dict(temporal="gap_mean",worst="gap_worst")))
        saved=pd.read_csv(out/f"summary_{track}.csv").set_index("model")
        assert np.allclose(combined.sort_index(),saved.loc[combined.index,combined.columns].sort_index(),rtol=0,atol=1e-12,equal_nan=True)
    pd.concat(scores_all,ignore_index=True).to_csv(out/"all_scores.csv",index=False)
    verification=dict(recipes=len(RECIPES),controls=len(CONTROLS),evaluations=count,
        same_fixed_splits=True,source_and_component_hashes_verified=True,independent_blend_replay=True,
        selected_for_export=None,reason="File 15 lost on Kaggle. A separate sharper-neighbor candidate was prepared, then held because evidence did not justify the final submission slot.")
    (out/"verification.json").write_text(json.dumps(verification,indent=2)+"\n")
    sections=["# Final-slot alternatives",
        "Nine fixed variations were declared while file 15's score was pending. Its **0.58492** result arrived before these candidates were evaluated and is explicitly recorded in the manifest. File 13 remains better at **0.58465**. No base models were refitted.",
        "Local loss continues to favor stronger tree/trend mixtures, even though the tested direction worsened Kaggle. The independent sharper-neighbor file 16 was prepared from the preceding refinement batch, but the final decision is to hold the remaining slot because the evidence is weak. None of this batch's candidates was exported or submitted.",
        "The smaller interpolations between files 13 and 15 improve local averages versus file 13, but their local averaging benefit relative to the weighted endpoint losses is only about 0.00003–0.00005 on the chronological means. This is evidence for the decision to explore a different direction after file 15's 0.00027 Kaggle loss; it is not a prediction of the unknown interpolation scores. All candidate results remain in the tables below."]
    for track in ["strict","published"]:
        d=pd.read_csv(out/f"summary_{track}.csv")
        sections.append("## "+track.capitalize()+"\n\n"+table(d[["model","official","temporal","gap_mean","gap_worst","cv","latest"]]))
    sections.append("[Every period](all_scores.csv), [interpolation diagnostics](interpolation_diagnostics.csv), [verification](verification.json) and [frozen manifest](manifest.json). The same historical labels were reused; these are adaptive comparisons, not independent validation.")
    sections.append("## Reproduce\n\n```sh\n.venv/bin/python part2/sprint_finish.py benchmark\n.venv/bin/python part2/sprint_finish_report.py\n```\n\nThis batch only evaluates local predictions. It neither exports nor uploads a Kaggle submission.")
    (out/"RESULTS.md").write_text("\n\n".join(sections)+"\n")
    print(json.dumps(verification,indent=2),flush=True)


if __name__=="__main__":run()


Overwriting src/sprint_finish_report.py


## `src/sprint_legacy_fit.py`

SHA-256: `6200057221e891e5b0c7f074a25abce678fb1734693ad96c0729c6a469100f1c`

In [53]:
%%writefile src/sprint_legacy_fit.py
"""Reproduce the original ensemble's full-data fit for sprint exports."""
import os
for key in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ.setdefault(key, "4")
import json
import joblib
from all_submission_models import LegacySubmissions
from benchmark_protocol import initialize, digest
from experiments import ROOT
from sprint_export import LEGACY


def run():
    train, _ = initialize()
    sources = {name: digest(ROOT/name) for name in [
        "all_submission_models.py", "benchmark_models.py", "experiments.py", "boosted.py"]}
    if LEGACY.exists():
        packet = joblib.load(LEGACY)
        assert packet["source_sha256"] == sources
        assert packet["model"].training_ids == train.ID.tolist()
        action = "Verified existing full-data fit"
    else:
        model = LegacySubmissions().fit(train, progress=lambda msg: print(msg, flush=True))
        packet = dict(model=model, source_sha256=sources)
        LEGACY.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(packet, LEGACY, compress=3)
        action = "Fitted original ensemble on full labeled data"
    print(json.dumps(dict(action=action, training_rows=len(train),
                         path=str(LEGACY.relative_to(ROOT)), sha256=digest(LEGACY)), indent=2))


if __name__ == "__main__":
    run()


Overwriting src/sprint_legacy_fit.py


## `src/sprint_models.py`

SHA-256: `3a5ff9cf01468bd5c46985fcb6fc1347be9ed9355edcccbceff790079077a0c7`

In [54]:
%%writefile src/sprint_models.py
"""Frozen fast iteration around the scored 0.58609 file 10."""
import numpy as np
from sklearn.linear_model import LogisticRegression
from extension_neighbors import x_meta
from improvement_models import recency_weights
from ani_models import onehot

BASELINE = "file10"
RECIPES = {}
for weight in [.25, .35, .65, .75, 1.]:
    RECIPES[f"fast_weight{weight:g}"] = dict(blend={"ani_stack": 1-weight, "fast2_neighbors": weight})
for name, weights in [("sharp", [.25, .5, .75]), ("trend", [.1, .25, .5]),
                      ("recent_half", [.25, .5, .75]), ("unweighted_nn", [.1, .25]),
                      ("legacy01", [.1, .2, .3]), ("legacy03", [.1, .2])]:
    for weight in weights:
        RECIPES[f"mix_{name}_{weight:g}"] = dict(blend={BASELINE: 1-weight, name: weight})
RECIPES["mix_sharp_trend"] = dict(blend={BASELINE: .5, "sharp": .25, "trend": .25})
for C, tau in [(.1, 2.), (.6, 2.), (1., 2.), (.3, 1.5), (.3, 2.5), (.3, 3.)]:
    RECIPES[f"meta_C{C:g}_tau{tau:g}"] = dict(meta=True, C=C, tau=tau, drop_knn=False, interactions=False)
for C in [.1, .3]:
    RECIPES[f"meta_drop_knn_C{C:g}"] = dict(meta=True, C=C, tau=2., drop_knn=True, interactions=False)
for C in [.05, .15]:
    RECIPES[f"meta_phase_C{C:g}"] = dict(meta=True, C=C, tau=2., drop_knn=False, interactions=True)


def design(old, extra, df, side, recipe):
    x = x_meta(old, extra, df, side, dict(bank="fast2", tau=2., trend=False, C=recipe["C"], neighbors=True))
    if recipe["drop_knn"]:
        x = np.delete(x, 7, axis=1)
    if recipe["interactions"]:
        # Shared effects plus strongly regularized phase-specific deviations.
        ph = onehot(df)
        x = np.column_stack([x] + [x*ph[:, i, None] for i in range(3)])
    return x


def predict_recipes(old, extra, train, ev, components):
    result = {}; fitted = {}
    for name, recipe in RECIPES.items():
        if "blend" in recipe:
            result[name] = sum(weight*components[key] for key, weight in recipe["blend"].items())
        else:
            xt, xe = [design(old, extra, df, side, recipe) for df, side in [(train, "oof"), (ev, "eval")]]
            weights = recency_weights(train, recipe["tau"], old["year_median"])
            model = LogisticRegression(C=recipe["C"], max_iter=3000).fit(xt, train.label, sample_weight=weights)
            result[name] = .5*components["ani_stack"] + .5*model.predict_proba(xe)[:, 1]
            fitted[name] = model
    return result, fitted


Overwriting src/sprint_models.py


## `src/sprint_overview.py`

SHA-256: `051c04edca75b5eb0fe75d202e209158c0e655f2990dc721f0751b6753df27a0`

In [55]:
%%writefile src/sprint_overview.py
"""Render the live submission record without changing frozen experiment evidence."""
import json
from pathlib import Path
import pandas as pd
from benchmark_protocol import digest
from experiments import ROOT
from sprint_benchmark import OUT,current as initial_current
from sprint_tree_export import current as tree_current
from sprint_refine import current as refine_current
from sprint_finish import current as finish_current


def run():
    ledger=json.loads((ROOT/"submissions/leaderboard_results.json").read_text())
    queue=json.loads((ROOT/"submissions/sprint_queue.json").read_text())
    for file,r in ledger["submissions"].items():assert digest(ROOT/"submissions"/file)==r["sha256"]
    best=ledger["current_best"]
    mapping=[(10,"10_ani_fast2_neighbors_half.csv",initial_current(),"file10"),
        (11,"11_file10_legacy20.csv",initial_current(),"mix_legacy01_0.2"),
        (12,"12_file10_sharp_trend.csv",initial_current(),"mix_sharp_trend"),
        (13,"13_file10_legacy30.csv",initial_current(),"mix_legacy01_0.3"),
        (14,"14_file10_recent_tree20.csv",tree_current(),"tree20"),
        (15,"15_file10_legacy20_tree10_trend10.csv",refine_current(),"legacy0.2_tree0.1_trend0.1"),
        (16,"16_file10_legacy30_sharp20.csv",refine_current(),"legacy0.3_sharp0.2")]
    rows=[]
    for number,file,path,model in mapping:
        r=pd.read_csv(path/"summary_strict.csv").set_index("model").loc[model]
        score=ledger["submissions"].get(file,{}).get("kaggle_score")
        rows.append(f"| {number} | {r.temporal:.6f} | {r.gap_mean:.6f} | {r.gap_worst:.6f} | {r.cv:.6f} | "+(f"{score:.5f}" if score is not None else "Unknown")+" |")
    sections=["# Part Two submission sprint",
        f"**Best scored submission: `{best['submission']}`, {best['kaggle_score']:.5f}.** Scores are user-reported and tied to exact CSV hashes in the [ledger](../../submissions/leaderboard_results.json). File 11 reached reported first place at 0.58474; file 13 subsequently improved to 0.58465.",
        f"{queue['confirmed_additional_uploads']} of the original four additional uploads were used; the remaining {queue['remaining_from_original_budget']} submission slot was deliberately left unused. The [live queue](../../submissions/sprint_queue.json) records the final hold decision. The user revised the active deadline to 2026-11-07 00:06:49 UTC.",
        "## Direct comparison\n\nAll entries use the same ten original splits plus three longer-gap checks, equal-phase log loss, and pooled repeated grouped CV. Lower is better. These are the primary strict-preprocessing results; both preprocessing tracks and every period remain in the linked reports.\n\n| File | Forward mean | Longer-gap mean | Worst gap | Grouped CV | Kaggle |\n| --- | ---: | ---: | ---: | ---: | ---: |\n"+"\n".join(rows),
        "## What the feedback changed\n\nFile 11's 20% original-ensemble contribution improved file 10 by 0.00135 on Kaggle. Increasing it to 30% in file 13 gained another 0.00009, showing a much smaller incremental benefit. File 15's recent-tree/phase-trend mixture improved local averages but scored 0.58492, losing 0.00027 to file 13. That result argues against increasing those additions solely because historical validation favors them.",
        "**Hold the final remaining submission.** The user confirmed first place with nine total uploads. The prepared file 16 retains 30% original ensemble, replaces 20% of file 10 with sharper-neighbor file 09, and leaves 50% file 10. Both chronological averages improve versus file 13, but the gains are small, intervals span zero, and grouped CV worsens slightly. Its initial exploratory upload recommendation was withdrawn after reviewing the value of the final slot. File 16 remains an unsubmitted reserve; file 13's exact CSV remains intact.",
        "## Experiments retained\n\n"+"\n".join([
            f"- [Initial 32-recipe batch]({initial_current().relative_to(OUT)}/RESULTS.md): 1,014 evaluations including controls.",
            f"- [Four recent-tree mixtures]({tree_current().relative_to(OUT)}/RESULTS.md): 182 evaluations including controls.",
            f"- [19 refinements around files 11 and 13]({refine_current().relative_to(OUT)}/RESULTS.md): 598 evaluations including controls and the final submission choice.",
            f"- [Nine final-slot alternatives]({finish_current().relative_to(OUT)}/RESULTS.md): 312 evaluations including controls; no candidate from this batch was submitted."]),
        "All local labels were repeatedly consulted. Paired bootstrap intervals share trial weights across phases, but do not correct for adaptive model selection. Later Kaggle results are separate observations; they do not retroactively make local comparisons independent.",
        "## Verification and reproduction\n\nExports preserve sample order, 1,500 unique IDs, valid probabilities and exact native inference after evaluation labels/IDs are replaced. All supervised final components use 9,050 labeled rows. Native caches are ignored by Git; source hashes, held-out predictions, component hashes and export receipts are retained. Frozen export receipts keep their pre-submission score state; the live ledger records later scores.\n\n```sh\nmake -C part2 sprint\nmake -C part2 sprint-final\nmake -C part2 sprint-check\nmake -C part2 sprint-refine\nmake -C part2 sprint-refine-check\n```\n\nA fresh checkout must first reproduce the earlier all-submission and final-neighbor native caches. None of these commands uploads to Kaggle."]
    (OUT/"RESULTS.md").write_text("\n\n".join(sections)+"\n")
    print("Updated live sprint overview:",best["submission"],best["kaggle_score"])


if __name__=="__main__":run()


Overwriting src/sprint_overview.py


## `src/sprint_refine.py`

SHA-256: `71ec1fb209e40f553a16a394cebf5bc66a486a297b697e48b8a38a1f48ff0cbb`

In [56]:
%%writefile src/sprint_refine.py
"""Frozen mixtures around the 0.58474 winner, with complete paired evaluation."""
import os
for key in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ.setdefault(key, "2")
import argparse
import hashlib
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from benchmark_protocol import initialize, resolve, digest, DIRECTORY
from experiments import ROOT, load_data
from improvement_report import audit_track, paired_interval
from local_benchmark import predict_and_score
from sprint_benchmark import OUT, DECISION, current as initial_current
from sprint_tree_export import current as tree_current
from sprint_export import BUNDLE, LEGACY, native_packets, original_components
from sprint_tree import dense

CONTROLS = {
    "file10": {}, "file11": {"legacy": .2},
    "file13": {"legacy": .3}, "file14": {"tree": .2},
}
RECIPES = {}
for l in [.25, .35, .4, .5]:
    RECIPES[f"legacy{l:g}"] = dict(legacy=l)
for l in [.2, .3, .4]:
    for t in [.05, .1, .15]:
        RECIPES[f"legacy{l:g}_tree{t:g}"] = dict(legacy=l, tree=t)
for l in [.2, .3]:
    for t in [.05, .1]:
        RECIPES[f"legacy{l:g}_tree{t:g}_trend0.1"] = dict(legacy=l, tree=t, trend=.1)
for s in [.1, .2]:
    RECIPES[f"legacy0.3_sharp{s:g}"] = dict(legacy=.3, sharp=s)
NAMES = list(CONTROLS) + list(RECIPES)
DEPENDENCIES = ["sprint_refine.py", "sprint_export.py", "sprint_models.py", "sprint_tree.py",
                "all_submission_models.py", "benchmark_models.py", "experiments.py", "boosted.py",
                "ani_models.py", "improvement_models.py", "extension_neighbors.py"]


def blend(p, recipe):
    assert all(w >= 0 for w in recipe.values()) and sum(recipe.values()) <= 1
    return (1-sum(recipe.values()))*p["file10"] + sum(w*p[k] for k,w in recipe.items())


def register():
    universe, protocol = initialize()
    gaps = json.loads((DECISION/"gaps/protocol.json").read_text())
    parents = dict(initial=initial_current(), tree=tree_current())
    for path in parents.values():
        m = json.loads((path/"manifest.json").read_text())
        for file, sha in m["source_sha256"].items():
            assert digest(ROOT/file) == sha, file
    manifest = dict(recipes=RECIPES, controls=CONTROLS,
        baseline=dict(file="11_file10_legacy20.csv", kaggle_score=.58474,
                      sha256=digest(ROOT/"submissions/11_file10_legacy20.csv")),
        source_sha256={f:digest(ROOT/f) for f in DEPENDENCIES},
        policy_sha256=digest(ROOT/"REFINE_PROTOCOL.md"),
        protocol_sha256=digest(DIRECTORY/"protocol.json"),
        gaps_sha256=digest(DECISION/"gaps/protocol.json"),
        parents={k:dict(path=str(v.relative_to(ROOT)), manifest_sha256=digest(v/"manifest.json")) for k,v in parents.items()})
    key = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:16]
    out = OUT/"refine"/key; out.mkdir(parents=True, exist_ok=True)
    dest = out/"manifest.json"
    if dest.exists(): assert json.loads(dest.read_text()) == manifest
    else: dest.write_text(json.dumps(manifest, indent=2)+"\n")
    (OUT/"refine_current.json").write_text(json.dumps(dict(run=key), indent=2)+"\n")
    return universe, protocol, gaps, out


def current():
    return OUT/"refine"/json.loads((OUT/"refine_current.json").read_text())["run"]


def load_components(track, split, is_gap, ev):
    parents = [initial_current(), tree_current()]
    parents = [p/"gaps"/track if is_gap else p/track for p in parents]
    mapping = {"file10":(0,"file10"), "legacy":(0,"legacy01"), "sharp":(0,"sharp"),
               "trend":(0,"trend"), "tree":(1,"recent_tree")}
    probabilities = {}; hashes = {}
    for key, (index,name) in mapping.items():
        path = parents[index]/"predictions"/split["name"]/f"{name}.csv"
        frame = pd.read_csv(path)
        for col in ["ID", "trial_id", "phase", "label"]:
            assert frame[col].tolist() == ev[col].tolist(), (key, col)
        probabilities[key] = frame.probability.to_numpy()
        hashes[str(path.relative_to(ROOT))] = digest(path)
    return probabilities, hashes


def benchmark():
    universe, protocol, gaps, out = register()
    for track in ["strict", "published"]:
        for is_gap, spec in [(False,protocol), (True,gaps)]:
            folder = out/"gaps"/track if is_gap else out/track
            for split in spec["splits"]:
                train, ev = resolve(universe, split)
                p, hashes = load_components(track, split, is_gap, ev)
                predictions = {name:blend(p, r) for name,r in (CONTROLS|RECIPES).items()}
                # Independently check the scored winner and previously exported controls.
                original = initial_current()/"gaps"/track if is_gap else initial_current()/track
                for name, alias in [("file11","mix_legacy01_0.2"), ("file13","mix_legacy01_0.3")]:
                    frame = pd.read_csv(original/"predictions"/split["name"]/f"{alias}.csv")
                    assert frame.ID.tolist() == ev.ID.tolist()
                    assert np.max(np.abs(frame.probability-predictions[name])) < 4e-15
                predict_and_score(folder, split, ev, predictions)
                audits = folder/"audits"; audits.mkdir(exist_ok=True)
                (audits/f"{split['name']}.json").write_text(json.dumps(dict(
                    train_ids=train.ID.tolist(), eval_ids=ev.ID.tolist(), component_sha256=hashes))+"\n")
        summarize(out, universe, protocol, gaps, track)
    print("Complete", out, flush=True)


def summarize(out, universe, protocol, gaps, track):
    a,scores,_ = audit_track(out/track, universe, protocol, names=NAMES)
    b,gap_scores,_ = audit_track(out/"gaps"/track, universe, gaps, names=NAMES)
    table = a.join(b[["temporal","worst"]].rename(columns=dict(temporal="gap_mean",worst="gap_worst")))
    for key in ["official","temporal","worst","cv","gap_mean","gap_worst"]:
        table[key+"_delta"] = table[key]-table.loc["file11", key]
    latest = scores[scores.split.eq("time_2011_to_2012_2013")].set_index("model").phase_mean_logloss
    table["latest_delta"] = latest-latest["file11"]
    table["mean_time_delta"] = .5*(table.temporal_delta+table.gap_mean_delta)
    table["eligible"] = (table.temporal_delta.lt(0) & table.gap_mean_delta.lt(0) &
        (table.temporal_delta.le(-.0002)|table.gap_mean_delta.le(-.0002)) &
        table.cv_delta.le(.001) & table.gap_worst_delta.le(.002))
    table.to_csv(out/f"summary_{track}.csv")
    scores.to_csv(out/track/"fold_scores.csv", index=False)
    gap_scores.to_csv(out/"gaps"/track/"fold_scores.csv", index=False)
    print(track,table.sort_values("mean_time_delta")[["temporal","gap_mean","cv","gap_worst_delta","latest_delta","eligible"]].to_string(),flush=True)


def native_components(df, cached):
    bundle = joblib.load(BUNDLE)
    old,extra = (bundle["old"],bundle["extra"]) if cached else native_packets(bundle,df)
    p = original_components(bundle, old, extra, df)
    saved = joblib.load(LEGACY)
    for file,sha in saved["source_sha256"].items(): assert digest(ROOT/file) == sha
    legacy = saved["model"]
    p["legacy"] = legacy.predict(df)["01_balanced_ensemble"]
    tree = joblib.load(OUT/"cache"/tree_current().name/"final_tree.joblib")
    p["tree"] = tree.predict_proba(dense(legacy,df))[:,1]
    return p


def export(name, filename):
    out = current(); universe,_ = initialize(); test = load_data()["test"]
    manifest = json.loads((out/"manifest.json").read_text())
    for file,sha in manifest["source_sha256"].items(): assert digest(ROOT/file) == sha
    assert name in RECIPES
    tables={t:pd.read_csv(out/f"summary_{t}.csv").set_index("model") for t in ["strict","published"]}
    for t,table in tables.items(): assert name in table.index and table.loc[name,"completed_splits"] == 10
    assert joblib.load(LEGACY)["model"].training_ids == universe.ID.tolist()
    original = joblib.load(BUNDLE)
    assert original["old"]["training_ids"] == original["extra"]["train_ids"] == universe.ID.tolist()
    p = native_components(test, cached=True)
    for control,file in [("file10","10_ani_fast2_neighbors_half.csv"),("file11","11_file10_legacy20.csv"),("file13","13_file10_legacy30.csv"),("file14","14_file10_recent_tree20.csv")]:
        saved=pd.read_csv(ROOT/"submissions"/file)
        assert saved.ID.tolist() == test.ID.tolist()
        assert np.max(np.abs(saved.label-np.clip(blend(p,CONTROLS[control]),.001,.999))) < 1e-14
    result=np.clip(blend(p,RECIPES[name]),.001,.999)
    changed=test.copy();changed["ID"]="unused";changed["trial_id"]="unused";changed["label"]=0
    replay=np.clip(blend(native_components(changed,cached=False),RECIPES[name]),.001,.999)
    error=float(np.max(np.abs(result-replay)));assert error<1e-12
    sample=pd.read_csv(ROOT/"part2_sample_submission.csv")
    frame=pd.DataFrame(dict(ID=test.ID,label=result)).set_index("ID").loc[sample.ID].reset_index()
    assert frame.ID.equals(sample.ID) and frame.ID.is_unique and len(frame)==1500
    assert np.isfinite(frame.label).all() and frame.label.between(0,1,inclusive="neither").all()
    dest=ROOT/"submissions"/filename
    if dest.exists(): assert np.max(np.abs(pd.read_csv(dest).label-frame.label))<1e-14
    else: frame.to_csv(dest,index=False)
    dependencies=[BUNDLE,LEGACY,OUT/"cache"/tree_current().name/"final_tree.joblib"]
    receipt=dict(file=filename,model=name,recipe=RECIPES[name],remaining_weight="file10",
        sha256=digest(dest),kaggle_score=None,training_rows=len(universe),native_replay_max_error=error,
        dependencies_sha256={str(p.relative_to(ROOT)):digest(p) for p in dependencies},
        source_sha256=manifest["source_sha256"],benchmark_manifest_sha256=digest(out/"manifest.json"),
        baseline=manifest["baseline"],metrics={t:table.loc[name].to_dict() for t,table in tables.items()})
    folder=out/"exports";folder.mkdir(exist_ok=True)
    (folder/(dest.stem+".json")).write_text(json.dumps(receipt,indent=2)+"\n")
    print(json.dumps(receipt,indent=2),flush=True)


if __name__ == "__main__":
    parser=argparse.ArgumentParser();parser.add_argument("action",choices=["benchmark","export"])
    parser.add_argument("--model",choices=list(RECIPES));parser.add_argument("--filename")
    args=parser.parse_args()
    if args.action=="benchmark": benchmark()
    else:
        assert args.model and args.filename
        export(args.model,args.filename)


Overwriting src/sprint_refine.py


## `src/sprint_refine_report.py`

SHA-256: `3eaf5793dbf7b86bae19346539e97bb49ef26bd785746f0c32e66a608e98abf3`

In [57]:
%%writefile src/sprint_refine_report.py
"""Audit every refinement and compare shortlisted mixtures to scored winners."""
import json
import numpy as np
import pandas as pd
from benchmark_protocol import initialize,resolve,digest,DIRECTORY
from experiments import ROOT
from improvement_report import audit_track,paired_interval
from sprint_refine import current,NAMES,CONTROLS,RECIPES
from sprint_benchmark import DECISION
from sprint_report import table


def run():
    out=current();universe,protocol=initialize()
    gaps=json.loads((DECISION/"gaps/protocol.json").read_text())
    manifest=json.loads((out/"manifest.json").read_text())
    for file,sha in manifest["source_sha256"].items():assert digest(ROOT/file)==sha,file
    assert digest(ROOT/"REFINE_PROTOCOL.md")==manifest["policy_sha256"]
    assert digest(DIRECTORY/"protocol.json")==manifest["protocol_sha256"]
    assert digest(DECISION/"gaps/protocol.json")==manifest["gaps_sha256"]
    for item in manifest["parents"].values():assert digest(ROOT/item["path"]/"manifest.json")==item["manifest_sha256"]
    pairs=[];all_scores=[];count=0
    selected=["legacy0.2_tree0.1_trend0.1","legacy0.3_tree0.05_trend0.1","legacy0.3_sharp0.2"]
    for track in ["strict","published"]:
        for is_gap,spec in [(False,protocol),(True,gaps)]:
            folder=out/"gaps"/track if is_gap else out/track
            _,scores,frames=audit_track(folder,universe,spec,names=NAMES)
            count+=len(scores);all_scores.append(scores.assign(track=track,suite="gaps" if is_gap else "main"))
            for split in spec["splits"]:
                train,ev=resolve(universe,split)
                a=json.loads((folder/"audits"/f"{split['name']}.json").read_text())
                assert a["train_ids"]==train.ID.tolist() and a["eval_ids"]==ev.ID.tolist()
                assert not set(train.trial_id)&set(ev.trial_id)
                components={}
                aliases={"file10":"file10","legacy01":"legacy","sharp":"sharp","trend":"trend","recent_tree":"tree"}
                for file,sha in a["component_sha256"].items():
                    path=ROOT/file;assert digest(path)==sha
                    d=pd.read_csv(path)
                    for col in ["ID","trial_id","phase","label"]:assert d[col].tolist()==ev[col].tolist()
                    components[aliases[path.stem]]=d.probability.to_numpy()
                for name,recipe in (CONTROLS|RECIPES).items():
                    expected=components["file10"].copy()*(1-sum(recipe.values()))
                    for component,weight in recipe.items():expected+=weight*components[component]
                    actual=frames[split["name"]][name].probability.to_numpy()
                    assert np.max(np.abs(expected-actual))<4e-15,name
            for name in ["file13"]+selected:
                for reference in ["file11","file13"]:
                    if name==reference:continue
                    for kind in (["temporal"] if is_gap else ["temporal","group_cv"]):
                        pairs.append(dict(track=track,suite="gaps" if is_gap else "main",**paired_interval(frames,name,reference,kind)))
    pd.concat(all_scores,ignore_index=True).to_csv(out/"all_scores.csv",index=False)
    pair=pd.DataFrame(pairs);pair.to_csv(out/"paired_comparisons.csv",index=False)
    exported=[]
    for path in sorted((out/"exports").glob("*.json")):
        item=json.loads(path.read_text());file=ROOT/"submissions"/item["file"]
        assert digest(file)==item["sha256"] and item["native_replay_max_error"]<1e-12
        assert item["source_sha256"]==manifest["source_sha256"]
        assert item["benchmark_manifest_sha256"]==digest(out/"manifest.json")
        for dependency,sha in item["dependencies_sha256"].items():assert digest(ROOT/dependency)==sha
        data=pd.read_csv(file);sample=pd.read_csv(ROOT/"part2_sample_submission.csv")
        assert data.ID.equals(sample.ID) and data.ID.is_unique and len(data)==1500
        assert np.isfinite(data.label).all() and data.label.between(0,1,inclusive="neither").all()
        exported.append(item["file"])
    verification=dict(recipes=len(RECIPES),controls=len(CONTROLS),evaluations=count,
        same_fixed_splits=True,source_and_component_hashes_verified=True,
        independent_blend_replay=True,exports_verified=exported)
    (out/"verification.json").write_text(json.dumps(verification,indent=2)+"\n")
    ledger=json.loads((ROOT/"submissions/leaderboard_results.json").read_text())
    sections=["# Refinement after file 11 reached 0.58474",
        "The next reported score is **file 13: 0.58465**, a further 0.00009 improvement. The 19 new mixtures were declared before their evaluation. The recipes reuse the exact 13 fixed splits and both original preprocessing policies; no additional base models are fitted.",
        "The stronger 30% legacy weight gave only a small Kaggle gain over 20%. **File 15** retained 20% legacy ensemble and added 10% recent-weighted tree and 10% Ani phase trend; 60% remained file 10. It had the best combined chronological mean among candidates satisfying the declared adverse-gap limit under both policies, but scored **0.58492**, worse than file 13 by 0.00027. The improved historical averages did not transfer on this upload.",
        "The prepared **file 16** keeps the successful 30% legacy weight and replaces 20% of file 10 with the sharper-neighbor file 09, leaving 50% file 10. It improves both chronological means and the latest forward window versus file 13 in both tracks, with small regressions in grouped CV, official loss and the worst gap. It adds neither the recent-tree nor phase-trend endpoint tested by file 15. The separate nine-recipe final-slot batch also evaluated smaller interpolations and nearby tree/trend weights.",
        "**Final decision: hold file 16; do not spend the last slot on current evidence.** The user confirmed first place with nine submissions. The small chronological gains and intervals spanning zero do not give a strong case for this upload, and a single returned score would leave no follow-up slot. The earlier exploratory upload recommendation was withdrawn; the frozen local selection remains as historical evidence, with the later decision in submission_decision.json. All local labels have been consulted repeatedly; bootstrap intervals do not correct for adaptive selection. File 13 at 0.58465 remains the best scored submission."]
    for track in ["strict","published"]:
        d=pd.read_csv(out/f"summary_{track}.csv").set_index("model")
        sections.append("## "+track.capitalize()+"\n\n"+table(d.loc[["file10","file11","file13","file14"]+selected,["official","temporal","gap_mean","gap_worst","cv","latest_delta"]].reset_index()))
    sections.append("## Paired differences from file 13\n\n"+table(pair[pair.track.eq("strict")&pair.reference.eq("file13")][["model","suite","kind","delta","lower","upper"]]))
    score_lines=[]
    for filename in ["11_file10_legacy20.csv","13_file10_legacy30.csv"]+exported:
        record=ledger["submissions"].get(filename)
        score_lines.append(f"- `{filename}`: "+(f"{record['kaggle_score']:.5f} (user-reported)" if record else "unknown Kaggle score"))
    sections.append("## Submission evidence\n\n"+"\n".join(score_lines))
    sections.append("[Every recipe, strict](summary_strict.csv), [published](summary_published.csv), [every period and phase](all_scores.csv), [paired uncertainty](paired_comparisons.csv), [verification](verification.json), and [frozen recipes/source hashes](manifest.json) retain the complete record. Receipts preserve native dependency hashes and their pre-submission score state. See the live queue and ledger for later observations.")
    sections.append("## Reproduce\n\n```sh\n.venv/bin/python part2/sprint_refine.py benchmark\n.venv/bin/python part2/sprint_refine.py export --model legacy0.2_tree0.1_trend0.1 --filename 15_file10_legacy20_tree10_trend10.csv\n.venv/bin/python part2/sprint_refine_report.py\n```\n\nExisting native component caches are required for export; the earlier benchmark workflows reproduce them. All supervised final components use 9,050 labeled rows. No new Kaggle upload is performed by these commands.")
    (out/"RESULTS.md").write_text("\n\n".join(sections)+"\n")
    print(json.dumps(verification,indent=2),flush=True)


if __name__=="__main__":run()


Overwriting src/sprint_refine_report.py


## `src/sprint_report.py`

SHA-256: `1504139e0f2452712ccc38f4d3cf724106caa56ab7fd35e3e7a9c1a1766b4fae`

In [58]:
%%writefile src/sprint_report.py
"""Audit sprint results and report comparisons against the scored file 10."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
from benchmark_protocol import initialize, resolve, digest
from experiments import ROOT
from improvement_report import audit_track, paired_interval
from sprint_benchmark import current, DECISION, CONTROLS
from sprint_models import RECIPES


def table(df):
    def cell(x):
        return f"{x:.6f}" if isinstance(x,(float,np.floating)) else str(x)
    return "\n".join(["| "+" | ".join(df.columns)+" |", "| "+" | ".join(["---"]*len(df.columns))+" |"]+
        ["| "+" | ".join(map(cell,row))+" |" for row in df.itertuples(index=False,name=None)])


def run():
    out=current();universe,protocol=initialize();gaps=json.loads((DECISION/"gaps/protocol.json").read_text())
    manifest=json.loads((out/"manifest.json").read_text())
    for file,sha in manifest["source_sha256"].items():assert digest(ROOT/file)==sha
    assert digest(ROOT/"SPRINT_PROTOCOL.md")==manifest["policy_sha256"]
    names=CONTROLS+list(RECIPES);checked=0;intervals=[];all_scores=[]
    selected=json.loads((out/"selection.json").read_text())
    compare=[selected["first_candidate"],selected["reserve_orthogonal"],selected["reserve_if_first_improves"]]
    for track in ["strict","published"]:
        for is_gap,spec in [(False,protocol),(True,gaps)]:
            folder=out/"gaps"/track if is_gap else out/track
            _,scores,frames=audit_track(folder,universe,spec,names=names)
            checked+=len(scores);all_scores.append(scores.assign(track=track,suite="gaps" if is_gap else "main"))
            for split in spec["splits"]:
                train,ev=resolve(universe,split)
                a=json.loads((folder/"audits"/f"{split['name']}.json").read_text())
                assert a["train_ids"]==train.ID.tolist() and a["eval_ids"]==ev.ID.tolist()
                for field in ["old_fold_audit","extra_fold_audit"]:
                    for f in a[field]:
                        hold=f.get("heldout_ids",f.get("eval_ids"));fit=f["fit_ids"]
                        indexed=train.set_index("ID")
                        assert not set(indexed.loc[fit,"trial_id"])&set(indexed.loc[hold,"trial_id"])
                        assert not set(fit)&set(ev.ID)
                        if track=="strict":assert set(f["preprocessing_fit_ids"])==set(fit)
            for name in compare:
                for kind in (["temporal"] if is_gap else ["temporal","group_cv"]):
                    intervals.append(dict(track=track,suite="gaps" if is_gap else "main",**paired_interval(frames,name,"file10",kind)))
    pd.concat(all_scores,ignore_index=True).to_csv(out/"all_scores.csv",index=False)
    pairs=pd.DataFrame(intervals);pairs.to_csv(out/"paired_comparisons.csv",index=False)
    exports=[]
    for path in sorted((out/"exports").glob("*.json")):
        item=json.loads(path.read_text());p=ROOT/"submissions"/item["file"]
        assert digest(p)==item["sha256"] and item["native_replay_max_error"]<1e-12
        d=pd.read_csv(p);sample=pd.read_csv(ROOT/"part2_sample_submission.csv")
        assert d.ID.equals(sample.ID) and d.ID.is_unique and len(d)==1500
        assert np.isfinite(d.label).all() and d.label.between(0,1,inclusive="neither").all()
        exports.append(item)
    verification=dict(recipes=len(RECIPES),controls=len(CONTROLS),evaluations=checked,
        same_fixed_splits=True,grouped_inner_boundaries_verified=True,source_hashes_match=True,
        exports_verified=len(exports),baseline_kaggle_score=.58609)
    (out/"verification.json").write_text(json.dumps(verification,indent=2)+"\n")
    ledger=json.loads((ROOT/"submissions/leaderboard_results.json").read_text())
    result=ledger["submissions"].get("11_file10_legacy20.csv")
    score_note=(f"File 11 scored **{result['kaggle_score']:.5f}**, with user-reported rank {result['reported_rank']}. "
        "Its exact hash is recorded in the current ledger; the export receipt retains its pre-submission state. "
        "The live queue and ledger track subsequent uploads, including file 13's later score."
        if result else "No new file has a known score until the user returns a Kaggle result.")
    sections=["# Iteration after file 10 scored 0.58609",
        "File 10 is this round's frozen baseline; these are adaptive local experiments. "+score_note,
        "## First candidate\n\nFile 11 uses **80% file 10 + 20% the original balanced ensemble**, with every final component trained on all 9,050 labeled rows. The 30% mixture has the best local means, but 20% limits regression on the latest forward window and worst gap. Use the first Kaggle result before increasing this weight. File 12 is a distinct reserve: 50% file 10, 25% sharper neighbors (file 09), and 25% Ani's Phase III trend control.",
        "The first batch freezes 32 proposals: five fast-neighbor blend weights, complementary mixtures, and ten small meta-model changes. Every recipe uses the same ten original splits plus three longer-gap checks. Strict preprocessing is the primary comparison; the published-recipe sensitivity track preserves each component's original feature policy. Equal-phase log loss is lower-is-better."]
    for track in ["strict","published"]:
        summary=pd.read_csv(out/f"summary_{track}.csv").set_index("model")
        view=summary.loc[["file10"]+compare,["official","temporal","gap_mean","gap_worst","cv"]].reset_index()
        sections.append("## "+track.capitalize()+"\n\n"+table(view))
    sections += ["## Paired uncertainty\n\nThe historical evaluation labels have been reused. Intervals are descriptive and do not correct for adaptive model selection. No local point improvement guarantees a better Kaggle score.\n\n"+table(pairs[pairs.track.eq("strict")][["model","suite","kind","delta","lower","upper"]]),
        "## Artifacts\n\n- [All strict results](summary_strict.csv)\n- [All published-recipe results](summary_published.csv)\n- [Every period and phase](all_scores.csv)\n- [Frozen selection](selection.json)\n- [Verification](verification.json)\n- [Recipe/source manifest](manifest.json)",
        "## Reproduce\n\n```sh\n.venv/bin/python part2/sprint_benchmark.py --track strict\n.venv/bin/python part2/sprint_benchmark.py --track published\n.venv/bin/python part2/sprint_benchmark.py --track strict --summarize\n.venv/bin/python part2/sprint_report.py\n```\n\nNative caches are ignored by Git. Export receipts record exact CSV hashes, recipes, training populations and native replay errors. Original submissions remain unchanged."]
    (out/"RESULTS.md").write_text("\n\n".join(sections)+"\n")
    print(json.dumps(verification,indent=2),flush=True)


if __name__=="__main__":run()


Overwriting src/sprint_report.py


## `src/sprint_tree.py`

SHA-256: `123edf072df10eac1a443042914eb5a8fbc8473299a6121b7f5a09d093bdf84d`

In [59]:
%%writefile src/sprint_tree.py
"""One fixed recent-weighted CatBoost endpoint on the same thirteen splits."""
import os
for key in ["OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","VECLIB_MAXIMUM_THREADS"]:os.environ.setdefault(key,"4")
import argparse
import hashlib
import importlib.metadata
import json
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from benchmark_protocol import initialize,resolve,digest,DIRECTORY
from experiments import ROOT
from boosted import transform_dense
from improvement_models import recency_weights
from improvement_report import audit_track
from local_benchmark import predict_and_score
from sprint_benchmark import OUT,OLD_ALL,current as sprint_current,DECISION

TREE=dict(iterations=1000,depth=6,learning_rate=.035,l2_leaf_reg=6,loss_function="Logloss",
          random_seed=42,thread_count=4,verbose=False,allow_writing_files=False)
RECIPES={"tree10":dict(tree=.1,linear=0.),"tree20":dict(tree=.2,linear=0.),
         "recent_legacy20":dict(tree=.1,linear=.1),"recent_legacy30":dict(tree=.15,linear=.15)}
NAMES=["file10","file11","recent_tree"]+list(RECIPES)


def register():
    universe,protocol=initialize();gaps=json.loads((DECISION/"gaps/protocol.json").read_text())
    old=json.loads((OLD_ALL/"manifest.json").read_text())
    for name,sha in old["source_sha256"].items():assert digest(ROOT/name)==sha
    manifest=dict(tree=TREE,tau=3.,recipes=RECIPES,source_sha256={name:digest(ROOT/name) for name in ["sprint_tree.py","all_submission_models.py","boosted.py","experiments.py"]},
        policy_sha256=digest(ROOT/"SPRINT_TREE_PROTOCOL.md"),protocol_sha256=digest(DIRECTORY/"protocol.json"),
        gaps_sha256=digest(DECISION/"gaps/protocol.json"),sprint_run=str(sprint_current().relative_to(ROOT)),
        versions={p:importlib.metadata.version(p) for p in ["numpy","pandas","scikit-learn","catboost","joblib"]})
    key=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()[:16]
    out=OUT/"recent_tree"/key;out.mkdir(parents=True,exist_ok=True)
    (out/"manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
    (OUT/"recent_tree_current.json").write_text(json.dumps(dict(run=key),indent=2)+"\n")
    return universe,protocol,gaps,out


def dense(model,df):
    blocks=model.features.transform(df)
    return transform_dense(df,blocks,model.svds,model.mask,model.features.year_median).drop(columns="year")


def one(task):
    split,is_gap,out_name=task;out=Path(out_name);parent=out/"gaps" if is_gap else out
    if all((parent/t/f"scores_{split['name']}.csv").exists() for t in ["strict","published"]):return
    started=time.time();universe,_=initialize();train,ev=resolve(universe,split)
    legacy_path=ROOT/"experiments/all_submissions/cache"/OLD_ALL.name/f"{split['name']}.joblib"
    audit_parent=OLD_ALL/"gaps" if is_gap else OLD_ALL
    audit=json.loads((audit_parent/"audits"/f"{split['name']}.json").read_text())
    assert digest(legacy_path)==audit["model_sha256"]
    legacy=joblib.load(legacy_path);assert legacy.training_ids==train.ID.tolist()
    xt,xe=dense(legacy,train),dense(legacy,ev)
    weights=recency_weights(train,3.,legacy.features.year_median)
    cache=OUT/"cache"/out.name;cache.mkdir(parents=True,exist_ok=True)
    path=cache/f"{split['name']}.joblib"
    if path.exists():tree=joblib.load(path)
    else:
        tree=CatBoostClassifier(**TREE).fit(xt,train.label,sample_weight=weights,cat_features=xt.select_dtypes("str").columns.tolist())
        joblib.dump(tree,path,compress=3)
    pt=tree.predict_proba(xe)[:,1];pl=legacy.linear.predict_proba(ev)
    for track in ["strict","published"]:
        source=sprint_current()/"gaps"/track if is_gap else sprint_current()/track
        folder=source/"predictions"/split["name"]
        baseline=pd.read_csv(folder/"file10.csv");previous=pd.read_csv(folder/"mix_legacy01_0.2.csv")
        assert baseline.ID.tolist()==previous.ID.tolist()==ev.ID.tolist()
        p=dict(file10=baseline.probability.to_numpy(),file11=previous.probability.to_numpy(),recent_tree=pt)
        for name,r in RECIPES.items():p[name]=(1-r["tree"]-r["linear"])*p["file10"]+r["tree"]*pt+r["linear"]*pl
        predict_and_score(parent/track,split,ev,p)
    folder=parent/"audits";folder.mkdir(exist_ok=True)
    (folder/f"{split['name']}.json").write_text(json.dumps(dict(train_ids=train.ID.tolist(),eval_ids=ev.ID.tolist(),
        preprocessing_fit_ids=legacy.training_ids,feature_model_sha256=digest(legacy_path),tree_model_sha256=digest(path))))
    print(split["name"],"complete",round(time.time()-started,1),flush=True)


def summarize(out,universe,protocol,gaps):
    for track in ["strict","published"]:
        a,scores,_=audit_track(out/track,universe,protocol,names=NAMES)
        b,gap_scores,_=audit_track(out/"gaps"/track,universe,gaps,names=NAMES)
        table=a.join(b[["temporal","worst"]].rename(columns=dict(temporal="gap_mean",worst="gap_worst")))
        for col in ["official","temporal","worst","cv","gap_mean","gap_worst"]:table[col+"_delta"]=table[col]-table.loc["file10",col]
        table.to_csv(out/f"summary_{track}.csv");scores.to_csv(out/track/"fold_scores.csv",index=False)
        gap_scores.to_csv(out/"gaps"/track/"fold_scores.csv",index=False)
        print(track,table[["official","temporal","gap_mean","cv"]].to_string(),flush=True)


if __name__=="__main__":
    parser=argparse.ArgumentParser();parser.add_argument("--workers",type=int,default=2);args=parser.parse_args()
    universe,protocol,gaps,out=register()
    tasks=[(s,False,str(out)) for s in protocol["splits"]]+[(s,True,str(out)) for s in gaps["splits"]]
    with ProcessPoolExecutor(max_workers=args.workers) as pool:list(pool.map(one,tasks))
    summarize(out,universe,protocol,gaps)


Overwriting src/sprint_tree.py


## `src/sprint_tree_export.py`

SHA-256: `1932c3c8a9407ba87e881dd4075a7bfa7618c3ecf9177aec873d75244b5f842f`

In [60]:
%%writefile src/sprint_tree_export.py
"""Full-data fit and native replay for the fixed recent-tree sprint endpoint."""
import argparse
import json
import time
import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from benchmark_protocol import initialize,digest
from experiments import ROOT,load_data
from improvement_models import recency_weights
from sprint_tree import TREE,RECIPES,dense
from sprint_benchmark import OUT
from sprint_export import BUNDLE,LEGACY,native_packets,original_components


def current():
    return OUT/"recent_tree"/json.loads((OUT/"recent_tree_current.json").read_text())["run"]


def infer(packet,df,native):
    bundle=joblib.load(BUNDLE);legacy=joblib.load(LEGACY)["model"]
    old,extra=native_packets(bundle,df) if native else (bundle["old"],bundle["extra"])
    base=original_components(bundle,old,extra,df)["file10"]
    tree=packet["tree"].predict_proba(dense(legacy,df))[:,1]
    linear=legacy.linear.predict_proba(df)
    recipe=packet["recipe"]
    return np.clip((1-recipe["tree"]-recipe["linear"])*base+recipe["tree"]*tree+recipe["linear"]*linear,.001,.999)


def export(name,filename):
    out=current();train,_=initialize();test=load_data()["test"]
    for track in ["strict","published"]:assert (out/f"summary_{track}.csv").exists(),"Finish every fixed split before export"
    manifest=json.loads((out/"manifest.json").read_text())
    for file,sha in manifest["source_sha256"].items():assert digest(ROOT/file)==sha
    legacy_packet=joblib.load(LEGACY);legacy=legacy_packet["model"]
    assert legacy.training_ids==train.ID.tolist()
    for file,sha in legacy_packet["source_sha256"].items():assert digest(ROOT/file)==sha
    cache=OUT/"cache"/out.name;cache.mkdir(exist_ok=True)
    path=cache/"final_tree.joblib"
    if path.exists():tree=joblib.load(path)
    else:
        start=time.time();x=dense(legacy,train)
        tree=CatBoostClassifier(**TREE).fit(x,train.label,
            sample_weight=recency_weights(train,3.,legacy.features.year_median),cat_features=x.select_dtypes("str").columns.tolist())
        joblib.dump(tree,path,compress=3);print("Final recent tree fitted",round(time.time()-start,1),flush=True)
    packet=dict(tree=tree,recipe=RECIPES[name],training_ids=train.ID.tolist())
    packet_path=cache/f"{name}_final.joblib";joblib.dump(packet,packet_path,compress=3)
    p=infer(packet,test,False)
    changed=test.copy();changed["ID"]="unused";changed["trial_id"]="unused";changed["label"]=0
    q=infer(joblib.load(packet_path),changed,True);error=float(np.max(np.abs(p-q)));assert error<1e-12
    sample=pd.read_csv(ROOT/"part2_sample_submission.csv")
    frame=pd.DataFrame(dict(ID=test.ID,label=p)).set_index("ID").loc[sample.ID].reset_index()
    assert frame.ID.equals(sample.ID) and frame.ID.is_unique and len(frame)==1500
    assert np.isfinite(frame.label).all() and frame.label.between(0,1,inclusive="neither").all()
    dest=ROOT/"submissions"/filename
    if dest.exists():assert np.allclose(pd.read_csv(dest).label,frame.label,rtol=0,atol=1e-14)
    else:frame.to_csv(dest,index=False)
    receipt=dict(file=filename,model=name,recipe=RECIPES[name],tree=TREE,tau=3.,training_rows=len(train),
        sha256=digest(dest),kaggle_score=None,native_replay_max_error=error,
        baseline_sha256=digest(ROOT/"submissions/10_ani_fast2_neighbors_half.csv"),
        original_bundle_sha256=digest(BUNDLE),legacy_bundle_sha256=digest(LEGACY),
        native_model_path=str(packet_path.relative_to(ROOT)),native_model_sha256=digest(packet_path),
        source_sha256={file:digest(ROOT/file) for file in ["sprint_tree_export.py","sprint_tree.py","sprint_export.py"]},
        benchmark_manifest_sha256=digest(out/"manifest.json"),
        metrics={t:pd.read_csv(out/f"summary_{t}.csv").set_index("model").loc[name].to_dict() for t in ["strict","published"]})
    exports=out/"exports";exports.mkdir(exist_ok=True)
    (exports/(dest.stem+".json")).write_text(json.dumps(receipt,indent=2)+"\n")
    print(json.dumps(receipt,indent=2),flush=True)


if __name__=="__main__":
    parser=argparse.ArgumentParser();parser.add_argument("model",choices=list(RECIPES));parser.add_argument("filename")
    args=parser.parse_args();export(args.model,args.filename)


Overwriting src/sprint_tree_export.py


## `src/sprint_tree_report.py`

SHA-256: `39621779f1f5bb97f80bc6dcd32e42adc8cbe2a46664e2178861a0dbe6213c99`

In [61]:
%%writefile src/sprint_tree_report.py
"""Independent audit and paired comparison of the fixed recency-tree batch."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
from benchmark_protocol import initialize,resolve,digest
from experiments import ROOT
from improvement_report import audit_track,paired_interval
from sprint_tree import NAMES,RECIPES
from sprint_benchmark import OUT,OLD_ALL,DECISION
from sprint_tree_export import current
from sprint_report import table


def run():
    out=current();universe,protocol=initialize();gaps=json.loads((DECISION/"gaps/protocol.json").read_text())
    manifest=json.loads((out/"manifest.json").read_text())
    for file,sha in manifest["source_sha256"].items():assert digest(ROOT/file)==sha
    assert digest(ROOT/"SPRINT_TREE_PROTOCOL.md")==manifest["policy_sha256"]
    scores_all=[];pairs=[];count=0
    for track in ["strict","published"]:
        for is_gap,spec in [(False,protocol),(True,gaps)]:
            folder=out/"gaps"/track if is_gap else out/track
            _,scores,frames=audit_track(folder,universe,spec,names=NAMES)
            scores_all.append(scores.assign(track=track,suite="gaps" if is_gap else "main"));count+=len(scores)
            for split in spec["splits"]:
                train,ev=resolve(universe,split);a=json.loads((folder.parent/"audits"/f"{split['name']}.json").read_text())
                assert a["train_ids"]==a["preprocessing_fit_ids"]==train.ID.tolist()
                assert a["eval_ids"]==ev.ID.tolist()
                assert not set(train.trial_id)&set(ev.trial_id)
                cache=OUT/"cache"/out.name/f"{split['name']}.joblib"
                assert digest(cache)==a["tree_model_sha256"]
                parent=OLD_ALL/"gaps"/track if is_gap else OLD_ALL/track
                legacy={}
                for number in ["01_balanced_ensemble","02_text_heavy"]:
                    d=pd.read_csv(parent/"predictions"/split["name"]/f"{number}.csv")
                    assert d.ID.tolist()==ev.ID.tolist();legacy[number]=d.probability.to_numpy()
                linear=2*legacy["02_text_heavy"]-legacy["01_balanced_ensemble"]
                p={name:frames[split["name"]][name].probability.to_numpy() for name in NAMES}
                assert np.allclose(p["file11"],.8*p["file10"]+.2*legacy["01_balanced_ensemble"],rtol=0,atol=4e-15)
                for name,r in RECIPES.items():
                    expected=(1-r["tree"]-r["linear"])*p["file10"]+r["tree"]*p["recent_tree"]+r["linear"]*linear
                    assert np.allclose(expected,p[name],rtol=0,atol=4e-15),name
            for ref in ["file10","file11"]:
                for kind in (["temporal"] if is_gap else ["temporal","group_cv"]):
                    pairs.append(dict(track=track,suite="gaps" if is_gap else "main",**paired_interval(frames,"tree20",ref,kind)))
    pd.concat(scores_all,ignore_index=True).to_csv(out/"all_scores.csv",index=False)
    paired=pd.DataFrame(pairs);paired.to_csv(out/"paired_comparisons.csv",index=False)
    exports=[]
    for path in sorted((out/"exports").glob("*.json")):
        item=json.loads(path.read_text());file=ROOT/"submissions"/item["file"]
        assert digest(file)==item["sha256"] and item["native_replay_max_error"]<1e-12
        data=pd.read_csv(file);sample=pd.read_csv(ROOT/"part2_sample_submission.csv")
        assert data.ID.equals(sample.ID) and data.ID.is_unique and len(data)==1500
        assert np.isfinite(data.label).all() and data.label.between(0,1,inclusive="neither").all()
        exports.append(item["file"])
    result=dict(new_tree_settings=1,blend_recipes=len(RECIPES),controls=3,evaluations=count,
        identical_fixed_splits=True,training_only_preprocessing_verified=True,source_hashes_match=True,
        independent_blend_replay=True,exports_verified=exports)
    (out/"verification.json").write_text(json.dumps(result,indent=2)+"\n")
    text=["# Recent-weighted tree follow-up",
        "One fixed recency-weighted CatBoost model is fitted per original split. Four prespecified probability blends are compared to files 10 and 11. No settings were adjusted inside this batch. This is adaptive follow-up research using previously consulted labels.",
        "File 14 (`tree20`) is 80% file 10 plus 20% the new depth-6 tree. Its stronger standard forward mean and latest-period performance come with a regression on the worst individual longer-gap window. Compare both the average and that adverse period before interpreting the result. The Kaggle score is unknown until reported."]
    for track in ["strict","published"]:
        d=pd.read_csv(out/f"summary_{track}.csv")
        text.append("## "+track.capitalize()+"\n\n"+table(d[["model","official","temporal","gap_mean","gap_worst","cv"]]))
    text.append("## Paired uncertainty\n\nIntervals do not correct for the adaptive research history.\n\n"+table(paired[paired.track.eq("strict")][["reference","suite","kind","delta","lower","upper"]]))
    text.append("[Every period](all_scores.csv), [verification](verification.json), and [frozen settings](manifest.json) retain the complete record. Final exports use all 9,050 labeled rows and preserve the 0.58609 baseline file.")
    (out/"RESULTS.md").write_text("\n\n".join(text)+"\n")
    print(json.dumps(result,indent=2),flush=True)


if __name__=="__main__":run()


Overwriting src/sprint_tree_report.py

## `src/submission_models.py`

SHA-256: `e4d0a5184cacaf446ef6905f461753da1d057990bd4053922f35a8c9f008eae5`

In [62]:
%%writefile src/submission_models.py
"""Refit, serialize, independently replay, and validate the four submissions."""
from pathlib import Path
import argparse
import hashlib
import importlib.metadata
import json
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from experiments import ROOT, OUT, PHASES, load_data, prepare, matrix, fit_linear, predict_linear, metrics
from boosted import transform_dense
from ensemble_search import collect, weighted
from refinements import interactions

SUBMISSIONS=ROOT/'submissions'


def reported_kaggle_score(filename,sha256):
    path=SUBMISSIONS/'leaderboard_results.json'
    if not path.exists():return None
    record=json.loads(path.read_text()).get('submissions',{}).get(filename,{})
    return record.get('kaggle_score') if record.get('sha256')==sha256 else None


def linear_inputs(component, df, blocks):
    c=component['config'];family=c['family']
    if family in ['lr','recency','balanced','interactions']:
        x=matrix(blocks,c['representation'],c.get('year','none'))
        if family=='interactions':x=interactions(x,df,c['weight'])
        return x
    if family=='text':
        t=component['vectorizer'].transform(df.criteria.fillna('').str.replace(r'\\[nr]',' ',regex=True))
        if c['representation']=='charcriteria':
            return sparse.hstack([blocks['criteria'],t*.7,blocks['phase']],format='csr')
        return sparse.hstack([t,blocks['phase']],format='csr')
    raise ValueError(f'Unsupported linear configuration {c}')


def predict_components(bundle, df):
    blocks=bundle['features'].transform(df)
    dense=None;result={}
    for name,c in bundle['components'].items():
        cfg=c['config'];family=cfg['family']
        if family in ['catboost','robust_catboost','histgb','lightgbm','extratrees']:
            if family=='lightgbm':
                x=matrix(blocks,'knowledge',cfg['year'])
                p=c['model'].predict_proba(x,num_iteration=cfg['iterations'])[:,1]
            else:
                if dense is None:
                    dense=transform_dense(df,blocks,bundle['dense_svds'],bundle['dense_mask'],bundle['features'].year_median)
                x=dense.drop(columns='year') if cfg.get('year','none')=='none' else dense
                if family in ['histgb','extratrees']:
                    x=x.drop(columns=x.select_dtypes('str').columns)
                    p=c['model'].predict_proba(x)[:,1]
                else:p=c['model'].predict_proba(x,ntree_end=cfg['iterations'])[:,1]
        else:
            x=linear_inputs(c,df,blocks)
            if family=='lr':
                a=cfg['phase_weight']
                p=(1-a)*predict_linear(c['pooled'],x,df)
                if a:p+=a*predict_linear(c['separate'],x,df)
            else:p=c['model'].predict_proba(x)[:,1]
        result[name]=p
    return result


def predict(bundle,df):
    components=predict_components(bundle,df)
    return {name:weighted(components,w,df.phase) for name,w in bundle['ensembles'].items()}


def fit_components(ensembles,configs):
    data=load_data();tr,ev,f,bt,be,bx=prepare(data,'validation')
    names=sorted({k for w in ensembles.values() for k in w})
    bundle={'features':f,'components':{},'ensembles':ensembles,'training_rows':len(tr),
            'training_trial_ids':sorted(tr.trial_id.unique().tolist())}
    dense_path=OUT/'dense_validation.joblib'
    if dense_path.exists():
        frames,svds,mask=joblib.load(dense_path)
        bundle.update(dense_svds=svds,dense_mask=mask)
    for name in names:
        cfg=configs[name];family=cfg['family'];component={'config':cfg}
        if family in ['catboost','robust_catboost','histgb','lightgbm','extratrees']:
            if family=='catboost':path=OUT/f"catboost_validation_d{cfg['depth']}_{cfg['year']}.joblib"
            elif family=='robust_catboost':path=OUT/f"robust_catboost_{cfg['variant']}_validation.joblib"
            elif family=='histgb':path=OUT/f"histgb_leaves{cfg['leaves']}_{cfg['year']}_validation.joblib"
            elif family=='lightgbm':path=OUT/f"lightgbm_validation_leaves{cfg['leaves']}_{cfg['year']}.joblib"
            else:path=OUT/f"extratrees_validation_leaf{cfg['leaf']}_{cfg['year']}.joblib"
            if family=='extratrees' and not path.exists():
                x=frames[0]
                if cfg['year']=='none':x=x.drop(columns='year')
                x=x.drop(columns=x.select_dtypes('str').columns)
                model=ExtraTreesClassifier(n_estimators=500,min_samples_leaf=cfg['leaf'],max_features=.5,n_jobs=4,random_state=42)
                model.fit(x,tr.label);joblib.dump(model,path,compress=3)
            component['model']=joblib.load(path)
        else:
            if family=='text':component['vectorizer']=joblib.load(OUT/f"text_{cfg['representation']}_validation.joblib")
            x=linear_inputs(component,tr,bt)
            if family=='lr':
                component['pooled']=fit_linear(x,tr,cfg['C'],False)
                if cfg['phase_weight']:component['separate']=fit_linear(x,tr,cfg['C'],True)
            else:
                w=None
                if family=='recency':
                    w=2**((tr.start_year.fillna(tr.start_year.median()).to_numpy()-tr.start_year.max())/cfg['half_life'])
                    w=w/w.mean()
                elif family=='balanced':w=tr.phase.map({p:len(tr)/(3*sum(tr.phase==p)) for p in PHASES}).to_numpy()
                m=LogisticRegression(C=cfg['C'],solver='liblinear',max_iter=1000,tol=1e-5,random_state=42)
                m.fit(x,tr.label,sample_weight=w);component['model']=m
        bundle['components'][name]=component
        print('prepared',name,flush=True)
    return bundle


def bootstrap_delta(df, candidate, baseline, seed=42, repetitions=2000):
    # Shared trial weights keep combined-phase rows together in every resample.
    ids,index=np.unique(df.trial_id.to_numpy(),return_inverse=True)
    rng=np.random.default_rng(seed)
    w=rng.poisson(1,size=(repetitions,len(ids)))[:,index]
    y=df.label.to_numpy()
    def loss(p):
        p=np.clip(p,1e-6,1-1e-6)
        return -(y*np.log(p)+(1-y)*np.log1p(-p))
    delta=loss(candidate)-loss(baseline)
    estimates=[]
    for phase in PHASES:
        m=df.phase.eq(phase).to_numpy()
        estimates.append((w[:,m]@delta[m])/w[:,m].sum(axis=1))
    d=np.mean(estimates,axis=0)
    return dict(delta_mean=float(d.mean()),delta_lower=float(np.quantile(d,.025)),delta_upper=float(np.quantile(d,.975)),fraction_lower_loss=float((d<0).mean()))


def build_and_verify():
    SUBMISSIONS.mkdir(exist_ok=True)
    selected=json.loads((OUT/'selected_ensembles.json').read_text())
    pred,tests,configs=collect();data=load_data()
    ensemble_defs=json.loads((OUT/'ensemble_configs.json').read_text())
    weights={name:ensemble_defs[source] for name,source in selected.items()}
    bundle=fit_components(weights,configs)
    val_components=predict_components(bundle,data['val'])
    test_components=predict_components(bundle,data['test'])
    verification=[]
    for name in bundle['components']:
        for split,new,old in [('validation',val_components[name],pred['validation'][name]),('test',test_components[name],tests['validation'][name])]:
            error=float(np.max(np.abs(new-old)))
            assert error<1e-8,(name,split,error)
            verification.append(dict(component=name,split=split,max_probability_difference=error))
    joblib.dump(bundle,OUT/'submission_models.joblib',compress=3)
    reloaded=joblib.load(OUT/'submission_models.joblib')
    vp=predict(reloaded,data['val']);tp=predict(reloaded,data['test'])
    sample=pd.read_csv(ROOT/'part2_sample_submission.csv')
    _,temporal,_,_,_,_=prepare(data,'temporal')
    manifest=[];validation={};historical={};bootstrap=[]
    baseline=pred['validation']['lr_criteria_none_C2_phase0']
    for name,source in selected.items():
        w=weights[name];p=vp[name];pt=tp[name]
        assert np.allclose(p,weighted(val_components,w,data['val'].phase),atol=1e-12,rtol=0)
        expected=weighted(tests['validation'],w,data['test'].phase)
        assert np.allclose(pt,expected,atol=1e-8,rtol=0)
        ph=weighted(pred['temporal'],w,temporal.phase)
        validation[name]=p;historical[name]=ph
        output=sample[['ID']].merge(pd.DataFrame({'ID':data['test'].ID,'label':pt}),on='ID',validate='one_to_one')
        assert len(output)==1500 and output.ID.equals(sample.ID)
        assert output.label.between(0,1,inclusive='neither').all()
        path=SUBMISSIONS/f'{name}.csv'
        output.to_csv(path,index=False,float_format='%.10f')
        reread=pd.read_csv(path)
        assert reread.columns.tolist()==['ID','label'] and not reread.isna().any().any()
        assert np.max(abs(reread.label-pt))<5.1e-11
        mv=metrics(data['val'],p);mt=metrics(temporal,ph)
        sha256=hashlib.sha256(path.read_bytes()).hexdigest()
        manifest.append(dict(submission=path.name,ensemble=source,
            validation_logloss=mv['phase_mean_logloss'],temporal_logloss=mt['phase_mean_logloss'],
            joint_logloss=.5*(mv['phase_mean_logloss']+mt['phase_mean_logloss']),
            mean_test_probability=float(pt.mean()),sha256=sha256,kaggle_score=reported_kaggle_score(path.name,sha256)))
        bootstrap.append(dict(model=name,**bootstrap_delta(data['val'],p,baseline)))
    np.savez_compressed(OUT/'selected_validation_predictions.npz',**validation)
    np.savez_compressed(OUT/'selected_temporal_predictions.npz',**historical)
    pd.DataFrame(manifest).to_csv(SUBMISSIONS/'index.csv',index=False)
    (SUBMISSIONS/'manifest.json').write_text(json.dumps({'training':'outcome_train.csv only; validation never fitted',
        'selection':'Equal-weight average of official-validation and chronological-holdout phase-mean log loss',
        'ensembles':weights,'files':manifest},indent=2))
    pd.DataFrame(verification).to_csv(OUT/'model_replay_verification.csv',index=False)
    pd.DataFrame(bootstrap).to_csv(OUT/'paired_validation_bootstrap.csv',index=False)
    versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','catboost','lightgbm','rdkit','joblib','matplotlib','nbformat','nbclient','ipykernel']}
    (OUT/'versions.json').write_text(json.dumps(versions,indent=2))
    (ROOT/'requirements.txt').write_text('\n'.join(f'{p}=={v}' for p,v in versions.items())+'\n')
    print(pd.DataFrame(manifest).to_string(index=False))


if __name__=='__main__':
    parser=argparse.ArgumentParser()
    parser.add_argument('--input',type=Path)
    parser.add_argument('--output',type=Path)
    args=parser.parse_args()
    if args.input:
        bundle=joblib.load(OUT/'submission_models.joblib')
        df=pd.read_csv(args.input)
        destination=args.output or ROOT/'replayed_predictions'
        destination.mkdir(parents=True,exist_ok=True)
        for name,p in predict(bundle,df).items():
            pd.DataFrame({'ID':df.ID,'label':p}).to_csv(destination/f'{name}.csv',index=False,float_format='%.10f')
    else:build_and_verify()


Overwriting src/submission_models.py


## `src/verify_benchmark.py`

SHA-256: `2802224753b7ef9b7c1d68ec2eadd23a94cd4b885f3bbc7b3d72535d01c6b9df`

In [63]:
%%writefile src/verify_benchmark.py
"""Verify cached execution, native model replay, protocol and candidate artifacts."""
import argparse,json,subprocess,sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from benchmark_protocol import DIRECTORY,initialize,resolve,digest
from experiments import ROOT


def run(verify_cache=False):
    universe,protocol=initialize();out=ROOT/json.loads((DIRECTORY/'current_run.json').read_text())['directory']
    manifest=json.loads((out/'run_manifest.json').read_text())
    assert all(digest(ROOT/f)==sha for f,sha in manifest['source_sha256'].items())
    assert digest(DIRECTORY/'protocol.json')==manifest['protocol_sha256']
    for track,models in [(out,6),(out/'published_recipe_track',5),(out/'derived_candidates',3),(out/'published_recipe_track/derived_candidates',3)]:
        summary=pd.read_csv(track/'summary.csv');scores=pd.read_csv(track/'fold_scores.csv')
        assert len(summary)==models and summary.complete_comparison.all()
        assert len(scores)==10*models and not scores.duplicated(['model','split']).any()
    official=next(s for s in protocol['splits'] if s['name']=='official')
    _,ev=resolve(universe,official);errors={}
    for name in ['ani_stack','our_linear']:
        model=joblib.load(out/'models/official'/f'{name}.joblib')
        expected=pd.read_csv(out/'predictions/official'/f'{name}.csv')
        assert expected.ID.tolist()==ev.ID.tolist()
        error=float(np.max(np.abs(model.predict_proba(ev)-expected.probability.to_numpy())))
        assert error<1e-12;errors[name]=error
    checks={'splits':10,'strict_recipes':6,'published_recipe_variants':5,'complete_comparisons':True,
            'dataset_and_source_hashes_match':True,'native_probability_replay_max_errors':errors}
    sample=pd.read_csv(ROOT/'part2_sample_submission.csv')
    files=list((ROOT/'submissions').glob('0[1-8]_*.csv'))
    for f in files:
        d=pd.read_csv(f)
        assert d.columns.tolist()==['ID','label'] and len(d)==1500 and d.ID.is_unique and d.ID.equals(sample.ID)
        assert np.isfinite(d.label).all() and d.label.between(0,1,inclusive='neither').all()
    checks['valid_submission_files']=len(files)
    if verify_cache:
        before={str(p):p.stat().st_mtime_ns for p in (out/'models').glob('**/*.joblib')}
        old_summary=pd.read_csv(out/'summary.csv').set_index('model')
        with (DIRECTORY/'cached_replay.log').open('w') as log:
            subprocess.run([sys.executable,str(ROOT/'local_benchmark.py'),'--suite','full'],stdout=log,stderr=subprocess.STDOUT,check=True)
        after={str(p):p.stat().st_mtime_ns for p in (out/'models').glob('**/*.joblib')}
        assert before==after,'Cached replay unexpectedly refitted or rewrote models.'
        new_summary=pd.read_csv(out/'summary.csv').set_index('model')
        for col in ['official_logloss','temporal_mean_logloss','cv_mean_logloss','temporal_delta_lower','group_cv_delta_upper']:
            assert np.allclose(old_summary[col],new_summary[col].reindex(old_summary.index),atol=1e-12,rtol=0,equal_nan=True)
        checks['cached_run_refits_no_models']=True;checks['cached_metrics_and_intervals_repeat']=True
    (DIRECTORY/'verification.json').write_text(json.dumps(checks,indent=2))
    print(json.dumps(checks,indent=2))

if __name__=='__main__':
    parser=argparse.ArgumentParser();parser.add_argument('--verify-cache',action='store_true')
    run(parser.parse_args().verify_cache)


Overwriting src/verify_benchmark.py


## `src/verify_improvements.py`

SHA-256: `d3a4c7162aa39c563f26c37b0162c2fc60e3034522af56cbc9f7e47cf04be966`

In [64]:
%%writefile src/verify_improvements.py
"""Independent artifact checks; no expensive model refits."""
import json
import numpy as np
import pandas as pd
from benchmark_protocol import initialize, resolve, digest
from experiments import ROOT
from improvement_models import RECIPES
from improvement_trends import TRENDS
from improvement_text import TEXT_RECIPES
from improvement_report import current, audit_track, REFERENCES


def verify():
    out = current()
    universe, protocol = initialize()
    manifest = json.loads((out / "manifest.json").read_text())
    for name, expected in manifest["source_sha256"].items():
        assert digest(ROOT / name) == expected, name
    for batch, filename in [("trend_followup", "improvement_trends.py"), ("text_followup", "improvement_text.py")]:
        assert digest(ROOT / filename) == json.loads((out / batch / "manifest.json").read_text())["source_sha256"]
    details = dict(run=out.name, recipe_count=len(RECIPES)+len(TRENDS)+len(TEXT_RECIPES),
                   tracks=2, splits_per_track=10, source_hashes_match=True,
                   original_protocol_unchanged=True, fitted_boundaries_verified=True)
    max_reproduction_error = 0.
    for track in ["strict", "published"]:
        audit_track(out / "text_followup" / track, universe, protocol, names=list(RECIPES)+list(TRENDS)+list(TEXT_RECIPES))
        for split in protocol["splits"]:
            train, ev = resolve(universe, split)
            bank = out / track / "bases" / f"{split['name']}.npz"
            arrays = np.load(bank)
            audit = json.loads(bank.with_name(split["name"]+"_audit.json").read_text())
            assert audit["training_ids"] == train.ID.tolist()
            assert audit["evaluation_ids"] == ev.ID.tolist()
            assert not set(train.trial_id) & set(ev.trial_id)
            for inner in audit["audit"]:
                assert not set(inner["fit_ids"]) & set(inner["heldout_ids"])
                if track == "strict":
                    assert set(inner["preprocessing_fit_ids"]) == set(inner["fit_ids"])
                    assert not set(inner["preprocessing_fit_ids"]) & set(ev.ID)
            extra = json.loads((out / "text_followup" / track / "bases" / f"{split['name']}_audit.json").read_text())
            for inner in extra:
                assert set(inner["outcome_ratio_fit_ids"]) == set(inner["fit_ids"])
                assert not set(inner["outcome_ratio_fit_ids"]) & (set(inner["eval_ids"]) | set(ev.ID))
                if track == "strict":
                    assert set(inner["feature_fit_ids"]) == set(inner["fit_ids"])
            archived = ROOT / "experiments/local_benchmark/runs/68db59271f5f0b9d"
            if track == "published":
                archived /= "published_recipe_track"
            old = pd.read_csv(archived / "predictions" / split["name"] / "ani_stack.csv")
            new = pd.read_csv(out / track / "predictions" / split["name"] / "ani_stack.csv")
            assert old.ID.equals(new.ID)
            error = float(np.abs(old.probability-new.probability).max())
            max_reproduction_error = max(max_reproduction_error, error)
            assert error < 1e-6, (track, split["name"], error)
    details["historical_baseline_max_probability_error"] = max_reproduction_error
    parity = json.loads((out / "final/parity.json").read_text())
    assert all(v["max_probability_error"] < 1e-6 for v in parity.values())
    details["both_kaggle_sources_reproduced"] = parity
    selection = json.loads((out / "selection.json").read_text())
    assert digest(ROOT / "IMPROVEMENT_PROTOCOL.md") == selection["protocol_document_sha256"]
    assert digest(ROOT / "improvement_text.py") == selection["selection_code_sha256"]
    for track, expected in selection["input_sha256"].items():
        assert digest(out / "text_followup" / track / "independent_summary.csv") == expected
    robustness = json.loads((out / "robustness/protocol.json").read_text())
    for track in ["strict", "published"]:
        audit_track(out / "robustness" / track, universe, robustness, names=REFERENCES+selection["shortlist"])
    details["gapped_windows_verified"] = 6
    details["shortlist"] = selection["shortlist"]
    export = json.loads((ROOT / "submissions/stack_improvements_manifest.json").read_text())
    for entry in export["submissions"]:
        assert digest(ROOT / "submissions" / entry["file"]) == entry["sha256"]
        assert entry["kaggle_score"] is None
    details["new_submissions"] = len(export["submissions"])
    details["independently_rescored_predictions"] = 2*10*details["recipe_count"]
    (out / "verification.json").write_text(json.dumps(details, indent=2))
    print(json.dumps(details, indent=2))


if __name__ == "__main__":
    verify()


Overwriting src/verify_improvements.py


## `src/verify_submission_tags.py`

SHA-256: `2e1434f966f4a615c5ce86fa9ef43b22d5a3cab8c56fc78e69d3570b6783b0bc`

In [65]:
%%writefile src/verify_submission_tags.py
"""Check that each confirmed submission has its own record commit and score tag."""
import csv
import hashlib
import io
import json
from pathlib import Path
import subprocess

REPO = Path(__file__).resolve().parent.parent


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO)


def digest(data):
    return hashlib.sha256(data).hexdigest()


def verify():
    ledger = json.loads((REPO/"part2/submissions/leaderboard_results.json").read_text())
    records = sorted((REPO/"part2/submissions/records").glob("*.json"))
    seen_files, seen_commits = set(), set()
    for path in records:
        record = json.loads(path.read_text())
        relative = str(path.relative_to(REPO))
        artifact, tag = record["artifact"], record["score_tag"]
        filename = Path(artifact).name
        entry = ledger["submissions"][filename]
        assert entry["kaggle_score"] == record["kaggle_score"]
        assert entry["sha256"] == record["sha256"]
        assert record["metric"] == ledger["metric"] and record["lower_is_better"]
        assert tag == f"part2-submission-{record['submission_id']}-kaggle-{record['kaggle_score']:.5f}"
        assert git("cat-file", "-t", "refs/tags/"+tag).strip() == b"tag", tag
        commit = git("rev-parse", tag+"^{commit}").decode().strip()
        assert commit not in seen_commits, "Each submission needs a distinct record commit"
        assert filename not in seen_files
        seen_files.add(filename); seen_commits.add(commit)
        assert git("show", tag+":"+relative) == path.read_bytes()
        assert record["record_kind"] == "retrospective_submission_record"
        changed = git("diff-tree", "--no-commit-id", "--name-only", "-r", commit).decode().splitlines()
        assert changed == [relative], (tag, changed)
        for revision in [tag, record["artifact_first_commit"], record["trial_phase_commit"]]:
            assert digest(git("show", revision+":"+artifact)) == record["sha256"], (tag, revision)
        current = (REPO/artifact).read_bytes()
        assert digest(current) == record["sha256"]
        rows = list(csv.DictReader(io.StringIO(current.decode())))
        assert len(rows) == 1500 and len({r["ID"] for r in rows}) == 1500
        assert all(0 < float(r["label"]) < 1 for r in rows)
        evidence = record["evidence_manifest"]
        assert digest(git("show", tag+":"+evidence)) == record["evidence_manifest_sha256"]
        original = json.loads(git("show", record["original_score_commit"]+":part2/submissions/leaderboard_results.json"))
        assert original["submissions"][filename]["kaggle_score"] == record["kaggle_score"]
        assert original["submissions"][filename]["sha256"] == record["sha256"]
        print(commit[:7], tag)
    assert seen_files == set(ledger["submissions"]), "Every confirmed submission must have a record and tag"
    print(f"Verified {len(records)} distinct submission commits, annotated score tags and unchanged CSV hashes.")


if __name__ == "__main__":
    verify()


Overwriting src/verify_submission_tags.py


## `src/year_cap.py`

SHA-256: `f557670c304fe9e32a25dc58a30a2f9fc0f5b068026114fa93fb77f5196c06b7`

In [66]:
%%writefile src/year_cap.py
"""Constrain linear year effects to the latest observed training year."""
import os
os.environ.setdefault('OMP_NUM_THREADS','4')
os.environ.setdefault('OPENBLAS_NUM_THREADS','4')
import json
import numpy as np
import pandas as pd
from experiments import OUT,load_data,prepare,matrix,fit_linear,predict_linear,metrics


def run():
    data=load_data();configs={}
    for split in ['temporal','validation']:
        tr,ev,f,bt,be,bx=prepare(data,split)
        pred={};test={};scores=[]
        for rep in ['criteria','clinical','rich']:
            for C in [.5,2.]:
                xs=[matrix(b,rep,'year_cap') for b in [bt,be,bx]]
                pair=[];pt=[]
                for separate in [False,True]:
                    models=fit_linear(xs[0],tr,C,separate)
                    pair.append(predict_linear(models,xs[1],ev));pt.append(predict_linear(models,xs[2],data['test']))
                for alpha in [0.,.25,.5]:
                    name=f'lr_{rep}_yearcap_C{C:g}_phase{alpha:g}'
                    p=(1-alpha)*pair[0]+alpha*pair[1]
                    pred[name]=p;test[name]=(1-alpha)*pt[0]+alpha*pt[1]
                    configs[name]=dict(family='lr',representation=rep,year='year_cap',C=C,phase_weight=alpha)
                    scores.append(dict(name=name,**metrics(ev,p)))
        np.savez_compressed(OUT/f'yearcap_{split}_predictions.npz',**pred)
        np.savez_compressed(OUT/f'yearcap_{split}_test_predictions.npz',**test)
        pd.DataFrame(scores).sort_values('phase_mean_logloss').to_csv(OUT/f'yearcap_{split}_scores.csv',index=False)
        print(split,pd.DataFrame(scores)[['name','phase_mean_logloss']].sort_values('phase_mean_logloss').head(8).to_string(index=False),flush=True)
    (OUT/'yearcap_configs.json').write_text(json.dumps(configs,indent=2))

if __name__=='__main__':run()


Overwriting src/year_cap.py


## `tests/test_extensions.py`

SHA-256: `5ffab0c3ce4d978308ee7b10628129e2dc87b2e255959cee93ad41a173497456`

In [67]:
%%writefile tests/test_extensions.py
"""Behavioral checks for the new neighborhood estimator."""
import unittest
import numpy as np
import pandas as pd
from scipy import sparse
from extension_neighbors import neighbor_predictions


class NeighborTests(unittest.TestCase):
    def frames(self):
        train=pd.DataFrame(dict(phase=["I","I","II","II","III","III"],label=[0,1]*3,
                                start_year=[2000,2010]*3,ID=list("abcdef"),trial_id=list("abcdef")))
        ev=pd.DataFrame(dict(phase=["I","II","III"],label=[0,0,0],start_year=[2014]*3))
        fa={k:sparse.csr_matrix(np.ones((6,1))) for k in ["knn","disease","drug","icd"]}
        fb={k:sparse.csr_matrix(np.ones((3,1))) for k in fa}
        return train,ev,fa,fb

    def test_recency_moves_identical_neighbors_toward_newer_outcomes(self):
        train,ev,fa,fb=self.frames();p=neighbor_predictions(fa,fb,train,ev)
        np.testing.assert_allclose(p[:,:3],.5)
        self.assertTrue((p[:,3]>.8).all())
        self.assertTrue((p[:,4]>p[:,3]).all())

    def test_zero_similarity_uses_training_phase_prior(self):
        train,ev,fa,fb=self.frames()
        fb={k:sparse.csr_matrix(v.shape) for k,v in fb.items()}
        p=neighbor_predictions(fa,fb,train,ev)
        np.testing.assert_allclose(p[:,:3],.5)
        self.assertTrue(np.isfinite(p).all())
        self.assertTrue((p[:,3]>.7).all())

    def test_heldout_labels_and_identifiers_do_not_enter_neighbors(self):
        train,ev,fa,fb=self.frames();before=neighbor_predictions(fa,fb,train,ev)
        ev["label"]=1;ev["ID"]="forbidden";ev["trial_id"]="forbidden";ev["start_year"]=9999
        np.testing.assert_array_equal(before,neighbor_predictions(fa,fb,train,ev))


if __name__=="__main__":unittest.main()


Overwriting tests/test_extensions.py


## `tests/test_improvements.py`

SHA-256: `341b29d8b8bb524764e829f6c25f959528774cc27a36e727edaeba7cd20be374`

In [68]:
%%writefile tests/test_improvements.py
"""Checks for paired inference, frozen selection, and outcome exclusion."""
import copy
import unittest
import numpy as np
import pandas as pd
from scipy import sparse
from improvement_models import TAUS, fit_recipes
from improvement_report import eligible, paired_interval
from improvement_text import fit_extra, predict_extra


class ComparisonTests(unittest.TestCase):
    def frames(self):
        rows = []
        for trial in range(40):
            for phase in ["I", "II", "III"]:
                y = trial % 2
                rows.append(dict(ID=f"{trial}_{phase}", trial_id=str(trial), phase=phase,
                                 label=y, probability=.8 if y else .2))
        ref = pd.DataFrame(rows)
        candidate = ref.copy()
        candidate["probability"] = np.where(candidate.label, .9, .1)
        return {"fold0": dict(split=dict(kind="group_cv"), ani_stack=ref, candidate=candidate)}

    def test_known_paired_effect(self):
        got = paired_interval(self.frames(), "candidate", "ani_stack", "group_cv", 100)
        for field in ["delta", "lower", "upper"]:
            self.assertAlmostEqual(got[field], np.log(.8/.9))

    def test_repeating_same_trials_does_not_narrow_interval(self):
        frames = self.frames()
        frames["fold0"]["candidate"]["probability"] = np.linspace(.15, .85, 120)
        original = paired_interval(frames, "candidate", "ani_stack", "group_cv", 100)
        frames["repeat1"] = copy.deepcopy(frames["fold0"])
        repeated = paired_interval(frames, "candidate", "ani_stack", "group_cv", 100)
        self.assertEqual(original, repeated)

    def test_misaligned_predictions_rejected(self):
        frames = self.frames()
        frames["fold0"]["candidate"] = frames["fold0"]["candidate"].iloc[::-1]
        with self.assertRaises(AssertionError):
            paired_interval(frames, "candidate", "ani_stack", "group_cv", 10)

    def test_selection_requires_beating_both_references(self):
        summary = pd.DataFrame([
            dict(model="ani_stack", completed_splits=10, temporal=.53, cv=.58, worst=.59),
            dict(model="ani_first", completed_splits=10, temporal=.54, cv=.57, worst=.60),
            dict(model="good", completed_splits=10, temporal=.52, cv=.5705, worst=.59),
            dict(model="cv_regression", completed_splits=10, temporal=.51, cv=.58, worst=.59),
            dict(model="partial", completed_splits=10, temporal=.535, cv=.57, worst=.59)
        ]).set_index("model")
        self.assertEqual(eligible(summary), ["good"])
        summary.loc["good", "completed_splits"] = 9
        with self.assertRaises(AssertionError):
            eligible(summary)


class OutcomeBoundaryTests(unittest.TestCase):
    def test_evaluation_labels_and_identifiers_cannot_change_predictions(self):
        rng = np.random.default_rng(12)
        train = pd.DataFrame(dict(phase=["I", "II", "III"]*40,
                                 label=rng.integers(0, 2, 120), start_year=rng.integers(2000, 2012, 120)))
        ev = train.iloc[:18].copy()
        ev["ID"] = np.arange(len(ev))
        ev["trial_id"] = np.arange(len(ev))
        packet = dict(year_median=2006, year_cap=2013, bank={key:
                      dict(oof=rng.uniform(.01, .99, (len(train), 8)),
                           eval=rng.uniform(.01, .99, (len(ev), 8))) for key in TAUS})
        before, _ = fit_recipes(packet, train, ev)
        ev["label"] = 1-ev.label
        ev["ID"] = "forbidden"
        ev["trial_id"] = "forbidden"
        after, _ = fit_recipes(packet, train, ev)
        for model in before:
            np.testing.assert_array_equal(before[model], after[model])

    def test_supervised_word_ratio_has_expected_direction_and_handles_unseen_text(self):
        # The first word belongs exclusively to successful training trials,
        # the second exclusively to failed trials. Missing/unknown text stays
        # a valid intercept/phase-only prediction.
        train = pd.DataFrame(dict(label=[1,1,1,0,0,0]))
        counts = sparse.csr_matrix([[3,0], [2,0], [1,0], [0,2], [0,1], [0,5]])
        original = counts.copy()
        features = dict(crit_counts=counts, phase=sparse.csr_matrix(np.ones((6,1))))
        model = fit_extra(features, train, np.ones(6), kind=1)
        self.assertGreater(model[1][0], 0)
        self.assertLess(model[1][1], 0)
        self.assertEqual((counts != original).nnz, 0)
        query = dict(crit_counts=sparse.csr_matrix([[1,0],[0,1],[0,0]]),
                     phase=sparse.csr_matrix(np.ones((3,1))))
        p = predict_extra(query, model, kind=1)
        self.assertGreater(p[0], p[1])
        self.assertTrue(np.isfinite(p).all())
        self.assertTrue(((p > 0) & (p < 1)).all())


if __name__ == "__main__":
    unittest.main()


Overwriting tests/test_improvements.py


## `tests/test_layout.py`

SHA-256: `9985275204e2791ea21465a885f3c1a6a68ffe6c04d420410f0528311fc78d82`

In [69]:
%%writefile tests/test_layout.py
"""Check source isolation, notebook startup, and migrated navigation."""
import json
import os
from pathlib import Path
import runpy
import sys
import tempfile
import unittest

PROJECT = Path(os.environ.get("PART2_PROJECT_ROOT", Path(__file__).resolve().parents[1]))
RUNNER = runpy.run_path(str(PROJECT / "run.py"))


class LayoutTests(unittest.TestCase):
    def fixture(self, root):
        for folder in ["src", "data", "tests", "notebooks", "experiments", "submissions"]:
            (root / folder).mkdir(parents=True)
        (root / "src/model.py").write_text("from pathlib import Path\nROOT = Path(__file__).resolve().parent\n")
        (root / "data/outcome_train.csv").write_text("ID,label\n1,0\n")
        (root / "layout.json").write_text(json.dumps({"legacy_paths": {
            "model.py": "src/model.py", "outcome_train.csv": "data/outcome_train.csv",
        }}))
        (root / "run.py").write_bytes((PROJECT / "run.py").read_bytes())

    def test_execution_copies_preserve_source_and_share_canonical_artifacts(self):
        with tempfile.TemporaryDirectory(prefix="part2 layout ") as directory:
            root = Path(directory); self.fixture(root)
            workspace = RUNNER["prepare_workspace"](root)
            original = (root / "src/model.py").read_bytes()
            self.assertEqual((workspace / "model.py").read_bytes(), original)
            self.assertEqual(runpy.run_path(str(workspace / "model.py"))["ROOT"], workspace)
            self.assertEqual((workspace / "outcome_train.csv").resolve(), (root / "data/outcome_train.csv").resolve())
            self.assertEqual((workspace / "submissions").resolve(), (root / "submissions").resolve())
            (workspace / "model.py").write_text("# notebook-generated copy\n")
            self.assertEqual((root / "src/model.py").read_bytes(), original)
            (root / "src/new_trial.py").write_text("result = 1\n")
            RUNNER["prepare_workspace"](root)
            self.assertEqual((workspace / "model.py").read_bytes(), original)
            self.assertEqual((workspace / "new_trial.py").read_bytes(), (root / "src/new_trial.py").read_bytes())

    def test_notebook_bootstrap_keeps_original_cells_and_is_repeatable(self):
        with tempfile.TemporaryDirectory(prefix="part2 notebook ") as directory:
            root = Path(directory); self.fixture(root)
            path = root / "notebooks/example.ipynb"
            original = {"cell_type": "code", "id": "original", "source": ["print(1)"],
                        "execution_count": 1, "outputs": [{"output_type": "stream", "name": "stdout", "text": ["1\n"]}], "metadata": {}}
            path.write_text(json.dumps({"cells": [original], "nbformat": 4, "nbformat_minor": 5, "metadata": {}}))
            RUNNER["install_notebook_bootstrap"](root)
            once = path.read_bytes(); RUNNER["install_notebook_bootstrap"](root)
            self.assertEqual(path.read_bytes(), once)
            cells = json.loads(once)["cells"]
            self.assertEqual(cells[1:], [original])
            previous, imports = Path.cwd(), sys.path[:]
            try:
                os.chdir(path.parent)
                exec("".join(cells[0]["source"]), {})
                self.assertEqual(Path.cwd(), root.resolve() / ".runtime/part2")
            finally:
                os.chdir(previous); sys.path[:] = imports

    def test_report_references_translate_once(self):
        with tempfile.TemporaryDirectory() as directory:
            root = Path(directory); self.fixture(root)
            path = root / "experiments/RESULTS.md"
            text = ("[Code](../model.py)\npython part2/model.py --input part2/outcome_test.csv\n"
                    "python -m unittest discover -s part2 -p 'test_*.py'\n")
            updated = RUNNER["update_references"](text, path, path, root)
            self.assertIn("[Code](../src/model.py)", updated)
            self.assertIn("python part2/run.py model.py --input part2/data/outcome_test.csv", updated)
            self.assertIn("python part2/run.py test 'test_*.py'", updated)
            self.assertEqual(RUNNER["update_references"](updated, path, path, root), updated)


if __name__ == "__main__":
    unittest.main()


Overwriting tests/test_layout.py


## `tests/test_local_benchmark.py`

SHA-256: `247e352c3e6db0c5575211f8438c01ebe28a717e17c7fc8dc3231c74084f29f5`

In [70]:
%%writefile tests/test_local_benchmark.py
"""Meaningful protocol/scoring regression checks; no model training required."""
import tempfile,unittest
from pathlib import Path
import numpy as np
import pandas as pd
from benchmark_protocol import DIRECTORY,initialize,resolve
from experiments import load_data,metrics
from local_benchmark import paired_bootstrap,summarize


class FrozenProtocolTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):cls.universe,cls.protocol=initialize()

    def test_no_trial_or_test_set_leakage(self):
        test_trials=set(load_data()['test'].trial_id)
        for split in self.protocol['splits']:
            train,ev=resolve(self.universe,split)
            self.assertFalse(set(train.trial_id)&set(ev.trial_id))
            self.assertFalse((set(train.trial_id)|set(ev.trial_id))&test_trials)

    def test_cv_covers_each_trial_once_per_repeat(self):
        expected=set(self.universe.ID)
        for repeat in [0,1]:
            ids=[i for s in self.protocol['splits'] if s['kind']=='group_cv' and s['repeat']==repeat for i in s['eval_ids']]
            self.assertEqual(len(ids),len(expected));self.assertEqual(set(ids),expected)

    def test_temporal_windows_are_forward_and_disjoint(self):
        seen=set()
        for split in self.protocol['splits']:
            if split['kind']!='temporal':continue
            train,ev=resolve(self.universe,split)
            self.assertLess(train.start_year.max(),ev.start_year.min())
            self.assertFalse(seen&set(ev.ID));seen.update(ev.ID)

    def test_protocol_reuse_is_identical(self):
        path=DIRECTORY/'protocol.json';before=path.read_bytes()
        initialize();self.assertEqual(before,path.read_bytes())


class ScoringTests(unittest.TestCase):
    def test_equal_phase_loss_differs_from_row_weighted_loss(self):
        rows=[];prob=[]
        for phase,n,correct in [('I',2,.9),('II',4,.5),('III',6,.2)]:
            for j in range(n):
                y=j%2;rows.append({'phase':phase,'label':y});prob.append(correct if y else 1-correct)
        result=metrics(pd.DataFrame(rows),np.array(prob))
        expected=np.mean([-np.log(.9),-np.log(.5),-np.log(.2)])
        self.assertAlmostEqual(result['phase_mean_logloss'],expected)
        self.assertNotAlmostEqual(result['logloss'],expected)

    def test_trial_paired_bootstrap_known_improvement(self):
        protocol={'splits':[{'name':'fake','kind':'group_cv','repeat':0}]}
        with tempfile.TemporaryDirectory() as tmp:
            out=Path(tmp);folder=out/'predictions/fake';folder.mkdir(parents=True)
            rows=[]
            for trial in range(40):
                for phase in ['I','II','III']:
                    y=trial%2
                    rows.append({'ID':f'{trial}_{phase}','trial_id':str(trial),'phase':phase,'label':y,'probability':.8 if y else .2})
            baseline=pd.DataFrame(rows);candidate=baseline.copy()
            candidate['probability']=np.where(candidate.label,.9,.1)
            baseline.to_csv(folder/'ani_stack.csv',index=False);candidate.to_csv(folder/'candidate.csv',index=False)
            got=paired_bootstrap(out,protocol,'candidate','group_cv',repetitions=100)
            expected=-np.log(.9)+np.log(.8)
            self.assertAlmostEqual(got['group_cv_delta'],expected)
            self.assertAlmostEqual(got['group_cv_delta_lower'],expected)
            self.assertAlmostEqual(got['group_cv_delta_upper'],expected)

    def test_partial_comparison_cannot_be_marked_complete(self):
        _,protocol=initialize()
        with tempfile.TemporaryDirectory() as tmp:
            out=Path(tmp)
            pd.DataFrame([{'model':'ani_stack','split':'official','kind':'official','repeat':None,'phase_mean_logloss':.5}]).to_csv(out/'scores_official.csv',index=False)
            summarize(out,protocol,bootstrap=False)
            self.assertFalse(pd.read_csv(out/'summary.csv').complete_comparison.iloc[0])


if __name__=='__main__':unittest.main()


Overwriting tests/test_local_benchmark.py


In [71]:
expected_sources = {'experiments/ani_comparison/source/part2_best_model.py': '253cc923e9e4867ddc5d9bbea91b36ccaf28b16ed1ce0de8a3067089778787e1', 'experiments/ani_comparison/source/part2_first_submission_model.py': '8a526bb1fa94ba24f48245b27add937c430ff65e3d805518627ce5c067ee8f6a', 'packaging/build_handin.py': 'cde541a7edf9580f3051601825fa93ab302181da8db04c42d4e905370aacb95a', 'packaging/capture_recorded.py': 'b3a2758cfb48612ef58b80fab7a0721fda94a1c76b586fc4e9b374ddf11fa46e', 'packaging/notebooks.py': '2c67543301b2054fad8d9c991a86bb2ae8f0db5106b791d424a78f136907ecb0', 'packaging/train_final.py': '4671e9b38a1016d8db23d3c8c490c7fadd31e0c6b70b6a08d1b7aab2d2a07966', 'packaging/verify_handin.py': '0d049dd5f90c45ca9cb9679e3e18ee38de9ff5aa8e5ee0374292d9a39e7ff4a9', 'run.py': 'd396a9f2ffc52db9f7b2fdd8b2fa5feca9539fcc5013607273ad5052378e4d93', 'src/all_submission_models.py': '7d8a51f78d13c64a7cdaef02a4b97b5b408d4164a7c806aa96593c001c8d8832', 'src/all_submissions_benchmark.py': 'fce474d88e9d29b68daf05521a63e4816e1c8a8f66c9cbd2ea03cfda7cc9cab2', 'src/all_submissions_report.py': '471bf520f9a60b6838223c01ee7e6ab26a74b284c0b223703350231bb04630b4', 'src/all_submissions_verify.py': '70b5d3298838a763e6d5e703429838a8f154d8e172737c685ef5f74c6f40fff3', 'src/ani_models.py': '13639f92744f3d4a1fc38113861bb15cc85d32e26964cf1e5f774a8974956c53', 'src/benchmark_ani_recipe.py': '3c1cc9719818cd841d58c1eb8a03274fa460608ee41558e567c75e21fe6a717c', 'src/benchmark_candidates.py': '6c5042a02e41e125c7370a7785c49885261d90fceb76bcb8a403dc8a1cdf1115', 'src/benchmark_final.py': '37a72d05acb222fc42ed080a1311e3059632f1f34b9b185b6abc6002cc162eb1', 'src/benchmark_models.py': '719918c15adb87b3797a1b57a7ce15c0e61e49535988994198acb26df76c741a', 'src/benchmark_protocol.py': 'e5eb9190d2681607fc2d5cd305f68c6c63158ab1dd248b49d9ecee8772102ebe', 'src/benchmark_report.py': '8b38e872c2fc5f1a59dcefbc01fe5fa2d9613fb9bb95137955f269ef6e408196', 'src/boosted.py': 'c2a3da5c682cb7f74fbb78649cd1f7a598b21848d667175aa24d3af039d1029f', 'src/boosted_robustness.py': '047eabbe06eb5afd40381957ef6b340baa9c429776c67b18e157cc5f08be0702', 'src/build_benchmark_notebook.py': 'c2fabcb3e7a0a209f94acc73f20b3c87deddbed5df4d94fb61546d75a6aa7d9d', 'src/build_improvement_report.py': '3c7fb1c4b5f6d8b0a65b1bf41a1e11ae46b2a7f389eec720d0141f8b845e656f', 'src/build_notebook.py': 'e7299f0a327a79a249e958ee61b10f69a6a7150aa36ed23ef75ecacfbc6c1829', 'src/ensemble_search.py': '7bcb0054fd1ddbc82629d59acb4dc0337838fa3ee1a1a8c3965e41bb57aee3b9', 'src/error_analysis.py': '115b477403a1e8cb7e90f5ba139c9653a209f4a0180298f22ef59c1b750e8791', 'src/experiments.py': '7346197ed8466c54bc4caad0ad31f4570e8838a9aa81663eab01903f901e4a70', 'src/extension_common.py': '762962c7790e2a22462015417c78b1416fcc2c677f3f91d675b76f2c0d9794b2', 'src/extension_followthrough.py': '06a5403476adc9191296b29362f74da150044fe30b99c30fc136982bfca50e0b', 'src/extension_meta.py': '975585971a711aa2bc08d93c097cba2453c490bef8ca6593cdba3acf8b2009b5', 'src/extension_neighbors.py': '0a4b3a8b3fb3019144470188fb7e6b5b4208f18063332abe64cd44fca6abcd38', 'src/extension_predict.py': 'fef63f8b77716e19a6a76e96e1faa196cb8e6cd4ad8bfac9ffad12f263472c5b', 'src/extension_report.py': '6cbc4a17afb5d78deb3cc2710bce5edc0a2d56e294603c0eaa4c5507327a6286', 'src/extension_verify.py': 'e86a721d0d02993a47cf72844247de005678a141a6e46518b35ad9fc90bf18ff', 'src/followup_ensembles.py': '822ea9b6fd39d4c4e84e9001eb37126ad1bdc6792291a10351f83fd0249c4453', 'src/improvement_benchmark.py': 'b25a472ef73f8c82007bc79997b4a05dadfa357dc17388d941f86eb2b284e88d', 'src/improvement_models.py': '79c482fa2a2fcb4327df03b44e484822f505f4500a76db31e95b919661790939', 'src/improvement_report.py': 'b84ba427e78b93fc21ec5ed73fdf8eba8fde0933a6a05ee85aa172141a4c99fa', 'src/improvement_robustness.py': 'b2261ae031933f7def8885272f8bcb214634d71dc29309790b4015cb0f5352f0', 'src/improvement_text.py': '383e85858a3990d07af6f96c2c5922005b1fd33a66c4724198dfd27f1c973307', 'src/improvement_trends.py': '0cfba4370fea616849d9f1945ccd2caa54a16b01cf3ca88862a9c8608d78a603', 'src/local_benchmark.py': '67ab861407c66f3572d8916ed933f04ce37a55c462ce00ebfecdf02a5bb4deae', 'src/recency_and_sparse_boost.py': '1f4f24e03547efe7eeaa383ab1faa0d5c69516dc32ac434823c855f6ff4a7c8c', 'src/refinements.py': '6d5bfc2f735ce7ef61015cb52a94274323c4b89ab11d0614407543e2f1e01d41', 'src/reporting.py': '3aafb5adf464a73f88c853a162951e0b6cf03b545f62a4299b5367a28cce90f2', 'src/reproduce_ani.py': '935de49af0c0bca694778ed0a86b737bd6fb32e300fd8b40de16fc2b035d480e', 'src/sprint_benchmark.py': '123448b2a5533adaa040346a81c5ceff7f1a8ec99c16fdc0cacf79c577324bab', 'src/sprint_export.py': 'ac57ad77a6e1a3ebd5e711509414b678fb642bd310ae2b320dd158669469a0fa', 'src/sprint_finish.py': 'f96af64a249bcb7d99828d409697d75e3c8b04bc446004741beca2cc75f49b0c', 'src/sprint_finish_report.py': 'b18cccd4baad33a4f07fa69fd2b1caa2aab981071f866b2db6efcf4ed95edc53', 'src/sprint_legacy_fit.py': '6200057221e891e5b0c7f074a25abce678fb1734693ad96c0729c6a469100f1c', 'src/sprint_models.py': '3a5ff9cf01468bd5c46985fcb6fc1347be9ed9355edcccbceff790079077a0c7', 'src/sprint_overview.py': '051c04edca75b5eb0fe75d202e209158c0e655f2990dc721f0751b6753df27a0', 'src/sprint_refine.py': '71ec1fb209e40f553a16a394cebf5bc66a486a297b697e48b8a38a1f48ff0cbb', 'src/sprint_refine_report.py': '3eaf5793dbf7b86bae19346539e97bb49ef26bd785746f0c32e66a608e98abf3', 'src/sprint_report.py': '1504139e0f2452712ccc38f4d3cf724106caa56ab7fd35e3e7a9c1a1766b4fae', 'src/sprint_tree.py': '123edf072df10eac1a443042914eb5a8fbc8473299a6121b7f5a09d093bdf84d', 'src/sprint_tree_export.py': '1932c3c8a9407ba87e881dd4075a7bfa7618c3ecf9177aec873d75244b5f842f', 'src/sprint_tree_report.py': '39621779f1f5bb97f80bc6dcd32e42adc8cbe2a46664e2178861a0dbe6213c99', 'src/submission_models.py': 'e4d0a5184cacaf446ef6905f461753da1d057990bd4053922f35a8c9f008eae5', 'src/verify_benchmark.py': '2802224753b7ef9b7c1d68ec2eadd23a94cd4b885f3bbc7b3d72535d01c6b9df', 'src/verify_improvements.py': 'd3a4c7162aa39c563f26c37b0162c2fc60e3034522af56cbc9f7e47cf04be966', 'src/verify_submission_tags.py': '2e1434f966f4a615c5ce86fa9ef43b22d5a3cab8c56fc78e69d3570b6783b0bc', 'src/year_cap.py': 'f557670c304fe9e32a25dc58a30a2f9fc0f5b068026114fa93fb77f5196c06b7', 'tests/test_extensions.py': '5ffab0c3ce4d978308ee7b10628129e2dc87b2e255959cee93ad41a173497456', 'tests/test_improvements.py': '341b29d8b8bb524764e829f6c25f959528774cc27a36e727edaeba7cd20be374', 'tests/test_layout.py': '9985275204e2791ea21465a885f3c1a6a68ffe6c04d420410f0528311fc78d82', 'tests/test_local_benchmark.py': '247e352c3e6db0c5575211f8438c01ebe28a717e17c7fc8dc3231c74084f29f5'}
source_lengths = {'experiments/ani_comparison/source/part2_best_model.py': 10683, 'experiments/ani_comparison/source/part2_first_submission_model.py': 9397, 'packaging/build_handin.py': 7624, 'packaging/capture_recorded.py': 3023, 'packaging/notebooks.py': 27721, 'packaging/train_final.py': 6169, 'packaging/verify_handin.py': 3080, 'run.py': 7044, 'src/all_submission_models.py': 3426, 'src/all_submissions_benchmark.py': 9142, 'src/all_submissions_report.py': 14520, 'src/all_submissions_verify.py': 7092, 'src/ani_models.py': 7783, 'src/benchmark_ani_recipe.py': 3122, 'src/benchmark_candidates.py': 2189, 'src/benchmark_final.py': 4124, 'src/benchmark_models.py': 1581, 'src/benchmark_protocol.py': 3647, 'src/benchmark_report.py': 8562, 'src/boosted.py': 6316, 'src/boosted_robustness.py': 2506, 'src/build_benchmark_notebook.py': 10024, 'src/build_improvement_report.py': 12056, 'src/build_notebook.py': 13398, 'src/ensemble_search.py': 7474, 'src/error_analysis.py': 4393, 'src/experiments.py': 10750, 'src/extension_common.py': 3537, 'src/extension_followthrough.py': 14216, 'src/extension_meta.py': 4374, 'src/extension_neighbors.py': 8506, 'src/extension_predict.py': 1369, 'src/extension_report.py': 15804, 'src/extension_verify.py': 4891, 'src/followup_ensembles.py': 4422, 'src/improvement_benchmark.py': 6508, 'src/improvement_models.py': 7243, 'src/improvement_report.py': 10259, 'src/improvement_robustness.py': 4510, 'src/improvement_text.py': 11295, 'src/improvement_trends.py': 6698, 'src/local_benchmark.py': 10982, 'src/recency_and_sparse_boost.py': 2805, 'src/refinements.py': 6286, 'src/reporting.py': 14140, 'src/reproduce_ani.py': 2328, 'src/sprint_benchmark.py': 6957, 'src/sprint_export.py': 6606, 'src/sprint_finish.py': 4718, 'src/sprint_finish_report.py': 5460, 'src/sprint_legacy_fit.py': 1337, 'src/sprint_models.py': 2444, 'src/sprint_overview.py': 5685, 'src/sprint_refine.py': 10427, 'src/sprint_refine_report.py': 8306, 'src/sprint_report.py': 6733, 'src/sprint_tree.py': 5825, 'src/sprint_tree_export.py': 4233, 'src/sprint_tree_report.py': 5307, 'src/submission_models.py': 11100, 'src/verify_benchmark.py': 3434, 'src/verify_improvements.py': 4718, 'src/verify_submission_tags.py': 3009, 'src/year_cap.py': 1823, 'tests/test_extensions.py': 1721, 'tests/test_improvements.py': 4760, 'tests/test_layout.py': 4210, 'tests/test_local_benchmark.py': 3938}

for relative, expected in expected_sources.items():
    # IPython adds a final newline to %%writefile bodies. Two archived Ani
    # sources originally had no terminal newline; restore that exact EOF.
    exported = export_directory / relative
    content = exported.read_bytes()
    if len(content) == source_lengths[relative] + 1 and content.endswith(b"\n"):
        exported.write_bytes(content[:-1])
    assert sha256(export_directory / relative) == expected, relative
    assert sha256(ROOT / relative) == expected, relative
os.chdir(HANDIN)
print(f"Verified {len(expected_sources)} complete source files.")


Verified 68 complete source files.
